# Kaggriculture V46: First-Turn Microstructure and Sale Timing

V46 keeps the complete production policy of the previous releases (all V43 farming,
service and warehouse layers, the clone-gated sale pre-emption of V44 and the mirror gate)
and changes how the agent trades against the copies and derivatives of its own public
lineage that now fill the ladder:

* **First market turn.** The market settles orders index by index and quotes both players'
  current units at the same inventory. The previous 70-unit wheat round trip lifted the
  quotes of a rival's second wheat purchase, but a rival selling a small lot at index 1
  rides the same lift; against `[BUY 5, SELL 5]` or `[BUY 50, SELL 50]` it lost 32-77
  cash at turn 1, below the plan's 8-cash day-0 slack, and one or two wheat seeds with it.
  V46 buys its five feed units once at index 0 of turn 0 (`[BUY_PRODUCT WHEAT 7, SELL WHEAT 2]`),
  cleans turn 1, and never falls below the slack against any opening observed on the
  ladder; rival round trips lose 10-45.
* **Turn 1 lift.** Every tape of this lineage buys its feed at index 1 of turn 1. V46 buys 30
  wheat at index 0 of turn 1 and sells them back at turn 2, when no tape trades; the rival's
  purchase is quoted 4 higher per unit, again below its slack, and the resale meets the
  lifted quotes.
* **Sale timing.** Tape sales due within three turns are executed as soon as the units are
  in the shed (not at dawn, first-listed sale protected), the market list is ordered sales,
  then product purchases, then fixed-price orders, and the clone-gated pre-emption horizon
  rises to a full day as soon as a rival is seen selling a product we hold five or more
  turns ahead of the common plan.

No rival private information is used. The mechanisms of the third point follow the public
analysis "Beyond 48-0" by sdy623 (jaxa623), re-implemented over this project's chassis;
the turn-0 finding (buy the feed units at index 0 of turn 0) follows the public
"Pipe-8 clean opening" by Nathan Jacob. The exact lockstep simulations, the slack analysis,
the turn-1 lift and the race detector are this project's own work.

This notebook contains the complete agent. It needs no attached datasets, donor
notebook, internet, GPU, training, installation or compressed source blob. Run the five
code cells to produce `submission_competitive_v46.tar.gz`. Optional full-game checks
are off by default; enable `RUN_GAME_CHECKS` to play. The notebook does not submit
anything automatically.


## Start the build


In [ ]:
from pathlib import Path
import os, sys, time

RUN_GAME_CHECKS = False  # Optional: set True to run four full games in the final cell.
WORKDIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
print('V46 started. No training, download or installation is needed.', flush=True)
print('Output directory:', WORKDIR, flush=True)


## Write the complete agent


In [ ]:
import hashlib

# Immutable tested V46 source; write exact bytes on Windows and Linux.
# Do not normalize line endings: the pinned digest includes them.
EXPECTED_MAIN_SHA256 = '735c370383b70d3bf3aac792f2c147e0afc99166fc9f253ede10e8a030acedb6'
SOURCE_BYTES = b''.join((
    b'# EXP260 combined; V41 remains the frozen primary control.\n# Earlier reservation activation adapted from aurax7 Reactive V5; upstream notices retained.\n# EXP257: V39 base with funded atomic opening.\n# Opening assignment adapted from Rayk Kretzschmar, Rank Your Agent.\n# Existing upstream attribution retained below; original funding guard by Ahmed Berat Ozer.\n# EXP-173 isolate opening market sequence inspired by yhay81/shop-router-0911-simple (Apache-2.0).\n# Kaggriculture EXP-167 candidate. Not submitted automatically.\n# Attribution: thomastschinkel, yhay81, destbreso, aurax7, tetsutani,\n# prvsiyan and Dmitrii Gluzdov. Apache-2.0 derivations; notices retained below.\n# Kaggriculture v31 / EXP-157, Ahmed Berat Ozer, September 9 2026.\n# Selected mechanism: crop_public_order. New independent confirmation is required.\n# Public V221B/V224C production/timing lineage: prvsiyan, Apache-2.0.\n# Original economics and integration; retained upstream licenses follow.\n# Kaggriculture v28 / EXP-154, Ahmed Berat Ozer, September 9 2026.\n# Changes: aurax7 day-end storage guard; Dmitrii Gluzdov physical terminal rescue\n# adapted to v27, with 64 deterministic simulations. Apache-2.0.\n# New action tapes and ordered shop-pair map: yhay81/shop-router-0909, Apache-2.0.\n# Kaggriculture v25, EXP-149: Shop0908 production, sale lead, terminal cargo rescue.\n# Runtime chassis: Apache-2.0; thomastschinkel, yhay81, tetsutani.\n# Routing and public action data: yhay81/shop-router-0908, frozen September 8, 2026.\n# \n#                                  Apache License\n#                            Version 2.0, January 2004\n#                         http://www.apache.org/licenses/\n# \n#    TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n# \n#    1. Definitions.\n# \n#       "License" shall mean the terms and conditions for use, reproduction,\n#       and distribution as defined by Sections 1 through 9 of this document.\n# \n#       "Licensor" shall mean the copyright owner or entity authorized by\n#       the copyright owner that is granting the License.\n# \n#       "Legal Entity" shall mean the union of the acting entity and all\n#       other entities that control, are controlled by, or are under common\n#       control with that entity. For the purposes of this definition,\n#       "control" means (i) the power, direct or indirect, to cause the\n#       direction or management of such entity, whether by contract or\n#       otherwise, or (ii) ownership of fifty percent (50%) or more of the\n#       outstanding shares, or (iii) beneficial ownership of such entity.\n# \n#       "You" (or "Your") shall mean an individual or Legal Entity\n#       exercising permissions granted by this License.\n# \n#       "Source" form shall mean the preferred form for making modifications,\n#       including but not limited to software source code, documentation\n#       source, and configuration files.\n# \n#       "Object" form shall mean any form resulting from mechanical\n#       transformation or translation of a Source form, including but\n#       not limited to compiled object code, generated documentation,\n#       and conversions to other media types.\n# \n#       "Work" shall mean the work of authorship, whether in Source or\n#       Object form, made available under the License, as indicated by a\n#       copyright notice that is included in or attached to the work\n#       (an example is provided in the Appendix below).\n# \n#       "Derivative Works" shall mean any work, whether in Source or Object\n#       form, that is based on (or derived from) the Work and for which the\n#       editorial revisions, annotations, elaborations, or other modifications\n#       represent, as a whole, an original work of authorship. For the purposes\n#       of this License, Derivative Works shall not include works that remain\n#       separable from, or merely link (or bind by name) to the interfaces of,\n#       the Work and Derivative Works thereof.\n# \n#       "Contribution" shall mean any work of authorship, including\n#       the original version of the Work and any modifications or additions\n#       to tha',
    b't Work or Derivative Works thereof, that is intentionally\n#       submitted to Licensor for inclusion in the Work by the copyright owner\n#       or by an individual or Legal Entity authorized to submit on behalf of\n#       the copyright owner. For the purposes of this definition, "submitted"\n#       means any form of electronic, verbal, or written communication sent\n#       to the Licensor or its representatives, including but not limited to\n#       communication on electronic mailing lists, source code control systems,\n#       and issue tracking systems that are managed by, or on behalf of, the\n#       Licensor for the purpose of discussing and improving the Work, but\n#       excluding communication that is conspicuously marked or otherwise\n#       designated in writing by the copyright owner as "Not a Contribution."\n# \n#       "Contributor" shall mean Licensor and any individual or Legal Entity\n#       on behalf of whom a Contribution has been received by Licensor and\n#       subsequently incorporated within the Work.\n# \n#    2. Grant of Copyright License. Subject to the terms and conditions of\n#       this License, each Contributor hereby grants to You a perpetual,\n#       worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n#       copyright license to reproduce, prepare Derivative Works of,\n#       publicly display, publicly perform, sublicense, and distribute the\n#       Work and such Derivative Works in Source or Object form.\n# \n#    3. Grant of Patent License. Subject to the terms and conditions of\n#       this License, each Contributor hereby grants to You a perpetual,\n#       worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n#       (except as stated in this section) patent license to make, have made,\n#       use, offer to sell, sell, import, and otherwise transfer the Work,\n#       where such license applies only to those patent claims licensable\n#       by such Contributor that are necessarily infringed by their\n#       Contribution(s) alone or by combination of their Contribution(s)\n#       with the Work to which such Contribution(s) was submitted. If You\n#       institute patent litigation against any entity (including a\n#       cross-claim or counterclaim in a lawsuit) alleging that the Work\n#       or a Contribution incorporated within the Work constitutes direct\n#       or contributory patent infringement, then any patent licenses\n#       granted to You under this License for that Work shall terminate\n#       as of the date such litigation is filed.\n# \n#    4. Redistribution. You may reproduce and distribute copies of the\n#       Work or Derivative Works thereof in any medium, with or without\n#       modifications, and in Source or Object form, provided that You\n#       meet the following conditions:\n# \n#       (a) You must give any other recipients of the Work or\n#           Derivative Works a copy of this License; and\n# \n#       (b) You must cause any modified files to carry prominent notices\n#           stating that You changed the files; and\n# \n#       (c) You must retain, in the Source form of any Derivative Works\n#           that You distribute, all copyright, patent, trademark, and\n#           attribution notices from the Source form of the Work,\n#           excluding those notices that do not pertain to any part of\n#           the Derivative Works; and\n# \n#       (d) If the Work includes a "NOTICE" text file as part of its\n#           distribution, then any Derivative Works that You distribute must\n#           include a readable copy of the attribution notices contained\n#           within such NOTICE file, excluding those notices that do not\n#           pertain to any part of the Derivative Works, in at least one\n#           of the following places: within a NOTICE text file distributed\n#           as part of the Derivative Works; within the Source form or\n#           documentation, if provided along with the Derivative Works; or,\n#           within a display generated by the Derivative Works, if and\n#           wherever such third-party notices normally appear. The content',
    b's\n#           of the NOTICE file are for informational purposes only and\n#           do not modify the License. You may add Your own attribution\n#           notices within Derivative Works that You distribute, alongside\n#           or as an addendum to the NOTICE text from the Work, provided\n#           that such additional attribution notices cannot be construed\n#           as modifying the License.\n# \n#       You may add Your own copyright statement to Your modifications and\n#       may provide additional or different license terms and conditions\n#       for use, reproduction, or distribution of Your modifications, or\n#       for any such Derivative Works as a whole, provided Your use,\n#       reproduction, and distribution of the Work otherwise complies with\n#       the conditions stated in this License.\n# \n#    5. Submission of Contributions. Unless You explicitly state otherwise,\n#       any Contribution intentionally submitted for inclusion in the Work\n#       by You to the Licensor shall be under the terms and conditions of\n#       this License, without any additional terms or conditions.\n#       Notwithstanding the above, nothing herein shall supersede or modify\n#       the terms of any separate license agreement you may have executed\n#       with Licensor regarding such Contributions.\n# \n#    6. Trademarks. This License does not grant permission to use the trade\n#       names, trademarks, service marks, or product names of the Licensor,\n#       except as required for reasonable and customary use in describing the\n#       origin of the Work and reproducing the content of the NOTICE file.\n# \n#    7. Disclaimer of Warranty. Unless required by applicable law or\n#       agreed to in writing, Licensor provides the Work (and each\n#       Contributor provides its Contributions) on an "AS IS" BASIS,\n#       WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n#       implied, including, without limitation, any warranties or conditions\n#       of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n#       PARTICULAR PURPOSE. You are solely responsible for determining the\n#       appropriateness of using or redistributing the Work and assume any\n#       risks associated with Your exercise of permissions under this License.\n# \n#    8. Limitation of Liability. In no event and under no legal theory,\n#       whether in tort (including negligence), contract, or otherwise,\n#       unless required by applicable law (such as deliberate and grossly\n#       negligent acts) or agreed to in writing, shall any Contributor be\n#       liable to You for damages, including any direct, indirect, special,\n#       incidental, or consequential damages of any character arising as a\n#       result of this License or out of the use or inability to use the\n#       Work (including but not limited to damages for loss of goodwill,\n#       work stoppage, computer failure or malfunction, or any and all\n#       other commercial damages or losses), even if such Contributor\n#       has been advised of the possibility of such damages.\n# \n#    9. Accepting Warranty or Additional Liability. While redistributing\n#       the Work or Derivative Works thereof, You may choose to offer,\n#       and charge a fee for, acceptance of support, warranty, indemnity,\n#       or other liability obligations and/or rights consistent with this\n#       License. However, in accepting such obligations, You may act only\n#       on Your own behalf and on Your sole responsibility, not on behalf\n#       of any other Contributor, and only if You agree to indemnify,\n#       defend, and hold each Contributor harmless for any liability\n#       incurred by, or claims asserted against, such Contributor by reason\n#       of your accepting any such warranty or additional liability.\n# \n#    END OF TERMS AND CONDITIONS\n# \n#    APPENDIX: How to apply the Apache License to your work.\n# \n#       To apply the Apache License to your work, attach the following\n#       boilerplate notice, with the fields enclosed by brackets "[]"\n#       replaced with your own identifying informati',
    b'on. (Don\'t include\n#       the brackets!)  The text should be enclosed in the appropriate\n#       comment syntax for the file format. We also recommend that a\n#       file or class name and description of purpose be included on the\n#       same "printed page" as the copyright notice for easier\n#       identification within third-party archives.\n# \n#    Copyright [yyyy] [name of copyright owner]\n# \n#    Licensed under the Apache License, Version 2.0 (the "License");\n#    you may not use this file except in compliance with the License.\n#    You may obtain a copy of the License at\n# \n#        http://www.apache.org/licenses/LICENSE-2.0\n# \n#    Unless required by applicable law or agreed to in writing, software\n#    distributed under the License is distributed on an "AS IS" BASIS,\n#    WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n#    See the License for the specific language governing permissions and\n#    limitations under the License.\n"""Kaggriculture route-replay chassis (pure Python, stdlib only).\n\nA *route* is a pre-computed tape of 719 Kaggle-format actions\n``{"farmer": [op, ...], "hands": [[op, ...], ...], "market": [[order, item, qty], ...]}``.\nThe chassis replays the tape chosen by a caller-supplied ``router`` and wraps it\nin small reactive layers (each independently switchable via ``settings``):\n\n    hand_align            pad/truncate hands to the real hand count      (fieldbook_logic)\n    weed_repair           DIG a weed that blocks PLANT/BUILD, replay      (tetsutani + task spec)\n    sell_lead             sell next step\'s lots one step early            (fieldbook _lead_sale)\n    front_run             sell before the opponent\'s scheduled SELL       (hook; opponent_plan)\n    budget_guard          fund each 72-step block\'s purchases             (six_day_budget_guard.hpp)\n    room_guard            keep shed <= 99 at hour 23                      (tetsutani)\n    clamp_sells           trim SELL orders to the projected shed          (tetsutani)\n    dead_stock            sell stock the route will never sell            (tetsutani)\n    terminal_liquidation  step >= 718: sell the whole projected shed      (fieldbook _terminal_sale)\n\nEngine facts (verified against kaggle_environments 1.32.7, env_1_32_7.py):\n  observation["farms"][p] = {"money", "tiles"[y][x], "farmer"[x,y], "hands"[[x,y]..],\n                             "unlocked_quadrants", "hires_today"}\n  tiles: None (empty) | "LOCKED" | {"kind": WEED|COOP|PASTURE|PLANT, "crop"/"animal", ...}\n  observation["private"] = {"shed": {item: n}, "seeds": {crop: n}, "inventories": [{}...]}\n  observation["market"] = {"inventory": {...}, "prices": {...}}\n  observation["town"] = {"unlocked_shops": [...]}\n  Agents act on steps 0..718 (interpreter marks DONE once step >= episodeSteps-2).\n"""\nfrom __future__ import annotations\n\nimport copy\n\nPRODUCTS = ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL", "FERTILIZER")\nSEED_PRICE = {"WHEAT": 10, "CARROT": 20, "TOMATO": 50, "STRAWBERRY": 100, "MELON": 80}\nANIMAL_COST = {"GOOSE": 300, "COW": 400, "SHEEP": 500}\nANIMAL_STRUCTURE = {"GOOSE": "COOP", "COW": "PASTURE", "SHEEP": "PASTURE"}\nLAND_PRICES = (1000, 2000, 4000)\nMOVES = {"NORTH": (0, -1), "SOUTH": (0, 1), "EAST": (1, 0), "WEST": (-1, 0)}\nFRONT_RUN_ITEMS = ("MILK", "WOOL", "STRAWBERRY", "MELON")\nLAST_ACT_STEP = 718\nPASS_ACTION = {"farmer": ["PASS"], "hands": [], "market": []}\n\nDEFAULT_SETTINGS = {\n    "hand_align": True,\n    "weed_repair": True,\n    "sell_lead": True,\n    "front_run": True,\n    "budget_guard": True,\n    "room_guard": True,\n    "clamp_sells": True,\n    "dead_stock": True,\n    "terminal_liquidation": True,\n    # tunables\n    "block_turns": 72,\n    "shed_capacity": 100,\n    "board_size": 10,\n    "max_orders": 10,\n    "turns_per_day": 24,\n    "min_sell_price": 2,\n}\n\n\n# --------------------------------------------------------------------------- helpers\ndef _get(value, key, default=None):\n    """Field access that works for dicts and Kaggle Struct/attribute objects."""\n    if isinstance(value, dict):\n        return value.get(key',
    b', default)\n    getter = getattr(value, "get", None)\n    if callable(getter):\n        return getter(key, default)\n    return getattr(value, key, default)\n\n\ndef _int(value, default=0):\n    try:\n        return int(value)\n    except (TypeError, ValueError):\n        return default\n\n\ndef _fib(n):\n    a, b = 1, 1\n    for _ in range(n):\n        a, b = b, a + b\n    return a\n\n\ndef _step_of(observation):\n    raw = _get(observation, "step")\n    if raw is not None:\n        return _int(raw)\n    return _int(_get(observation, "day", 0)) * 24 + _int(_get(observation, "hour", 0))\n\n\ndef _shed_adjacent(pos, board):\n    if not isinstance(pos, (list, tuple)) or len(pos) < 2:\n        return False\n    half = board // 2\n    return pos[0] in (half - 1, half) and pos[1] in (half - 1, half)\n\n\ndef _tile_at(tiles, pos):\n    try:\n        x, y = int(pos[0]), int(pos[1])\n        return tiles[y][x]\n    except (TypeError, ValueError, IndexError):\n        return "LOCKED"\n\n\ndef _is_noop(act, tile, inv, seeds, pos, board):\n    """True when the engine will certainly ignore ``act`` (mirrors _apply_unit_action)."""\n    if not act:\n        return True\n    op = act[0]\n    x, y = pos[0], pos[1]\n    if op in MOVES:\n        dx, dy = MOVES[op]\n        return not (0 <= x + dx < board and 0 <= y + dy < board)\n    if op == "PASS":\n        return True\n    adjacent = _shed_adjacent(pos, board)\n    if op == "DROP":\n        return (not adjacent) or (not inv)\n    if op == "PICKUP":\n        return not adjacent\n    if op == "PLACE":\n        item = act[1] if len(act) > 1 else None\n        if item in ANIMAL_STRUCTURE and isinstance(tile, dict) \\\n                and _get(tile, "kind") == ANIMAL_STRUCTURE[item] and _get(tile, "animal") is None:\n            return _int(_get(inv, item, 0)) <= 0\n        return (not adjacent) or _int(_get(inv, item, 0)) <= 0\n    if tile == "LOCKED":\n        return True\n    is_dict = isinstance(tile, dict)\n    kind = _get(tile, "kind") if is_dict else None\n    animal = is_dict and _get(tile, "animal") is not None\n    if op == "PLANT":\n        return tile is not None or _int(_get(seeds, act[1] if len(act) > 1 else None, 0)) <= 0\n    if op == "WATER":\n        return kind != "PLANT" or bool(_get(tile, "watered_today"))\n    if op == "HARVEST":\n        return (not is_dict) or _int(_get(tile, "yield_units", 0)) <= 0\n    if op == "FERTILIZE":\n        return kind != "PLANT" or _int(_get(inv, "FERTILIZER", 0)) <= 0\n    if op == "DIG":\n        return tile is None or animal\n    if op in ("BUILD_COOP", "BUILD_PASTURE"):\n        return tile is not None\n    if op == "FEED":\n        return (not animal) or bool(_get(tile, "fed_today")) or _int(_get(inv, "WHEAT", 0)) <= 0\n    if op == "COLLECT_FERTILIZER":\n        return (not animal) or (not _get(tile, "fertilizer_available"))\n    if op == "CARE":\n        return (not animal) or bool(_get(tile, "cared_today"))\n    return True\n\n\nclass _View:\n    """Cheap per-step snapshot of everything the layers read from the observation."""\n\n    def __init__(self, observation, player, cfg):\n        farms = list(_get(observation, "farms", []) or [])\n        self.farm = farms[player] if player < len(farms) else {}\n        self.rival = farms[1 - player] if len(farms) >= 2 and 1 - player < len(farms) else {}\n        private = _get(observation, "private", {}) or {}\n        self.shed = {k: max(0, _int(v)) for k, v in dict(_get(private, "shed", {}) or {}).items()}\n        self.seeds = dict(_get(private, "seeds", {}) or {})\n        self.invs = [dict(i or {}) for i in (_get(private, "inventories", []) or [])]\n        market = _get(observation, "market", {}) or {}\n        self.prices = {k: _int(v) for k, v in dict(_get(market, "prices", {}) or {}).items()}\n        self.money = float(_get(self.farm, "money", 0.0) or 0.0)\n        self.tiles = _get(self.farm, "tiles", []) or []\n        self.board = len(self.tiles) or cfg["board_size"]\n        self.positions = [_get(self.farm, "farmer", None)] + [list(p) for p in (_get(self.farm, "hands", []) or [])]\n        self.hires_today = _int(_get(self.farm, "hires_today", 0))\n        self.quadrants = len(list',
    b'(_get(self.farm, "unlocked_quadrants", []) or []))\n\n    def inv(self, idx):\n        return self.invs[idx] if idx < len(self.invs) else {}\n\n    def in_hands(self, item):\n        return sum(max(0, _int(_get(inv, item, 0))) for inv in self.invs)\n\n\n# --------------------------------------------------------------------------- chassis\nclass Chassis:\n    """Replays ``routes[router(...)]`` with reactive safety/market layers.\n\n    routes         : {route_id: list of >= 719 Kaggle action dicts}\n    router         : callable(observation, step, state_dict) -> route_id, called every\n                     step; ``state_dict`` is per-player and persists across the game.\n    settings       : overrides for DEFAULT_SETTINGS (layer switches + tunables)\n    opponent_plan  : optional list of the opponent\'s expected actions (front_run hook)\n    """\n\n    def __init__(self, routes, router=None, settings=None, opponent_plan=None):\n        self.routes = {rid: list(tape) for rid, tape in routes.items()}\n        self.router = router or (lambda observation, step, state: next(iter(self.routes)))\n        self.cfg = dict(DEFAULT_SETTINGS)\n        self.cfg.update(settings or {})\n        self.opponent_plan = opponent_plan\n        self.players = {}\n        self.diagnostics = {"layer_fallbacks": 0, "entry_fallbacks": 0}\n        self._future_sells = {}   # route id -> {item: [remaining planned SELL qty from step t]}\n\n    # ---- state -----------------------------------------------------------------\n    def _state(self, player, step):\n        st = self.players.get(player)\n        if st is None or step == 0 or step <= st["last_step"]:\n            st = {"last_step": -1, "route": None, "router_state": {},\n                  "pending": {}, "sell_state": {"due_step": -1, "suppress": {}}}\n            self.players[player] = st\n        st["last_step"] = step\n        return st\n\n    def _route_action(self, route, step):\n        tape = self.routes[route]\n        if 0 <= step < len(tape) and isinstance(tape[step], dict):\n            return copy.deepcopy(tape[step])\n        return copy.deepcopy(PASS_ACTION)\n\n    def future_sells(self, route, item, step):\n        """Planned SELL quantity of ``item`` in route steps >= ``step`` (suffix sums)."""\n        table = self._future_sells.get(route)\n        if table is None:\n            tape = self.routes[route]\n            n = len(tape)\n            table = {p: [0] * (n + 1) for p in PRODUCTS}\n            for t in range(n - 1, -1, -1):\n                for p in PRODUCTS:\n                    table[p][t] = table[p][t + 1]\n                for o in (tape[t].get("market") or []) if isinstance(tape[t], dict) else []:\n                    if o and o[0] == "SELL" and len(o) >= 3 and o[1] in table:\n                        table[o[1]][t] += max(0, _int(o[2]))\n            self._future_sells[route] = table\n        col = table.get(item)\n        return col[step] if col and 0 <= step < len(col) else 0\n\n    # ---- main entry -----------------------------------------------------------\n    def act(self, observation, configuration=None):\n        if len(_get(observation, "farms", []) or []) < 2:\n            raise ValueError("incomplete observation")  # factory falls back to tape\n        step = _step_of(observation)\n        player = _int(_get(observation, "player", 0))\n        st = self._state(player, step)\n        cfg = self.cfg\n        view = _View(observation, player, cfg)\n\n        route = self.router(observation, step, st["router_state"])\n        if route not in self.routes:\n            route = st["route"] if st["route"] in self.routes else next(iter(self.routes))\n        st["route"] = route\n        action = self._route_action(route, step)\n        raw = copy.deepcopy(action)\n        try:\n            if cfg["hand_align"]:\n                self._hand_align(action, view)\n            if cfg["weed_repair"]:\n                self._weed_repair(action, view, st, route, step)\n            if cfg["sell_lead"] or cfg["front_run"]:\n                self._apply_suppression(action, st["sell_state"], step)\n            projected = self._projected_shed(action, view)\n       ',
    b'     lead_available = dict(projected)\n            next_sup = {"due_step": -1, "suppress": {}, "r36_debts": st["sell_state"].get("r36_debts", {})}\n            if cfg["sell_lead"]:\n                self._sell_lead(action, view, lead_available, route, step, next_sup)\n            if cfg["front_run"] and self.opponent_plan:\n                self._front_run(action, view, lead_available, route, step, next_sup)\n            st["sell_state"] = next_sup\n            if cfg["budget_guard"]:\n                self._budget_guard(action, view, route, step)\n            if cfg["room_guard"]:\n                self._room_guard(action, view, route, step)\n            if cfg["clamp_sells"]:\n                self._clamp_sells(action, projected)\n            if cfg["dead_stock"]:\n                self._dead_stock(action, view, projected, route, step)\n            if cfg["terminal_liquidation"]:\n                self._terminal_liquidation(action, projected, step)\n            action["market"] = action["market"][: cfg["max_orders"]]\n            return action\n        except Exception:\n            self.diagnostics["layer_fallbacks"] += 1\n            return raw\n\n    # ---- layer: hand_align ----------------------------------------------------\n    def _hand_align(self, action, view):\n        """Pad with PASS / truncate the tape\'s hand list to the real number of hands\n        (fieldbook_logic.act). Extra hands would be ignored by the engine anyway;\n        missing ones just idle, so alignment only tidies the action."""\n        expected = max(0, len(view.positions) - 1)\n        hands = list(action.get("hands") or [])\n        hands.extend([["PASS"] for _ in range(max(0, expected - len(hands)))])\n        action["hands"] = hands[:expected]\n\n    # ---- layer: weed_repair ---------------------------------------------------\n    def _weed_repair(self, action, view, st, route, step):\n        """If a PLANT/BUILD_* target tile is a WEED, DIG now and queue the intended\n        action for that unit; the queue replays on a later step when the unit still\n        stands there and its tape action would be a no-op (the displaced no-op is\n        queued behind it, so PLANT -> WATER chains survive). A PLANT is only replayed\n        when the unit\'s next tape action is not a move, so the mandatory same-day\n        WATER can follow; otherwise the seed is kept. A no-op turn spent on a weed\n        is also converted to DIG (tetsutani weed_dig)."""\n        units = [action.get("farmer") or ["PASS"]] + list(action.get("hands") or [])\n        pending = st["pending"]\n        tape = self.routes[route]\n        nxt = tape[step + 1] if step + 1 < len(tape) and isinstance(tape[step + 1], dict) else {}\n        next_units = [nxt.get("farmer") or ["PASS"]] + list(nxt.get("hands") or [])\n        for i in range(min(len(units), len(view.positions))):\n            pos = view.positions[i]\n            if not isinstance(pos, (list, tuple)):\n                continue\n            pos = (int(pos[0]), int(pos[1]))\n            tile = _tile_at(view.tiles, pos)\n            act = list(units[i])\n            queue = pending.get(i)\n            if queue and queue[0][0] != pos:\n                pending.pop(i, None)\n                queue = None\n            is_weed = isinstance(tile, dict) and _get(tile, "kind") == "WEED"\n            noop = _is_noop(act, tile, view.inv(i), view.seeds, pos, view.board)\n            next_op = next_units[i][0] if i < len(next_units) and next_units[i] else "PASS"\n            if act and act[0] in ("PLANT", "BUILD_COOP", "BUILD_PASTURE") and is_weed:\n                pending.setdefault(i, []).append((pos, act))\n                act = ["DIG"]\n            elif queue and noop:\n                _, replay = queue[0]\n                if replay[0] == "PLANT" and next_op in MOVES:\n                    pending.pop(i, None)          # WATER could never follow: keep the seed\n                else:\n                    queue.pop(0)\n                    if act and act[0] != "PASS" and act[0] not in MOVES:\n                        queue.append((pos, act))\n                    act = replay\n                    if not queue:',
    b'\n                        pending.pop(i, None)\n            elif is_weed and noop:\n                act = ["DIG"]\n            units[i] = act\n        action["farmer"] = units[0]\n        action["hands"] = units[1:]\n\n    # ---- projected shed -------------------------------------------------------\n    def _projected_shed(self, action, view):\n        """Shed contents after this step\'s unit actions but before the market runs:\n        PICKUP removes, DROP/PLACE(non-animal) near the shed adds up to capacity\n        (fieldbook _projected_shed / tetsutani projected shed)."""\n        cap = self.cfg["shed_capacity"]\n        proj = {p: view.shed.get(p, 0) for p in PRODUCTS}\n        for k, v in view.shed.items():\n            proj.setdefault(k, v)\n        total = sum(proj.values())\n        units = [action.get("farmer") or ["PASS"]] + list(action.get("hands") or [])\n        for i in range(min(len(units), len(view.positions))):\n            if not _shed_adjacent(view.positions[i], view.board):\n                continue\n            act = units[i]\n            op = act[0] if act else "PASS"\n            inv = view.inv(i)\n            if op == "PICKUP" and len(act) >= 2 and act[1] in proj:\n                qty = min(proj[act[1]], max(0, _int(act[2]) if len(act) >= 3 else 1))\n                proj[act[1]] -= qty\n                total -= qty\n            elif op == "DROP":\n                for item, held in inv.items():\n                    take = min(max(0, _int(held)), max(0, cap - total))\n                    if take > 0:\n                        proj[item] = proj.get(item, 0) + take\n                        total += take\n            elif op == "PLACE" and len(act) >= 2 and act[1] not in ANIMAL_STRUCTURE:\n                item = act[1]\n                take = min(max(0, _int(act[2]) if len(act) >= 3 else 1),\n                           max(0, _int(_get(inv, item, 0))), max(0, cap - total))\n                if take > 0:\n                    proj[item] = proj.get(item, 0) + take\n                    total += take\n        return proj\n\n    # ---- layer: sell_lead / front_run suppression ------------------------------\n    @staticmethod\n    def _apply_suppression(action, sell_state, step):\n        """Remove from this step\'s SELLs the quantities already sold a step early."""\n        if sell_state.get("due_step") != step:\n            return\n        remaining = dict(sell_state.get("suppress", {}))\n        kept = []\n        for order in action.get("market") or []:\n            order = list(order)\n            if order and order[0] == "SELL" and len(order) >= 3 and remaining.get(order[1], 0) > 0:\n                removed = min(max(0, _int(order[2])), remaining[order[1]])\n                order[2] = _int(order[2]) - removed\n                remaining[order[1]] -= removed\n                # A zero-quantity order keeps later market race slots intact.\n            kept.append(order)\n        action["market"] = kept\n\n    @staticmethod\n    def _add_sell(action, item, qty, max_orders, merge=True):\n        market = action.setdefault("market", [])\n        if merge:\n            for order in market:\n                if order and order[0] == "SELL" and order[1] == item:\n                    order[2] = _int(order[2]) + qty\n                    return True\n        if len(market) >= max_orders:\n            return False\n        market.append(["SELL", item, qty])\n        return True\n\n    def _sell_lead(self, action, view, projected, route, step, next_sup):\n        """fieldbook _lead_sale: when step % 4 != 0 (no town consumption between the\n        two steps) sell the lots the tape plans to SELL next step now, for products\n        other than WHEAT/FERTILIZER we already hold, and suppress them next step.\n        Skipped at the last step, at shop-unlock boundaries and if a SELL for that\n        product is already queued this step."""\n        cfg = self.cfg\n        nxt = step + 1\n        unlock_period = 3 * cfg["turns_per_day"]\n        if nxt > LAST_ACT_STEP or nxt % unlock_period == 0 or step % 4 == 0:\n            return\n        tape = self.routes[route]\n        future = tape[nxt] if nxt < len(tape) a',
    b'nd isinstance(tape[nxt], dict) else {}\n        planned = {}\n        for o in future.get("market") or []:\n            if o and o[0] == "SELL" and len(o) >= 3 and o[1] in PRODUCTS:\n                planned[o[1]] = planned.get(o[1], 0) + max(0, _int(o[2]))\n        already = {o[1] for o in action.get("market") or [] if o and o[0] == "SELL" and len(o) > 1}\n        for item in PRODUCTS:\n            if item in ("WHEAT", "FERTILIZER") or planned.get(item, 0) <= 0 or item in already:\n                continue\n            qty = min(projected.get(item, 0), planned[item])\n            if qty <= 0 or view.prices.get(item, 0) < cfg["min_sell_price"]:\n                continue\n            if not self._add_sell(action, item, qty, cfg["max_orders"], merge=False):\n                break\n            projected[item] -= qty\n            next_sup["suppress"][item] = next_sup["suppress"].get(item, 0) + qty\n        if next_sup["suppress"]:\n            next_sup["due_step"] = nxt\n\n    def _front_run(self, action, view, projected, route, step, next_sup):\n        """Hook: if ``opponent_plan`` (their expected tape) schedules a SELL of\n        MILK/WOOL/STRAWBERRY/MELON next step, sell what we hold of it now (before\n        their supply depresses the price) and suppress our own SELL of that quantity\n        next step. Bounded by our own remaining planned sales so it never dumps."""\n        cfg = self.cfg\n        nxt = step + 1\n        plan = self.opponent_plan\n        if nxt > LAST_ACT_STEP or nxt >= len(plan) or not isinstance(plan[nxt], dict):\n            return\n        already = {o[1] for o in action.get("market") or [] if o and o[0] == "SELL" and len(o) > 1}\n        for o in plan[nxt].get("market") or []:\n            if not (o and o[0] == "SELL" and len(o) >= 3 and o[1] in FRONT_RUN_ITEMS):\n                continue\n            item = o[1]\n            if item in already or view.prices.get(item, 0) < cfg["min_sell_price"]:\n                continue\n            own_next = sum(max(0, _int(x[2])) for x in self.routes[route][nxt].get("market", [])\n                           if len(x) >= 3 and x[0] == "SELL" and x[1] == item)\n            qty = min(projected.get(item, 0), max(0, _int(o[2])), own_next)\n            if qty <= 0:\n                continue\n            if not self._add_sell(action, item, qty, cfg["max_orders"], merge=False):\n                break\n            projected[item] -= qty\n            already.add(item)\n            next_sup["suppress"][item] = next_sup["suppress"].get(item, 0) + qty\n        if next_sup["suppress"]:\n            next_sup["due_step"] = nxt\n\n    # ---- layer: budget_guard --------------------------------------------------\n    def _block_requirements(self, view, route, start, end):\n        """Planned purchase cost and item reserves for tape steps [start, end)\n        (six_day_budget_guard.hpp calculate_six_day_requirements)."""\n        tape = self.routes[route]\n        budget = 0.0\n        seed_bal, item_bal = {}, {}\n        seed_need, item_need = {}, {}\n        hires_by_day = {}\n        quadrants = view.quadrants\n        for t in range(start, min(end, len(tape))):\n            a = tape[t] if isinstance(tape[t], dict) else {}\n            for u in [a.get("farmer") or ["PASS"]] + list(a.get("hands") or []):\n                if not u:\n                    continue\n                op = u[0]\n                arg = u[1] if len(u) > 1 else None\n                qty = max(1, _int(u[2]) if len(u) > 2 else 1)\n                if op == "PLANT" and arg in SEED_PRICE:\n                    seed_bal[arg] = seed_bal.get(arg, 0) - 1\n                    seed_need[arg] = max(seed_need.get(arg, 0), -seed_bal[arg])\n                elif op == "FEED":\n                    item_bal["WHEAT"] = item_bal.get("WHEAT", 0) - 1\n                    item_need["WHEAT"] = max(item_need.get("WHEAT", 0), -item_bal["WHEAT"])\n                elif op == "FERTILIZE":\n                    item_bal["FERTILIZER"] = item_bal.get("FERTILIZER", 0) - 1\n                    item_need["FERTILIZER"] = max(item_need.get("FERTILIZER", 0), -item_bal["FERTILIZER"])\n                elif op == "PLA',
    b'CE" and arg is not None:\n                    item_bal[arg] = item_bal.get(arg, 0) - qty\n                    item_need[arg] = max(item_need.get(arg, 0), -item_bal[arg])\n            for o in a.get("market") or []:\n                if not o:\n                    continue\n                op = o[0]\n                item = o[1] if len(o) > 1 else None\n                qty = max(1, _int(o[2]) if len(o) > 2 else 1)\n                if op == "HIRE":\n                    day = (t - start) // self.cfg["turns_per_day"]\n                    hires_by_day[day] = hires_by_day.get(day, 0) + 1\n                elif op == "BUY_LAND":\n                    extra = quadrants - 1\n                    if 0 <= extra < len(LAND_PRICES):\n                        budget += LAND_PRICES[extra]\n                        quadrants += 1\n                elif op == "BUY_SEED" and item in SEED_PRICE:\n                    budget += SEED_PRICE[item] * qty\n                    seed_bal[item] = seed_bal.get(item, 0) + qty\n                elif op == "BUY_PRODUCT" and item in ("WHEAT", "FERTILIZER"):\n                    budget += view.prices.get(item, 0) * qty\n                    item_bal[item] = item_bal.get(item, 0) + qty\n                elif op == "BUY_ANIMAL" and item in ANIMAL_COST:\n                    budget += ANIMAL_COST[item] * qty\n                    item_bal[item] = item_bal.get(item, 0) + qty\n        for day, n in hires_by_day.items():\n            first = view.hires_today if day == 0 else 0\n            for k in range(n):\n                budget += _fib(first + k)\n        return budget, item_need\n\n    def _budget_guard(self, action, view, route, step):\n        """At every block boundary (step % 72 == 0) make sure cash + the value of\n        stock the block already plans to sell covers the block\'s purchases (hires,\n        land, seeds, animals, products). A shortfall is covered by extra SELLs of\n        unprotected shed stock, highest price first; SELLs are moved in front of\n        the buys so the money is there when they execute."""\n        cfg = self.cfg\n        block = cfg["block_turns"]\n        if block <= 0 or step % block != 0:\n            return\n        budget, item_need = self._block_requirements(view, route, step, step + block)\n        market = action.setdefault("market", [])\n        existing = {}\n        for o in market:\n            if o and o[0] == "SELL" and len(o) >= 3:\n                existing[o[1]] = existing.get(o[1], 0) + max(0, _int(o[2]))\n        cash = view.money\n        for item in PRODUCTS:\n            planned = max(existing.get(item, 0), self.future_sells(route, item, step)\n                          - self.future_sells(route, item, step + block))\n            cash += min(view.shed.get(item, 0), planned) * view.prices.get(item, 0)\n        shortfall = budget - cash\n        if shortfall <= 0:\n            return\n        candidates = []\n        for item in PRODUCTS:\n            price = view.prices.get(item, 0)\n            if price < cfg["min_sell_price"]:\n                continue\n            protected = max(0, item_need.get(item, 0) - view.in_hands(item))\n            avail = view.shed.get(item, 0) - protected - existing.get(item, 0)\n            if avail > 0:\n                candidates.append((-price, item, avail, price))\n        candidates.sort()\n        added = False\n        for _, item, avail, price in candidates:\n            if shortfall <= 0:\n                break\n            qty = min(avail, -(-int(shortfall) // price))\n            if self._add_sell(action, item, qty, cfg["max_orders"]):\n                shortfall -= qty * price\n                added = True\n        if added:\n            sells = [o for o in market if o and o[0] == "SELL"]\n            others = [o for o in market if not (o and o[0] == "SELL")]\n            action["market"] = sells + others\n\n    # ---- layer: room_guard ----------------------------------------------------\n    def _room_guard(self, action, view, route, step):\n        """tetsutani room_guard: at hour 23 the end-of-day drop pushes every unit\'s\n        inventory into the shed and overflow is destroyed. Estimate the shed a',
    b'fter\n        this step (stock + carried + harvest/collect - feed/fertilize/place + buys -\n        sells) and, if it exceeds capacity-1, add SELLs preferring products with no\n        future planned sale, then highest price."""\n        cfg = self.cfg\n        if step % cfg["turns_per_day"] != cfg["turns_per_day"] - 1:\n            return\n        cap = cfg["shed_capacity"]\n        units = [action.get("farmer") or ["PASS"]] + list(action.get("hands") or [])\n        carried = sum(max(0, _int(n)) for inv in view.invs for n in inv.values())\n        produced = consumed = 0\n        for i in range(min(len(units), len(view.positions))):\n            tile = _tile_at(view.tiles, view.positions[i])\n            a = units[i]\n            if not a:\n                continue\n            op = a[0]\n            if op == "HARVEST" and isinstance(tile, dict):\n                produced += max(0, _int(_get(tile, "yield_units", 0)))\n            elif op == "COLLECT_FERTILIZER" and isinstance(tile, dict) and _get(tile, "fertilizer_available"):\n                produced += 1\n            elif op in ("FEED", "FERTILIZE"):\n                consumed += 1\n            elif op == "PLACE" and len(a) > 1 and a[1] in ANIMAL_STRUCTURE:\n                consumed += 1\n        market = action.setdefault("market", [])\n        planned_sells, planned_buys = {}, 0\n        for o in market:\n            if not o:\n                continue\n            if o[0] == "SELL" and len(o) >= 3:\n                planned_sells[o[1]] = planned_sells.get(o[1], 0) + max(0, _int(o[2]))\n            elif o[0] in ("BUY_PRODUCT", "BUY_ANIMAL") and len(o) >= 3:\n                planned_buys += max(0, _int(o[2]))\n        shed_total = sum(view.shed.values())\n        fillable = sum(min(view.shed.get(it, 0), n) for it, n in planned_sells.items())\n        needed = shed_total + carried + produced - consumed + planned_buys - fillable - (cap - 1)\n        if needed <= 0:\n            return\n        priority = sorted(PRODUCTS, key=lambda it: (self.future_sells(route, it, step + 1) > 0,\n                                                   -view.prices.get(it, 0), it))\n        for item in priority:\n            avail = max(0, view.shed.get(item, 0) - planned_sells.get(item, 0))\n            qty = min(needed, avail)\n            if qty <= 0 or view.prices.get(item, 0) < 1:\n                continue\n            if not self._add_sell(action, item, qty, cfg["max_orders"]):\n                continue\n            planned_sells[item] = planned_sells.get(item, 0) + qty\n            needed -= qty\n            if needed <= 0:\n                break\n\n    # ---- layer: clamp_sells ---------------------------------------------------\n    @staticmethod\n    def _clamp_sells(action, projected):\n        """Clamp against a sequential stock upper bound, retaining market slots.\n\n        Earlier BUY_PRODUCT orders can fund a wheat wash\'s sell leg. Their full\n        quantity is an upper bound; the engine enforces actual cash/capacity.\n        Removing empty orders would change the later lockstep market races.\n        """\n        avail = dict(projected)\n        kept = []\n        for o in action.get("market") or []:\n            if o and o[0] == "SELL" and len(o) >= 3:\n                have = avail.get(o[1], 0)\n                n = min(_int(o[2]), have)\n                n = max(0, n)\n                avail[o[1]] = have - n\n                kept.append(["SELL", o[1], n])\n            else:\n                kept.append(o)\n                if o and o[0] in ("BUY_PRODUCT", "BUY_ANIMAL") and len(o) >= 3:\n                    avail[o[1]] = avail.get(o[1], 0) + max(0, _int(o[2]))\n        action["market"] = kept\n\n    # ---- layer: dead_stock ----------------------------------------------------\n    def _dead_stock(self, action, view, projected, route, step):\n        """tetsutani dead_stock: stock beyond everything the rest of the route still\n        plans to SELL is dead; sell it now when price > 1 (on day 29 everything not\n        already in this step\'s orders is dead). Highest value lots first."""\n        planned = {}\n        for o in action.get("market") or []:\n   ',
    b'         if o and o[0] == "SELL" and len(o) >= 3:\n                planned[o[1]] = planned.get(o[1], 0) + _int(o[2])\n        day = step // self.cfg["turns_per_day"]\n        extra = []\n        for item in PRODUCTS:\n            have = projected.get(item, 0) - planned.get(item, 0)\n            if have <= 0:\n                continue\n            surplus = have if day >= 29 else have - self.future_sells(route, item, step + 1)\n            if surplus > 0 and view.prices.get(item, 0) > 1:\n                extra.append(["SELL", item, surplus])\n        extra.sort(key=lambda o: -view.prices.get(o[1], 0) * o[2])\n        action["market"] = (action.get("market") or []) + extra\n\n    # ---- layer: terminal_liquidation -----------------------------------------\n    def _terminal_liquidation(self, action, projected, step):\n        """fieldbook _terminal_sale: on the final acting step (>= 718) replace the\n        market orders with a SELL of the whole projected shed."""\n        if step < LAST_ACT_STEP:\n            return\n        action["market"] = [["SELL", item, qty] for item, qty in projected.items()\n                            if qty > 0 and item in PRODUCTS][: self.cfg["max_orders"]]\n\n\n# --------------------------------------------------------------------------- factory\ndef make_agent(routes, router=None, opponent_plan=None, **settings):\n    """Build a Kaggle ``agent(observation, configuration)`` closure that never raises:\n    a failure inside a layer falls back to the raw tape action, and a failure even\n    before that falls back to PASS (with hands padded when possible)."""\n    chassis = Chassis(routes, router, settings, opponent_plan)\n\n    def agent(observation, configuration=None):\n        try:\n            return chassis.act(observation, configuration)\n        except Exception:\n            chassis.diagnostics["entry_fallbacks"] += 1\n            try:\n                step = _step_of(observation)\n                player = _int(_get(observation, "player", 0))\n                tape = chassis.routes.get(chassis.players.get(player, {}).get("route"),\n                                          next(iter(chassis.routes.values())))\n                if 0 <= step < len(tape):\n                    return copy.deepcopy(tape[step])\n            except Exception:\n                pass\n            try:\n                farms = _get(observation, "farms", []) or []\n                hands = _get(farms[_int(_get(observation, "player", 0))], "hands", []) or []\n                return {"farmer": ["PASS"], "hands": [["PASS"] for _ in hands], "market": []}\n            except Exception:\n                return copy.deepcopy(PASS_ACTION)\n\n    agent.chassis = chassis\n    return agent\n\n\nimport base64\nimport json\nimport zlib\n_R108_DATA=json.loads(zlib.decompress(base64.b85decode(\'c-ri}-EL&p(j53My5<F0e<Xd^M=BpR+!6(;<$}j(9DMM2FoS_Tz-QkXes^~_Syg-QjEsoPwX0hP<10~YlC}2Q>nAfZGU9*y@Gt-AzyCk~-+%pYKm42j_&<L5zy9T4|I2^=*Uw-6@Y}mT{`le3-4Flwzx>z#^UJ?|{_?;4%fJ4=|M|av{`x=u@V7tz!#{re{pF`WfBg8v-4CaqkMBPJ_hI|#F8QbJ{g;3G<M`pj?0cX7=iT%(e|`D=<Inkr&VQYJ+WyPG{QUm+;}7l^U;fOyU*G@o?#l=K_;UK;ZWF%!$Ir*(Z(sglG3q~F{+y5c^W?q%@!$RV+uNV|@`v7L^ZJO>ujW5Jedfg{U4HO&D6@~8{5kenfBW<OhoAoO`A0tf`Q_1_4||=|*@rFuihRHi?|wWQ&lleR;#cwKoQ{8d{QAX@@5Cd${iHi<mp{Cm_qZ4SI39oc{O`XUKfL@3mdJ9x_y|5f^RFK-e=YgW;_av*JuHWGo>;Jyz^9#uc6#^m`1|s!uhT@P{oj5W$?OxZzkL16=gGF#s`ZGk>tXh}mp7Wv_4Q}wGgN-*akW_!8-M6^{yJ}X_J_>*zy4d?PqWWGpTps~-~Pbl^Ug;knE3PQG95u!P~PW;`HrtUPV@5f)iiGl)6Cv?obJU>uQBi7J+pa#{ps=tFGGdfO#DWFESYb;(b!cO9}r9|xIk_=q2z_W4M2TtVM4Ee+nG>sB@IpJ@}o+BnEbiL7p+)mKFL{xspA}Q{ejo0Z<ysk;kOz$_H|fq|7QG^c>i90_wApYKl1Y7!|}(D|M<7Z-#@<p@c#c;9&wky1b?s+@Par#`J3m+VDW9Z@7|J@$&Y^C?`fK}Tq(zQ%Qt+zN|%AElc8l%U^ct_Nt2HTkL;W^=iL~Si}V~d9x(r$Fq6G=<@#rCGfc35hsJw3X<paX%A>uW4kKrEvhN1gIr*%2`6_GNE<^8T<(n?P;r4auUygS>1>l6UP$D$)dZO@yekJjyVtFY$=d;*KU1RL?<Q<7{6ZZGy;vy5g$kK+V7Xb=#eD-X<-Jhk|1X0!4XszW+vfshk9F4H(VCBOjbQBEt{0D!2_xr!^rT=7>FRP%Ft60;08TLTdXTOIDn&RYsGNc?_-XKPt-2r**k-u+nQ|vHKe+i~lPCbX@`O(%`CiDyjaGRV~_6GrI{I)2J_e3Tjj#GvZWP5z01ilcOY|ELuj73;Y!TPEOoTt%<xRFEktN@l*KdX4It@SwwRf!i&_C@bCfFq-F^Ts)k1HR71?2_qSlImTAWL819=&oCoc|TR&Q+!5z&z8Z>c<oIKJ@~PfMNIQ5VD+x+S7fSt$88Ak7t;xDh;`!<^la',
    b'9WDTIuCf$B@p``Zv?YXw4E|D)K!0VASq77>jHmMCxPJimy?m)Hxu(-Yjl-x!yVYeWz%3cLQ@l<pz_zWr{_lMCv0<kK40K8V_k59)+*8MUJv*6oMFI%jpUg2-Bs6ODsz9A>qmtIO`SPz(u?#W~Jv9lhvkQFU4uCdp&g9_R%A-V(<s#UKC#X~b3K%jGiXAQa(I)ye^Fj6||3LnlB%*ElUOxs?l&d|cirEkSjU2>=F=RRSL5Zr$}Wdy}XrNtcpy$j3dHtPntF8mRpd;1Yb0f%7=EO7{2Mf!9&^!c!2KGXA-N*(aa#)9)|8RU;FS0_^*676(Gd%F5l3m)pjUz??r0Ue6-<+o<yE^%FmdIC6u@Dhq;fvjPZCm-C;4kCdD}E1`x%)w|0t$0-_*Q!|c`b-R&CM)1gR(04hC;!Ly<{#s_N382@9=F2JB{*(LrkB`5<JN@nW`|tk&tZkGWsLvaFFaS`98eD-{6%o(L(|h539P#Bg$uZ;l+3QtTuc1GW@u%;&5~jFM<bIS@as@Ab6`UpKwN4j7FLGh%b`!MBR;Yi*y)Dea0yo4b8?3bCdB!Mjt-7}8Ml1D~S;3FfL4qO$cD<+U-z17z4&nOVVdX^9jTa-lT$2V-xwOSDe>g>o?qDo{&V|}eobHS?9=dBXyyM=C5dwJh_;2btui@*<%jf?bj5z!ZNWc989>AoV2nPhq{GJRdPkb)Tnx7AkpHxHV^|y<e$93wHM^Pcq<qss4gAQ}Nn?0RTcTn>Ifdk=#uvg-QFd%k}3~(^Yn5KiSRFIBu*1fo{AVGpk<=iK;Pv^nCE@1!8ti+Nt_@>{A^H}H_4r2hvlSA}iZNq8z#GRkA#DRj_VaaAU0+raTDL)u&JX)!c4Vt>I1P-3=Ky~<`ce*(5%w*us;&ZT^kjIcVxy4`g@tjSYSkkB(jjlo92Z=G=^@7%(yrr$nx)kdyc~gv$L$=~1maL_0v~`xIoIa6f2@pPi^4}#mArOP4v)?xEn@pQS<-4Pm$8YuS@TSXC!~*?bvvQ8ut5s!5+1+EJHZl#=#3^tA&~pm*EK=Wm>R{3#HjU{Ni=A}&-@my~)sf6w36YMN@#F+K@>|x<=QMbS!_{j$ZHPeBZ?FW8#;x&9N!YrZfyLl5Pl#Eb)!wD<P++EG)t@+`#7YF0#TD<J{I-d6CfJB<POcTPMdEVJvQnxY8jkgj!2Kb1A=n#6@1>7_gOpfsQ_thb=ShZ=^d3bYO0|>JC0>pHojIaA>h-)vonUEybph($l*CwgXudpe7s-3E1A?U&Xhhd7=+^@u+A$;=LE5vavhUtzBVsGM4hV)nRsb%ft~p1$f-MzZ5qw(kJ&WC^fpt{BWb-{FJ;3lV@GX857ZPJ2uID9C;pWM;O3YVKPNb}rQ3#D!t*;GcQ{X_P(v34R5-ouZb53UYE4#eFKVJ4oj><MU6CB;0ju}j7oUzg<E>REq=xfRG?LPhG{fB?|R<P0}m7K=@`ez<CX;JEc66TADXswVF>wvMu2hign7@8&<*I!0zgI0}%lIc_=t>EQYF}ps}a<6s(adZ{O?ZX-442zSc&{~+lYKmYtEC@iAI^MS5A`|mTR#{vMl5!k;giHjA(_pG%)~!?DQc0>#h4(3S@hQMDA#_FQSP86!Pkm2mx4(hSo(vhtQ_|2d>UJFxAXKSYTEc{#X^OjHTENb^j(CndPOXUo`u$bl&7?g62)tTr$;XZa{8lml@AA*DZ(ubV0EpvFQsW&}>dn@LjvFa|k?l9*ABTmKKu;P`kJ2r+n@>$XC@zkrcOn~;KLRPk41=`7XIt!2^cCV>40nm&x^&m$d4lSUrfsHuBkXMjJ)AlDGs#w}w6nPJjInGOZ*^Iv^h7a7Lt4941%!_=(~ifA<_ZZ#1=w7xQZ1#J+e)X}Knuu|j%F$tk-uVm=8Oj9n>eh!#8Hb4c-tH+(OjYRM4zW>VgZ!{0A#0gCh?A|Rwm1GMn8Bm*b+==$wjrQ%Z(LZ0YzTa+dauCm6*b_zR5r#IJw%?c}Z*y9|THHOyw24f{M)f3l&T+2CO{8@N!hQMOlz(0Fk3%B2q($vAw)<!K~o(p56HsvTl8qc<e}de8kDoO(K3;TC*pitW}A+76`MjpUJI_6A?kp)XfWiW>u4+MYMUwXHctr^s;y^w>XQ1WZd9&8D+&#zHTjxlxFiWPh2a|#&>H!T%C1FzF-@O-}b_&@9!@!e;j2oZCiMXqIN4yX!i~Icqs1&{0+a&6M?hG2mKhY&K&^3qEZ6a>%0#rF5#9})_=h757`1qI!ODisH(n@{VYDvfi8s_AL*Y=svt)~HeE(=Mn2oofS`m|v-yGnaE??#aa6b!B?1}3%jnHz?SP2snEx>O1-!9#{9N8=W<=M&oHTcZ0n-UWzd-*8Pz^U}?q{pI9k-?hhyDJO?}a)YG}xm2$-Q%)CwuJKA2t<g1xjkss43~0^}R3Wr?-^%)*a!Iv0j4~`A{|oJ@T8*Pa~go#$<9T>beGzoO`eBkUzdGC2(X6AE``l^Jm=bdlQJ^8W)S9^L3gNl>rA&X1LtsH(9)*Z<f7PLE*?-^$-#f7pQK0Z*Dx&t$fm*e$Jan4;G_%b0rFbv;2AhUWU9_aMaogASmhySo-lg)aN#xKfdbob6!t9R(gg#nxIXNdOg0*+zQNhkZZ(ZUeZ?@#byKUmdz12Q@C5rL3YvU;l3KxnYj33h4<jN&{F-b@Nb|N`mQW&VOiEXU(}izC2$l@coD88*w40xAQf7X5Q4|541M5>%L2W@N0Qx8Q5B&iXi%jI-(jNaRWYv)Csl@pu?9Uo_`t=hHNv)eD}`)_$_XPSW*BWTBCLSxy(Uk{g%Z%(f;P5PX(sCA6lcMX<<h%-a^+PoGoG_iBX*hv@fE|zJ6rn)V$^&$L>v|DCj1OYzK&7ZZG|F#kufVQAMm7n9a&SZv6A|20w7OA89Nk8Fb}O#!F8;60O00|3Qt{_g`A5N%<W!IMg-{_x${2OnCc_!226(}8L%XrJA}oPPZGJ8XLg^$vZ=~DVNo}}Iap(xI_2el>WhnU+Lga&NFl4=)xik%xISQ!8q5P=U5x;-q#{<`KF64^VV=mF9pB??iuB7`-Pkpmd|jH%o$BtCREX|e>bJ|Eq^S@-u2W~nj^$;KmS~wl%74w@K}mkjY@my21eIRSpTRhn$5a#8dZ0cFmD4~Q*$l`k7@KnRB={?^j2`O=(_RrjG&2YNAng<^RgIVuI90^0czJ;>8{<Pv=%A_=m1&4<1;rQ!RVM!@*-etvUoLG0m3+YKNuW|iOac_QDbPlL&`lo}Tl^=VpbFPBNH=KPR^+~%*73Wr7nKZNi6wY*)62!g4g+in;zrm+-+8WeLeD5#=rW4FoomrLyNnCa%*(pw*ThgtDwVi~ll(kcFWgiE1_?I{l2cWxQsM!Keen<hBWpQSgGsR$T}FW{c4Do*-ax_AlLAxv<7<rF_Rmy_Chd7ytxpUz;TetcRC1!F0tl|%_!)7Gxqo|YPuNS2m4nQDq#=tg+dcdpkr(9~()*ysBt6vGxL0E%beIbLoDrpXht+aOt??1-&H28-&=9OoOJVd_I&9kCSRFK|0FjM<S1ypNHaRNOK;M9o57(kbN(dG$2U#7t8Mm?~1;o@7$E@Te5mi9I!vu3-tM1wMVF_sj>?nPp`!d`Vd}hmQTxuG{$}!)&kQw5NYq&rqlT16N33r0k9B2i<Bz*>LH`M3YH7O=qqOXiP-H}Q1)^$m&WvEL0d>7DqLFTz!9-~*-rpnIt8tiUoPioqnIhR<_R0-r{b=orNcM<}FZ&L2Dix-RIWhKR9-<K)-njSSJ?1I&tkkdd5N@I#1&9q#+!ejAGYp{>e##q-UFHZ285osk`^!2X*K&5A;wCtVsdRpG`h2T98fWac>h~d`u8`Uf2Dh9`H+C{n%WfA!EuwNJ^mXB$|d8NT)g8Rnmv4V?{JwMeeAp#-i{FzoXw(`DID)cADiHZf|O0q*Vc2DR+iXagKY&30oTd_FFgI2IaLk=PH02#D0rbv{V0T3H$;Zdg6rq9w<+0Saze5)46S6JxtAy2Vw>s7G!5rAPpY+ge&uIP0v?jX~>*!<XRD`WS)!w)J@)fW%6?(R_Bd2d6ll8~~%l2yyNL<vo}Yh@Jla%4C{D&GLbYyctDE#g$HYkR74hUFr^Loc)C?FG}&a%5-',
    b'N9?E7q(<KPVV2O#30mzWy`f{-q#RAj!%hn$H(Ng9`VM@{Z<C1@T%Z6dIagH6a#aE%(v9ngeBbT+M^`VU8?E3aI>&5ic#73V!0wV9z^|`k7U!tMgU6Q+rgm<Tkt;fIr{VneN&O}{80L&<aNnCH~0rJdt7iuQY!K%8a4w$RY&3l=Ho}X8vw1i)%#-S@v#Oa(&Y1nDvR<21M3n_zYO3USP6vQwp=EhO>g5q-FEF1`dY;)qMcA+eqsO+^bS0~TZn(_s&+{4R_1twSPJ`v(E+^-;=$XXmTy_j{vQAFIxWvS)?Ud36W5Jxc2cNE3NkMJ4aM>l+kOL;yQz(I1Xhr!Q)*Mdq-Q{8+Gfc!+@LtcbU7A<KxLk3{+B{8ID@}!kH_VZJa<I(0DUV-kgNeR*>PiQ;4X7Y*{l@_?AZ&P#1A;nJjNj&a-eUzWx{~hcfOWdasm(b135`U?r)@vrTC1~c;<GP89qeqy8u3jFWKLp_Zv3JoWuW(*f8F+xvli+K&z;=4@s`|bxzz~Ryhk{duf+sT^^pAe`E7LK+F5rWcHAf9LYoZ3QZq7x~1}Mb7`#MU<MO&lHxu2=zoN1LuxuxuiV_7epf@`kfpa7-&(dr@g@DtGp*5I;n*;Gy#u*-mj0TxI<peO*YnSx~wQR=HU126>zCISqG9rBf`V!p6K@H>HR>qNyav7H@JgD!RxK9WX)5{$xH${Q&%Rl+)%*a+<CRqv1{8*L9ZR)7O@FDMNHlF)@8FhW)D5xp*P=dyLSLWFvf{p~m=wTjM;(m^w3<+Vt>cI~4EEHjcqzt$=qc5R1Rag<QdI(D&K)XOK~^G-Yhr5q9!5#9Nl^)yUZ5Br_D*2kelAp--iKq@XN#-Re2F}FDOqBs^)SfYQydIFH5O$&N&Pjx6YRI+>?E$V|#-cZ|@dvxBxr#yV}ithP<iKD6Tc*y;B1@&|}2=fAFP58W_iQ?COY}W;1P^|!+wJsmyj2yDVnb<L(%iF$S8aZHrc~ph3k)nIY0UpO{{EXF_7z%71Wwvos_{CIHV}L?zY6vt7-O!Mb=)ZzcS-bop@HKWeU1eXn864TTjSP@z6{kb;V%myvml_TF`y>bN431SZ;2Fn$F7+i|NsWs&m^ymny<~0~tvr_SKXAo$s?2@?tp^3g%+hJY%*rR)esqlnewh2R=1c2B&#|zYf?}iBARY@4Q!#)+zhd%mkE6V8&0&rC!7OvLJHW}Ps;tQvrYY8hX%YqLc_*o`mJ~i1*>OkFZ>G`444k_QU5SBZJQcIiMYIk(b4d+g-HJm=-Z$BmJ-q{FDo?R0x=hGXy2hZ>2m#-eSr!s25thbx+hc+y<FrRMRvjv|Oi>fz3f;7|yi{zc2;mVtDup_zD1D5Om4UK<_Z>`tgWv?we<=lpx$JL>!x;sr<$Vqj0z=aegG{WHOGF!HgWB}{UE&N!^DAP(CKX<lQXarDOs+@#M@fP?|H>7m55rsbixh|&-xb1$n+Rm-^8!O=25xpO_!f0$$>TLv9;J&|-teqknU%|=|9Ocb!rm8+F9>lH0}0`)?x>gy4ORRlE!uiWg02s_kG}R4m{-xSaH~Qz&1XzWH-}cisVcl&m;zBnAt}-lTdaXn%`HFKVYN>l*A*?yIPIhG!l_rru%4rA*=FWgBBg`e6{3XJq$ty(I76rk5&fwwm?qIq#<ixvI)EGGk;eI0O)d-Fl5$FuHhTu(H{>kEKIAg+KGclc@z#}jz$0U;pOvvLk#Typx6A^K-D$MW2w2x`*=TAAuscFH2mv+th%^*4(L;i^xu~gbF$G{~nfZ2?kW?yy7rG4svw*;vf%KgiV&usTACkx1gwm|x7A&0XASoy7Q;h`TSero2IudcXk1V52MNX<FN1F3&pvQv^o7{)@Q*L6M<`2B)!cIKDdoCU+J=ZE1rZTv8Lo$|<((t;&?g7a`<Obgrjgg)U22%oJ47|=j@~Vsc$r4%t^S{#)AiQBIhHZx2ho$cXg#=rfd2IFx=onK2L`r<<g*mFHrB0$aw|1L9^)?<6!&Y&yS@&aE&2ZNcYk?N`63nA)+5gcR2y2a&>KN%WHN`U+PekT+;2NuydYejlU=!&ND@4i!F$RrWPP-QC(aS=PR`f(#+&q;;vFkD(g5@PF_n_OuoRMzIwk4kJA08r61U8WtNO6~K_w2l@xJQ!UB9X74nt(^c^@nV%rc!VLP$t+zfi#^V>vI^zF=3o-pQDh85IWxJYp8J#rkgmVmz$bQ3bTk}w9&afL9KY?S$BYI3!!5}6Nni?NM2g5LJ5QQc~Wdc1Nl3{7?L&n4XHWg01}YS<~eY5rg9O6r$V!d<Yz@X24v(1(RYKl&NM-pMWU(M(S~C~0_J4*4>!;5sl^^lCl9<cgt;kFDb0=hP1RU3AREe?QJzxdKuRE`TH<$VzLT6YO~H3feG@|&6*7d4rfsVBLyNx+mnOv~hBaHs9tG^oqm(y=F>a!YCCV?HG0M_+-bk3W)q-O*laS6O6Sryf)=96MSw8b1=kqJvM3Xw%DzUs^-$*SoN?1RQ@v5vUc{QDf=`pMRmjDMrg(f-LiOZDOOr<uqV8ft(546lXdnOaK?pdmUl7#?kqU+e3u-0@&+^3fek^e7r*gsWzTeAg3^w|A0E8D6D9l>h-&nPTXs=;XL^RI5+z-G}2l>&?Os<#tHcEJ@iK&LRTHo{5J88er3!VO>o=8<Ux>3~WXXCfrM^8<?A)!SfG`5mI~*6nJ_`w!LmMs}b#`-Uh4F9R7BEruWGlCC5{PgO+kyo20tZmDxJ@Ld__(4xn6^8>TprGP@Og}{6fN8lLDtT!XNi)_AK{|oo+;PMX!U7;o7@i^qA8qX_w%g~dum&e33L9}+~IvRqzTpE?K0U>R%;(Ro<k!k=;MU<o2G4Ca=wEA0@<>)r0!wF{w215$<?x)cl1qE3Bx(eA=sQMTfKrJ74*ZO+7680;Jfb+25y3tt?-YVp4j;O;K=->>TUBJ`_I$rWR_yaIla`ceH^t9IUuk$RZCrE``w4#Rt+q)P84r2T04!`K&sguW7@ZS`jM=*}d&wLORkixKYAkEnZ1sP^Wf8+~^<Zzu!Nm8nF2e-;Oe=2$p89OvC@}9^qLr#dKD-obw-=xBC4dd=2sgUKW$~B_t<Dk8NbEYhR`mfKWuT)nT#5Hmuo<?F<O*kHJuVhq|r)sW1hT?ahTIxKj><M_|s@e$CKt8d2L9}MgfQ4uf`wEyKU6CvX5`3%VK*-ceN+PsXS&>DIdTYi@)q}+%SZpQx-=t*3E<C^l@(C?(o<Aks1l!}*?FkX>Ck3>DUJ*1rBV1VW7i>$9ZDCBQIFi|2Vv5h8f2dtcJ(4xUAxfRXBv1q7)pCuW&g;NkBICu5dMlnn#3e4$hT*i$WjJ^FH9{D(od)%+*#H)_nOH|&Gvr+L{7|F~R%)f^IGkZSR!F?!B4-MoOfPRW^rN&hd}$kA;x<FZat9Dk8G2}?f*%~l6O(!%b$vl_gQVT87F_^cHx-H=Q_DyPgR0mWV?c(a`%e{$xu<4hX{mRjA&6ke+eI2!X*G2@joUttLU22dhTU$bd~9i5(}<MPo#kAgljrP=8El4#p)n|MlGB69r#Dk}Db<B~ZFA2-&y7W}JFq1M*KZ)LF}8rY$I`qrE5ht}*3U!NktX!kG=w{X9HppE=TG}@?#ZGAM_rxHU6O3`)&3PFc;Zi{Fb5ry91Q~#%CWF4ED{xv0`-kcERox02Bnc-ZFJ|96w|pwhl&9VImDItAfzG$LM1>5rNo-1R>jGkCtDCtnj(|_ocS6UsCP2u3_Rb01a+#ZZ#c5vHo4!Jaq+xuuP<(igF?2r%FjPl;6Sa;RzCjGFo;v{!q1Pb#ZP7SPIW6N`gTlbgjE~PEdx$9`?TMG87J4W$?Hr+o(2Kbg{+BH1KYZCoJd|GU5>n-yh~C>#%MR|l}quKyM8`I%B(SyGi@{NRlTWLPL@l(<u;luJ+0-$IrG164?){~JW3P9prTHvjAZyFR3Z9NJ#ylRkPMZbgho<W=*!K|#$^U<9ye46ofaS&bHH(;liHY~CF6WXod1irnEB4>8FyhW1z^Dlzn#gkF%;5sNt>kE+5JUuvxJW*(nU=*RxIQW^l^SSw<cZ?7K@M^*(UJTm9LVp6PUZJQF`lBE*zh7-qZuMIK!0qDl8ko)5Skxo0+g4<V+ysl#>L6',
    b'sBGH4OqTSn(l(9XhSXgG(U<|YwAvq;N_;j@UtXhT55ptV2^5dtz93&~PtV-z>h;y%x6?&cxTiH8;+C*#z6kg)^Qm6`-?B6kzi{PMf>^pIRHyKoOi~jC;^d1On8T76)*W?nTYUOe5EIaJ^5NPCXF3u<+R`_gmRul8%FKP1Yo|s$KwIZ9qq7MNn<u!1EL>X_xN?zNqs4Ygq#1JhOrQVi?=R>8Jnzo8ACQy)Zl9)>EzFkn8%akm(y=+k?#Y#_5Cm48Bs$(w)DmXCHj-Wgyud6flS!@_1LvqQq4BN<;5$^kN>V-6UkTPjWwDl2D<19(s2}EO%{OvKMSro88-dEUxJBu;!>bDTiZFYM?gQBOOi3uTPV8)W6$v2lqy%Dt1+e@U!4ESw`(_22$_iiU7fLN)^6!iZ$fiifEom&9Jf{;DFc87v?k{*vcn`?ooK1lkE&v2_f9Gy1n-0gt@Z*Bq1xEkl0BZK=wJkxP0r;~J8^wY^RAxOqBF%!2(S<>GO(qurO|p*aM5hH)Q~=UyAbW*mw|Bb<UF7q#l-IgS31YJzw3;i>&@zbT&glLGES-u*=6Ts{Y6WS@R82O>TL4vnSOr*>F8f$-YK}P+u_vL8PddNdRwf)d#gIN}84@qSfaiXto=SEA^Imb21}+7J9wmv)#2bh(Wz;_bm#VnK>I;R=brB>CDfV>zbMZfr3lKE_Et|WqwJ-}rUYVo;>_S!?p3)_6-UUwcA<L++?MnQ)xa82Yl{WRAR9K^JGBnX<I~_b-<EbiHq-M>~9FzTH9PcoqEBpFb=d4N0d<^H~)?M))z>B;ztH)2IFWM2rozlCgqexJ4BWWXu&!@s{`Al1&WNM>ElYzKG(&41{5Fzp7ZqBf=&)>fedUVbr1cAnAds&&mlR{8RFwGKC*)b{)xqylFj}7?T<t-NuPQjQZCkJUc9UX&rIAb<GWJ1Q2B{R3`hP6$BYo+yAuK{NC3H*v6AX!kCGFzU-kaS=SpOT~k80YfJsSg;4T+%cwuaT1!Z(J)GwJ`?(DB&u3+>(*J-SkW8sAF{M^M&jl&<`G*>l7hu#$v{eXoPCj)~!bVfCS$Gp&fC$Hu^%IM=20Yb+^?HaPqK*O(;-4mevC^&Shm}+pOIOt_spBP*q#pC8kz#i<l%ehmPe=GM32%5rHNM<w2-SMp|S{CGS{}&Q@?VYSfpBZRIH}d$Q85JPTUL&_iJcst1$>S|!OfBBV{Hte~Yv@fZLZyNg@}COlW-68%Re{qOXIFze=%b$k%%;{>NbL6SR}ys^zs(bd0G*Pqu6(&T-iUJr7{Io)6E_rXEaX%<@*)+j2r+$EN@n#1L?V%F_!@_lYV5)YF|nd^HXL8`XOWV+4~Tab-S((R}DZqSg*#bx%cn_s4FLVL}h=~H!ht)CND%051f;me)g#sqYYtcy9aUaR*hZ$a^Dn-U=rGKk+v63vj@uc}%f6GU48r{W*O=oITnua$g3Q2;ZchcjR3ith63pY(<FUF!oRf?8s(GfuQH_jarkUGo_0^>2zrEKggHDL-dz2BPYJ*WH7FJ`|5!tdAAlpt*McDtx<AH)7mirTxjTwX5}?*myy8M0$964nUntjA)6lEi?(IJ(4@kv{s-O(5}P!SG_BZb88AE2QK1A%$7<WOpNv8&Qv_$v_z5fYANTA8zF(VI|9H-tpx!`0h*0@X=$i34(7s%9#o-m-gARh9N!~jAxR4s+Ro?uojp9!#-L0`==CRe*ippcXr7Ele-J%^4xvyyb|^i%yX84@ke;8XQ*$|t@9*yi8m7H|Fs<QIuNZ$BcX)sEZ?b1x=DUZACC_(p<RK;FG>bnJOp)2?{tAa|?kTa2)Iw9-<Y8<omI_YsQDFx(<M_oXte=+HUNI-XFcFIYCV0fe4n2X#{(t-P{fD3a@#D+eHRZWGONqsJ5t4cGO0xz|;A$n1P$$-RH(K3^Uf3XL%Q68)-mYZ|IGV{{Cng;LveV+<`b1vd2u*LHLO6v8uw1s!tb3MBG)b=sm|1((Ua79tJY1kjw2hN8m5nfY29iCsRtJhXV|ID|q!}ZaOBTtZT%|kkWMm!SzNm@&usWyo{2f$)qKk47xLv@m7`O4#Cnfw#>uV|AV=H9CC-DMP=I%``&kXEYDYND0U;Q~4S;25vd)Q>jENK(qvL)?D;cJeOI%%g4$%RRyNqzVrfM)Z`4ZmSFpE{wh^aORqsi<nb_*C~f3f;mtT3-Mem>~*ZpAMzQ;0mPf=YyN3-xk1~>*A~YT8ck^CfHEmDV6wOCZw^-ZcFQJ(}lUqbX?;XvtLZ+puGR)@X^zf`SARFsLFYrhjK+WL$iMT;_}1rPYrHD0w7)2R7)@Tjf!))IWTmx+8vLD7}60{xkSZVikULgq)w;NsMRrre5jU_C4ap@vI=jC#SCt2ampG1Y8*FSAz~~7K{}qF#?{<k(iO}625^tcp$V`AN8u-ujo8c#UC*KwepC2SRib(My0*}0VG6z|7Qm!G)L&@&&05Rpj;jWhY+yk!VTXo1ssg{dP-Ut~(=y|B)qM{iDaDsoz&F-`W7xpL&AF=u{bJ1#ry*G;(nZxr&?K=0ySD|~IEHnzZFE_Xz~_!prGgwJ)yI1<l!M;};PcmT=7I3roOgM7aJsPe;nnPQj5abCqWOT$1#KUWxxk`DwUL<46Pb@Mzr6eS5`K8P_Me|-5$^HB%daF=5cTIbBunHVrTmyyE|CNpo8M)&IiC3&GrXL*e}XAyD);|;e(&###OZjf%UgI^ojDJP;0I*NjjF>x+mwT3Ac|}PoU=Z`3xw(acQZXNNxuFv>X&EN8Tb<^P|bkA`4^0r>CFFkICFY$AUDlo=fS+Zw4C($cgN;DTlMH+NvX5+oCg0jmJ0A62lkDQ<{yRb0OPFNmFP2@Y**}#uC6y~F%28xW@YKpWP!Tsw~Nf3#pl)#fI|7ayNUp+blKmY#i-2dz?uihfgv-dCTzW4+@2&oL&S|fTT7Wrr_n^)gn9*y@?PfW{z3A|Og$`SCV9_t?s;`&`kuc0%ZGQr{&Xp#y!-Y2FYmryGcTvf&*0jMV=sDs*<b!%f>MyXA@d#qPovL}SB~lDzy0ygoM;&D_V{b4F21SpQ1cK1%J4DjUb)=}jOHAmzrCc~Qf&III5CUwG;Z=RG(Z4GJ(0_f&Pf0%s`8Xi;E`+<9_O{lq?%Odb)ko}JfT>JDEhngIU6$P&H0qWVd0wBqZX|ybOGoa5Er~JXZ0UHACJFzUPPAc1=Y}~{=DYl5;;c0OOSRil$sW-;2_maT*cAEibWb~iM{%k`tan+s>vrLe}orfSWIG4#p2_t29V1#f8B2dzaCp}wVGAqxdUNMfQM2ky5g9(_Gv^dM<eO*I>~Z{cnpWG!%d_JX8Qr1^{ILcYlU0LFciB*DHDjP-Xw8_(If7eNP-QNdxw;CAc8%D<M&)Namhu(7W0+1<c`l^x8uIX^vMzY>x*NmNOp$OdTm~K37l&}<crrie{UljGbWjFsibQu-yhv&rTozxK}qfV;b)<7dPwpcvzG!_%S-D(F2WP4E__{?3C2UG<%8BMNx9#~@*#QzjThusP4GLcgU&b5Gqg0Z9l^b=?xtO}K4JOhQIBEocBt9k5ogNct;EZ)8!Iak8qbvRv53=Iv2~Lox(A66NIVzR6IGRI=VT;il4nsg3+s4Wvf7wOr%Uq;Cq9*9gzs%zFp{OQy)}1|mR_@fb+v(xWPy9L(zJt(p{wcmP8hqrfRHA@T(i6r#HNaAV5$rGCRv;^nmxHH(!4{)9jddEsiK1a7?sJs*}B#sIu)-eVS~sn_LxSkj6mL>xwsSo>mUmSMYmDom=v9%EKE%|dUN5s<O~I6#4#&@%x+QAiZI#BjOhk|L3PF)*Hlb+PSz_`<$alvfa8n(B{4tL<Gj&5q}CWXjcU{8r*2rEG?eJK_FT#QyB>L>0qt&(;rc_|jWi=aY|=W*w-c>{SuwHflPYyyS5vJBOInwJhs!Z_Aa=$kKRQ+~fvVZ%P>68-Dub}S-fjOR6YJ;}JZ1L!4Urfo1V!hAr;sJ(b+WFg3UnwO!#WU=*)e#c#QVKt8djSapSaqA+zrjPVkv-;RC1%9oKG}iJw8eA_HmO0Z?Ulop2#j!pggx8h$k$Ln6+1yWszlw>x$Esu*S$2z5s*2CumDIU4;H<m~;T7aKwTlBaN!8t)~Fb*',
    b'SWut7LoXx%RY^_#WtWSe`Vv`ZADqUz8}0b*eba*N}yNX&T^LF(tmWIZmhV@SSax(&E}}m$Z#6RhpD^rrhk0jm^C=cy|L`ChF*<a@-J6NZ4hzE7fc9=^AJ~{1=of%khVfh^<j|8?0&`QyQkyIp{x59i1%Ib`mV|pkF*QuA+w7srEc0j0mIn^5$A%%JYnT3C0fRM1vcTfTV`j3rc`j8Ayz}^lt|bWFSMn&?2~{>-nz68VDd4!;r#O%6XWvtAWaH<DpjLGzkt4X{R9LxS`B({e{Qq$Tx-J+xbz_U4T2^rM9p{*LO^*K;eu0gFn_`1YG<9F#={;~cvgqyC3(`^&r4>5aT>ym4o<^>YbWh3LJuAr578!(-7AY(E_*m~BbLn3gzs@;O^Mo*M57AiFN+hGy(!8$P1^~PlrV9$YDQaeQLxeS*v#HV?G{`{FeGGq(2E>}>M1!(J(^*|w=ws8B=N?V4Z3~u1%+8d_T@&qlUvUN5{Q^*yN0+FV4P3W8qDq}e@z6m*-pljo!L`2S^1^nIFOU7wIn9=`EdI^wMyu-Knm6T>JD&lbbCVU+YB@j^Pt$lEY?7TR2$K&tVIcDSG+$-2c#$sh*v?NCo32v|KyMYplME{LGQ^5bj(eqbWNxb;9Nw7+EuB#^ZWUrIJ>fWOVk<`zo&Q+<f^CsVnbb}h0<m6SlmgAd2+Dr;jh1{X8y>|$w9`onhC+)QM*{JS8lj^Y&CUO))jzbYa2u^QE8o|Bdf|LBDi(Mh?$A#Co|;rP*>b#)a96<Q`#jyzrM;DIcx9XiuGVzks{~%eXcTYFvydUJgC8vU#0V3Q8^9=QXy|(eZ6o_>df=O9n8)v)=^;Rsdf)Q37Ap{fIL#b_?PzyT91RwLZ~C*083y@YWwCIF6D%qMCr8RuelYrwWQIU)BtUX+XCsUVOK(|eXZvK1CT$qjM(ioEO_9pZ7^bXH>q7sTn>zqOoXrylxi8NxsaC}CwTD|rG;Th#*aH(ihW%zb+1SV)IL7_LE77gL8o1%BTJD=p~pMtIk*G0(Pvu}8qLPzw__-ol%5jvr;fVGVaDs9+FGEyw58L>H6WNPI8}p485gTinNdtZI})m`F11l|*Ikt?)Aw9xtt4$D)D)1S5ji)y*ONpWAcPxu+Fia5AKa99<Uw#^&h53MtPw#nc4J{Tu{^PiiYm<$b`>%Rni|}v!KN)tbHGa&v-kkQ?sd)RIoxJfvN5%lCcMQ9S3w3i^v(5kn{?04Nbw6a=nUm?TS_zD_Cq&IEQaf*J|fzI7A)<_K&2YLO`|Ey!3eNKsz>$qkBM{?n8j9@UYkj8nNhQ1&}(&c-2%iWsL8}w;AYI-Z@=#7vPU?6PW&(oVGNmi&PXady4>0N7wW2#5@+O5*eT39f#Mg244oxYS;RV8?oQ`iGE=YkhAlqAwHPRcqQsdgq`5sH5IJ~eMl6<Jy#@@HL7=!9DL+KCag0XJCftG00vONY28H>IY#09+42nk22L_Im(kp5ea)gF{2TLWg`4|9?rMmWX-c!~D4HVuc$h-0KMh9TQ1fh>uHhq^m*PQ>B2BgVUO$G#VtwN^_O`6I09ZgKIoc-}w4UIC>%~k?B6b$bL0U05Gpq>`FoxLLNfz|Q7L=?MFAjK$8+$(;3O}3J1aHROy5lh6=H9N3EmDN#MbVU!zUys67o!6Vu9$b<7LNwmN{Ls33m1HuCUenCWw+U+|vT8*Zz<-#A<@NILmseS&AdR7sl0nU3<048!*rJS-0u-zxZ9k2zL&8v^y-!yyanuVpR|y?p!bHqufc7%w*k4U(MBQdSI|pRoYjYgHhjEqJ>gXLDz^$o&wdLM@hgy2jS>Y@RPObNxupUc2IyFCEK?e!CIC=KFuYI#SF<Guu^jrdOC^nYOmOphP)=DGR_<*1QV8#&rxfp$w#x%X>#kdqvM_&IaP~TTWnVjF#=P1M^ti)29TNZtSTOEh*#0kZXuC&x{mo$*gu|?7+Rx9xfqEjLUPU-I_hj39_-LQK&%Q}V&Zi_c?be9IyG;kOuB{n$49LS+7@gNsWo`K{K7C1GX&!67^JwIEWg-p0n<%pb7e9<<WJ_upQ_E$II;}P`u*+GxRvg&AB8LpW%hz9a@;BL(IhvDkKxVC5Vw+}m0Okv;TAJO#pWNZK)c(w8<8c#=ZS~n!g!KtC8&{?s(PU7ed7fui`sSw6v`ts?jSpl8p{k<g2Agc5o6DB3PGVS<!N}fR3oY)536m`CBhYn}VX|)?cwxCQ6cJeLezguZRh=@bU!C0Q|4k_%~91B^`9wP%q7)B$N8ckVEjvp)pbY;+Y-v&+|Dw=xN8ntMnArulWwDtOja;3Ri7R`%yj-NM3Q5mysu>bdz$&p%Oq3RsvD(yW);`qnMN=zhAOzO^EnO));Q%D<Cu`#B;Ab``66`xznrjo>~X^?ib6JUBt#u$m9bfx>gV=71cC3tk?m&8p2n444onKo^K<2q0HF)}&{HDUT^6}bvhCY-`dsq$vKBzrXrmS#IR-ld@fDKutBeMD;gex~KxULjerBnzmZbA?&aI=3QgUVgX`%LdOg-6L>oEQe0YJ+X&4QtMA-EMB`5?f9q=5u}45A)74Fo~Lxv6XcO3@3f|N*Tf=?=2syy)AX_`XTI|S$l=q98$)bQ1n4N|nu_quqO@~=f1mNi=l?WQskx7LY20X>6d#pErysBpFcR(l>L_a6Y+A$?C!o~1u%NhFQ52Yphq&0)4JU=Qm5|P1q(i0jRkN-;w`u1aJiAYi!RLU@vmx%XM!(fJnF6T%J3&vL%fh?g`+b}JCkz0iD3EV2n`459<WZkO5|uEi97`yl_7>ME>Ake0dLl$5-u~ym{QFD%_Q#iB-hF)03*s*5F3-RF^%X3C^0hWM|K=r+rAu>6t%x!SFGWZ7;x0dE`iJEYI?Df>92?)*_seCo!JiYjN`4U7M2F3Km^X-qnZg6v%&y6Z&!>xFDZ$&WpaLZE8Je_I$oG~V*Tl%FDx-58v%$ciuP@h6r`giK>6SDUa7`@BNq$4qw2n*xDysOcM{|%xpo5MMA`vgZ6G~R-iHMF6d5v$gu^Q`Et*8z~%_U<pCh<vm=!q`J`65C|sny^#st`l(4ZHKoZ<DKZc~~kk%tO%`j{zUm#Yjd)XQ<#DWBo#ijI@ig5dN_y1XgL9=UR3W#1_7a{_2NkHzSYJqqK?z)9TyCslJgN!izezIb0pSbpOk{ZuB-`6@R8|%%9I_c_gic@fvOdQR%Ial7Uya3|NILyWKqPuv}<rO{FK+QO*uaw1*8$9gydewg~`d?8<iqXgy4a2DWv~5Gg+4-4OAUXD8Lurxiq39Zxg5(lRG^cIRyj?1y)1DqN>&-Tn#(m6UuABdwM$%ECZbvqModojNE~#+d-$w^Vhiy_>L6k*QLJ5wYeRKdtFKMg(s>)n#xr8S)lZcV=YLFb-O+rad;=jGa_q#4t5bv)QLZV=4?Rr(-b&M>jn+o7+kqe}d@WkZ!2AwUGh~r@a)({D~4yw?hTc+5E;Cx}+<~KU@@vkWmB`p*xgV$H!_9?4S;gLc_QyPFT_@aKz%B?NI;miDF%-!MoiAxn748HRUQex@8xE$dp?wcaX$h>VtIem9&;dwFnFCdElm3On1Q1A^i50#;@Q*D$C@1kq>I$po3T~X$XBK)dou}ka0@OD}~?wKCO?4XUoKPlQ9yK?;k2>50MpjTSW67r8TJ4xeplYNOg25DvMpWdy;|1scpj53$C2e5V?4ax5F9|NWc?rYThn=k!?Zc`MP;G1@75`v~nk&^MHTUk-}P2;OdPjbaR4BL|%mq)S&{v<bPoWX4^WFSlm1uhf~y+s%B6nAAGPoK89EsoKL-4D#luLh<V>z(iv~Jm1mbisS1~a3<b!7&>n<l7ZHh0@xr8+A(kY##3f)&!N=!_$GSqjst&<bDzMgrCEF{xEbU||nN@(W@_?~a%aa>Hc^|N3Ki@K@4C3<Ok<0(^snyZ0GJbAqLlu<ydntjW<OGU=I@|z~{9}tZ8Sqj(LWR6tEP^Y!D&KJVsBeH~IXAV3aJVj_;g(z%lK7}j$P8*^&f=_m2J)=3UBH}4|EYTvX*ncya^=u$Q=5;uI-OK2`Ke**QODGsrgk<#iem&aYI*yBdUh^ePHPU4f^bi;8IEZ%JA?#l!5GY=$?OCx->haNYs-?k2MXxbJ1',
    b'o9FYCzMWH#b0p&Q=Bsp7&gzavM3*cQ3i6B9EdehJhM|&_>y*tF7cIY4JzcJ?CLq5%y8Tnf^ONhKw)l3ojGDd@>+ULw35Tn{`|8Me8(J5`I)jU_>cnAeLer+_Y}GIFAu<j6s-ln>bjyUGclkNPcY_i&AzYH6|<ce-OMkWrw04P)CF%+SgIJEd#>wu>Qu4k6d-mB;vJCo=7O92ss0Qjkm*4@yXSO4rki$K@r*t#v0F63V7sjR;EBm(O|28Oq?y04^WCV1_a7X4M}LbS#6pWK39NF(Y6t(Bo+nnY&rdm*ir9@QK%6?Oc_IzDcmQub-DRi;x7su3CF<H;TqV0Vr|-07Jj-}DqnQAX~W~_;w8=Eo|nelqPMiBX`#?AUgYX>E5M%AfGp&|gSyZ#a4Q4St(uXPhjdsaxM8Qtwd3;le6Lx4i=MfeY(n8{8}uUu3*C^h@GCu>-)=L|zET=xMr)P{tP5(bbaKa5s*25l$bRAt>;U@RK6daD&Pgy>53=;7%QWh2+3GcqTD<bC*KJrOjJ1O1FVV;ZtK9BWrw)YO{z;{nNqXD3qyg&*m0MLdxIKq_bPKe#^a&3U*=?zrkGHRDJeEA=Mn6-zC2;^Y_6Pp3xd{|A%LDh{M{x?Ah!SndA>Ai;$V0?B))Os|B$@zQzhMYk%LhQr`h50nnf&rl4X(82xa64+BLJlQ<x*--1Jv`SPn;K>bYj%ihxGI(f}L*3jPMn3zWQ}Wj8(A?>Tk+*C`FF&K0$;*yTXh|fuLJMHkzSI7Ef@@go8@|(CtuwLY8aHkcGG#R&ugzK?@m+Qgw4Ny<I`IMhP`f@eiEXQmmWX9^X#ci6tQ5lH!b<P9Kd#%so6nAZ6-y;8oReIi$GT8H9S->>46tnGLK~RXT9Jo&P#ozVY25^Q<LbhUGhBIS8(BXmO}Sog8cq>CGMUsg8yNbLA?~p$n`7$bTzPC|i^RQ&>)#2+WhmCue)-TR`C`>RJ(o5(JN<*hKRfb9N1WB7xmme&GdIYJcYlvL6u?W|9X~#PE&wr9q>|8q=*M|0F76CexDLBXJ3OYprA?P!_o|Cr-6xmOJ-pGx|4k?_6|ZLXef>DVBoPCM6&me@vlOhhc1LT69l3KXcV7%G5|f9ZxeU9$u7)6BT2!rbV7Sy1>(Or8qAG7N!@|@*30}6ZqOGxLEY2q9rMU5f=INJW7BvYQmo05{j2)0GiK%*W7jf5<xfxzAZ=tE|eB2o=MXI-VKUXdk;a^4JBWfUP{(j*69~$YV?V;(^uY}yYD?0YT$1#!DU`23oIsXmd!1RoY{Mp6=b@CY+!{PLzH8|iSN3os!S#D_#^LL7L3iKv%f6J`>j{K5lU3|le?HFl}?>dT6p_2$tKQLOUw&h40sxzI!{wK%r@6FLx;fRI7K!d+az>PU(CX;FVr?JT`%cnsMGg}DqSEE*=yT_LB-hv7xuyoGH@s-@BZXahBRTa`c$}aFe`w;hzC84C#bNv<La>w5udf|A##VC;yG>$S19JD{JJt3hcUlSh9in|R9M8!X@#MVsW6ApqWzs<BE^NRwfP?CLuKDNSY5n}^{m&=!>#B@lPCj$H%wuiBSxO)dhkptJ!6g$ZA$I<RM5*((h#h!HTBi0b5G=jGYG5CS#n$qI&lf<$=9Y)N%LDz<B7W*s<=_5Q(TVybnHdz;fCfm^(uen%@ryCuPH9Un#7BW)uOEI^eIuLUXg4vMPXmE+T=7a&e(aOY@DZq_{lQhlRDcm|B17LNCfB4s4sf@rl{L`LBE(~xG=A+KZhjH#0}#46~K5s69ul18^~P553cW+mVZG-aQq=f4U7m5zAR3YLAwa(0CAabX$aMOV-!2GUEM%Hl`ck>PVBwe&R4!^ZWvLU4*9Tnb(}vc;ux(~kfGlAi<7zwpy@M;XXBGSfJs7}B}#wC<Y)w)b?3Q37)~E)hW;&U5GSwe{^W?>NJmY?VtrJXW!xAoQm@bNT4j^bO9UJVqU@UL=B*r}7_)?qSaHQvf*Xf4yHB=C*Y+YWd&x6`<B-f7&AEeu)?Zx>s6d4lIz^_DVma)&X%pRHWwH~g+VGU$HwdhB*B#bCr%V|dPqzugU8do=XF2G(osT}x>1gmVP3h@nJiFc#Sl+f>pEdr9Qe33oVMv<EMInx<CeWdpYT%o8GdVBhpbc>}?6`~SsUT1^wUq#PRk<>{1%9%gYmNzWNki1a!dT=}Z7M>QG*)1+KNz0@v8*Dh#uTY2#KoT4Ry@6?i4{{&Q*)QMh!8ZSzA9V+Cm+A^(_KMd@@Se99=@};)%r3E`g9r6MnY4Q2+QZvq{$0@OnA*i+Ryd*yY9(KKbz5~Rik!JHmX?WF(IW=H>ORjgL${#6PO?WDrPXqS-PFk$-c#@Ri+m_bm#>xz65EAa{%jMlbsTj<5PalXZ$l=xE@3~?6itO8C@q-)w<N#mp*{!|AcQ^St?;ii&sir;T8FNmTf%WZ6j&eJpS>q#PTX`-KOk6!k>dw92F6C)I1sKxK`Bn0IcioqP~o|I_9VyFXv&c+rgKiTM#T<VlX?EdM|(BkbaYR#IY;tEA*A2Bs<XYy8O<*fsIYa-4jk)OaUj{a0^;`-ZPS9Mg*mgGQW0rswq9oskv^k@Gm@htdr;d>mhUzy9O0V*C5vKQ)p7%iP$2md(^plhEYdl{Xd@8N_PTEBm$v^848j+iPgdNN#w}alR5%dBQTsk>LxC}vwQ&~&)=$8`77kZ>*y854PRRRSr^D;flca`Kyl63;ucLNzUOcrO{J2H3T({xKH+YRE%v6(`XWhusQC7nSXRCvs11!wgDl$P@*_&@B@n#(X$tj>GlTk;%nEIDgCPbNP)d>Ym6(zM^b4Yk5N2ciJiEkhY(S#90}gMS9;S*!glXO+YP0Sm<%nSEme=ZQ;20AWc1Feq%arZ;0xE}&0e+{S@+h7YuS%Wx>`l9p;U&Kg`E|Fx8Z=5-iJGQ?)(t#w<h#F*iR;(zR`fy{CTG3q4usWv4qL~>GGLRve2$z`??C`cB1E1hO^-K)hR|X^Y7aKz>dbolbVuYAnz>xgdkTF9wr$tk%_Hmdj^S`_g!6-ny_D3<4~}&CU9E|@Mwe#{Nhuyl)Snxyq~v+Fm3Ac&onA-vw?E&1_~{=%zP!1_#&-+#qCD0TUGgiwGS(W`Ft6cM?b1_Evh6};B%7-Wlb9AG0Vh04!U)abwvNB#0fYF{*KuWyn2Uci==DWd`t?;lC2<VR9b38td1^oKZ06ns{9_uFaoJIn6`_`--iDMz=Y~)XCkCY|uC;DhL_D&RCJFx6H9Korh%X7wE^2e2i^X%tkmVYxO6dw4*4(57-Iq4dSPm*oZ|zF0ntjLcUjp%*;M%Ef8q!#}q=<BVAp~ARHCzNgOtMbx#LYaF1gF{qRBA7kn^&ieTY`EiUnv+B`+F5ha7saU@43RgC75@Z3C1LOTvBn(X$&}A&npdYSw1kRjH4x5@_lg-VjS`o?!FCC``C~Ho43_`Qsz<jeNBG}QUE7p6c??T{c7^SWPuMq5ggU46QJo!cllxXr+gB9?MYatl(dFx+s9qPo$1BPJmuEtabvY7gDNj<tl5oL-S(=xfewwI`xaqzAstr}E)=1L7^tEO$vOjdLgq|s_=!~bRLErL6-X<9B5wk5QyLBRSD=2iaF}3lvoh)LGkC1zA?-%u<<|~=)1ye>J_j$1qlp6#-8ygE4%YZ8WNXhL*bs}{wAn(0a7yPRX~AM4CaXG<h2Z%Onz%m2b3}AaW|_mQk85>^FwZ9X*o*T4Ps!znoYN#WK~Ry%TRa}iRq3}H`1#j|3k8n1iGnq$^Bj8PWg?ZQ93u$Z{F&3Rhg4qKBayf@M&upwMf%i->QlddXV;qt1*=@1Gfqm}V$njKQ-|n0JUqmS*y{&Ujep<`vf|t2#}W0-NMcq~CC-;3%zn9<<?<fzMJmAX3!4pGu9q)xjBu%#<%>->`_1y*+0!Hh4MZ}SefQ(bFYi9SEWf)Te@x%~@x#ln%Wr-?hZjTeo9{qd30R|)VzeX%Y|~QXIoWm->UfzTtDYbx>BGHtkxB>NoW~ZcXO|V2QsT<IOiC%cEOf?xKDj-Y1iCwiK|LW0^Q&CFk#^Dv!qoQLb<jB`X0!qq`!tZ%9ScIx&traFQK3A(>KJVETZUFab^rOcpn9lmrl-m=RMHjw=rB~n0%Wc$Yw-J@OH4',
    b'@qRlympElC7=Njbmqnt$DJ+de#L#2fe>M^h5k;JN6L7Kf8hMvTSjqUsqf@WRLdx^He(LYulfI4ne&RL}SByzF(bxFj@G@-;`AMsqRgWSxiL`FOk3Wpc=wyV+IH$ryyl14N1I5_U$Z<CR;+qA;0XskQ|zI}IG|!e@A7@v{IVV!NJVhbVIRe2cWAQ`#GlAI>197)OOO(_x`|nxa94GqQ)QyJuhOWkwIS$*LFp?7nUT=jW!esKy2ktE}TEJvB&5ixF+|(sUYxZ<~ABW8kvVIFSH4ss1HuiBa2zB+90FUBU8Lvc$Fh@-ok&MN>JTBxzu=JR^=Idg!WM9S0rkg|OohfNzWS1T)5EW4dmGu^pUw$Jmi}P%!V~lDYwfT;}V5yB~;3JUDoL8A-p9o?q-lxJo(v2BD1CZhxfE+T{AyIy9*pYAz?BxhxytqgULxQ}W^>Em_QU{ElzR1U7?%g=drrn`I=KN2*h|;g?fI!Cwy~S}cbby&$&dXZ!`ISVlZnn!YQmN4-9v*;Q#I?A3tSlAq_{TzrWwo``j(33h!`v?o3R?-sEf$yAJ#-v4^|k;Bns;X?Xp%T;1Yu3`AZr%DaS*zvpc7@+cnW-<^xSSN+wI?gxCEs&&LDc!Jfy+Wk|MK~o919~FPA+N1`>mbVY(D|@&tGr0Ty$Wi2+0J1+6e(f7)!{co8fhpc)!nqv2MIKYoE1~k%SEWM#2e8Rg>m675(G9Cv12Cfn-0=)ifEg|`7V#(3iIq;V)G6;E5Gpi3ra<{t-5U#%4EEPMiW{%<L26rE9-V>Ty?8U&(^Q(k~@IEdDhU;ErvZ8ie3!T0f}(V`u49jM^Xai;N*H0;!W&Vc{JnOUs0M#ssyV#?9q`fj95zf?PVlT`7s0z4Wmp~(dNn2P9&via$%~s!{s3Hg)e(l!BATQoK}xF&(fWVCE{aDPS$3qt;(dJ9{1o%R#jTo+*YylJ0vv7%=UC&lZl91vLMpI={CAV$(u`-f=$fGR}KbE^rwz4uL1p_7$?aD%g%feChBFpk3iMkV2Dx73{*<PB;}=K__r5uFaDAF&0f!hnQB;uJQHO$ft;r7rfOP@G6y_FK^QUN@TSfrs3*<Gx%|In8Wg{1UPvVMq1v0a_?A;JrgP2e3s0tay)6O4MC0Pxf~T+csZS;Lm%_ZZ6Bz9#bzV$(5MI?W9l^w9HV3JmWEGLonk@s=q=!TRYzfrn*$II$>jg6N5gOaX7zj%BpLuD)wR`FX!Xg|;U!h1IZ)~85j}$+VF;|1V;>(@0pr51xhBYUba%~@w1Kk*v;%h(?3h>t9TcKK;Jbn;`Ul{w2lhX@ML%3W@OcNmWgrZjz$0|g6O~;Mcb18#5G<Ier<elKFz~q4BUlH0H4!!(J55{amUzxjNcx+Ni3M4eo4VUIYIn@Ug(KuQj=fZ6_IB|$cO0ceLRzg*3vs^{!7&U^8nt=x6`9<R$s*6yI|I3F1o!+iJxoL(>e?jw6i<ALRbt#>D05C@R{6@{1#a(cq;OF8(BFg@m*->Q&B7sQOhjGejz|aNOG$t6y7ArEzN5=vlu50-k9*gIREv00Nt;iyXha6DZV)HnNV&e8yH=F@hP6FXbZwrLH-OBr95oa>VkABvu&3tLrHxZsx8KJqg3zS%$i|tpY8svE)RpWMvc{^5^AP0rRRIMgwk1cx?mYSW8PR9V8lU9oZp_@f8HSa0gP?3xk$0X7NhqrBk8^E32)lIHgTm-_tvB5>EP475Voq#P)W<p-SQ@C3eQ;U%ES_`u|Zw@?4zf*#_H-bwv#<Z5n_%$FMELtFO7OM6kwb;F6%c%+RGD<kH8aKGbr@ZKH^kaXvDA*fF^{~wcO@v{3rZx-%Ld^;=&SB3{Uo@;2u2e{7Q|}@(si0&omARgh%Q}iLbeY7-rs5}+pvH@al&FIaf1f~9w`lC&yoQE_=rHOuR`teR{czY#Lo2fd=fx*bYty{0C_tHcA0=zPX)*|06IU@DytJC&ml+icHb^jVgBk~9GR0|dRKNOMyI%+#-Y2CBYGLH=rcL5+3H1{lA4GKDC#Bi14|~>7yia@`m>jz>ve1tc^n#0Ii(HEI_FfQZmCz@@&h$TnO+?;$S2d0(3T^QX?8>+MlT+{qj?&$R)XK?eP-6m~QMt9;6!?%FrI}cFqC`Oyu5qBK6yjqd0;6K0wSI2mWPzfuMINW-yh{w>War39oR7F4lKg_pKs#sC<dbRp=)=ZgDfmKhtcI&)i8t{Onnp6SbL!o%?|*ssbw~evJpMK*;bf{xrH-_3<Q(nylbKOm=`0M}hEDK9ae`S4Yd94v#-DEho2@BFS~XQP7S!<9IV+4^UON&6Z1BU));(|bz>x9hgRorDn#)7(1twOQ1`H@kg1XL3(jMa|zk2SXzAVC1W_;3If{b;qDil-^qc)x<tAsH*a>uA{4;rT{a~@Ra-oH>H^VsQ$9C+!`UUELCG6`U9c&@W!9CF_8FevVv4L6_RI<s7z?@Y>v<R3%`OW%HlmO`eXk)jZt!;v9{G#~{f(l|>^r4vs<q_Fl_IdpWZAXsc`Y#ju#ff;mr|Gk|;Ib+fKrC^GjdAydi;=u6Yl*G}YSFj%$A63#Nst+fpsZ9TyN}d`C+cr&HU!7wP$)ty$GC|_$FVH^%bGGH-g0tM$KeySXH?+q@<?{M51u?WbjJPd`^jv0#J?z1$t6G#FaJc{N&-WjG`p4&O@$pZhS0kQlIbp`7v^Nhg9!AB{M>K!B0#9jCnM#&s@r2_&F#Y(hvPgVPU?2|acK!t#m#7O4>`Ula%&RU>5+4$Dz~tQm#<S`A+_T$>8IVrMAzwUdAaNr3UZ`y&x8h>*>akAryd#_<u+|X&V$vuX#IUky;_Zy)+xfOePUU{WU9y8KXe7azAnFK*%4Li|_6g`L;rz+((Ul-Ek{tu@C@L+g%|Vs|i#oOQMVF~t&hoA%aK+BYXCXfwL5UnVr2bPORvH!CXU2Ib#*d<y)**BK@i7cMYUu#Hlj-=9(yf#ukxjJm;|gdu)w1ma5<YH1SBKFKP&1ODmU{%ZarQF<CWh9SDHcsDLvm&s%S^^Fs_)06NkW9Ak22b_bA-|W!aafm+rlyr6(j|{2a2@8NG(^n^_DI5n6TU!9OuK5*gvyID|BI`{no2&qNCYpSQp$gmX%#K0`UwHg=9n@PA~0%pcV;<9+>>baU`ILAQVEAO^I2%7r~PL)|8$M`nE&}m4>CRsZuU`((#Ic;<R+psEk**P!yo9{bSdZI%2mIltLv0`3r0~y>C?VzW=t8*EyFnv&m%~2+Qru)|Ilg9*DmhJD4)P)zm*>Y2gHggxSibLIq4mq;J5q)#^yRU{>kZ<St)Qv(oLiI8zmk2Z3UO1#L&ceV5U(e28QXZj4S4ss$V)Kg_=+fl5IjjW{q8F_n2U%z(pRyz?awfg~wh|5&5ykA|TiU~~9jM8yA?J&UO^;;F1dQ3z@JHr@HWil-|#){aXBbcCEy--glV;K*Bzs)f;mxfG8?Kf)9kjC}gH!>c3&?{(T0nk{pqbXiqV4>n`Q&t}SK#Vg9zD9vWr92}#D@_ufzTY2S<bAPn*6c^nxg~bqUJ~mUc<mqgNqQD5_3gik*eW^4+++)uu8s9x#V(V@3yWv@Ewq#GKXuuTr;@Z-*^*#+(S$m4rz22G-(9?a(qOaMxM|>$oJ32AdQ3Z}0^mUNu04;c+0!MGQhLt$opPV_)Ln=xO_s>;sah_l-)H6(uwS|IO#&}sQm-AUH`M`O1?DyTz9Zd>XS%RTJk*nemI*ONpMd?sRT8rN>NE5QI;V2gu#iWQ*VTT>TO7W^fZ8?<Hd#;>80tPFNrK3tfzlsm!&On5i9DTk)xxWrU^2MC_<e{7bJvVM8vTi?x1C%=%%a_Yv>oy#y6$;r`Q4&<56xGt8w)|j=nxe|BYMVkqFRu-XiY$+wy2IsPH-_1_EL%b9n|GDppkZFoOwq%epU(NE)JZgYx*Pmg4H7s6QSy<R(}A4{gg6{fQW_cr+#>XdwMNiU4xoJ6MRP!bh-i&cJDXqSGCuGApGcpT-(Jot50v_#xyZ|)F7a)p$^a}x%qSy1knnb~Qs|CJJNqgroZ*Rxpl|2<WNdBv1gD&`rlbmkWBGS@LN~o+K`TwvGe`lHPNVGXkE3<TCWRO=m;@_#?Zp(OP-R3bNa;o^&K(_X1l<B4yBG_',
    b'O&3rziJcHuFM=4&UxhsMqLeU^p{d*}1<w+;g%9vvAy3B@1b~!QQ#xL4xq0<MCPxwrm?Jatp^{VZrFZAlD7HSCn7jHfLc2pe8mq1U9+G7J26c-9KPNJ+v+VWj(&n=A<y3LoCEZq8Uvp0~zH#uOKc=JX_%3tvSbQpv5`Yam~(e1y`XeBZ=x&W7c5hnH89oXNE%u50fcbgi>he#(smL`z;$&W9mkshDsuxc7^oPJ%eqcO1`-)#%Y)tq<1ms#z)NhVJ#%LS)nE(3!jP<H|ywIp6?CThgz(UK|i(*v5KN)|tp84ZDFmNauTK<qm@en+^n+0#PZW3p8(aewPlSn1`O4Y?|t0cP1T{MrtdBv=<Dn9!_Ytk@BBOg0>UhX7w3O5kIF2+AZ39aKSU8?gxko1C~T`AD5N2LosD=LnqQ#?C`;>Zo&m2@?=Wx6YJ&i@dOv{t<n(kxyg~ccYK&DvZMtzb1@%W*F@h&Cx!4&B-ee`<FszryR(1%V3iY6aL_VoyS<2OI00Iq7;5ybR>B7vZ-)8oHmtFd0M>mA5*faLr2@_!;}q?dDf-OhSGLtB{?AdJTHtb`L+etDp3g&?!X1N0N?7ie;3kxKxGy5A5ogps4B-17c`1fA{Zvm3BEf#c}fLRCRM=JOh8YPx3z3PX`zjIX3jDaU)9Izmn>@2GqriJ9smqtTO#Rfj<nzjnN?K7wKIZZdDPEIKvNUs`CdkfY_i;eT8v02J@gLggwh<#rW4onc#e72*AZSWE=abq&GvjkjrP+X)?%k5Zs@5YUte7>hM4kBY6D9Zx={UM+S^T9YeBY*mK!?@Pjg6FYD?2s&|cvb9_7cgzvs^dtQ>+@KBQ7sm`akKI~IUixqa(vqWB7|fCRgyd`p8{{|(hzdA|SG5o`S}qNv1~z>!3~jqoWulc?^g7qfQdP&q<6ric1h78w1udY$W(+xx1VYbpwUGg;2>C$hQ7H2yjwo7~mJtf7ocX&PmPZ>OY5=Hz<_X%^?^d+KPWG4aqcAJ@yI<s83McvEdm)^1~K4BuWdldeWrC)KZtfm(1uL^DpMOAhUcXto%ZJ)+2MbVS-L%hDHR|4+A{&~m$?B07B+UJr+31d(#MPg-m%ks9|kaw(zIN!MU0Eh@P!49un_DSVSversKx)4!y`SN!jIEa-+&ML-rW+6X7q?xU^}ML5_ph65x}I51{r&+7b^e^$#)f(?!7kn8qwW^;!s9Kza-jU9a!8mep2$L;u*H52w2*EU+o4Zw^PX4U3C4oH9E5#3Ob2+tT+;KoK9!C?UUa6@$*KPLE1^b6o-02+zL79=z9FmAd?ADT5`AfI?e;AbW9A^SyQrx@d6Iv=g0M{m_bQS5|Ot7y2w#0@%HW}A}FU@Y(Ex*iFvo1550A3V}0|2DjSa~oJvPW{pALYK~-d9~h7d(j3ZF!lAviS4BVmPq;6CuzX|I)IAA(lXvZ?RtIA$-<#yto_As5gdooUA(+}*^YvUTO>58mmT+`;FkF1Ro1k?i~s4N6A_cR+ZwBub`OJZy>6}_UtWGIvH^;xc#IY9TQW(rAOzLYw(IzN><3E6X+*tuYshOLOhL}ZVN_X6@qNNvI;Ib1I%tRbN8WJOXHjPTvq`x|bWyRV7-~f!j;B6*ja5A8$dCP&HO8i|wZ;}A6<73^^A}0vcBt$k>KtRTq@g1dddE(GR_{%M$B3IpuEEFyrFHI>I*2p$xP52^i*w~V(o$oi@9rF1_){KmuHo0fQ_}U2t$)uf!!&gp6(z~LV#&QzwNU7tXEv)EnDCIj)3`J(H^tlWM^wF?8Q0TNE!rVg-Mu3CDoR!ArlhJ4nov`JD{(c&o#0t!*``6J&|Y^{y-T&r;s}`nFD=o*YWdj8>G}zpBW8UytH|~-4HMBmWBZ<^?JTaaXlqHq;zo`lINqtrdq{eg`_~#c{sGK@qA8yh76GEw7Ep{%Bl8S#_e8!$NMtL3X-a{JEFHk^X|~s=z)80&(_-)x20J$DA^+LKci;nIt^(tHaPX!=V*x%XGq(95><MI$Vl{q4Qb&oP3MXQYT5Jofg}g#d;Dl}4yT$VpVcJ*tX_qgo*2WrvCMtw5|FEr1N9*eN%0K`S^d+IKwCX&=Qd3fFoC{rFPi9ix)y8)g5RUEecTG*CO}`64Wm<%+&2k$02)!21Fs1Zj4Uxep{l9=gWu)N>iG9=VRie>8iKR}`KYES+l7d4n>ozfb;!7)z375rPeHjL>SDRU{p^;9lSCgsys)XkjaJ$)GXEo+loM<&#236z79ybkTQs7NzDc0+Rh7c1{C#Tk}<kiKNw_TGvb&Wo!j{i~KfMwW3)fW+3+WA=M)#Rpj00kHopVI(K<Ug#Oa&Es}rpfrUP7g|&=Zy!A;0p92qQX!bz6ex##t)Q+2FVMWgLU<auLN`BaD~HI7d%zw>ZG#ZMu80vOx3j;Ex1&Iql=}I2%`5o8B2iHby)c+Z6dB<DwK)Bd6R$|@1^amUlcP*{MK7-hqOlfzOAI|$Ks6nfz}F{>4K`Q43m5-)OaTNUB(5CBF|Q-r4YvAtWMSkNDN0-ii>pD_1MN_!J*5;!(}4Do9>I5TFU;@I!h61r;xowsKojzWSnhZosM$*)eh+>Rr6>BDI6Jfqt0ycUDqsNx$*-$(v_B_8H?ZlF1f~C*VX5>YJ=$Xu)2fFO+3~-YLn)>#kPO!bnoLx^2mBJPaFehN=3nz+)wsE&4`%?acV+E6MI{r<iH9YV6Q>IcKLXAZ!#Vx$}N%u5H3#<(s!U0fu@IB|Lrg9YoQ=7!El3`y+>Sh^Sa_bS(!|PBCY}*wy`9V)U_zki&TgG3T^V{ntaIi0+jGj?R`_KS$wHVNSPn@tqhGUHn>>R(jOY-?H8g*99Wg+CJRvZr>bgyrT{Wdv-FlHdZAVpZ@46t@<=K`7?&nS3sUvx^3Guk6={T~n)ZC}+-}8Iny5QTdIf;<bd5YWKP{QlB(yWmD>VQk<V&_)3OOZ-c|oIjSc+<|aC6)qgPylARsAPZc5wqK@_7sVvIF@tH62s#Pps%?X6}U)p4I=T)ZOyr*17oZ=D9co4EBW%VZlL#WJ*IIZbQ@agMKhi#ST9HRz#t~$Ew2d^33Zap6<RAI8;v&XhQB?e5Y{VLjWYoRcIRNPMj(gdFxs<;1>e~sWE%BP4$5`P5avEbJrBRtw#{1N&FE=R0|dY2qjR^hVC9r^OmUp&MfR@Yb&oYvE6_kfTWmGyc|Uv0w0j<A}m<2><>7z@ABYA9YTYG#@AH6v<b8as)4b{_d?6;q<~G8yI_Y^CDT-EnQ1^$k}ADOn^|XJX;0>Hx)#*>hHYP{L(XB|HrKOK6P+RBZcYG}4A`c9igvP&$b@ktOGZO2wttr@6U9`GXadkCIN#;m&&Me5foX&hML@%<1HN~aQ`b^lxGS;fSmDNL>PuUkNk)>84RBtYc-54YGNNc|tk9aC-W_Azfw8g~tz38=k5N@ep|=M;$~v}Z7M@Y3iB}vlFc`9Jfgx|gn){o>AS80M+iiKiM41bQo7}Yb{VCJOo6#0>yEXse(FoJWp$QBxgInS%8h2n4{spj~$gdZQ+Cp_Vmg-V`K0XUMC|exuj;55)8&k;2=%jMaD=vHyuBWLmBX^OwYb`GN<cn#p;>ujrw7W?hu2d;l`-M}2u0^>m-?>6(L>UolyMSbH1c-82Oqr|>hX_Qc7p%x>nf6u0aB?&4l<#_8aF(~;&RWC_l<=`G&{ciJUJ}FJD*jqq$8E8?R3U^<<gc?^EW@F1(bdlw7UcY82OwR+Nvc-0D@|}5cllCAdFF+(0_IH!uyw4IJMs8*3%%N@!8~n*7Gji?{oA+eL+9<n5&;dCsk$;h6}E>u-NICIj)w?D;)3e(=(rc-seJ@0R*n5l+hwbvO1C9JLUt*o&l76x8+x{5$I5f@9;c`@g{k;gQh?vntp5s8?EflF6O6ST+bBYp36XhE^y||W5@0Ofc0sfVBSKqD6O4N_GA6ao_YB2Q{IPj_%Xq-!^pt=N%<R+Z=x;Q_Q8mnJVBf_=e58X=yYEWH8KyFKJ!FkZj}TTCpijRRAv%pF)-X+msJTIA^fK2$uuQ>OsMDR6ywy5xRfV>nN<aiOL|mr&6FSCxg*4QgOQoX*2MwNcaxc=A<Ya?wnG+Q7pp$_ant)^TSOzIdVJ@5-e0Hc+UZKUSkTIn?%IvKq',
    b'v9PB1IwSmWt!1XhMGnw4&*@W+5M^|m*o0qW(DXI+m^R7Dv02NN(j|!bu=!N;Ha7bQflNTNt*QseFB8G(GaqP$5A$Mp7c|G)!e{rSc%nE2#o|GG1CviL@FWoKbC367ISG!t&{AW;Yx}-0bQu-t$f5u{(xNnw1pc`tBv%!KC^9eDqNwW0y(JCze-)sj(?Eits}#y$C$b_<YzogC<#SHN<vlokPg(8cIcw;JQi5P2m$?VM653#xVVbI~ukmE!3C+oF_4J_N6T>fn6kt_G#)PG5BTh#<d>-nlJ}FD;e{?a}4zoP$#?K<vMAnj4n}amD*BCEKnDvu&B#(Vw235*u#n}YPEYz#06k~~B&j}|PjfYeZdpntz?J4VxR=6A58k@pO=LXVLiWxi!hYt%}!{O71IUi8VOb!cm8sJa$Vx?#$$`GFvm{?UETvXoq8sO=5ZlJKHh&{!v46zUtRxU{b@MZXE^urpy3;EcrZm~+&zVkqLAq8B1=c%vy#o!}0d}I+<&kk!U==FYprmNUWIqczk=CBK$1V52wm3d3oaL@4z%=`xl@22zoZUx>d*$1(O2mx2}8EdpLon=47dgE3aPz2oQfSPD-Q-UJtk*-#7q@euZKnEGIdBvM46m|1EShE5VA5ZaEJYeY;tx~GO>mzO-uKPWxWz=EDaDsp@W>hRJR!f<jCP4Fexsb>EODU1krUEed!lQ`#HI#%JGt0$E#KshOhxnrF?MWI!Rv0UIYl}>})wTtg47Ain<qcb%aO!Adbuwki3ZyW6IvRVwu#?!Nk*F@^!M-fee52MN#HsMQla!rfM`mUEzT>E2&ma>|yrbfW-99W*47J5^S7hvgvBm;!+%nobeweO@Foq{QP;^d)8y0GTy0lXhi+K7}=<5g1w{i9vRdz7F8~jJwd}>G)Kqo)UCW${vOxbUVMQm=4MKBv!m{A`pU_|~G+N}T)kcgU=bEINFq|}fx4~cLit@#r78$JVtk+K$9`8=$wT`_j20}PX=G;xn^w;mYl>$8^%U3u<q8|~2xTW~35^H9Y5Io%MgLtIsX_<$+kuzGdZ)pwOb$}wvsu-IU}(Q3Lc$Oo`;eF-*{QdwwEX6uG_@-5Gl8mdAzBQMIziXC7h`mQan!a<foU21$SC>6<?a#YMsOF6i5nk7e^7?8#qwEzw0IqR?#r%Z-!A@V|A-9{R!VO6P9v!wb$uM;YM6t_4cow^+8g07_1#grYO>gvtmowd#z06}}}SLn0p!<z^M`q+2I^-h&PE%tfbe;I%KuIupOcHA7z<B}P@^Cqs^J2Fk?3)Dwa@2?QjJiAcS3C%|1*1LQV1pzJYow|!k4OvA)cXwm#3UG#fxb(izQT;r(et7z{Ce)$3`fsGi$k_(GsXiwj+zRA)90GWXBz30ZlExjS)K&D-ddf1)1ZtdsN!%ZWFN3dmSOHfm3J9eR<*GwtWzZtj;y8)J`OoPTO{;32RhoQZ7Ay8!e91Oq`o>bHp|DIxJAUn5SJ_Wl^1ulGz2)N5HO*FFT7f<t>l(HU+|RW+n4$+b_EKxs6_zxnAZJh>ujQ9c$MN<t+6-Z>-K5xvpY!a1;;IIQKWgAuLLC?GQo8g#dkaHgppf0r9yHd=gbK!fw;|}`6m1WrsjyC4=xh1LO&QTiXFLGP%U?AEAy&U;l0=HO3UONAb+ui#<#AY`k@k?m<o8gxaX>#Voq(?orc+y2Ac0rls=}~z)1F_+aP$&32456pD7KvAdgecs%A3zg^!h2eeQS@ZEqk*d<*E#(7`_y5GyYuZdVn^jUYkc7W0>vJE$vsDfe(iW)I2<l5ltqz8p2^bD(@+eVD*w>o0;@)m~AK+sNkZ##r;ZihZ@LPr&{z1<|40v7l2EVXHCQvx_LoJ9nU7k{VU)}`-bFf1G=Vl5|jsrOd-+f^DP^Q4s0H=5^biBWyjtWxGtoTYE61wYmJl!&to}gqcPL1gd{}W(-(AP)kfB5)u5MKft2DK8!j5{MF6_OgYj-aA{c)yUY<<KmeQaZg^qew9*y#eThU$V(DH~T+FJld!SRFY<yzw5a2|R{hD1HZWgB@z-WG(nh(96bQLrO{v?wrM{McfqzSbg>UjE_LPhuug07D4&_O&FhKgp;s1>r8+k`RKM7vEf{VSw}CC?Q^R#ZksUtcA}%;_&VC8m759(@pi@*l`{MQ+<hD8Bj5?`tuE=D%4JxPnn3!VpQ4bP4JE9tg-=<-0S3eG;llLoV;G@$FjEC@YY0kP09!5>)YKPN(~Vp*N&)|TdU@<xv%+meyLKUq5LRbV@?36MZ0<-EA19?h#np6Pp`WMaA6VATEC8{o8^|oSOJDi#t2e}W*VG!387f_wv5Y1#O2L^n#<}o{NG7%39;<K0uzYdIxfS^>_*jd{FW$Ia_2y$G4u5HqofTF3V2-v!hT8S#)?6mYe^Db?B?DiXDnBz;!#nxdHBXirpoGL?N!a_a!mL_G`Mt(?`2hyC#XoM)9Z<^Xv+7R;B9%nY72Z)7rjyfuKzSk=rr-6m^6)~epv%Mb+d<Qk|n~oN%`c)ACp4u`>do3xc3hmr=)u=*kJc*>^3KQ<;kyUC%t&OfZznAaLNhY%B^(9^i*L5$lIP8oR<64oxBvjaMAHwlCL^ZoR@$D#pXGq7@POsu2y^IJ=x%`GUY(<QzN?C59W4n^;9thOi+O(6TU(%0u(5P(!CHapSy27<G)i*Y~=1EB`W9;l0|dmXC&B1_nJJ=v`Pu1$8zd(%^1r!<&4a0S#!vmv`Mu0Xlo6an^ZJiv1Hd(r)=*~x`uL~4Q$_R^?-$TBjH55BFx%^7^aglE$bIKhulpbj<aW)dzLN?(>_h~E1Y-plD<+tt;P8Q*XrB1LM5DSMur#mK{VW7oi*NINfpal0L;VUu3Cd^pw^=u!<gtNU2aX8hg6}XYdV^nzLu~N&p4sgIxoAJ&PyQDm$zzF(7(62RpEHDZ&3>G`nX5jKo`pQLytmPvP>t86n0Am=`MrUGOJH($akcq-Zv1wNy1D8QB)}Qd6s^EL5J!2WPJMpl|-w|$gSm<{1C{Y8MB0#?0Eu7qZYPq8WBT1rnVqy3pxZizj`NvIl)|azCe0{(C5eJn&9UTsjp-DO1N&o(Jxei$;g03kzu3{0<=rwhnCkETKWa*T>EJXEyZ-90k(Nnc5{taaWbJI%3ENpe6|sssb%CP(<xE3>WrR_=f4>fu@--Fn2b_ex<)57SjV_jF7zo-uI|<$W~9m2I`d`$Fs088qAmJJGo)e(i=MoWDUi{#0_o+E4Egp)bthOCGUMoo7qmj`pz8YK7_HoJzN*AyvY@uc)ttlCa@U~eqpYrz7i7$`%gv`gZORsfq*=7MdjTQZ>r$@Ex4mVck2lw7^IL39!@H8p`8F$_@7F;FSX+@-^<zG~CSY(rnf=jH&P~^DTvnM#b50?C^`%PaXr*|o)m-kGA-gJUYseyVWfN0%O=x1&6juEavo>S~Ia6VQ;%MFeSU-RBBT7-q9LI&!tDx5~7p`V+rRBp#Ln=l<Dl7IE=*#KMdS=w9O+6;_f`k~0h(=!N%PFx?(-R1zIlslC&CHjdD_A{Rui$oGgrSpM=e3|<T-s|a7rI-b7z2Atj5*&CRq;ip)zv&g318!U^A@YZ>~RXi+X@Db7L>XZBXk<2aC9QS-J0#HG%boGZ;@Ipk4lHQnWa#l696^Ms)W6}xy={MPnSu-R#C=y$uj-|IhSWC)B-7(O?>)DtUSRvqpYCPqH&%N``4@GfLlqz58)yMk+Js1wsfi4q$_tz=Kb<-?|yy%%e$}J;pgM=H$)XVrtQ6Udr@5*+rx0dBkR!%$`W`(ZUbTE@?^wt%X~(f>qxSjF?;dEhAh&r3t!XN&v(USAtoEZa;cf0G5)em<>-p`Tr}QWch=RSWh=(Ec<uDnpN_=vhkYxi5sOt}1bKQYJ*8lx07xlVj6|qIGo%9(*P}dRS+0HgdzZtVWFmStks>-fE#va!5-0^hM_-(sm3#!Rg;wiU+Sv%q4Q9x3Ip!8|i*0nNiG5uGO+gQkXcm}F{KQ;D6Qyy1t7j4AQcHM-R#*aL^Uv)}qgxGS@V91la&76Hut|9g2oAZfGGdNYpj4Ng;8;s=zDbrWuo6KQY|eO$H>|Tvr|m9rl31>zZMDY>BNg+K`(>~Dn}7QK<u^H2hi=?=^4=EG{DYV0erxfno',
    b'$E%;WvA^xd^%(+!-U`kOX6h*F|Lii9Zu`Skt~t7>Y}`3FhL+*;pSaUG^D5k0{0tZ9$g02Ps(Ku>9C)1pvgu~Gf2zUm-aR$szviR87|Lt=GfC1GZcc|623by-;2$!^eYLjb%J*nN@5Lu#1w@kBddDbsbG#)5_LJ&7T=@R%DJmF*yGMTCAk7+_xcj~1c_la2%10u!Iw1Oq5SR=5n`2bg2?-#1AGqZNG!*##=w|%9Ril~1-*Bu#wQMas5Z`1#VVQgVzD18#eK$N$Zn*)*9%akJ>^uW<m^{=Dc;~kyKhAUQZd2M=>wAK_PE*ki4O}{ZliXTRh!{mM&<tH_~GT(Wz7vd0Us;#!+^iZg2wBRprNt%Uck;~qh*($jE>@5hSykp(9+|ekLQG|7%ju0!}_-6TmSawX2Mtj>=-!J(6CR8!O!RD<Y?mAd+z%1hW_~?cV0lyWX#|<(Jz2Q+-nlW@3MU;UY_c2f4=|l(?32(A|L-;#?2B#$v85FeW@4iVQ_8Ih{iL8X8qtrh;q)F`0^4n|HIU7jsgNi)gnch6w%O|{FnxTrKMF~;F2uJGDrHT<g@2>mg`=o4v6u`<0IHYu?y-<Y21{$4lOZZtqh*Lblfc1*=qT8p)yKZ*5W^*g15vQeI~x^qe@!#Q@7-$`m9{JFk>!Qvv>ziOk88hNiJ4PXoXi!UpLi{FK_v@Sua|%o}?-w`)@>Qq!c>SFLN6IZMv15LZHF&4u0Thc7#P(jCzKUx=0xwG%y?HC)UbH4aau6ZV)Z)n5G`3t<HwR_$sN!O$X18!?JWmOJ*H*Sjd)$Q>-Cn`K0DgI}GSQqGDBzF;L*i9CeUF!KZbqoffhfLrqETBj^Kv-OI#F*AhG0FNSo`N<2Dli^L#<NoF1cfPqC|l|RiFoQa9}ksTBL<GsQ0`pTGcdF<BTI0z09K|K}KBlrN_Fknj)DlXtHrKKlyJG${@5_}axWvWhi%&C>kE+Xf(2m)IisKv4~oeTBbSa)hkeJ<uZ9rr@2*tjA^3bn-n;(u<oiuESuKUN_|jW=(Bw!WOi)3F|o=|}uX`7>4ZD!`<S#<$=+w?EaUCV)R)@NHn%FC}(8+}Bu9Of%Mv24)1Khx;_(&a${me)swA_^Dg@eyH4}q$(CurgpQUInYq&puJLzEV$;7m+wYMDzk7pjt`V}6qc91yj5B};!VZ0avCHy*1;O?)Fsvp*Kh>H9f;g0gBWKGTd5HgVjzAY)$(bW0~AM<c-qa2q#TI;^mq%jV^JE*1h(e@scH`OE^a!!E5?;e%Z{bokV|_ci{s?@)VijRu-CdQY#$>mRzswE4n60-NUUNfYuZLtdU>i2m^Zq?x9WGCcHXUf!?Mo56Kms!WrEUcWxFFvwUda_zq%2VL{nE4F2hV%E3eW?jG0zR;%OacuKUXJaE!^JvdH}weMXcb!V+wnhMaXP;mA;C@lfuF%3AaoCQ$ovnTQgBLrSw@^}+SB-$3N>bm|1nN%#u+McH-=sj?K7*Z@*yC%Dy32@V24<<OJg85s69-+L^qEhwI_vW5!LJ<Y(bYDi3ovduz^x~?Fjkr+6#Unw{y#>s=Iu65TH6DAwXk{)Xc>VOaekU3Q?s}mnS^y9-Ez>|(pzH<ZJv8;I@lp-KEP<J;BuPJKEC6fa8oi^M?VbzA}A9<;GZ|BA6l*%sYM){o7LMm|<l9hRlD-g43U1k(s*O~{=7nfEDS2JmoY(P@<t&ha=qz6h|_D2d5;%(W6P-%WQgx+sDgJl&vnl^Cyp`YP1hGFCZ@%FvgDRku_S^@+f(@<oQF3k|UeQIKGy%z!Q!UB^NU|WfD1i)}47v@D#EnQ6l;M0#7ABZhh;9~Bib%2h$DSYYFp?nTMY1}YL=_DL=S)OiCYOQ3*Hlrem#5TCMf*T2jG?B5hv(?u(f7J{NGjmH%aonYNdn%72by`6{)zW|iyY5)2kWrU;cUuv<C$ke|QcP(zW1W>sodt47u{-61KHMAy0<h{>Rf)1wfuX`W4C!|CV4TPu9F~$v<s2+r=wu#hm9xsY?#g2MYt5hG$)YH_%m^7Q2FTSm_5HS$W+)nFTsh>DC|gNoHJumC=dM<&COSB|^KfIbcV+(`%Xp!nsn|OJ`ig1+f4sGz+CPaR$+-N;UtO;wdBQtWW%Qp*aG)Y<LE8`hN^RPl`EP2pJY=-0ht%nokT|z1lOwpPilfH5MmvP0WmnnGK1k}MFcc$WiQ5<lvy-+(bQK)spdmu`o@PmDjmJR{S0#Dv8AC1JVyoYnLB=IUNkRSug}1zlYOk&Hb5yj}&8A~Wf^gZ5YgmvJ>kEo*&3L4q=}RHK1s?_&(IPiDX~QecZF~9K221iFTFZHpSCs{<VY3>JYn(>2fXF-Rxw<L^>FfihK+C3(j(r<hfc{MtlU>R;2&%dAZR(B7OPkrs81X&E2n)roBspcw!3()E5fm4^1pj*O@YLfnC-e~jM5lI;==kp~h_>^%H#XKHKo&=*_W;XH$c3zHT9$!Qp=SYXZXy;ggt1N8lqEUcUm)N(23Q&jMttKh2@_3{-g8!6IVWPbKQqu$`3A(4jpoH%;76&P5mcDFQ`ZqrTkFR{R>8NTB~IVlW?xWpd%C%WZQS!1p<=>r13iTcU^o??denu}IU;=si+(Zejji<s701a#96og`mr>)vHbBT^<+c&5Ka>V<N=AST58rKF6HMl;0<IR8`;v5`<h#xyxq|mD^DSS)frLf{@sz%tbzoe6^y8=7Jc*(T8=1-|liKJoFi_mct);?4jV>%19RBqF?*%c{cB@Wj)j=&kHMCmx==U?{PL2w(qF`*#+KP`M#;#86&7`$_nh#%sd1z{O!*o{_fd5sfM`TLaRGqKS@r721af??HYnSej2#P%zc3_J(#~#-apTD1D<6e}@Ovgr3_;|v$uMV8~Q8?~OY1CV9*k|6<%%mI>gfn+yYct?L)yMLeA~uMLHyyUz_Z_yN2M#hA6a0mB+DdSE18hT(V!jDkuF*!%ITr}QKDxehm8j0;Aba3$!D`Vlz&q2hWPsw^Yc8IHFHK@J=esoc>x|ZG)AMR1Uh%WP`0>6EVqqTRa^wZrmZ^MN#~_PvUs!|$Wg`7_cSL)qxAM?tARd}tnUGFZ0)aQzAv!_WwgQnqlU6851eh*%@&M{wFoB7@tLRcdXf$@3F)ngNZS4(}5o*j`G`pkWaaYug&-mO51AqPw7&wgiQR}SJsJzi`2dSf}F9*e2?zpZoMrRAgMRryUz-hodM^P}?^kZs;ocImM9>-MkXYw>Id7SIv0P=~V0nZ@*eKMWQpf)}WMUwF44MtAu>KsKy@K~YI*-jZq0ft=bd~`Y_y9{C-azpL+@RrBhzY|^3#39Zu)$AMAFD6*7p~YsrcA6Ll+ueIdhREs^6MZktUY?HdQ%mcWz2JhnYZU~|@iEd%D_{*$CK%Ro?QU7dx5?lQ3T@vR-)Wci`03~g0xb~xCK7H2%9TbuT0l<%W2uxVn`9qu9tp!>eP2M$1x*BYej7$_8`({`DDrmPNzz8KSr}kGK0J>rd<S-q!C0R)yhV1%bn5b*YR<Z-dRD?5AN-OjKMrQ*6R@d1_NLIZ1m+rCksR0%E2;tV`pLz~W~Sl>s3jfiOi{@1#8QZf3R~BuR)X^x<z38;sW(>Y^<kAWPg$j_ft_vE$m$^H4%34m{n@L?h09kxUn6Qgi!O2v`ZghX{QivcqsOcPJJ6w=C94C8B~b@wF5KalJsiFHw|n~W(_pMl+LO5ENCDq}=!e_1wB(VpPIXNlV?m7PJ#Y_Bzj<XWj`?n|h?FucSdW8b5Buq?u&m~SJkDKLXl^m!RB+sYvGP`7uP~8Nmpc}~M-vyqVLd<2jGiY$*tKCe?`69hGRSh<!)R)I`#uK}K2o0tt~x@?KZZ-_>$E-v+~r~DfzSx-YSlj5l(?+L)cs7F>^<;!^&~JXH*APG#4f8=CpPr}@OuILD!9s;mlseaF@#ghPqY+lJ~~E#W9gr`4J{ht<cn!m;G4T3bu~n;@MSW(OCUd};2N09@H}}dL1o#ha;Nh3lh<TLb*4PSZGQ?|FVk#m{kjXix##Srh#O@4a+<kLr`|-sZ1+LNfg(GUC6AO7&<m`S#9B1CcGgBorry_eu<BK2bB#>29E;Pb)a^!gx-e}UaBNf<cJj=7aH#AUk69smRiVMD(?<p#fK^?I{w',
    b')=urKID9vG$I(;&~AW?anf(Z~QAFfmpJw`ln&CF-i8wZCeochI?Mp%F$Re3<JjbnnhFO82ttK?(tKLj1o^cp%O{1P<H-NEWOe`L;1tB6|kv9<UUo+q|HyeI9yeLWDR=IqxEJY3Z2D#)2~I;N3Y^_J>ywd2qt8<Z(S#X;TaDKUV?kaq|iB^=7xe((E-~cpcrzSr4R;6N8{x+s54pV&s8DBkyusmFuBy3<*YdlX;9I6s$3)}13t)46i=C}d{SzS>FQARv>-6SaC$J!ge+KbO#uT`MFVCLE*Yy*{7!DBO~pF^<$MQKa`N2keZigihZh(Y05GL8DQLq1Cog+z!HoIdE~xRZ7#hlDYEhzm-!Do>#RcGa54vv!0<&zT8dXRzlgLY%Sv~Edzp8@4kBEi+Wzx$I8<ctCgap)XtcjJiLm-PjVIH(OCB#xb?OsTzfn+Ll4^63&m6vK#uIol3h=ztHO`r|Q93@ZZ+ty@(3T=W}OPtFSk1+fn_yvHU#cfi{7^F@iHcLSTgCa{I7?SB82iw}}qPc=izf<mly>h4}i!jgpb5U&$QJbwW9v(ww`b$JdUOUa|xuod94yKAo0k74er=<ZNg96u3)Z%g}R0a$@0jGiv3A&x?RTM#oPS2V~2Q<@m6QwpjXFN{?VXC32QrGayDtx%ZFU0odS|>%(a_vsL)&)yAmKhk<@!b}|hIjm+$lGgzQeh>K$bZo)=7;~R3T0B&n1$EA(-vlhHk=8^ej#{7Ml7sE#dvCsKSuiOhZu{zts7^JGolxCoK0M;t<pg2?Jc@cUaDaY)f$R~pg0Igc)-Z+bE~hqJwhzt#sXA1dv&ha#;nWD0#86JM-x;KCa|jzoLgizZ!$g|Qr#|8{WOUXEV!dEZuIIw0R6pSb}oO4B~-46(xcN%D%O*h?*PE6rNpOT0Jb(@9!=Yjbic3FHM&?2&qe}DOK6pnC|<>q;-!e%4|}V!0c&lEI=Led!adr$oM<My=!yZgI0~$m6(dpBic4IOTAj@(w|c@E0yR2dh>}feB_#_=8ISsO!76&@{91MH8gF`%&$RjEcVjU_n5-}wf0P)#`@=rO_8AFReg9GtD|D#_VV+IGWb3d3Z*iM;zcC@5w=;Y*RAq`+ZAQ8|<#inja1o7FbCQa2BzT?q3j-&9^4N@!SaBkHuGI>X(a3n%Ya#X2%8U%=PPw7Px<b`)GXqs&ZfQ2lu~g<@q~+vIaR}S5W~Eb7!i?SG%ml&VOvAdK$ui6uz>%|b+|mbOXH=YQwQ6eAJIfQ2^PCAv(a^1=)~ZOo5+~Po+et<AuC(8@>bg+|OFKH7au0shJ)>=<(k+Zfu;daqs8|SsG#Rc(^8{gt2WR-!-iB#~G^FsKJHQPq=bPHWk^feZPwXsmxMFw8y&>}$ONCb4qLjPRZFM=4qp4%W$-eH&<++;~CT+j5VNzW8yza&Osqzxo8d;h7NaCvELAXN_%d0aUF8^3?C*-;ooUg%dUyI(B`^8yqPH8ypcT)LWYn$$XLF<fvheIq+f6ZJJ5juGL)_tD<VKza~lE<kfDt-T{VCkB7u(7^3W>iurb!>E648ja>jiwdzYG;wx#&MqMP1Prc($lORkbv0u(a%Dl;?NlpE8&R)-2|SzV@ZSX&D^>WSyE~<BWW^&vqrrP@Y@Be4n*69qdrq3EL}zsCn*{Com2r@WO)O35uRMm(c}$`lM?rN*Hs#0DPb2T7l*>#0(uVUqkeq^ojn|C;B8e=hi<m>EEnRFuzvX)B_C=7Je`h91~$mNN=bt9WUS)?!G$GFMq)cTxddu=A+~7r6rp7RvoV8}vW^q%?8zKsu;)*OWlva$CcP5Z$SJ;8LhAYX0Rlht6@9v46G{;n+qEg)VdP0Ky+#-N+PMrS$~~CrY1wfd1^bC$+3tniRj(k5I#~u&SS$rYT!$TFrAj9+`hS>~*3*B;*wK{yC6@Wo5&O=&^4zzVNbH%NY9Log6EjxMjR$6D&xov7X6U6BpLNg6lZyihX*~%%H>?%G+Ya$xf%nJi0FAb%6SQ1Oo0{B)F+#d47@bpkgJ-(oX1~kJYn3`W3mx2dDrI?=iU)N_2As1nRoo`cY^|10U5Q%5&B!wvv(p~G+;p?L(*8zuCEq8B$=+~*iR#arMl%N09dX%n4vTwPpN6etz1;CM15~71x*rD`2v}Y^!kfjePE)l<d?Ds|5}o9gnyY=mhVyG0ksM+aOmO$nl~E`Yc%(~uoe;4DRlcQiRZp_eRcN@Ey4}2&7@Q|bfFX9ElP@0h`Nm*?%hJ%<D+@m$sO394g>uU4u%MrsiK=rB5?CJ<pIX=M%T|1oDk4_#(4IsU1FNm5{#}O2%lGA<2f~(#z!E8*T66)n;S(&r!R9UP&t?T~;JIShC;99tWv{GU?0n$gu}m5eR%o~AoNMt~mhdmsjI-##pyom_z_xT7HYp*@_+u_mc<<C%pnDx_0(6>c$gi1}NY6&r2|@>uP$+{NEAB6~piRHOGiRI!>`l#Ze$+SmO|pvKgBG{IEhEH_h*mP1l2FOy+xZozI8%*m!fRo3ttS=~hJ>t{5Z0OtGd`E$TDigr><wiiWkBMfwsK-YcZ8qV>Fzv64x*pa*H`w%{UaMqLZn$Y5RC-f(~!LMI>zhjoNM5QuaO+_06Q~V9bh53!X^oQjkqLA^qY~SLG8CGPzgGQCs7jLwkm+B5Sl3%<m<W!IYp3tB!XjXwV9&bf&_Zr_k~K0ICJ1qfBRM_(Xn02Dq)t_8Egc~*var>_moVT=ethmL+VyCs;%`LHC^H~IQ)P+`zLwa=K=iwa(1}|-+#E3I(Kf~y03U4#r(}{HE%x&wPIt<<D>*rswmZ3>EC9h<|t?RK&6GDJ5oNY0Kc>m)4wT5$D@`^rtH0yWZ8HaG@NVF4%=R6l~ZwI&SMJ*SCC*T<0L}q7#@LL%h-}lYC0jGydAb(sj>5uaizNRhtyXsoo(J$c=XEzU@|g<s3+M7LkZZ0ktec~!D+Twso(-Wp}df}=jsYHwL(7=UE6>wWU9>c%&Sf&)<RoUnKZ4n$ub2jewv+MZcGW*bnZ5>uZj(z(bohP2`)T}MpYs*$)F}jHkS@@mRK*bIVu86rAl?<?Q;n1cO^T6oO!O!lamy?UHRUYWzzs|Ob_Y81?2a6K4e9O$K?Bt|5uK%%H08eP&VN?;X$89BATb*H!TaXZzf@JPJo}2?{9C;LG*iwOhtM#mjnVr-6Y!Qi?a@AT6tnNCM|HlAG61JV1jh1FJ^t5R?C`a-6Av5<vmg#E!C&zB1<kAjTPJK118M`HGM_jT}%0}ghXk8#!ro>*+%qwIXmd4ewSR>KOPJu#c20eMqFGcaDFf>iiQOkhelqED5AnI&q}D-_fn#n+VxORc4HN#1QwE>Tyk)pM_cwpSB)EHFF7T2acK~S<#QWoxTxlt)5FHR=xtS^x2Nx^$>QTi9pS4nUj2o}){C#GX^H~*nbI!gsn3I5OQ^1IrWMP)w#VEQ;?Ub<YOhtmB$_XWt!bS8x(gmtv4iXPkAMIB8$<3KxSz9~-+q7j>CYcOs^%l`@;qknJZO@j6yula$$7Bws#p(@PK~9``K3AAXwfBi6yp9u@e2~gU)t4SBVLWH#haL!6%2g+mOJJK4?OxQ<OF^Y2!lhGc6jltLEqim{`3oaC;IA)G~?*Y>ru-btqtDfh%Pv5?9RxD_<a^|F@MoJ2v-LQBS-(&BYq<+E-FabGed-`-~SfA?;FCXUHs^>=^I_u%-1nh0EsWeduDzeZwe&jzC&cr)Y)|CDGzQtJHMn=d0Z4vGb;#iM4GzL+mu1oDyPT0kF`Dau9dwzWcVt;g$;;9k(?leK>Xz5fL&DE50s*@#olOu)JTzsZKt1r-Cxx9dh&De{piNAM|Ya*@d!-eJYtI>rAf~2XoB>RR(|XdMknDk1ew@&sqjSn_#lP;+XgZ5g(N}DB1t#2Gw{{3W#7M@u~-2^Ti(1o{Y}_DYRb%fo$uXPXmZ!D1m%=jMzBq1E1j#gBNp~7m)%MDha%ESMOm7{sJ^$1(=1nOjVARpiNpHvYoy-mgOHXcY44-%+G!Rp3|aO2<g#+=+1!r7rlC4&B~*YjUci+*YMrn6J2doR?wFtT*K_sDaq`LGgHFmKhXf(>-;-ma98UBli1s<KczvB;{^#AV?|*ss<=6lI^76-V@>BRmcqj<F)HV',
    b'tT*JDS;>mc=Ef6t%dzF<Z3_K|nOBQ5$aq0B9RG0mIlZ-2i3@Y6qjd<ov-aXc4qrPts2{QZxA{yz6sSlsbymbZpi(%WCY&LSnWT)q{@IDOJm%?CsqPB+C}7`*W+m+(Q+{Wq?B-je87Q#W*ACN6!{<hE&U$!cMmth|g+F9+3+FQ<7l_xgnQ6)sinXsVe8Eq4-KH{sBNhnWn!K&-4I>etu6Tx=t<8lUCzUjF#`c>HaWaGMthQNk7@=U`|o)rxlMd_%9H*vHzFB3zjMP>Fg4juY26_;OiLU&P*MK!WtTb_`16VzsXM2AL7f)r}qOZzJ`x-lt4k&k>%#gAm_g7Lx8{XFDq^<MFWM1Po}TbF#}s%-bIdXFUQQ;x&3-l3Hr}0T+)nW1OUohPUh|dnC@a@9v5Qyd5e*aancN`OR$JI(!7yo=uBHL_9kqNC+sO0wb3*FsZ`k(Bxe7EzZ%`)p915#q4I91%OKE?Dx0&jj9~GfOb4=irRbU-#&hL`E@~YjD0L6sCuCfQ2-E;*)jveaF}j=A<+jJJXX@H7!~qn(w-$y9&vkjdpVDMD0DRxz%gZj%>`>wl)#^C?3vFe>*R;rKK<FRaK;)TjV=C4=K|aGNVFBeGxuF##;V1+j<;1ZNiINJ^221Y&vL%aY8Jb;YvO2@SBYV5q#o@7^#Jq)uekgGZ4jy$V3|%yM3aO9iNem-K)b?~sgLu=o9pPF%2X=Epy#A2+Oc1rMlfP^zraoaw%=%4VkK!&V#^USit&12Z-Dd5+F+H^re2r_@OU>`p%IJ#krF_#q25IX@NLUc7LJphoR{;z<J||4vqc!GR2q$41y0X77n1sI`}&JeZ<Us?_a(m=shzeFuBn_REZ4>(^B65qc4dnfvHtR<`*BNUC-f|VpQ!F>H6twAgOU^h1Zjox!2p}Ul~O8c2+L)bFhv2ii68J+cGsda@7skQZmX`+<qMJgEN)R^uNe;bm}3xbU!3ENnENGf&Gtfo`KBB)HW5ah5D5CY6(O&=KuWUK`66BB6_f-URuDQLpl;EeU4NN21eF$vD23N*RSAI?^0S7~-t?fKBeDkzeWh`rFCbucvC-;!0F>d?0*W!2IDMjzS(Q7Z5u(G$jVZKGdbF*e2ooiVxnX$*sk~3;l&TP3LU5tcf$kl@amDSXjTup$&}fQP+g5!X8|QX96z6h&HMi%P3j)-2pPs8%{=<liIbeKK5$TdE`{LeZLK(YSUla}5QTsg7$y}D)+zph}r<ipeqX~*z6SgT0v=`mCw>(Es@qlG|LM|0~3?B>vol9xLk^rf|5so1P#U?0_kY~!&N@&tf#QHqAo?(*oFH~&1Qu~ffzViN4t9!(>G(G6c3ZY{r^ZsB*)tRwPGGM}?0imaKqID3x&T|nDYsJ%mKNvhgz*{Cnv?O8!=%WI$nC_YC#Lc@@`bH|eWW)|E6*G@HwhCRMO%`r&>>=-<+X~QCqwAaDn2Wa3W4*Vs5O!uFW>9^?{2TjNitI9i8K<UkQyncUVy7*INZ6=g3n=Mw`y&yDV>y=D`GP^{!Vtb5f-zX_=5j^Ddl%&reiF6(T#F7EuEmrf%9$aBRrz@Ll_XjNYRTC_Y)!o$cOl;9K^-mzpoN_3r45zq@W%n|URI-zEbTfC%AzQ;Ecki~0yUT<G=m#pe`>14CS_N<qf8S&6bix`GT0S$2Uj<_xk|u5%uSd7n8FSKL2{XbLb;(5J(CK^sqQUG;e-(@o5<%6J%c_x3|9T}ThvV!uZB&L=?@x<Bb*8%{ajeexX84Y*!27R;+2spV41wpNLFd}$iKCi`ZP+YPd=+5&C9<^s-n;!e_uJ|8TbD?ko`<{j8v)BiD3cD&oAOby#l|Q_yG0Yz}^~Ld6y?KOv+x`k)WU^USfn8iO#K6z+Xv}2JsipYmrb7z^8lcsJ7sLa@eqj4<?j1shaQ{C)2CCK)QZwCnyeDM2M_1-6vOcslgS5feUMKj;7G}(_D3Y2MgyAh{smAA4tG@SR)Ei^5CdRbGj7-(Y%UDGZf^Dba^lC)aG#_0lh2`LvExBI3D3SCFhTUg;{N6i7Yv{=U@@|tqzY9LHa>KA=$(fM?Sec5gFG}$I3!Ko^5`iOB2+$S)YB1s{>DB{-o92S*l(_Z(6(9n-{fu&Ed2ef)>`%LqEyhc9u0QFiyQp{N2R>8c48>#KP6OyDB{Vl=kKsJh@g{OeaDW$PDlv{$fTdt!_S1w&u=LU<EM=XH$+aa72t{LiXsDr$+ie=L`RWwn!!z=|y+<>JF#^MC&MHJ9oMy0Q@9%CTz{d^&LZ_WbwCYjT8QCpw0&F1CX?OKG4+LR>fB^-d4>7SzN&AQ>4TypXzZ9Ha@_v@#|s&H<HEKXA{S(r^g~>=NK{lA*b)-4X6h3X>~twPQc^EaKFRUHy%s<K+zA_R?JMZIUH}5o>!w{yGyZD8grUJa%6ZQ_+3EC!L$W($4e|~yV<MI=jy3AntJEYB`bMq^hoNH<S*2>P30W$9eB~BmHJ(D{&_OFUNR3lTJ~e|;N9p*Pm%(d_#(i<wTfAdtVtVuWc<8{yMeMr?FCEtFLfNUEzn~uXy!GEB^yayY;QK(QGjauC5h6yHYTf<GI|p7sqoHX<+M8(9`-AwCi4Pqx#YanDh<}J?2-$w*nc;DnkPejZoKmwAymg)n+WnpjR-m;i_l?Xvo*1ISv@w2KiWwq?=YS(cfT=5iPu${zB#qGqR-2<#^|$(J_U6Gt7o)3DvGSO?WYn7h;HDx0M9=q<1`Q7h^SWw0wj`K#EVqDuK25(uPhsNwgZdqI_H<3A;KqwPy_9s>a?_H)VYFvwP?)4jfSUMUxTUICocM;Nig@-j5ff;caS$RBp33)k;g!F0f&=v1_vaOQA-|<E@Ep(N)e>ZzPS|Oqoxp$=@^_vN|bC$9m@+n>lio`?wV#Puc~Q%-T>$S;&no+8A41l;<!9BgEmo8($G%YP;CvxqfE4COiWb4mbEq=x<Y*5RdiNvGq+^LrDY~1MZt;!ih0ns5R4F(RVX78b-hV>YfmQIk;#SU&w3~sGD2e*NskwvwKD94OXyI9G-4$ll_M66JvB0Tye$b>Riq0+S$Nrqju1Vu%stIxq__#N&rEPFrFlEClWVa1H+G{e$#{^ON4~IR4Bp22G!mDpELF3w2<KV_R~~)VVR~>a)k2^&=rguPh*TRwM)<S6XCYXYtX{e4pGzf7s&`a4n&NTIj@5SRngHEL1Ek}*ki`>T`p1lA=59ZpW@JVlRMUnR7AJw_Zdy;d_^}JETuyOQ`$baQ9j15sEFxu^TZBDH@*``B%Ee7f^;AbhDW~I~b3X#u;ce&L9ds8bMAkZ*q_n>>N4&5Y?wShtVmD*>&(!+?OGtu<y@#tes}dKO%V8iW5qvo;f0Z4{E<Px>IK31jdZ=rz=bx+r^2y6^_P>f$5cuJsDghsyykqdK6;22ri0TT5QYGVN+i~Io$K^aAh~mkc2N@>Ua4eJXcdHA>LXR=Dr=^0~{8vq`m&Nx<wZI`ROH0|bsBHp3OkE4Vvc2ZQGBab4#!1RAk!D$pEZ_T*NuAj7T=XzSdoO=M0uK`QIJ0|o!N2TFo&ii&1I68(+1x^veUG^;q0!ylIizw`<IFiwk0c_;wQu*#!lkpRRW2pdjGI>#0AJm-FE#?bHe!*d`DCH?ISD>|BOkmG{jj>53OLegB5#CEBlRIul}WN;n4HvTFuz{}^WFK5Sm;DqaUPh9e#9Y^K}#KF8Njymwu;zgNUf(kNFq-t4JZ~V9wW`;;l9>xASw?^43YNe+2G}!v_B@7X5^i)Bzgfy8OioCs90s+QDQpHq1mw!w<I6D*ldB0$oqY0E~>=_T;=e5r%fUyCwH!c$e4aY{X)S#B}=6qB8&tdt0xlGAipINEb8PSkVlx15yrc){MLtxCUYujSLBTMU?A^X(oXTO3mXZdjm)p4wA*T(4GA44^co0F<Q&u5)ui4wj%Mv)6(B3S38*|QrPl<B+gNNASe7ZA!*HR8o=fMUXM|xlJvl1H*dQ=x4SCoEk!q3>Kq^QEPk`{6CaMB-ylGmM*S|HL#X(UNI0xd42=pYZD4U_ufa8pn&23C==eK5^-Rf4oCPQ}~c0`HSzCCU#;lOEKC%>n`L%q-$K~Ns3thCIkxcm|)`O4g(N`e3bh4tNtwkR~rO_XX&h$)4$;;-5bT<^F',
    b'Mwi8L{f`DS~tb^a3MR@YlPQ}3Vy=g7AyU|GpV1S3of^rg0r|g9lOLnzNfuO?CRvMbKipTMH?H!RcD8@z#4p6;%Rhc?3w_}bbbc$tnq~vC&C{=w=sQ8IzR@94y3)@wvn;9H<W?&x7<<joG>5ZeVxWM@T4vI5f4m$`%*(u2=c~0dEoLUpe+2<5*qE66g%TxU;UBQrao>EUAuh_9PEOGyeQha0~U9)!I4iU5%av@w1+cWdW%9V;Eq2q^_U(XhKX$9=-YJ1GCx9ymPM1Reex@OF7J?{n&Cy4$NhZW#MK9k7v((UR!)y4R@G%wseyQdL35U<aHw<MSyyukP%jy-7oV(I-vlAcQ~INxC+W(hy~RA#kgz32{9l!u^}4vV>j{LUff7RDI3EyQT|G(PzNe93;2igny}{@wixI+m4xhSo8!DEd^E&?$%3NTUaMVHLAo5SW7VgJkcQ&(AwUYqdPJLWRt%<MCB$>W|&?YU>nSPh&NK5K!d6K$QZ675S$!&D515l1hbd&B%muy@X0f+!IK52p(St$0?jy0Wx=Xk0Hv|dmNrL5djAcZvu{|2DLG+Tn-0|B<G<$apY!=s&OO4aaT==&E%BUm(3~dxn6g@hzrbWHK(BQ#y_o|Zk|3d;dN6TIde6=Qw>V;@$Zx5`^)zg`J&ndVKvRtn)qvDNKPf$>Q2sxJzu&_PE?#6DWhd9fyxNCl>(Pz;#(4R&CuC!UhEy^fkOSw%2uK6QmO=)j?};Fl$r0Tq?*;g@b=H;-Noeo0K&mMxI1%9X&C&DvoWNGWapV<f7cZYVR0Fy)e=P3*lv%dy0=~97*g6yhn5JcGw6dhgOz%fN$@+pt{!iX57&Q*dNtpbE^I~m{KC>~dIMeik-v@AGwDA{rPdPup%RmS_8!w$s7rc!NDDG~NXx%it6;>#<+0FC9C0cj>VMRp7+!nI<0`&I6e`YHO%n(j94eLj`{}vZ#^%Vp)(woa(g0OEd)7OYxtanDe2VoaE{4gI^CEU;khA?%@av9fMadzBMv0Otwk@onxDkBiG29~@vxzh_8(lHtYE@RkU{R&<?wY%nTd9}hKTm+S0W>`g<UF;;7^l0^Hah{}h0<OD7e7d4oL^mhGdnv?Y1=o!<4m^YsS#J)SG>dFIYePZ8(x-7xfo#VM!-doqf%o?NoU0B+Y)JfG~78nvYd!bw3&O=p+b^w4yKkCb4Dk`L~!I;V!rWj3i_Zv(~Vuk;fX<SNw7kUEA^h~1TBx8;$FiB#g@l5xbd!HJ(;J3*<`;<GhW{#hd~aWpE>;t6(7*)quqnjP=zbNi*cr##mSTSUURF(Hzmxm1!Q&cXCaH#$Z+nNr2H8eC`-fbRmvV~JOM?}^_{(jx<dVZ4_bs-8}VG<Yw^eK9Y1+=i8*v<x)O5H2_qzl5wa~)<n?s%+uVpueEYZ1G?=psO(5b#d`zaW>_i*)kn6=1sC*LYCBe4S<ScB)byq44rgLGZ+pb7OZ_5;<z)Lmz%WC;QV+@~2>uDxL1ZJFocO$O=ja-DPj9Nll%<>k*+H9M*EZEwo+*<rZvav#f4`RYEr<ZLpK))gwp0$2wquTVB0yk-o41#**Ek#M<8W4^xn=HY0EKF^y(~XG#b3>peL}Ingu&%}|11-C}Qk-b>sqp@2Qc3h$sqkT`JL$2i%Ew!j3Tm#ha_!WVoJKvBkD$a0+1=SDMW)H>rY4qmqTQk?<-4jp>($&VyVC^D8*M}!9&}cJO`@?!#G+)K{9BD$!DESWMzJ4ocgqbMk=`{C2T~+!dEg|nHjDzq^SB(AY|(ES%*C*X3u=dSg30Kw=z+$UUDB1|cwkgnMS3oCzCzZpY1TW<8#;Em;C2OF!Fbw~XnI-FrG&MRBxwv78~ycnrpt3g#qaxS4GPm>tJAN@$Cqq|$(}cjn?hAI6<pd|iFE$@V(+zxVJzDS%{}eUu(araB}LL(6I^dI=i%V!F*_rtsgA?!i+5bSm+S^?>Ss}JD|s?x?`*{=+fL|+S@HB}G)kQ%L3zDT^2~D1i`wca!sj`;MobKn#SgrI;s&C)ok=K`gds*ktGLbShPtO6IK!Ii&1QruT4D`{=-lT^BCrv}@+c*S;T_{)Bb7j-^|w^4ugQl=BV=0NgBK9A$*U8Xbiw2)tl52Xc^z1Vv`v)0cXfGBw9^GbO2Q}-pIklrnK(M$B5U^_mo(a%V8+w7l``ktV;w;<7{g1Jw@uMkBcUOKgei_U?e^8TEzv+&S&8lnE)V5YowjM1+v2g{UQ19dItcx_v?Iu8=NxJ!atm2i)T*NHinFnIp9<$xK~s?^^?-duO<0N%Rkr5tjWu@xxT??*Cj(kK$RQ2vnU(?!B`fDM39FQsM`*Ey!s)Ied|@hcTl?L8DR{S7a!<Z(VnZ1!?`|L|-`;Tfy*%Na_rh#>V%inZYLBd7v5tOTEHF7HqnUgojN`XMJUxyXMIPdgsU^ONb&g0;NX#$qMYPpCD%#~TVtO|C+%dE6QMcKJN8|w2i?_D#^L@7$37k2iA{md}#5Ml5uZ_cfWRyarvA_^(ycUYN85neq3s9)jq?=?dk`Gj6JeK*cM@=}4+$QM~kAYKcBo)tX9Xt89!1%GAdfH@^>JpDyiimTI@h=IQyhiW21nIW$6e$DG2lh{6(nSxogR=(}pUAJ9SA&0z0ox^!Uq*(HH_-!}1bMB3x9=9^bINmEDn%4NyD=tA_g#K<l6qTVnop*Ac;;?EC{Hx|mF75?$4|Rj8z!hGkzP8aSfuHyml)YW%xo*J0|+r{2bK&Th^Iqo@A>>W4iS|(9Ge{a+*c_Yyf9fwNAUJX=aS_Jy-2+GEk!p1Xi8JpDIE-BJ*HZMXv_2|T%x(-+Pi^=ivXiz!%)hxtEUA>kMzj&GfWFL6VnAzB}{S+r=0U?W~aj-4Dl3NG8}j7gIT-MTZ0(x(GWW+fLLB*`nHr7<m%eah>+(nUK1-=@WyN<wMwfhy7h>HChQ<}6amf<6{9Gi4JpvxPT9|6t?N^;+&pvr&FLO?qo!>L%i8;ht{>P$`oS93&`wW7#q`AxHN~%6-JB+fGYe}&i}tXlak=+hHGjEHQ@aGG5i|h`sz)tDF+V-i>3bvUBX|xYB2bQ|iI!|CJ-YF5H#QI$cme;<e5|uISuyyr1Qjno1S-eS<htN!9pYOa^b|{6C-=o*uY&?FIP0(QusWj`qyW~iFg%NTmXUbTpx}S8_cpzeEIGE|e|hUzgwgzSSzRS`7f@BV(bWYl9xi(I)&mS?R`chFhpq@kxY0<OMJO|r*MaU^O36cVcbbv3L-L%+9tt-N1#I9vc(CI*a?XgQN!R<;GR<<6U9M`NTL`JF7tA`bBgDCZdDiMKa%>q#De5Z2?M)Y9u_XJrU^a|7@W~_aR!pg}-702lbzOqBYs)Ck;Pc&GnTw3+JT=&QP*uYTyn@cT+oZIOvDojdrGq=~ZDWl7h+_|im4kg*dY)0QMVH6nAU#*-R-%Sup9hN?+{4kvMi<WPYEd?Y>$L)28?T1fquJUYTb|c$7vmW1%d!=a%Xr&BUu93*hgF1YYLJFEZ=q1}yW}IIGkbI)n2MS+{$~B&q8E&ip!u)Af7@tmdYAu=u@j$SK&x|5R{~EisVxKTeO(Ht?oQE8Kc=v<rI~uw96bH+heaqNtO{}SMNz6!giw(|40KB>N}aWyp6V#MO{_C3Ope|t6Is-UojgRE6ujAq>U8nswz)D=CiwJAJUWrqwkK^e$1Fe?3PdM&#@o^ZY3YWlkOo)}qnoobdQ_j+jA3j0l1f|E#$v;$-Eo}2!0JMEGxM=6->zt3yxWMsTSHT%V=RtleX*EXk6AiZ{<W_&m0xJY+C9nvZOBiECdWKtTVFH~6Gswg>nLQh4!#A+6Z*?afdhP>URm31ROaY4!9kZPwWIOP?47RD{kCmsOQq-ug>a+W(~%ugJ!q8ME>KdSK<7@Bqrji8jjI*Aqf-8c;J@)-ziDvIWy^nD;P!{uM1|h!cZD0ACXwo6I48IJ{?sKptmw3qdlu;6?XoGa?iWyH=0snGVG2lxqo=bM>|x;2<n;8~UOt!20~n@Obg7&it+8{90Izv7yS1Eq(yw^9H`HdYo}!0%sMZMHk%+aTH)eRWt33KPQZ&MXO!U?Bwk>6udYAycL`FkynqW9(T`1qO@Sc*=%qBZ&Z)bYWS8ZLh7G+!HI~Vp*N2fH_',
    b'3$<2WZUOcr`cW-VR6lh(v~wy17Np;-*{DggwU4#u+%Wv%pe7-C)5s)ux5yARD7LBv2(xX=Y~TtJIMvjI%x5rET`gTypY5PVIv#}&=juN5W<V<L|5JqOg5NZ7j63>_4#GXRpU>_Ro=9MHo0b&-%d0Uz1G@-PmbQ2by(9GUP|dr{vZxVXQj~edB0DiKJ5@{5-eH}pU6-|1AH*6PF<7@1ys|}A30e~vx7%`R5zyIa)u#@t@3_NsZ}n6%K55d}?h`>3!soz<!jS_eXaN0F?AN*-m7|{}7xNa=(rf7r83Rw%M<H55=%wVdl5sc`cMq*G(*lj)bb+MYZX}zPsQf@1b9y?r@l`mgt{$JOf^gl$yvUUHQr#j^4z<AOwslBWpIf=ET$j5htQbzTKUJP$!=C2&oe8N}E<$-s<?&R^pc4&t_mnM6Y+-xxCHQbH!5wV22n`}L2)Aot?&lYbRTY#5ux1xl_6axDdF!ENt>%C8`Z)4<x97_fh~p9hN0I2>Q6w<u)#N&ABqCOBKgyeVizr^(pF(J(b$PrMyawvDQfvMIbkYpXgTiLzG^_=o#6qW`Xta9bRAn9b^h7;F6k>zbU03P}dfZ>rS3rm<3N2cVSn+bL(uJc=_U$Mw1H>{l-<sC|=Zw{q)x`3mJi^`?TN^8{;X;kokFVLI=zZvLqqd}{RZlF;GM!r}S{ft_>yobR^Ip|^wbkb=#aWXcQO!)O$Q{F+?wsw?V3KmDc<G`ynrFF;!c1*8QKoXO9NW&$gQ-$wzTwfWV2_=|gTME4ll6EDDdFMHDb#>%z;The4D0jCW($Jj>EW4x_OBXfm#dkwK2cYoD5qH4Z)mck2x{dQ%Hj31Bvk1<Z9URjf1~M(HaN`HfG*;Kp7p-{?YDni3gfnjV=WWj7&W>R-(qX3_gS!A-`&CVXfhtB$hMm2Y|0a#C7gV6lY$PTa++%#P>DuYLt|Z;=Gmh&#t9=jfUBHq0t=_Qbmy;HI0l=5BF1OTtptJ5@(|m2C@U$XvhS5+iHk!_?zYYPB0E*bRdHn$Ty5nOt8E`)+OI2~XS^+As}uuk7jrkaCygHeP3hB$#nP>MP&`J>sofIp7Tc~iqIdg80@<w7&q1}Of(U_CnCOZ^Ge#7R$ZHoU)9|Xye&t#_P}LI@Uqw{d$HwBQ-P+&KALL4!2Tt+x_>J|x=@BvSpRI)b@dV#S2fNB4VaKt4VeFzw&?{E^v*t=W!YA?<L079i=HEsfx+VpEll+?E(9|gotvt7xk_M*swYsL82@D-m<#Wqac{w6pL~%>-Lp#N&A}o#CBK6kLS5fiq6QlTHqTSEXrFgHOUOnfg2KyMW+{}7r95%|*iRMbn=@`3*yOHlO1-79^!x*Lx=#yNDw+MU>h4%&)w+0QZ8q+{m^e!Ed*N{<55WTHi?I5*MV6%hTXVVJ`JLD$mFS`P04X4#QfA`(1K@%vlC!a6YE?8A2y!#}RO87n2**N-bvP#LsYS*hN1*<!^XNukML8Z?(2*A2DH4XSV%u23t$f$%XwWE^ixHVCbnsu69p)G(y5oQ>(jZghne0Xx{WnGBS+v&M%oyfKG#p><7+89eiYihu2JegJ2NUbf}zIS6STRM|oBJ0!_YqK&@O2;eeQ%`@}7q2m9Z%gxdWWq|}qu#OAOV^@QiaW%Pd-We$y3s2H^G!T(xyoA+5XjUI(e8qoT*AGw^0X8&etcq^VUu2!)}#~d@YRHZovE8B3U>aUdgV(SUg^SFW2~vS*5I9nuPXokLdxU1&Un(b$$NV~UP>jo?0N3fP1aSCrw3G$UsFZ+8>$7T2UuucPL1_DsjjZSj+E)Ip;!8X@|WLR<T4~_`F*t&zm<UEo?AXDmKf6E+w!2ZnI@FVe19MzzPqx0GEaGr3-JxVz9ihQApMs5^)_BxOYNH~rZsjD{Yt87^o?Z4eg$!{@2gDpee|bl<c2lHsfuLN>#7}nPfa6TzvxxveomeAN3SLe@yfjYef<32fX}|#vKyi{`VJ)WPB-%M9r)H;+4dgE5WfL&`Ez;8zXOAJD_?h&!8@g=Q+VG{b%%$Wn!%IrPc7Wd@%eqIg3~vm3by(2e?<!6Z^;w<YRtHyQ^mJr#oa&bFF=a>{M_oh@a0Cx2fiO!ZjBK1*CEl3v&{O>TGMR#nJl^wu~YwaCf&X=!Ve(Pee5Xly_j@6c&2N-8?VQd+p<5h(#`gr8E`FCt{QvU3$ovaD=!KYUK+j$&+H4Z$|}?cW}^FM3|3W>=*AiNu^Ye8T7!zia79476?Cs57Ngr=8;Q~;Ifq?EaJvWlG-<8rG?)6cnkAcMMLnzsfO}=i&V~it@~4`UXq1ZvL8cN4jbV5Dec!*=_0gJ1wYSCX*YL{LJ&Q$WVtWb3<wpdKe=5K7v#QG`fp6T0I4NQiudjbHi>XjqB*{)v`oI*Ya*%jYvg=l<wOv<g1L=}^i*9FF@Ibmdhj*<tFCnp$OOtOx^}qGx%p7L_iPYTA%8$I{pT)*q_d^CR_lt6HKW7%=Rj9Y~lc~3Z(#DoZuSK-=V4IG&mX$QrB=Fbi0G0HlHbOYtXJiWSTVBt(*h7E%$6x>cC*0bfe|z7Z=)=^7N?9YIaPYto$X;LPb7S5twH6kfY8Rn3c39j*6ql{dGPS>ldP}Rt=JH;w-Wun7_+==Y$l?mMzcR%z+yG@xOWdAEZiYsu5a!_WV5hpw?Nn^IfBwhY+kdLcxMoK}jK02r8-L8KIh;t1YDZ~>zAV3RcSh~)M!eL&{p+uP`^*3E-~8`?6*G3_EnQbp`LWB=*>OC0cc1+RUB#~z_oMuEy_Gvu4lmY8m-n?;0*&l-=d4}M{<D847I>h6Tf6IxKfW)ihlW}j*imDv|0{<0zB1P+U!j?O^o&J5;~hOlAW<$lB?i?-4<3<s3*^%GF1C6SbEveo=Xsh3`5`HHEKs~hT6xTAOP9jv-Qo$4-KDBeBw=9NSyL~=>3-fk$WIcD${#;rfB%=to_{#~siyH85uYSkM%*-e8zM)!Zd#AK2|w-D%;i@d95&-;J%njzi<<;FQE05$!Jth7l1M1NO|$Pq%lT}l-g?;S%)yJpgO&?9Q=?|BSHmP-va4*p70)BrQJA7UY;)?^yffY)TdwRUPdBMyPCv*6Rp-E}Q?`$7-+c5{HMp3d&B{XkUF2<!Zov%Hba2Yhr`2PtCpvGI-oR)g8_L+BPCBdHvD=OAK@LhOG0s!((0gj=bel+x^6bt$w0FzO=CqD<QLbvQWZu{wfAONxe77>as=sw>qS@F1<!dr_!ZWD%tq>mXjR{q6Ypu%#;bjw0sJCFKtZEHAA;vBjD+t+r?6H2}DslU%QGX!X-I!8HK1CawwL6B4HE)X~HIO*d?`>?~#s0i~EJGlpe>Z548k%#f&6y%eY+H9d6~@5HMgLYem2qV&D}PMMFg}@cE=W#E^8WGqhOh<hlJff#&Dhp>;>tNXNs`S8={r-C55{`c4<|&965w;zO3wnfr#mf`g|`O;2RwS`daLkzX!8FN{`;7qqeXON^flWpI=gyKPhqvtaabyk;<dJfo10@Yi+jvACjS<AX4B(DWny|X#cLe#<qv~TeC~&#jkaRV7%FTYQiHe5*f;L>W>S>)q_SrF))xU$mea8EKBJB6(J@wI7dCNXUExSq7t9SdHppR@JiU(MDCfYsPCir1wH@qS9hW(u&utvdHN?20zmv!bfFb#WxLjy`sXzQ$ni-!>0!X#-y=dImnsy`O5kKmsSZ!DaHKZn><8(un+!u9Sm+&ij^0UV!Vw^g#+Zt9BmiO`cr@sQ_i@UnbTDhqHMWemT6Zg{}Khz;{!=?0;-iUsF=+OfJY+>Nx2@)iOr4MwkIiWtea^)B~KA<C7Wv#8YGkCKT7GsKAxK1^;&O}{PF=01hkdKx{gcyOT`k2>|hWicm-@pC+PyhW}@oi&c{{8K5zx|{5?gtsWg5|7h$?UN%)8kv+IIWnANcrIv)$+`?qaU+jt(K=T4>grpwK(ya3+S69FUC`@e)6uCPL1k}nP7zoLz|9bYm^b)8Afc+6hiT|{KDuxyJ>#1`Gs5B8{Q3iyl5B;_5g%Y;zGCXCbbBKsG&N|)7KqgNrPAuQ87iuP-Dl`*H=)bNO*$$u7>;uR2jNE4&rnPL0v0Vd%?q}t>Nl8?4*0T`zN>bTkOu6-XeXu$y+Pa77%O^@IIqNUkQE|;',
    b'X%W!Z)|9&sr|z&b(B4H4W*c<oLfT?OkFS)-i%V|TNSo+7JCydU?9X>-8NwyRgwG)RY5GE)$B&DG`}}B*}R)P<62)+cftOvifKJ;5PpVibAc&)x{tm^>6N=4HOx(G@wfCaS=DRu`S<?1v661+>&qhguL{>UmtI;beU-XinYXEZ%VFM>r*#{|7r%NQs|YhW%sTHqq^yfS0O&3;J0FP`c^nD%7&Gv>k*~DLsVPZ{A(Nq=j!h2k=Q`sO=L&UPL^?cU25N3#Fz&c~<DJk<Sz56I`~AJ{9_S-FsNk3Ktkj^YrUMZkn$f}jTVYfPP5oAD8HJ2?d8rNC?7lD-Rh)6B78l(%sLBj+v9+~Zd5QpAk8Vv3?yovz+kfplh?|(Ci{<c`u%p8oR(gsO-uE`rzWcH<O`E!>)?En$XDHSgPi-g>mh)_D6S`&MPBv!QT}>;sXN~sOhljR2*Yj?`(%K2n8nS30Idie;!4&$b95kAdcW-^hi`-pPmyU>7>0Y$cL_HvQC#R6ccYrRyV#W0zDa_t(Q1V;I`Fl{c$Ap&4YL(epZOa&jyFVj6&mkXP_Jyg(Yb(EYX)=Q+#4Er1b7hEKm8g10Q;UYhdnsO49F=OAPczEcB8+XBuG}~lKff|`6PxmD>0(!yGOR75(ocqG=A&n)sGhe`x{ggFHq><D@`Eji%HTT)C;IE`MB33gO!d2m?b^LupHiGvdQyUAjK8@7Qf*a&v+-+t&Le!V52}Y+DDbIsPSBduym|>AskHJ_b!@^KO0)|qF}JbZyX`fi4`9QPfwkHkj4T!KGEP`~D%j_vZ6(7)_(y@BTE8`l?4cVfO++8+T>~vDT|3suw)4_k=4hM9m)j3W;qB9+gQ@L^r~-XV++{UYmYlJku5G73pJJ2fLEfwox?L^do#%jMHN^|3JYKv3NB-Tj)$9XeE)M2ldXAuU98!-h(I6nPJ~lJZ+};>}jXvfQd%_`rAerP+o<L8uT_Os+ti8aCN!RBEMym&<Yd+Fs6>@->&)2KLm}-YW+Z>}QTW_OmZl*l+E0q~NHZEg>KRMFE9{X+{PZ~HBT`3GZXROw>n@^6FRTVzD+ch3jS6oY-9(^2hQCUfPo1q%%uY4(skk*6Y$eaU}mnW9Ql9$;9rwPV79WKE{ud)QolZe%IZ8RZ<fXZj;ZnPr|){?TINZSMZokEUCu_Jii(D-ZD?EBkM!|qFUqp^~nP0mstmg0KGO^a#|<KGC%D7(M5oXP`EAEvtJ>upyB^3tc^7O+s0R_WEojH}Xt5z~ZdLiF89hhA4@#bD=<deesc;SYiw`8DC<KEfIO++42r`|8`NRoKaln@@YbkS2<Hpt9UUP(7=RrQKPWG2Rm{MYwK3^UL9A8D|_Q&eMb5tP-O1R`#MrQg7vNJ^XorURDD-xO*?9*7#YI`D&ovXdF#;T3OUx;-V*$Y1{HU&QejpTxEbpIU6a_!Wiy(Mq>+g<11E2WIPxe4T3J7Q4{li0Bvz5wM=ZM6Lo1LNpDXtV=BYplcdeBq!r?teQ%5Vw3xmc=go|PmB4^Q%jAr6z_Lbt;sa+^S6p)!(x+r>;5k)zVV|A2^KxZgU%nql2sd)XY*8qV<xwruMb@d#o=9weAh*w=+)J?$W;N&ZhjE(9RH^^O8CQZfHelD>bI+Kel2mP&T=>lGkoh~yPe~e;Z#DF5b*@&co6bET8_j77g|oGwTN)S?in=joocbO!lSIWt4r4&HGq5q9UH)hZ33O5MG*s4tfK@s_w0i;*Y|7fvdr+eKvX#%o1}@8T@DhCiF){2)L+!qZVC$AEeA17#c`7?_j!>@6UYHLPj@j02>o$tc*7mUgYhkSXp_=XoNGhJ3Zv|eQ$6#v#e}P{gV$ukrC}l_6SlOKsYJqw&%=JY!YWU`JC*OR~1zFa>5O|x=MAx*PNq1*|<vt>rsFBf^(<IUL{uby8Gx3EwQCAEwdSF5-FkD~;jo#J@lbcMF{r2_0V&HE%X%H>7RF%BEhu9tD0yga(qO@UoOWN6@H#F;FZJJ<q-cfQWc*Gc15x%AH72gDHLR0QOP|x;JNFw-n6xain5|s(FT2U^BNVx0BTmI%r)S{lVm62kNh0Zajg>o-AvTt|L+n_)T1V{d^u-iC%geQJ@b9!Xn+ge($4aUJkSd_ruDfgNS)tu|<XUJNb5_=PvQK|{%vTKICZz{WeUfR41tBcs1!8N;Cg69w43wzP7Q@z=iT&lmbj=<t2Doss6fq~EJ5z@#sgKhhf?^Mr6?{5*&N<-yqmjc}C){#pvAD$R;m_MBFto>yHOFG92;#Px=nS)mgLyt->TD#bb=x&+9TFp&@nXyCgGeTi(pL#`7-4mK)eoY-F{Tuz%%jsLnCe%5Q-6-@PX`Xd?!Gbz!<+n-^3<ulM+CpQRa4E;<867?wgNn`unzVy+<T=bky&}qPtHIRaohM4cl#Y3KpF!e$^14CxAISHw@UqYT^pC&({ZDxMfB)^bf4<#Bm>`_>UT3f^C2O|7QOA8g)i*3e=l<?)hx9v}6RkG1OK#9Y+*-@-^qa0)A@Ve~{J7CdVi69Ov}|4P@ZbLR*T4PcfByWNt*Bx%9aPIJ%De1)H5UVau1)RCRZ~Q6LFay;(}@dFSq<@Rz^u0Fb!xl)YA4(0<etZgmI1V{REEF+TIU)#mk(NNocN+gGnhkLge{B?#ps>98)eqd&8b<!7qlo|Z)?*~b8~uA={D59+%^-^!f{snTI;o}bhS0nf+_yg0ZL-+OF``a(R<YblqsZPWn~Uh_t^dEN4YQw%vA((({-CFkfe$7+<a8uh(lxFeR)$AO^)K$%~z#;!2Bg;I&Jn+TW^O3BMbzE6!os9;`Ytq82bZO%y3gVTQTk;92k*ZQO*wK829ZH)RmgrTg@$132&vxS|OFgpOS}sKD+sJZn3m@W@8E1Pu<rRUp}njB6r7z_welLCumZ>H+5Uqw_vX-R_*L#FIk0ZIx^nxjR`3XsG+6o45M4Nb|Rhvp+#iD%Fu7O@jS2}cXlgJ$@fah*{?)~_hP6usX~l>e^-grz@VIPcaOVQs&<Fboo9<twB8g0y6EcF#`fJ4->O4|W}cPAr}DJBci2mAT0WO@9{P0LC9rC5G)s`6cv~Trv0ig2vhoGm9Ws|EyH<XovC?bLaPm_Xg(|I5ALv(lEeq<qH)ZMEG>srPebLx!5>`<u0*eor<%iHPuhga#N~=ktnRdiSgj+!4k5y|5xr86Flqd1LyjCd<jtf+Gpgk8`MC)t4K-C=Dm$YxQUfHQPf6Y;Q^=9dt4VC(p<MmU%bsTsPtGCs%Wc3M7oh%tWGbLd@4RiG92^1d<0>>ULtrgokbs#KYwminWg0y!3Tw7<dqTlvb??ZVva4a7;(<o}arjWP#1frt}Xdpf{qSdfWibwwu*ZJM~^`_L}=J;ABFf}Q=sN_>a2X9;Y)FI^7QL~%%_KB~k-j%IT%30qOiB&3^DzF*o?WO7VhJWvFDX-oLRltlo)rNAKZmXy$hdzr`BRnPPiSjv}E4hhmJ5DfR&am`SXEbzd*~6n{EV!n7Wh@D{p7!0RnO%l^4828Si2Plq?HV_kH`3Eq9<*5w8m8Q=dY{b>)X<`0?IHCYzy15aKht%``@hVYAI&EkT7~NMHu`(Fmj`RwaW{JduW#1!sC}r_=s2-dWA+_nLFzMmCa?&$pQp_?i!bIqT8H<2R=?Wgdt`{+2%pb)Mm<xm*Eg;tpgT;=XuqGD$mwnfKmW__T-$H9;R!RmEsqiIVX-<)m6XTi<mk-YBckPcxN0JQHd$cpue@MsAG#VM(cncQ+fk+T<C+5%b)=Gs)vRFOpAzeVH<5ayjbgZ=rn!1|S*{MkXKxt&=FGc$AJ3Se_bMH2?3qEF^Eu*s<$jMB4TCg7eGODTA9qukLrn$LkzzX?t1yl?bs-o8CQ3uW{lI;I`l&{x?({>K%Q{B!(Lp(A{@ila`{I6deoh?#_lK+6vcafSuEnLIjWE{7_V3@v^r)U<jeNLq5Z+ui4f&TD@tdLdct>%7b4MRMskfW#Y|+N}>k(6a>3P_i(s~FJjC^$WO}C%ybH}+s{dQh%%WzbAvi78>4Ok0_$wbG^O0uAvUWbl%^m{{uu-orVwxfrp`oqzp(wQ99o@~2Gi}cwz4|m9kz8kWI&hUUblkMdfPZx!Dl*Jm^c&?a$MyogkoA*w2b@',
    b'B)@r4$eMQ~ULjvP(MYeCD1jVQNdh4XSMIj;+~i^EXsbFgxLP?_OU?Q8BwhnMHZaZVvq(3Ej`GLdNj|bzIBWCaj&hhZx8`TDZcwva9Mia41SglE049klks}+QVS?Z}zq%n8b2rmeIn@5jZB75h3s-En?BNqg=ent)WWgO!e7aI31Fx=EL{7axD-}F13)ws}?`Fx#Sj85JoYay$SY3l4=EEj~~~xw+o+UgS5f3W{Ck*e^*37s=K4n+ec+@YWd#7o3e+r#UWoYsa)JD(&Zzcsik*^mR5ioNU7QsW+LGA(_MCo(`}XJEtjOoiFy0pqeo_wpVJ}<)%#523Wq96th%$Ow0c;{wUw4#m!ig%?y-7Js+3$ec4?*lR$z%WmD)L4JLyo6GSMO$h)VmaaX}~2?eN~#7HR7nG8Im%15HhZ1l1G8gz4|-gUmK0x6fdx9PAg<vE44GStHrn_D-$Q(#B8AcpfNkTB2sv7dV*X9TmTv{gai|NnVe2R>y;TtWnoSus5qB!B_jq+}1-EPL$#gYjIyiaFpKKbybGd6%{c$ni?o<!tF{{p-gv#D*WS>@clus{v(W~F<%TtDI=r$V@9WZ`+9Db-|k}JO364<`g?1fb9=^?^;#R;Nv-M>?=AYLYq*5Qv2#mtMit!cO$m(l)(E?LEwhYIB-Frg188=)f6Y@wjq<dvdhL7BDLRX&yumaIS+TA_uL0z@FU)eyfqhp_RcqQ4ZAgLDr5x#j?5qcRJ0#u~WgVm%WFKRjxRNpkrWz|MA?ok;`Nk}(xl7!KN(-*Z`;qO4AWnIhRYsI*ZaAo{Mvcv7mZXi!gi=;HG#b@Mm$IY%;%NZr*Y?a}KVy2=NbRlas5NY30u8szCT0Mq@2OTqiNYe~%$I_;8ldC-LA;_Bet82n_d_N4n01(PTtS(RjlF2Iw_T$Ik9%w7_qd;q57icj@o@Y5mzJ+h-N?ps{NPQ}ntkUK9)I;3=xZgg8W-)O4flHMoXuBR;tcj{H;KP$NK{8V8hO)f@jeLUsqdXf1Eq>3+GhcqJD9p<Y{x5+tIb@s)T^v5c96<eP6(O~i{GGT<|biMDZiy?`n|2PSc^2y$zCJ3bdeHg1|xzG^wPhm*rP?Tq-SjY>DtCV+t^FoQ@b}x7fTl&!<yWj#m8e*AVb;A0eN5-KXN)<j8pQv-BwzoR(y&CVw*mtEtuv&<H1R(W+*JEj8OCFu~4oG3U<<h_0SzPYde_)8AfTaQJ~e+IW^`geMhJdq&E!AnQzqXk<c5#LN#s!9J3~)?*CX5MXyCa)lxQS5CN$6GBl-UHX*&+|5V(A*-~f<3$!gHWtu<Mz~Nv8F)5N-VZP7JD7&+k7Fdin)hPm9j<vQE-6tJ>Zy-N%rr!<7E;N4Nisd$tx%r(a-gZ!)TFK}Ff<(?5ayosBjNj$S*5hlnCaZc@KgkRqS%Te=4WN{Jsekv%JVIpZu}Zx#d=(Z|qSz`N6q$vH_4u?7Q+yJOyC9RmQH#>LNzR+ub%S=3^%{`3L46Q8Hak+cb?|7a&%a*gsY(dCspg<$;5y`g_t(NoSoaU}z(J~P;Z_`!Q@CreQ-iB>*{PZ;Lsmb7Ij!0<#1oN)>792`-@HW@(Bha4>>~Pi%_E$aVJa`8eb}@BF)V(_viq_zO9{<OKGJ#NlDcU4%DdasrsD>&PVV~QpzWaD1IXBSVKKb5b_rNCfbN0bQ~atXdfrHE*A8g!C^F^4JCELv_Ip~*43+*M?eMnLa*ONuZv|?N_v$afQ3|IpujA)-giz8AqxYcpqo!_-Sf{eRqKp%33#6fNnmX)p27g+M_m3Z)JGXGT_#?L$IsZG7k2c2lga8(m$BFX`Rp<F#GN(S2z?u1(+=Xp)vfJ<B@`O!;MA3=cD8-Wk)C4<<OZDpz{MxyN?CAyLrkt~m1*v^y=BLJMr=iY>1vZ4c@0V&4Cr~94x@qhx4#O25XiH9;X4C4;Mdo-+um$FB_m1l9raQn}AJ-ipXf9F{7+a9F)<4uby``++X0h*~SJm7M0%kI1baC6ip@JB8QYG4d%i=3&8wNAE5}V>p-vEIHPgor~O|;teWKC%)ZEmTk%R4@47(FP`U0W;Or!C2AbEd%QZLh`R8}6&29z=|;ZR<6A5V+l`=eDR==ZQc`_WpIA+HW>CN$Kq+69DW5U#u-s6&rDAY#u+`m>p9-f3e3FWxmOVxBHawJNqcohn0E|#~0As8;vdI-#vL_LPHc$ak&NK)Ejl-6-`6D_<LGIfl1R5KV}ZUw+u&Ev*St&aA+jHUCq=zK6$Oj5An&B_uJiJXZPh8Z}|PKeWWmU-?=G=Hk<3+JGva){nB6$)s(Wq%DAabK?GVxlug&UEZkbDHi}=g<(eHwYOAqwyA3m43dO;L9k^Dd#UN!3JchY6{NC+;@q)+p=I=d(rByw}J8Xgq-uUr{6PkLwXHlxOp@KHVv>|c<y*RBk&y!Kn3N1zxctD@{Q5=O@a1?9gOrr|28&a%~#@T_WF^eeuXiL?`njNd33tQV}Jilr6#9S{@`h~--ShXCui%91^`c8Ebm<9@_GWzw-cr@S}Rxa!`n^kv8^Zh#gv0g;>-nR4PK?cXxZE(FSEoUq@!{w@m=CXYz_zZ>fdfpPYLi{wZEcswtcst*=X@wxg@-wQu){u?WL*u>vAOWz}&e9fwZZr~$wicrX;g4)ug}@kps74i2U6giheW;vSt&*(yGv$#Uh*NsttrwchW27l$Wfel3&0-wNJUGvwL$$jTL+D)35#5{*Y2=~y2FW_OpzRpb(iUloYCS}#{oYWUZ<qr&9iVnv4f7H!PB7RYXSGLI5Y3xTe%?Q;bMXV0|J8R-(E3dvfHK_@coku^#-7qOhw%-Ud~;I&{ZW<KW_k6%Sk>dO%N_UE@a23vE@tSAo#YD6rk=>F33IS3=0^VR4su#5?J#R(0BU*wlTn(8>=7}yxuc28h?MqcQzEXF3Yxm8px@sdqMi48uZBY!ZVlo$vO`V1909t8U2Zic2@D`Z)mfyrkYlm!*KR$|?CUg%c20R|r3rl%Iqmbk#5)ZS@ZJoz*?bljwU)yM(>q1gkr3AQ_x@k|%2N47WtgM)S!~=VG$Cb6eHmZ4qwSI7y>&FR#_8Up<6!+|DZ8WT8Q4HeUrEhqL2?^c`Y&QEvz1tzbJ3{I!jhr^NzC5jc4!(zEgIXJ>QNHYXJF{&X|qBW>TO<cPlsg!_z(TpZ<@op?Dvlg+|~}8vejGst_G~n1|51TPqgc<b8~5?l5ygs=W^S7B_ou+_19P*cv04@HTu-_5WyC#hjCQJUV-!=u%;fsT{RC`Ini6g#>%=gn;4(=P(~J@xBT7vkRBNqO@=UJyS<QUH;I!|2U6*%l@gM^m9Bd*%#$4P0zu_hr;ud$1;w=5x8e`g-pBNuufCX{I!*CWsnHS!`w?wRWYpPgAv6_G3$jqFuvd4NQp<pW1E3gDzODjBJ-tf~r*ulbb<1G|wcW&O614&HccY!D5$GBlrNN5ol-Z5#6J6HjR37Gss5stIo}hu+TS4sfP1?+CxUV5s^X3LnK1{Cjj01nmIWs;ZI<z&$+55(^nS0+>d!hkbx#P}Cb5;a}YDtx;trx-VqOR<%l7GO1xl4}{Pn6xFW#?lQoeVd5RD2>LGU?ZL+UY&kn4@~usE?WZciYLV%p3d28qJK{Y5ate9&*N-MnUO`OEw9ocdZT{IE{pThG>*CD{z3@J!7Z+<p+Y8T5HiLS~&6KLQTTshMSBG+*-#x_2!l9%B9xA1NW456~w%)Ev`J6O<V9xXwuym8ef~`W6cAwYlnwnZ+EoSN@>e{rLAyf1KFCGb*pzNn_LwS9A6*Bk+8R5WtaJ{_OA@Dl+WJOB+!y<<gW4K4p(kMRgK7&<sf#QFm4xF`u!WgV(v{Vj=^s%OGOXWTEofc$sK2PV9z5u3{fx*R>ztDdTHMGjLlEQ$E?Pp_XuqC?xKa&xAGi#Z0c$S+o?jTMPvu&GbJ0I+Tj(iq$)H0twpCcN))SotBHuDNu8?!KvAe@&7<6#AxT_4Cbo=1#zmycDF6-nxoeS&w*T~GBz^i&JybKDZ4VChR|KLm2RZZvwuyk>Zkl_4d+e~O@eUerl9C#hXF&C$PDK{GH>B{tPV#AyF+T)ySsXqpp?tWtGBU*5*)`?TU%fSt1v<xw+B=Q{S;~<;27X!;>(1dNzFE',
    b'!52@}OKl|xPXR`rDGJkdjld>PQaL%LTNv2qq|_4}>N7MHyF;u;77O^C$p+lnrDFY?=`4O_tC%@W2imB0eb2G)DBL8u^(V7Dwq->*I87(Y3CKUXRm#?q*c4@NR7%OOVQ0<F(0-l<{ajUJwNYyYZ?2dr!a3r!y0bb?Nmw-wI-OqHI`wx9~&_wf@p<+kkbk3aq6uYdm&KK|$5-c`8p4l^k}_3yv^_RqJ_7P_8fw~y5Hij^_y`xCa$=@f@N6?*RvgMHp)tfBtj{`J?t{pEl7_x<<3K1{XyPJ?b7H8EHz+^ApF=T+@<n!TaNFoVwa1ZNG0+Aq{+zC`+}Db&kyYEnE3)4&A>*&f(2xuFfN?1TB|f4sf@rxByC?M%D4@zfhGqlrZ(rW(xa*L{2VruS=ndvi^Z-R@m?pDRWX8h=Q?1AHTNfBV~S|B%S2c2pHtshtsbqzA<Z%wM85q=v%n$FHMWAMRK+_ZaHLC?iUlm0dSb*9=xjxqeqo$-kS-{E5_P`soO+&#*n-pMU#!0Hu6rw#9Ub)!#iyR63?OqFbxst|R$wU*w`Nm7iH{I+4Sm_#RR3eUESr`mgrvd%BO8en+|4(3EToRtM@vb!yFvKfSrNu;fV$x36m~Xq(rFub)Y(uLWQ5|5V67o6XiQom<QB+uEATR@IVc>q9Z-Tt~Kwg?L*j4N#UUp13Tw)RF1xV$&X~em32Cw#Mz(H|SAL{_GycB7C&e9CyFw^PSnGVCLF13XcsqL7Z&UEML_g<T0}uqOp!1b&(6kU7_i9f}5^aEVe{(clYb#K4xQE<(GOKb?vdQPqELhY@3sxCE#HKvBoY8ls2NWD#a9nZUb|+rQ0qZtKi*D#7H%(8>oy3#{BEvuZH(r@Bx*ha&S&HEGcS<qc#{UPyp2CRs#Is0jH+5_qO=Cwh`&tYExQ;dcB`D24Alcd<A)G0A}$<!&=Ggdijn#p?lNUsS3z#q^C_Y^%3z*lZW`3B(z_m0~7qIX%)Zdv;N+~R%|J>nZ)?b444nIE?CRKVur5gU?Cs!Eb*GI=+x8q?A~UXeJfyuNr$F+1P#0h!Z0<A_F7(XaHZT*N14Bz*0VgoS>!pkc1juXZ6*Wv@vRr$*iP+EN{7gU%nsEJvhZ*O69=(mS?wkiNBdGG7TFnRv~lX%3MzN#Cw8MXzgW!KrBJiODm%4d>qU7w(p9W1>w4So!%PKc)Uw1?t~XVaw=xREQ=l%U9S%PWj+L{MI$cnjyK%nMdz>8F{c1ciaC&Y0Ksyz+^qfcg{K`D1DiWv<Ks-;y)JjdlM0qioBG%8`v7=m3#?U6Tk)kJPxmeBEHlbWl5$6%+%7%t(1A{ToOS`7`PEBB~E2@z<%v+H$4c%fZ*&#hdvaIqu#)fnr*-{?;V61^xvr)qvZl}YsP#48%YRGN-+HCAGsV)F(sa7rxQe3P}BScO$#+zaYyQp3*GUa4zwL{wdh$f|Kx71ag#l+&RS8_cg6)LB7FUippG$!>aC%;Xg#^C02zvaqH*?af<cX54AYfLpx*=E?w(nNeW=FQp%QVk|Hjf_D*p+pHYzL0#aOl2&J952u`+;7=S?e$6YPC)94k{$)N*_8ef`Y4D|TKG=w?<?EVs<0_Er)Iz3<0QQib+P-1*wPMMXCG<Wl(uks`NtIB;v7yI&q?L7-|=AFflpJL@`dI?nhqppYv{)vHK1kMdv#N+N)5{1BVLYmgP%o`mk4!Fi3aFud3$qOM|WSNp?n@2ZHpfwl+?RE!S4+&wvIz9xvqh>xB()3AhNi@io!!1o)sTxFqlN%jcLW$n=NBcydP0#j|}zNHju7a=$7I)tYcftP;2*E4Yz`-q+usd_0aKo1|9Fi0GC-@u2G08?$0{mKLRb-?cuguyoqr|K3bQbw~maqI9?hoExg9_Ve%_2_G!bGQ`ZibMYZF3WYKE1N}Z0ehAdjNQQcd%DJ)HBsO;zV8oP7l<W(n*&y9NU@wxx-?4DN&^>qaH`I7dgqf*7a#Mw3WtClLKlF)dq8G6ocaL!ooA<i4tTzj&iN7YL4rKUJ%V|?;vvi7kwSk(itMx#x=nQKM1Dv2H(2|ZU-F9v<?>P|)?Fubl6PEEo?O`OG6WW#H1**E@t#^?ckHmWzonq)@9(N2}19hWm1dnVP*(KfpV{$9tw-rvHqnTk(S_l=u{+b2d1Vc%cY)}kNNi8uRn9D?dy#@n3PmYdT&wrbh8S&BAD=$o<nvFYaDjKOB5j&9W~AM8hJZ%uQKXTjQ;->lNxxbjevVQLV>)r76)mZtpKf;!5B#=EEg{>UtKEd*Co`;1bCZGt$UW$JtOHW=7EZCxEL>M&Mq+Vg3+K!XC0^&-uKn<sSlEHoyl+GsgNcWJzL==EBKu(<_fGBal-$5B%ZSGcFQ=w-mAU33(co)D|?p|$hQfKT<_-(rRJHLdA&)o8cq18`6b3$~63*8`fBi^OV5r1o~y3Qg@Z_!gFG6*MeD&<2>?KNSQ4<zRYR0%es_X0y;maS)#>s7Ca3(oAp{CY|(-{P%1C-SmmI&NX#xN3E?XZcJ<5wrjq|=Bhn(nUtNQ<6;FS?#$NJ**cYehZ4)Zo02^@`IqVgm(FDOR~jOO--uEP4o0BfAKTlW?!n=KIAv*5)H4wX7Y!;$*GhcolZLo3_40e`KwgEO(CzKBp$`KF{|IL~+%;5VWNl#2f|}GbKY4>A+Ng&11|suzU(w!JriS5OBwLZTz`i<~9izh?R!@xXY1HgKM)&jtkj*HrD2G*}$F8E!@n{R4vP@c}Ejzd@DxFKYr9JYJ7&mn1sqNgofvDU#4^51O6-jp8TaoE;cPr=%E6|)8sYp64yatVxes)xH0j(9ZV;g+FSo@q?=~Rj%oa*+C%y52CMK;YrShef=h*@WSyCvORS|Cp2=-JSPW#GUXR2z8DS~I0_)^SF*Evxmm{+3L(#mO$-)2IwgDc`bS@B5UzQ_(UCmzC*QgNTZL5N>$idJ)gAhr8KVgp#8ZCN{i?GPP4rSl*2lZXR$W3*0%Q)fx!vRo;6A(F<m`cjoTOn3|Y)G?}Q_9eR`W*8Sd$h1EbF2Xi+yek(b@*(6~Rv{;>ik?u*gPG0M=*xt?3H8NN0)Tniqx=p=RZY#GgT>;Fh=ggs@o%HtM0zEYz&zD;Dss+v_&Wl)PS81VkuwygMcxOtZRq5l^PmMi(uD{co3aqL{la69>*m64CjD)kqP}D5hc!SN}_$|WqW3F6#XgHb%Q32wOf!kMzVPd74AkSO$?gy)VurKX!5RU+gV;#!sQw6~(r~*~)d&)SAnvxnc9C~^Zg^xH8K=~cIP?JH~s>nayLN@jDA}cjadr7{B)p9M6JdBm9NpKV0rhA{U7*V~-?&R2~UE{uHO^so&g<qvo;jPbCI-<6b@MCm*L~q9oL|P0`@|P!&LMaRh<?W~s7wOfH@TUe=*chEVh<b#Uy<SaWLroUO_uOSS_Z4Mrywx1R4WkiXIRa^|M1Ao69)_$`x304X>W;F|@#5D9DgX|yVM=wnsjI=@&<@B1bJze>X4Vu-M`H@H+C)_C67}Wk>YFvP=+E?RV!?PNkn;pDXA2#BRT!(Sz8K$m0!(RD_<}3sa$E5pR;vf_k)l&eM`F1mU9i~L?)E9rXDS#2ilE*YLU=Yz{ZOa_sN}}_7+mwQPP3L7=yj>%n{_UuI2czF4PT0bMJoL;l9}rC%nd{>_FT9!-EA?r6+69~V*32Zo^<1YNt@F|F=8bg`>FmmaJJf9@a9!ZP4vl=S}uFF;)<ze>yPq7-N6(-FtCUk7mFQ5`;J6jE@+SOhi77SrlA@Su%6YZm7LnVvN_)v>q`yvx|`D3g_><~lgi?%f~{`WoGOnIVOdb~QrEY*GZcnsO!UYT#YD|wqN()Y-uxxQ>mlCL0VsF;OVgw;<`=cqQ!c*91|Ie9^n2zPO{Jnm%EUw0!<-!!q8i%v*Rau4X7_ut`&k~l0QNHNOsnUzeC6l@K3yug&ckZ$%|r^>7ZVA6WT>=W2PkXK_LDX8;VN;`LdxFiszm`!wHU!lBr#Wftu{~+>20sD?ie5K=r0JLI9f^}AqrnhdBZ76MR*_s4dXt*tn~24l~wC~eUDMa^n#pDTf1Tzr_{Jxvat$w@7-Vjs+Mw;3+{4NLnY3vcSu~EO%$N3TwN!Iv9v&(^%_46g;u87N^0uMmC(M?Y3z5(H|V;',
    b'zGuj((&z=MvYCIP`iiGwQz^PvPP`*F*_e_|KkX{>BshVD=>S-D$tbIS;%%gE$UH(@uLIy=P{C3b3IyN+m2u3D7gAY{8pg-Zwf-*ghO*`$YxpI2>M4>B9n056vYU)T0k#{!cu-cfm7Ef(3u$-0D$3m+@7be?YD9uyO*ZMs>b<`kRV~`%O0YLDww%W9p=Jbp&lwaN=sun+EzXkH@pge*RO_$1`dS8Iu6+I=6-gJXUnj}y5DbOHdEJycuuDHb%6|}WWf4EKv7krKzV_c@rqYB>6F`?3eXEgwrg*$u6S?P1@gE^}&s8Ob!H1z!_S2m?G<?*5PZ{~Tt1CM*cSHZ(k4SO%C6C&TaLQ^(_cng%bmgGIBCg6vD6tGA2XzGwYs2P(3-#PXxFp9u~d$Pi(7^mZVfNU_Tjiu$8=11S#)s-tF#D)}>txg)|c&`D4!7_1FExLcsw6d7XgMxp^=^_PhFFqS<Psh&-HRGuD%2X|=p5BVlXfyr$bkXfAQjoxC23&X;CE8PWzGC($1e9Ys@d+J}{!Ct4C_4?i#KhD0m|IlR@TRjnwAxbiaUJVf0t96vq0_DrPc@UClyyIlH596+W%?6nVmaR7IKr@II*LdqP|>9F@qS!7?VIaLdm7K?P^Z=M&iP5}(wyvbT6OClsl%^CkWe#)Dhy#e9<6g#v88o#+QTOe$bHpZ?Poi+tF=;HEuYWJH%tN<{h4x{DfH$>T(rIXNI@QP`?I#odnmSTU-LQ#*h)w5poK*T@pj+Ou>UKQm1IA$hopX5_U2nc4cCJO(i`6W@zxP)JiSok_fre3{~kHPy}w16R@FD$)biVUmivnDpIhEuO{_@|OHMsxcRG1q<GN}8xN=oVnq(60cehTLkb^m#($sdZ5i9N@?DKoQHM*7J#9b*>#fmmvjYrjt#bj%?YfaA;&kprXUoiDD0iB}f^jM;)Exr;_u-kmX4E4@PU-eNm0U{*_ehUz_+v9(Jq8EjPnqUT#W=AbKnnwWZ$vocuh&h_uBig^}pz$ag8Opw$DtOYME+1htw#gUm;=;cFBanGL&pcfkzwUYL{>_`s8h;!;+A8_lHx^plU)#|AYY{YP(aPx$bV(ZD)y?MdlfAf>VvQ_cj(4NpZB-ulD(Z?21h&00vE|VB^}+xBx8MHxrudckz{36=&R*kMoR{h1p#SubzyAGCqCD{1yWUx7&GWatwHCRskF9Jv=%4u_HJD-lE+crc5(<?2`saVVz5Qo&C?#-oEXFAR{`R-u{!tZ8y#Cac*wu2z5NM&4KmP4sfBoBE{^!rXeR%h{a8VidKHgtR|L~cgG%^$257H;vXZl+px|;3!c-uqqvGc$GwXc@iGuvTUz(0Ua(PuO>oA<XW_pl|T@E#dRs_i({WcxE&QKbuZ(7bz8FL4b)<+xP_tvi>oEtSs$jh;B?>f<O_f#N+%2Ya`*E#DGQ{Vc<w>rE<q@H=`=Z|hw>>{oltTj%lwr&&A0KIA@iHm$d+le8D^dlhUj<+*BXz^zlP`vu)IrdB?((xz6bv2@8%`Wv`~nBwIlVMwTE?Us&H8yt2IZISV-7?@JEs61`?8hdqy%12Yh7%KX1E<0JtHDJ83khV>~=3#LMK{;kGr2C~vel35gM*n0-|70XO&5A{J_Eh^+TSSG(!LCLn*ObdeFn0e=%#nlYLbR~@HgRng`-<fzVa4WZysl-)u>+56=$p?*-h2^}?>^rI<Ti0*1_^Z|JjPWIs}>TkXyoqrsvsI&Iu6RH<i~FY>JZldD<5k29a~4&Tnf7<_u{q5Tdn0IZ7Vr2Y#N`Gt&704i6j32gzrAWY+$4^I7AI)f91wLKu)!Axj&(p5+M6Osv&?jby4LLd!>y4XNzB*Qo2RTm|2(hDA#joD0dNQ6TBa<;L|Ff#gWotyRI3Z$0*<|gF`0PZIyS^H)Cmq((*<5Yg8-g7kx)2b*QCh(N__*bgImmgf8x9Z^Y9Y+E*)&tz?as1ed}dQCEKn1{B{YngR4oN8=`_<V5%Hvxyd4Vc#{#5j5#`V_2XTP&LQvvGltzl%u0&3sTn)7oizhOJuyBQ#EUgG#}9)075@K-K&+5N`8<YwD6j4{!FN_3m$B577xOOmbdH&32j}Emo6do)Q?<K^a3fL8`3=kyPD@u$59;BsMkR$eCH7h+7_>_N}(Ma6$xmmwF{(KZ9|&!uGjoI1ysbWjy01*y?WHm{t5O`;!K5&%wn&P$wV6FH5<0eR&#9`z6I~ag&q|f7K_Epv8sznJ=FMsc~=qQ>w0qgGVJTo^*uJ)uB?F2s6(yXR=%LR3s#8q-kiyY2aadfBL{;)Zz-m)4x~!9#g>et9Aq5fto&D}_j1`Vtj<yesnTj1;UnjJPz1|FD|%=}U%h2qi&nFj4-ZW|;`5`Uol4K_$HZ!!U$)HEs|13TfwXXUl&^wQs_azukb#txE?bcUusYFAMd4OOL+3PB`frE#JE&LK#%&NeJl91%-BV873s}v`roOd7EbLY?Jr`MmsXLX!5+?N$FjGdR&relye@h`MqP=QbUWq&%DyZc5A-_k6Ijl35g(kNvF;?y}F8rrNaZ{7o)MSjPX~$s?7xlc5dfYFI5WwS>g37vKTm(@Y(FqCOJzYhx{a{lS*=sx9C~m$^wvf)xOg|!A6W@83uF3@+NpZ?D{q(5Jrd=T|-JJ^wxRR|6GJVR?MA?+ujR%B);<s!QbD8X~N?k-LhL#PyO48O=f^JHz-dc5`Vin(}1kDyUeNfF;+4@Fv!wrkHofMTBU051bz0edmxZf0yX#{?YS<n60W=V5YJ-uG0LPK@ZqUs&dJ-E+Wv=WE{`Sn$I%T?xxA5hh01|++}(q7ckB6MyKbxltA-53`ykHqhubxlXu-YI5NX4r^VL-&AEpth^DPUY;Q5wH-c3$x8R1oc*LQe5M^gR`Zg933j+Hq^>1R6&;BT)Dy?NS|Vg0hZgp7Bv*~J?71-T|u{H+{tY*v){i@U{sQ0h3n1A&M259>@>lw|J^SVwN$DC=~|DL()VCX!^a<!dOC#doK9<=iZtD5D`*Qa?@Dy`<cMQUMZxp8FYVA?ZV2T}e(vKI?<-y8ZmmATH|J4jS*88Ktx72Ss}wmZ5+qaka@QPQd-e#0Qd@g_k&K3cXtNE+ATM;+iFf2zJdr(ERaS`JW8~UP!vV|!5)ExWr)*MdDs@u$($E2bq9MK+e4*t*v#nFbF=eRxNPnswB=UrFku6qF5lJzVxO)FBcfnF)bg7*x?as7Mp6u)RM1<w@J_BnJc~_n4WWxhipVxh^+K$Qkk|ay(`LjlLR*+Gpp(>;4o|E(%Sb1Kp0}oaK$ndUh9O$+6so<6>b6c}uU%#W(+CSbR@)l~(Pl@arb4;f2oj<KvW+OciQ<dw^K)!ohR2}a70G?u1TI-Ury+@cx_7IS5I#uA)fDUOZGAa&cjy3F<-dy2u@bixbt!oQIs9x2C&A1IRCdq6Fps_NTP!{Ay*}3PKR_57#H4v*W(GUamxZ&;nxB0*M_m(^}aPX{W6y|v&Sfeew$8HoI8gqwBiM%NnsP)W*7|2x6tVx<xj5{{Bn=0W_Q~EUov`P|;7P;H~0yhC^FCv9Yp&FIe&D{0vpEbAA<8mhbN_IT}RYNbZUmhBt*SOl4g145sqSdjXi!$t^z+yjWgJQTrF=169Or62%kXKBrCh!<@%P@}(e;Y$vcZGKg3yih{;l7)>5;U@+!fCcdeoG%F#7P-aaCf|;l@DGox}p^}lfcue=I_xrr?w!qpsr;=i;9;-$!eUAA1{n~Pi29h4b|z^Wp%wRO)9%wT$igF4k`{wPD|M@J{n<+V|w<nu~chDCQt>teV$UFbM>1I-=mJf7C|?07Dt)0(>o~p=krgNa)ZVox}LXN?Z(X|a|?TM)B|)d=)J8&!1`IR(i&xD)T2$gJ}xFQ^hE7)g#vW+v@+BV98uakrO<678QLg9bnd`kR&~%VFng@bMg-dHn>^F)_j3#O+tLrL!y=BDu6@?-+vK-@^0MGB|EsSOvpS?f+$N!F(f{=!1QWnf%bD`Pcia?9OSbB4YH{zhJ}f*GX&H@Ms)uZ;tSA_)TXl$9qob;fK1A8P0<#Y-SlZ-+s21zJitH<G9>pt)mo`Z$mFbs6D0GxAnTDgYMRR=X?QNK`Y^TwChOCj_z>&Er8MqnL)o<RZ^t)#T(>&d&u<iv+',
    b'FXl+U+q4n!BTYItMbxsT^4<{pf9(O_a?tt)_^7z;5Y^A}dpEs~m$vPeT9|v@oH%r+8O>^mIeyerd4D&n5WUn3Muo{y5m7UpuJrvBp)71t8r8mvuL|`H-11~~QTs?seqn`ro5>~X3=vC}neyYLt&bm6&8e`6!})yKr^mdiL!wZ@ITWg;7`biBBz^BQ2Ee1C;q<KON|Z9O?R}@^&g>bBxgZ*C2RPGD+xVEiwJuj$$>Xh&)_$7P@KNpE;slHw0Si#$#@@=eYWueGsPu#?>vO&`PI&u`^{Jd&HeZzE47+a)?G;d}r_4_Ozj{!t*Exr(m4X%D7A4`p=%9T>R6J{aIDmS}qhxv4p0i)sb*`4UN(|LTS7MB-y%kdG>5;N#>zvZB!_xN_)+*?va4LJY-BevOikd&ht6kMh((U?DP7og6-ruO=JW#Emy6fVK`v$$Mqd2%m_lsquPPw4+ixMD-FL%FB(p^E+R?>0%tSY-kB@fc3RF!US8=$Xlg%alOy!oOEbd%A%hx@SHhQU~tuLhjn6N}cGZk@nPKC151uQ_gSY?x|aymyKSmKpXcn74LleB~A;%kE&{*U%g4>3#3K5p65Z^SArMPOG&eLa-a+a@0FNv@4>Zt+uk95~InKSv?!n&gr$j^p7ApHbhBP>au!T*BEMVt}9osOLK3ywc0p=dGK@VkY-)`Jw*HC3aV10M^F}wUdT4xg7Vg*b^6o?rfT%Vtq!1U*l8*y)M3<SNXL;qa@Kxiup_nyavapT`TK5ThXTs%8<A|l#=1O^#8l3U-vQRSJwZ72s(mfLv9bE0_W7G~^KQ!3=45nvKk6vb;D6IqTnLeag@B`zdyF)JCad259zk4UD(~v+>cE+bXAHcfvE`W!J_g0}JM9I^ysL&Xjcu?%tCpeFix=k~GGneo+Y`BQL$A}L7IO1)<&NtVUhTK;A!!yrsit3{vT%8a^e13vChwEU7V27{?L?*IhHa3}NjW98bd&0tXwwX`dVYv-EH-gK+Nr0J_?2fMt>EX*J+F^;Bq(>JR)R~=BStOLwpfbO=%qT`vUn^>CaXRi<}C{M?)DxO%(;g6o^{S1-F`|FdJ7?wma36+)CuhUSl5xwt*LcFnf>+ViPL$ejvZ1t>z5U-;F;JRg$m*nQK9wfn(|_}eMC?~qSM*Z(^qE^NWF*lu0gX<Io9_g_3o{=P!Gm`uE6-6Y+YJUw&7%~PP0h)MA`4VWbtc6q`pvsPEDS(rW1Qo$FEf2_m^TSJ~+(!ILxLff5IP(x5aHE`iaaIdMR=4nH^Oz*k=!qs_Dv;r$?>BQb#TR$B$&1?pNzF(J-_S5N}L0$8NP{=uf$oW9si2qO)p0pRzhk1!hy##a3dj`_k$62Y5t&Bnw?={BF@o%3iQ|IVv~(h#K{+^#Rk`au18)s2>V(*p|}<Te&LG32iAF+H#&s!%%gyz~u?l%9k%g4yUx5U%%7SAYv*PdZ*F?GB5`+Jpdxktm0)39_xp)H;uJC22L@m`*0nlNBU#Q&1+2{k5GF26yCs3p5l-H`_7zu|67^=7`_YNoqgbGB)NMrcDTqO(kru-)$YGI2nI+^elE*blek~LS$H1F_={0}nQ`zZFkJ<rL}U`rcr!nc<0>FZa&mc#8gY!D$;?zFr@pB6k7q8b2Eq@5qQ=<%#b^84-F_S!1I-n`BB9M|k-b=r*)PQP@<^5BHMw5;C-A=;&r1wf{pn;ck5P7fA<s(#D^P^%<u#Z`!u=Je=Wp*9Wjgq!M)zN4B)=KKA1!)@-C^oGjl%c#_`T{Rd~k^d2H{_BQv1m*QZG2J-9*Urx})09Y(4$jQEl^N@_oj%AKDo&JF6X;j{QDk+QvitLPOa-9;`N%Z8q$gXCKQS*lkYj&+%%b<X*eWYs{1H)?@oP`3nwXM;S_T+k|gDCr_3}`ZEWBhgX^L>}WhXJ%8f!Ri^fOX0U3Q^3NQE>+R5cooTpexFB=3zBUeLNQA3G#m>RT&W#m}!NmYyXzHrR5|xnnz2`nAtF+XvO8+CpxC{*A{>G~)=y%k^*!;n4k|!7Q8HHsF%<iU*=9{TOEOy<^#-}O+|39Zh_!Bw4TU6hY0d)j7+RviB4p2XT?xtUm`nuMU`O>u4hS5rz>yI^;UX#K)&b0SQb{#l|>3;h~%0}V!^>U25LD_;1i1&*S^<YMn9>uDE1Q6O1k!Gd3*jg6;QP5|cpjLBi^Q!=I;YgZ8W_V#<L5s@L;}rbHe0Np?6p1xf|GRbQIoZVgWhgFp(x3kE*T4S>cm3zz-fz5J4IPd9h%zAIFVwfvwKrk+L~%v$&wH*Z$OMtcOUv`+hON=Yz0)ut9mRgyG12%N7FSDP4LLJAwU>Aw0L>DDGP4}${!y0d7(@?@U*JZ<;?O@^yS%Xj$0EVf+Ak`d_Dd}<D+deg(X3iS{mrySV2D8BJampVMngf%2w)f%zU|vn*Yh&G{1~xu96I}LiuLDomaGj3>r*pkKkoY;QYcMtxTHBa%i%qinnZhLh?4C=ajRNZ5+fGfp78t&vA@*7Q~5@IVEFK-%xbqQpDn&uY4RcQnx~;U-qb>N8=zIsIpuFP0v@k|PPw;r-I#B+a<#HnS~X5-g_oXR4|mH{au6k3$>ES!z8cexC5llk)V3nb&AZOXS6*Y@KWx~urlokczf*fsL%}p$HV89^@8qlW9_pIS9R&MHG-oXo{O%Xugvp>f9e?{WmPhywYPH^Tz|pG7Wv6@9W1O~`+muTRmol8cQ-8t@KoKWO(D;dJ4bA6QryppjxHL_jTIEykv9#Vft937);0*H%Ff4?_-DHsygytrPvOnegZO9u!qXez&l1jLbP7dAjK|YULvNl6ntle>bC8mi6zmX#x8#SiBF|dN4NxT_L!K1XBwR6_`0yG>^NKiQotqN<$w{77Yp4PBu<bh3I*8~A~ia1^x<jm6s=Q-xKq|R3&cDwP7FU&^{c%v6#L}+rKoh%7~j=1uvD(CVW_Y*=VqR(2k&Dz~r%d0mrRyZ9_3YbyHE$q_xtF-{e8nn)h-zi(HFWDW%8$4Y2W9{wsdl>Zi5uJ|D>T5J(TM*-T75<EBg(<=ca{g9)8ES?#Tkm7Ay2|_7uwTA^^xnQb=ZzBjE)xNla=T0bTXLP+)Rf+NLod%C^1y6*rkC#n4s@V5Fo~RBp4y`HF-?#CYji^o0iR=<^=EP{#7w(BWWt}=FddBrK}*GkofROenUtjjsdv@JCJ*YhY%b?-K<&^%u%pc4E@x%hxO;l$7ig)ESGQjEU|Qz-m!o7Oy+@6|;~59Y+DY<7rx(iNX>gvQ<1Cq?XUv}}`ToE>0j)2l0dE!EXpJ5QUx)L<K)QD^FF*qFq6*lpe6*1o<IiR@3HDhkb;A6;rLzC`gN4uULgX^mn}P!fqIW3pI%lllps&za+wb}JZ=qN1QAh-A@)OmpE0QIK7|h8x)C*}{dyB2VNc1E4kRn<{DMVBqAGvG@Ny^HyW~z1#_Q8z!GhyHl{NMlA|NH;^AHL83?|+^7Q|D7U(J7yX(|DRr^JzJ)r^Me(r$l@#Ii-Q$&M8f&G@sIPO6!TvC*sf0iSmht6OAXDPBfor;f(B*&!?PDnSW(I<>8daQ=U$FKIP?<*VAx54e2z{X~;MK9Zth|8m7}QpN8c$tf%pO8q;Z{)0j^q-`RK?CyuSBaXF1EZ<f=PP7|G`e42*S#5X>jruj52r)fRS=hK`{Go9vqnupUoo@Rco`7|%5dF4a(X-TJrPD?&5!)X~$%XC`!v6s`bp4RheO{bMkYd)>RX&q1NbXw=r%5Md)h2P7WU(A``%$Z-!ncvNsU(T7|&Y54&ncvTuuao!3*WnG~72+M@CE_jOHR3(;i%k3`6X8Ykb^J0Dzs<z2Gx7UO{6Z7I(ZsJb@jFfYQd5RS_&R>CiC=8uH=Fp?CVsbxUvA>JoA~u6ey3@G>G(Q+vx#4A;&+?)<tBc+iC=Hx_nY_yCw{|egpK(+e#wd7a^lyV_&q0n(TU%5;#ZycT_=9oX~JUQ>-c>qe&LDVc;Z){_?;(y>51QZ;@6(|y{8$ghOgsSpZMJ;e));te&W}k`28m~Kw=9dHbGjjy!bk{LSi!{wnJh=B(_9iQzW)UVq+w>Mq05Bu@HGBvOS3HL2M6Vdl1`$*dE08AhrjwJt$$t@^x$vVtWwVgV-L#_8_(gu{{V28!KCuHeZLuj',
    b'n$3ijrENMjup=KAhrjwJt$)j@O5ktVtWwVgV-L#_8_(gu|0_GL2M5iuvz#zwg<62i0wga4`O=|+k@C1#P%Sz2aVWOd>z|^*dE08AhrjwJ&5f=Y!7045Zi+$Y(Kt^?Lll0VtWwVgV-L#_8_(gu|0_GK{NI#U&r<!wg<62i0wga4`O=|+k@C1#P*;C8=9|Udl1`$*dE08AhrjwJ&5f=Y!704(2AXpt<QU(?UC6YneCC;9+~Zt*&dngk=Y)Z?U56F1YgJY$ZU_y_Q-6H%=XA^kIeSSY>&+L$OLD?*Ree^+at3*GTS4wJu=%P!{xx|!0CwB;p^ac;CSG9;CkSD*dCefk=Y)Z?U4t#C%%sDk=Y)Z?UC6YneCC;9+~Zt*&dngkw^G3zK-pY*&dngk=Y)Z?UC6YneCC;9+~ZtCpbL5j_r}z9+~Zt*&dngk=Y)Z?UC6YneCBhct*aC?UC6YneCC;9+~Zt*&dngk=Y)Z?U5I_P`-}sk=Y)Z?UC6YneCC;9+~Zt*&dngkyrRyI9v9%Y>$EMF|a)bw#UHs7}y>I+hbsR3~Y}f!7KB1Y>$EMF|a)bw#UHs7}y>I+hbsR3~Y~q;J*1fg6rn%*d7DhV_<s>Y>$EMF|a)bw#Sg+;rTkY$H4X&*d7DhV_<s>2nL7-2nP}a_&NjvL;{2Y!~z5Zw#UHs7}y>I+hbsR3?l*tU&r<s*d7DhV_<s>Y>$EMF|a)bw#UHs7$!s(zK-oNussI0$H4X&*d7DhV_<s>Y>$EMG0X@-d>z|kV0#Q~kAdwmussI0$H4X&*d7DhV^|QY_&T=7!1fr}9s}EBV0#Q~kAdwmussI0$FL%(A*yj$V|$EjkCE*$vOPw&$H?{=*&ZX?V`O`b3E_{gV|$EjkCE*$vOPw&$H?{=*&ZX?V`O`bggD99u{}n%$H?{=*&ZX?V`O`bY>$!cF|s|zjDX75u{}n%$H?{=*&ZX?V`O`bY>$!cF|s|z0g;!lV|$EjkCE*$vOPvbXM|_OX9Q?OXbI7L9fCBXG{Q97V`O`bY>$!cF|s{Iw#PUjmh*LNkCE*$vOPw&$H?{=*&ZX?V`O`bY>#n9(C6#e9wXahWP6NkkCE*$vOPw&$H?{=*&gG9?0~OhdyH(4k?k?EJw~?2$o3f79wXahWP6M&(g@@coJg=eCbq}K_L$fn6We2AdrWMPiS045J*I?&gRf(IOl*&d?J==ECbq}K_L$fn6We2AdrXAPgs)?JOl*&d?J==ECbq}K_L$fn6We2AdrTQA3}46gnAjc@+hbyTOl*&d?J==ECbq}K_Lv6bI(!}5V`6(uY>$cUF|j=+w#UTwnAjc@+hZD$6!CRzkBRLuu{|c_O-P)OIU#jI?u6t?vM0U{`4bW-w#UTwnAjc@+hbyTOl*&d?J><rv-mo;$HexS*d7zxV`6(uY>$cUF|j=+w#T#}ALHxT9uwPRVtY(%kBRLuu{|cX$HexS*dEi0#0{Anr*3SIne8#NJ!ZDY%=Vbs9y8lxW_!$Rk2xX5<LlTSGuvZkd(24T|5y5j651K7t2d*aQ<2in{IDxt)cW+Kc19x0zkZfvwoGbg`FcxgXZhZi)XqqFD@pB)MEE1Aoe6(qzDsIn!rxmZsh#n=*hq86JIa#O&bgk{&iFeMuI1m0zm@Sj_(kJyu{me9;XL4F^S$G5u_<S^<&3|@)|}a#Guv}!gU<L{Bdo&L@!y*HZ_WI-X8v0<{uWzy#@}Mw&TQOyg8BG5-f%P9cZPvsVVD>;hLK@qm|5)1*TK?k>X~glv$1Ek_RQv<+1@i7e1^prEC{}iZ9cQnXSVtbv-5_VVRzneGaG(p%g-xT4weqD9k%_#wqMxx3)_BS+b?YUg>ApE?H9KFlCZ}3I=217wqMxx3)_BS+b>vYZ2N`R)xx%42#b-gV>>Tw--YeCu>BUc-@-Oq*k%jcYe51%;%ls2zLxE^uw53m$-*{S*d`0xWMP{uY?Fm;vJ6<_d>z|lVVf*$lZ9=vuuT@W$-*{S*d`0xWErs+__}f0?jMPCAgNuDY&Ryg3!Z~*y|Aqpw)MicUf9;lgw4o*pKZOctrxcS!nR)6)(hKuVOuY3>xFH-%s7Ygb!_Ve+Zy{C8yh<tTN`^Dn;Yj<Ik56|{2U8E$HLFC@N+Eu91B0kg6CjcFZ>+K3O4{hz>a{QW98>q`8ig8j+LKd<>y%WIaYp-m2JHyI1IjyZN0LsSGM)awqDuRE8BX78-WjDTdxGq!q>5_SGM)awqDuRE8BWyTd!>Em2JJUt=EiGHDAYeUfIqo+j(U>uWaX)?Yy#`SGM!Yc3uY@zxg`0^U8K!+0N?#_r=$-omaN=%64Aa&g-}(wJQ?BCnvS*i2vum&1+&E;WznjvyE4_@ya${*~aSxkbtja8?S8Rm2JGTjaRnu$~Iov#w*)+WgD+E94}wTHeT7rE8BQo()Rn=&MVt_Wjn8I=auceF7VR$8D9%W4o?nr0`LUz2>=vu=J4j?&c&a@p^HZsmo7eCoVs{*aqHsO#j(S)pOb_Dd;-q>4DWu1dq2a!pAjn02$pAr%QJwBGo1XHBx>Lj@abnb^E15m8Ls{ee|<&<x&@atobqxM!f!{y>Lk5<k^BdL9q#@NzkWslIRh3s0~a|XhMa+noB@rTfsLF8iF5b_;E^-%kuw02GZ2z9Ad)jMk~2V(Gf<K<V3PAFp%I@zP&p&2oDo*eh%0BHC1=1TXW%7g048T3Cg(|_DLw(v<P6m04A|rh+~f@4<P7BG4Cv$x?Bopa<UC6N#wUQEoB^PmfuNiLp`3xCoB^VofufuNqnv@GoEM4N_ynMoGq991z?3u4lr!LzGw_r%0F^Tkl`|lf^D1Fp;yi*mj3z08<OGrwNLC<ef#d~}7;KrrmKxv_k{n2OAnAeR2ZDMD>?ODt){i6!k|ju*AbEnIUjlzgsvrQEAYhU$2nHq~m?R96F-XcFIfEn(0)+_{CV7LPVFHH<9wxbiBo6|K2_`0>n4n?;iwQ0!z?dLo0*whaCV7M;5|T+sDj~UqAY=lO2}UL$nV@6>lL<~HK$#$A0+k6?CSaMMWdfH8UM7H<AZ7xY31%jsnV@C@n+a|vz?mRt0-XtVCg7Q%X9AxIekK5#AZP-i35F&hnxJR`qX~`%AT0q&OMudlo+sdGm;5{dSW7_G5}>sNY%Kv?OF-8W;I#yNEs-SHus<NS1dJ^KWJ^HV5@5CjoGp<6pOPd};*^~Z4En&J4-ERipnog|o$!M9AaueT&*HCS8vtu=z@Zb2bOE9h$?n0AKA`ACf>pO;(Fw)|7@c783mTnB{tEUYjGsu{oQ+5)EDu1^32OwHbizW3fYJ%8sRflzSb8n6biz9QN?bZ&6C~_{Ju;oJEpDOdM1n)(u?38l$C(A4P9(52UMBn)iGkqBNGv4yG%o+<Aa%m-e=KEBa2~fgdm>4hW!)l-N}Pd{hgTUT(f}^LfvXez%qYtn0P+oBo#1&AT#p7>C-|cThXl|%!7C+_b&og!$8-U%6Fk%fxlV9Z3BD@9StWR@J$jwsz7qV`1;0*kWVZlzf>%o<y9QYG2Zo(UKx_HYASiYsfv-0hJJAP@ooM^daz(NeTwsC^OeFtAR3h1Ah)X2kwp=j@qMhIp6MSMK0lE>W1?@(p#s%<?2@W#BL#Di4fmps>f@lsmiCSHPpG-Mj{wCaI5`T%RT@r_hx?Muy4nTK;*GzDmlcWtKc?13+eWKkdZ|I+VHK~Aif_DYto#0}h33(^@>s!n_p@{e_&^u+xlmPFq@<rYfYZNA1fbW#$JxPiMn6iR>^A8u!r1OzU|7w}^uLgW4q(09?z7z7OghVPKlS)~<2HsQ_w}Dd!(y4@eDj}gtl2M_&pCqM1fj=RqN|IEe#-C8+zhL1B)&51CB2eBBG&~{G^78A1RP2&qC*))aNm)WxM)GInl}vIa{s404n>_n~#Sd8gfW;43{D8#|Sp2nM@gpY>=H$bXlOH+xAFCN93BHJ)*CYTVx?U3y#sr8l0b@)6852;(1eh@aXG{PZlOT=ge@%cI6R^gV<(I;V18z)NHZz<!5XY2dHzUhCnpBQ#{K&?SZ2ZW^k8J$N#*b{g%*Mwf8$Yt~=V#*y<?f?5{7AdCv^!xs@5sBqHt$a8Gx?E;cS2LkqnUR?SImz}y;J`B+&jy!f2G`ekZ1kna_>Q2Sz',
    b'qowh|!;yd!vOb4Z+-7`n&?U_aMLJKKCBvm)_^z!}&{d??LYV$i1JCdrwF1edOMMG$j1zG^ZTt_<@2SDENVbA1L^Nf*&aOk%{jz@%hNak4*dzmWdBCeu)<&GnnPfWkT|ikbPKFmkFuKe(W+KSHWQ2k$WGx_mO)ax%ZKKAG!CDdmp*Cl6x;l?tSFmf4JNml{8pkkP;wlF-Vb5@@^8c7`n2aq(O>()eIcp%<>uhT6yuX+90GTM-qM{;YSjFB;iLAek9>X5`HA%Y7)L4N%*fvIhf@?$P2SD55j~3&f*hLo=D*LN+9@3!tgc7$Aqd-xC*!Q9iI@oLTUdJ!a`vz#GZo=MS>7Tf)Yg%nlETkB=CGC5Pc;ueI-zRC2)NukbNbveI?L+CGdSE5Pl^vekD+TC2)QvkbWhwekIU;CGdVF5Pu~wf2BcM{@`ap{*}P~l|cWM!2gv%0G7Z2mOufPzyX#(0+t3zzNO`F5#mLB0)7@$U<q7c31naiY+wm=U<rI+34~w?j9>|rU<sUH38Y|2f+Msm5~5=og*R%Jf8r#A)i6pc$}C?kAH$spnF%p7A!sIvnozY!qwFy(5X8+SjA4kJ386C~b|xVWljsS3iv)p-B&=bOxJb~rNJ1M1m5T)4ums|;B+OyscGkitumr#zmOvhsggp#;7YTwF3G`tJ{9y?MVhIdlNhri1dy$}fk%U8R6xK?itrXr$A+8kW%8j}bpOE96oaf{~ha~_Jj06>o1R}8nCb0x6u_Rn#kitmN!bl(!OTs1wIgA84u{6rt#S(xdMuH|r8s+U`2?(VabTN`}ia{A838@&gF%nqClF*7l9wP~_7z8pB6fzQs#gZ_KK_nwVB_ly5BS9x)kz2+xfLca^Tt*UlF$iWP@QW?Nyosd%-HZg`44e}Zh{kYGNRZG-!ZrpGjRX~q1R0G49gPGbjRYl)1SyR)iKoRHfSN{voJNA4MuMP5f}%!(q(*|KMuMnDf~rP>tVR;LF$ilUC~G9)8-un+g1AP4x<&%wSQ5rD2y7%MY$Ql*Bv6heaE>LAjwP^;C7~UI&_<GQ3aO2Rc$yGU6C!FtNKJ^T2|+a>swRZhgt(dzSQ8>^LTF8soa%+dR9H-f##DGrg~)VcGQ}t4jS875)&L|p(j>1GYhaRhiZy^pn-FRfVr@dOO^CJ$;Wi=OCIsAsh?@{{6Jl;c&`pTCNy08vITB<!lEhsIb0i78kmg8;yh+G^5qlGYZ$k7<2)_yOHz5EgMBs!FoFoQArXxY8BSEMmAqXc#;e;@p5Qh^2ags<p<Cr$dzwjv>)+YI4xt>fCq$-kVONg+9s7Q!>gy=?yKm@>-B<Kqumn4`A0G9;BB>{3tg1HFbBFGA0l>}TR0a!^uRuZ6<1Z*V%TuDGz65y2td?m?<4-l3Fh$R7INdQ?AP?iLkB>`ti09q1|mISCJ0c%MBTN2Qg1h^#uZ%F`L(k#zDT@Efnjy200z(N4_l4f}a!fPwUw!&;H)V9KHE9ABtyDdH;{I*yKz+n=Am;@vy0g6e$Vv>L{7zaw22TB+SN|*>r7zs+42}&3WN|*{t7z;|63rZLaN|+2v7!6984N4deN{cK6JTGR15{85l8WWRrCSpz~Noyh|g%WxblQ3rioJotEo$z9T(IkL02`EhhOp_K#y5I(Y)FePP30O@6Sd)O(B)~NZcufLelYrPHKsE`OO#*0>fZA+l8Q?Z)k)zinKgDOU7#2ByUC%PWauUFt1T-fB&Pl*?5&)e9L?;2#Nx*c{B6p8v0azyi*GT|&5|Ev=$VqIKukf)^Xom4ItPKD>3Fu7%9FxT10+2~SWD+2m1WYCYlu3&`0+t1EnY73w2#vVBPOJ?8G)WxqBF}-fvCNmV44|3>tR^kOfGiZq!htL#$ijjwgyQlhg$!BPkT*JHd<?4u;7wZOiLgq5-=sxY#qZBDi;#@VTf@o$u9E=lq(x3_cx%9S5&)hAgeL*wNx*m#K%NAY=Qzs*&ob%I@H{FCFIG`Vz-SXd+60s~0j5pBX%m3j1f(_rs!hOZ6TsR8v^K4>Ht|vc*d`#h36O09W}5)oCZM(nux$cvn*iJ<Ah!w7Z31?i0Ny5`w+ZlV0)CqS;HEXt1@p`*X92lCy!=%f_V1Zz0Q9C+`uDNRf%hf=zG;<1!u4>nif5DKg4~~+8RY&D82|+*V8IDsZ~_{f00*aayc|w|;3h!02^ebvh?{`ord8f4B7<a1KynkH+@ln6lp>B&#8HYkN)bmX;wVLY&-6VV>HCqszgqeZr;EUx5Sf!;afr=HNGqhafDoM!qZ5L3LX=Lc6emPC3GR25{8}m$*V+OicUq-ZA^J)1#)#f&m2~?UmpxMSBSk+_^dm(-QuHH5KT`D9NYM|4(qFG#Xcc4^UImdTt%B@|?5E_fBK#@JpCbJ!+MklYiu$L>e{TApA^-{nP>}!?4NwsQ6%|mC0Tmrk5dsw@P)LCyDiNX*Au3(ccS3X`L?=RYB19)bbRtA2I?I~EszP)kL?=RYB19)bbRtA2LUbZTCqiT*L>xlIAw(2HL?J{JLPQ}%6hcHHL=-|qAw(2HL?J{JI?Js=)d&%V5K#ybg%D8)5rq&@2oZ%4Q3w%*B%*xc@t;iag2#{YR>twEc>Jze$j0MGc`tI>lJ~N~?m8Y%=L#NA=k0qRcswCc5&|Wi<)_d`MWCef_ES7GG*S^L34xLjC<%d*5GV<Ok`O2ffszm?34xLjC<%d*5GV<Ok`O2ffszm?34xL%P>%8oq~WJ*7NYak1SG!z|HNqlVU!R?31O5FMhRh*5Jm}Mln_P<VU)1J2|Jvy#R+?yWRol1LWG@9Y3l}(i!pYMosdKncT>g1uM%Bc5yll|T#?3=Ic*u#mPu{wfFypEL`WB(zz%>%ptQ9FeKZVF+M0tt8iojtg3{I|Bwwvt!w|u1P`We>QQH24Uy4lv|3Ppd1P?-RA(XbRA^eR&hRWqD+8Tx^ZT&;I7<(pd-@`B1zK36cJ%fl$h{%M9Oo+$?4P*ieK!5=VH~;|%ARqw*D1bI8WGo~g0t7{5f+R9+n#i*Zx64oGn}#8RN-{wvnV^$QB9x4XOOQ(@(Mv|qC8#D-l26PsBab_nKN^Mz`pJ~!V^}_rP$tn(M(`ylDU(PkUmJ!9s>%deWrD6UCHXB_aFA9eXe$%Ml?m#~l;pQy!9ie|ps-9Lu`KH3cMU@X$z_~PNOYG)cv+N}MWcKc4dt`^6q7^JjU@9#SaYIQKFdfE7M=()Z;H&Td=J-Q^$AOW=#~R`At4J8@$zfS5D9Sr{5O3JFUeQ97x`;4z9jQY^4Da732RQYn}tO{Xas~uK=hkMz?q~$X_V{mG3in&TZTx~=Sb3jxizdqQK1tVI?<sMAv#fd7O7{Eq7$)aQF|7-XA!McT82nOphX2*WS~U{T7;lQp-v>~L<?HPphXQ@<e)_lS_Gj*5n3dnMST@5LnO-3+k3`36oF`x-j%zSArcwZMXV+LEMh1uOOcNj{b&)876oaMkQNPT5s?-ZX_1i@9cdAg7I=iDl|_t#btr18%QjYq&tfr(ys7wU(N$f<GV8dt43TI}3xXobs@Ikw0-8dAQwVqp0Z<_zDg;P{fT<8b6#}Y4LJACQheTi%=ng?;l|*G#JTatJ2`a0UH@*oOdBZP&Gla@2L1C4kuu4!@B~e$swhR$;Rtfs5lx18Fix66>goK|&s~RW`Wl<3qK8uae0v{74agh=iEph3ITI4!>7OPHxI>0srw2pw*5zsmUT1P<Z2xuJvts_C}fZ>opc0h3`%R|d~M1CJ!Aiz5k@DAV(34*uCi^b9dj7M4Gu)N(-rv1)a%Md|m@i@?gTZWFFp<i$SLPlcmiVhovt+f811O|}ibxD$?=#a*G(P2w8q=g#!%xTLnr0p2l0OC7Zd{2@}NN1|_%p!6jNTC<?B$<PB97@|KyzNQyg+-bn;j7?>;7Wd4&(Ie6CGS#sfi6?Y7Aug%^BpOWgmUqp@LP+k6)OSyLn>CHGlDbRBD^A^FoG}~A^hNtMy!NiL|+77#9p|61TX|%WPOOZh_wi{aQQf4Aj?BEMKDDyMJPoig_lPhMHodCg@;EBMF<TP7$iP678no%5&aPS5cUxD5cCl95b_Z55bzN15bhA|5bO})5a1Bs5Z(~o;QZmQ;iBPLvv?!E4*m!Z2_6Y92|kJKfiQ',
    b'*uhWLf>h3JLgh1iAAg~)}Ng=mFfg)oIEg&>9SgusNjgs_Bwgm{EV1f^dH{X*v#62DORg}^WLeIbqpZ8RK+aUO;+8g$Vhi-xl>j>0$zLlO;&Xb?n$9vTk8I0NGd3@tQBq2c@sAvEZqK?V&fXpo|S;s6No!BG#2c`(0&njK{5U=9ZZI9R(u+KmDO$hMzdV8CEL=JGL?kJ&7YW??c5gISo%!dMojvM`i|nJkQCVIm6yS%bs|z77Eb5dt9sF#<t??SVNgjA3C43qx3#!NLd@Ca^Goh50LtUt#(R!&jKS!srzyuP}IpxhsraVd@G)SD3lN$Q358FmQ!=D~wxV+6u!~n6<*F6(+4PXoWc|j9H-)iXkh^SYgBp6IK|oLJbt-RhX{Aa24vGD1TzI3WHUsePXN%Q&p%TV5TZsLcrt{2B*9N0w55KOJQ0H_y=aCfPP?73fKqcq=0;2N(w_#n2`eNfe9%LNMSw-<58H7!f+I3qc9qU$tVm)VJ-?|QJ9LtP!wjOFcO7{s8K>UUx(n0=#B8r_P{6<CZRA0g*hmUL179CLr|E3!Uz;5pfCW1`6rA&Vfu-%p=9_8o(XUUVS5qw*C?4OehGfWM#0<@#-0ef4pUDUdcw>TMxJEiNn9BABZ=TPb|hg-680oz$%_%q;Lb4YBy&!Z@x!T2l5ZmElRC2G`?GwDuTJtW{AOf<Y$h3Pl01J)@!@5&+%o(uHa1~rQ<h{}ZX3=PyBk2oB<VUo0Op#eEfqleBcI@Vkz@cd4sH-LO&DpyL=*4{%ri}rm9OU`JC0|^M@!BhCOf7q$89;BuCh~PBo$v^E6NC@og-705xLJKnX;f?_gONDGRia3WT{a|YK$ZnIV=Eepwqyh(Qt!8#wl%;36u>^0+C5o0;sM4(I)}Mq#z-H?n$69fnc$E@j)aCGzeyk%=9eZN4|>RfQ*3#L2Bgt$Xf-i*(Jt9`pArhb`$vo3B~3!lVhRetYybS&568)khrJ`GP0Q>Lq>W-$ZsS^Mv`+)l7*{ikmtrezvRinOY}gZOvsuDX%iuD(sE@Ij-w&hvt>e>rKQV++$)+d(;!K+pD+_Lxl6`ONa?(knUM1ll0HJ#M@aj!7z)4oBX6cb)&c&=N9Ih!wjzdW?o3DwYsoVWaz9An7rAt~%by8JBq58uWYC0k@-~Gg<d%fw@{&XovP>G}8IWhn_CT_ElSn_>sYA$INmAEJDov8pD!FvRe%anDypqrjkmSzrq>@16sgMPKWYi=?1hS=&6qB$K$gYyyShkkz4`j%}%$kr&(;)UiGL6(~ko_bn^_E@}a%<X>YuVelX|bAYFJwo9m<`zx@@yJpgNfy^!;otel5HBqcGzXew+RV14Prj*H00cbq?-mY;3^(OTnK!~ZTAjsIe3|SlO*QwUcz^vHt>*qaF>9S)Hvl=!873Z(e?`w2XSn_q~RpFJ@%dS2;n4un~5h9JK^n0^8b;Hlbiwu`L6s2e@$)%{tS0RayuV6IpMxYP6py9NxlWoDWO%8>sfvv|H1Ep6UT9Z<VLXz<<Kz6cf|SPci`+W%NL{M=!APG+&tm#2?q(nNrG^cAlyD}_YViZJpfiC4itnF1&!k6#m$SK7e_CTu;C#11QH+EbQFg#9v?0rCk(<7gK)+m95M)}3>xM2!R>Dkf`z?32^K648-&vaZI1$cWP28QC0L-SpOVy1<-~#d=_KzEixlS$!oh>&<N?A3!qJ0p_8=TS2&WIi@q<9PKp<Qo5H1i17YJkvv^~%&(M!S?&>grL96*lzJ)FPiBY!{g_djy}{v+Y+Ne%`#Df=Xcghv2vJmOa&+Tf)jAU9^J4$c|^XAOa~MzVE*cEDS6&DX(SL%?_lAP<4Rh9)^=;C+L?Msjwvq}&4BB<(5Sv#BQSB#~v3Y+b$>C2gn4$lFQ!S_Che<VUz0`MHt06Cfo*<I4r^Cc%~_adGk^{Mqey;n!!m0`Em^kKgx!dlS$m0^D>#zG;%>fHeIE`zA2zNCJ;Qh6J7v%`O={A*>PN+Lps3x=9X?5O>Mp39*h4>~2wTLgc&T@PrUZh=GJ4cunFF4JC<}Gwviejy(1zi6=zKS{6@}tV(&jZC!4eJRyo+QhAzWnIdogK*b4sJ|yiqh^*Iaev;>(<$_W2d79)cU=1VI5`rxu+Fk&1n&fpX8$?dhwlm2~k^ETh40!YhPELr%gm5ewJ^FYqP&t7Bh!C4^Q+fg!&?ToQ1Z)B+&>k-*MDSa{oF>^OfVbu5*H6rx5Zy1JIU&qTE{|A$fz1g)pAhvWkw@q!#C`&)5P?*PKq^EC{)Ff+dHlTP<@iAAhtLTb1kLiDEr%yG8xb0fZeer+OA*Zi7r>oLo6#)L06Z#~iAdUvX1Q3>2W(2rei4Sh2$Nrg(J#X67h(8|F!@Co@giw6!hjcH!i&&mMABx2AumFkk)*=XD~ee!!mt-%+KVvmMVR*@415tLz6c{<G|MMoc}#r~#=Z!1UxdLg!sHi8s}W|u2*Y1APamK=+KmY9Mg(#q0yz<ZoQP&gp5?~G6^kzxXDr@WJ}drM9P;Lo@iFnqSk7oSBD5P3+KousjlfVuvn&gEW-t`dEDHl$8w^D>%gVs!215~np@_gxL|`Z)Fcc9OiU<rvB<)6EC?b#(5y*)MV_<|%BSNPUNv9FWiAXw)FiHi-EB%7fafG3;3knaXdI903QxMZ)gmE$WNO6&v8zT&kT|juk1Q}t3j4(q+7$Q6J_;4OS9C`ea#~*q8_i9VQkSt+JmX<Ad{kqPSFyekq;w!x=b^?!4U#ml<in+&=cLCj(?J6U8CrlgnG^%W<`yyVD+Im%-eEqgvMNihXh84o7^uA-|9&Nu%)CFwEX!Tv5jyY=&Z8x&?edkIoM_&YKmmA1iioOK<R|r$zfZJ)wA9y>VUs%cym;%3TVj+x#(;{hu*c!WWk>7&-xX7=;j>Kp<N&oO7zX!V#<Kcw)a9ZS7jncS<Phf9u1LCqf@m4X<O&I8wL>*JzT62q@r{A`>NR}>W`y#30B}o^s9lgkx9v70HWQ>b{WtR&r(<ME}L^>_vUzGG5bLzBgUPmV9@!G`sT+?%muG1op=$f8mjGdPGE<MLcJ7K2%(*Ck#=h7~{NP;UC6R#z1OR`+CnDo_2s(a1Om(7tax7qn3+hx6E=a`4DX6J&p@3M1gyxwK!i=@wZI%&q1?0k_#T3oGU)8cF;rI!0%#l5auc0O)jT#|Fl-xCJ#Nm{cpf=}!DlAKF_Hb(IY9iD^^PeO+$t>Tpt?^c0T5b@CAN$Bt-ba>J#`SW)1a!t<RQsGSrZJ&g;PeR)#q3x5<_DN{_B(#0fD%Zixt0aI@&X7<5TVp<rFrP-4-zN<46Xw&7Bt4v@k4KXJD<<i*ZjAX)?<FT;VT8+)QO^gv$q7@UBvYdBr>opK0;f!c?s9bcU~N2_WsYW<qgm!?mN}Ybj%JyoS>|Y#`9tUL(~-abYWX{XT+U433EU_IZWQ{UMUfnyHf9t9WEU3m50IU}RY$^R{zWsPw6UZc89Z&2ElvuLOykJmj~xEU;g1~t$l;G1{>b5v9R63z;pZcVKQQ<v41Uw7d=3V0waCyWXLR82NA7;)?nmx^<nBlAe&p^)?tbL%*W7(Ma`z*5Kl*2m{Qbz^kNo|}-+$Ho{pf%{I^d5E_+K)EUyls_kD%ZxW2%rd{819F1i>MK;82f<E5`WYA>km!Kd$mkzK{(JzlgDYcttox_(Y8C!z0dOXub}nW?N%eAHEQy`tXDp)W@9uq1O7n)mjMxR<9`6N~;`mrNeAnxY!Gm`~a^)?i~pXbVvz8qy!~Wf)pu1i<BTnN>C#u$dMBCNC|?Z1VvI><=l&y4na~9MbdzZD~W8WA>&F=lA~4n*2J!2Sh1{_R%|QAU7xGxN~?r(Su60doszD!ZmaBUin?Yf-QiDUsNLZ;=CiCuS&p(EWkJe{6n=jY{6q0BL-8&{@h(H@E<@=qL&+#Z=`KU*E<?#EpJnnI=Se8tWhmX{O^5YSnb#~&xT(3`%Dm<ean}!-*L*MYD#`m->AViIt>sT4(BtoYvCeD8lZei1-fr_w=QZQ5GqeITv;y;XS4co`R~dJeaaS34m2p=Yca?Eh8F!U&R~dJeaaS34m2p=Yca?EhdAqASo!5+;%(%(C-6Y7Iwwr{)!dX@v=GM2H#N;|O_A)f~GBoxwH1;y',
    b'In=&-^GBoxwH1;wy_A)f~vWU^%>%3-2?qx_hWk~L2NbY4w?qx{sWk~Mjv#fgvG2ou_Sr)zAGiKW%xtF1~m!YheA*z=ls+S?<lp(5@MVt0s=QTrPFGFK5Lt`&PV=qHvFGFK5BTLFilQQz8j6^9TQ_4t{G9Dr05wbi&5)F-wq5z)|tVX~Z(9;~5d@z$whb+>s*$e-bx=7Lm|CPc>MHBopsb+E#ELU`Pgkk2MZ6=HE<_+CU77fl1zL}ha+K>btCYN$1!=^b2o67cS`5v4YpkY`xC)q9nD`x?m@d<1j*f%HnE!aFTafXdEjGR$%$*^*UnKSI1VdxA?XHnjq=o6LCFnosPbJ{5KOfs4*Ohg_VjY}%&ve97qWvsu91(>k{GnQb+8q8RP8LKd38D^}*4Va%U$UIgg$vc*3oCKwxY~>`Z+ABJb<$DLuV=ZSa=8V-W06ms;#+uGp)H%ufLCn0>P{JD5xM>pBN1>vG-H@>zGWJ8phPblRU}t1(jf}k^1U1+o89PKMYOqZ*_K7GAqrRi!^r&Ttq!M;g##YMMOBtI<kb3MWkzTnX_1IrI2_XLo)hCkmhG3mURY|_sQPpHRe^S+C+P-toDVec5Z<GEkTNSbOlJ;k8+<o4kv6C~la>icHB)=<(b(4VhnWQfRp`?b|U-)XSn7vd#NbUkjIov?nN?;e%KEo|!_yrU%5{c@PT#uWO6(&n;TVwcbvdUzcNfAh@K(f%J4kR~<#Xu4rrCx;14u_H9F|xP})a<et6;-<|FCTTgOmfGAtYmxwn;nIRjOty+z31(=VNF!<GKvkE<eo=a{KCtGPhfYUj+X%gWK{ApI$kn-Mn;267N>z)UPdu5lWZESKDdnxzmefMGCW6y>&WmOS)2!IdKum$!+m7<j|>Nr;XyK7NQMu|a3UFMYZ+c7!;NJ4k<F%9zVId)?j*yXWH^)zkCFiwWVn<JpOWEJvUnA7D*_q_Y9O$I;06L52y!6Mfe5jQ5}OcRiC>Y5pEwq&`AN}F1ldH9P0D`aS;V!7ZxQDr-bLJt_!p`EiGvXjvlRfv#fXpDYJm8JNVG{6P|AQ(2b4k}>@KB1sRfFg5kDi2M#_QWX~fltuMuY>B|)hPin|eiBMwJAjtIAja+^rE2_KfY9Pv3)85B+|A;l7>Beg-{#S*V0DsEC86dgAaaudHJjz>I?xE}F6!jC1+M+DtO(M=@Xq)aH@M@X`SB}-_s#Q%u0n@GEf0}>A;Tv>VR#E~E1V=%lpBGGx1Gp?K}#TB8bp5cqKI3pAeGYaa*u^@abIP{?YILCsFAT8;k^qI(3Tep9%&1ZuQWG^Gi|Fpxw;{&o;8fjn~X{$j5kggg80O^LgJswC0Oofgz51WE@{csTJ&dHL*SK4#(u-(<F_UGh{n;W-D(pDm0;Zq1#KvMG{1d9k)(iS7JS6X1C)kWG`q+MleMUfViER>A+1op19m`HocAgwF%1oX8Ms<hpvoP0VcDnpGr(hVbBFoPID5_{Pfh_;e5OIHgzS+Zn?=wQjniZl8-G8#HEIyy31I`Sa)K(rmiAc(d|HgBa>g|ITC(<5&m!LOO*Tlh5y!RP?V=m5#+_sD4Y$mssa==#WmoJ-^^f)I?3kBp9wjE;|tj*pCvk3w#Bkk%diD&ek_MkD;DMNTvLO?O>9gUGMq^6g(_uvli%1HI|u$s!K=aWcrF5?Uz-WRVZOsRv}y5^d05&1g!=XiCXwO37#{&*&x30Jbxb?F?u;1KZ93w=>Y~40t;O-_8KIGZ5~K{*;Ucl?;eG1LKw#GN!GaCj;#k^8@nE!m5t4SqAc*rRN-#vyA?gj0TpB4wj6L?u?G^JPJ`M*5N3JM%+I-xidPs^C&?R^qMd<y12#8=;Y350ueK#n>(Ws<RaFZ(bX-sL}&Mf&oZOKyCk&C((`>Gw9IJ!-d-KlPj6IPGn&COn!)oZ=UnMVlXoE}T{&`GPXgn%z;V-}9xR8CMJ~oKkQ2!DB!CX_j1KWElBx27&~X2u<(fr6^`kLAi;}9m8*CKOQ~hYn&*&h}=pfG`ttzq&&{q|EKx7pg8%;GC-Q;<ccPe))FBSV^lolUcA#Yah7>cW#{IPt0xC*@HQSKg%G9MDJS(I3BG+wjFvd(htdfNUr1X|@!ON$R*D}Q*UPm$M*R`gqy*NnFGi^^+8d%7gXXmH8WqAoc*`qc5WjJ}<WUiFM#^^9Ki3<Z;m%xi{vNrnPRhEmA({~*KN{sT%}-V4wI$$)V(zPiY~W~hT?sDfmCD}Q_sI0a5jAQjF+DlCd1PyxwM0Rh2$Mq^Kw&Ylm7`7A9y$Z+$<FrS4#SlH+*GOt-^>W{QOn%2{i)*tlphs^8IdEeA||I2De<w*pIzEnFZ<G22B?WimPwMRQD%j10{^I0Cx){e?}`X8hnbu{N6!?nk7?J-<?r1;eo|4@ScW2N|6a*k)E_!-H|KE=;SW)20ZKbV5lF-Cig(H>*8N2Y#c>PM!2Wa?j#sUKReM~}?|JvP%9@>u>_JvK=)x0WWGJW0MM(cretW|Glgy!0%dS!=YJB<cJvjW$O*f28wAI)9||M>>C`^G7=WNIHM0!XA^ff7D6ZuLRw<8CvO-XrG})PetB(Dv;w#4@KH?dr9Z<6%sP(n^0zG^EBQL(M~d{S{tIxaD;bLw9;ZBtrZ6(f8_8-4u9nEM-G4F@J9~+syY0j40|-s{Gpp?j{f+Anf}P%kNo|}-;ezL$ls6r{hQ_QhbrtJQ&}oclAj9%yWFAh8Joa@;qxTlz?GQdpXA$H`kt|2E=YWy1OUd?5oCAz%-}Qjk(R<|Y_3ZVpJkWbB=K2>+*$>uvP{LDB{?EV<TG}wp2>ee@)>)%hRJ7a@*mL&pC@st^5FB<R5OWd^%D3Dx3l5z%h~USmjqsHv}_!f8LsLMeZNcJm-g&EPr{`h<Ejyoxk(5YZ&LS3C>^)lUFhZDKP7jcga)sHtY#AGw0-hEf#ze{DAS7cJx{{hf|s<?J!d5x^<Dlxtsnd}@Z1@$J5Rz6vVL+_!jb1mm?l0bE8)^Je0qjc&+zIQZas@%|0J!%i@M9<@z%gh0~V4DkDuZ4uSq<7e}?nV@ctR@Us&UWkmfUs=O19Yb^IJ<73#|MGll0Xru#wG8F{&q!{=F&^>&u}eG1RNXjN1AJmWVYs6e_%PDYR~k~S|$7zxo4WQ;BuJpV&JC0}9Pg8yZdtCx+7W(En)C<i4N0_z0I9l}He`3@l>f_}#(iO&%1z)4_`Q`bihpJ(BjN!JuUL(2nc{FcSbOfvsTwwUnGK;c7JhGyZQ;a|_|9H~1p_bdAh^goccBWo9SAP@q<SP%sf>X}*iX%INi!aS296LQN%pP?CYo33XFh*(lme8F6j(S!E=A|}OGELWx(<iACpo}o)3`R+}cE;KWerX$b2Wa&t9g=7XYCV3X)U`x$nol%ai@>AIWxV=&4y4MeqNvN9SS*(P^@jP8fCTB60>@l2?A#jpK|7rUzqW>f>S<cI-OG58N=w{~aKluE7s~?r;ZTsQ#B5rljkIJ)bMX~JWP4Q>Noo)wip_0PyTjU4$EsOjBzZ5JtVoYq#8M|}F_MByZ!mG?;bL`N0yG~fCgq8|h9(G^gJr;NmXr$yt`bv}hV8VZph{01pD<v=DDdeZ{5YS7>i+Bil14yRiMW{Sixeg!0tA=(;#tc@*3|3w?H7Q(-_YD=5yvX5bmG9wC!Usc6B|}doFWa>Jk*hc4>W2#K!9{a$(HvO&fwdo4`>SE?fkb_g@BW}f{T6KR&eNgt_KW1{5Ucy@JbjVW;n6&OkwYE!*CKiLU7ntoe96-xwU?o_cOdEqqW(fz`l0lCWa&qieq`xKmVRXEM@P)j5p#6J933%7{{C(9cRKW4|Ck*yy;G?K{dY;-*C+7H(551fn}#n~<<8;sASS<0-<w-uj{N<|-;ezL$ls6r{Z5-{!CDh-rcHdRPn&5O21A=^0jOVSGeO;|)Mi>_>7ujdqRq6(rn=K+T96OiX)`Ty%d@O<`p{-tWcQ)va*>?jqRq6(16;J3mQgO9Ked^bQ9iPMYBMd<`Cgl8!65`4FUv$n5`QG|uav|eYOF^Re<blo5`QG|M-qP|@dpThfbdZeJ};6m-zD(;J?y&e6foT+@cd8g&q*#>)dW5-vfGtr_XXmz*Jk$x6Jv|q#U%d}?X~eHf#-kXlp>8;',
    b"a-b14W;x8r@e93Ki?|5!5u(bxOqXVN(Q!-XErI8ME}sc}UL-p&6L@wjG7N-E;8>O|ZI`d|O#;vV9E=2h&2q_jB=GSB{?J`L68Iy5KN9#Ofj<)XBZ2>A6L{p5gq-rW0e+vqBipvx-;pivQ+FiCm*#gumP!~&CZwt)sp>551iKPb2sDcu!HZiq7X$|aM<gjJ1`P;<286tnBrnAzFk!NQkeAXTGIH2X7!;sIhC=yW0>2jc8h!t?$UMEwg~Iw<&brE1LVznj#jnIG#@s8RpPrV@pM7+~(<};jc<o3~39a;m1XU(h7rf@JlOFH-b7FN_FDU#r@p_q9rCA&<o>S6V8FZD5789$4#Fb>Ib%n(+)XHDUf3Qu@@-g`-zwPY{d|>muaJ}2LaJIM>myt}dt}=-_%XM(WNYg+5vu+cs{2%-YQOHJsS!F_f`vL9@vk|xiNj$`?p<@jRYyL5Q10u&t;`!nFw;5IZ8mx&`u7jCZ$qwa^Bq1S7$}YKzM2x&;@cf@6gKx;-4_VbCgKx{=alx}Pc&h`R|I++**}KRzv0KWTPSa}XH04$P(IV6oN!;&rnnL2Lm&9F?7~KDg`-p0*Uf5)+;;Ki=ex&S2%6?$(2j+hC%p5&4FO#qz>Z(V=en9RA<bFWzNB(}~@4s~Zo`IjXK<@m3<(j)^z^fVXYF;I47bk)Pzx<GHXlDd;736&uR6s5*k@qaP>@1lBHt4#2cG{5l|G&NS>22c%qWE|5*@xkdwY%(1H5Awg92v5ENPt480=*?j+Ed~0UjE+Ns#YRbELyZ^!9xT&;&5j<{PXz8nQ_I5jnfa~5%cb_whm$IB5Yya9oE*x+7fc-59*yTI~}2QM7_H%Wkglt{Y1Sx3z|n^6OR-1?yN|z)#yOIJL{51RXU^IqaM-g<26{MM?F!v`Hgx{|JR>V@9F=BSL%J3coS4%?uUBsh$s0<z2E3hB&4ziFxW`cyG{%g+*OJ7G_qF;>xwFR0a5RaH#qt9Exu9jy(Vqy><`p?uMf=zv_-Gzu!wQ*CjzlvsrU2;PgQ*80{NbPMIG+>#K9->J$*x#c`Odc<a_#4zoMAV9ste_%hzK0mRhx8`G(~imTy?TVflvT8yDZW_{POIF8;rR-&<PM{~dmZ-GJOU3nIy89Q+Q|!5u?l8571B{0`Mo!tXF3;CJE}6S|nohcF<-CngLr^asob#0R{`Fg*-NhjcO~qzB=9iQda)LkJH-_Y%1mwga+*u)PHAh3Oz%FVT9TIK*&7{bwqe3=FeDU|x6)Xbo5m5@*0^3^H^ohpCgTp)+7J$cq7&K|%~%gaYU+0M7#WECA5b2aT$^gWV3PcLEqK0MY^|EdbL3I4uCw0!S?Y)zSwN=~YODF=`;DcOnNyAR7-i2yqG`0S5UmNPj{03zA=u`-0RLB)uT#1t~AcctOGo@?DVWf<zbOxgf&@2`<QQL3)c(OCsOt&Z%rNBN3R1kW74IA{!IQm>9-{FeU&6p(h9-L7)i2LJ$Ok7!ZVhAl3uX9f<2dL<eFy5W#`C4Gpz7)ZS2gL+uT<H`Lxx`@f3XTQb##+8b(bsJ#NUOaIkS1=>fQ*3Lour4*`!ZX;qFLE4DVMpOr)IN)%Hza1`dIK#;oj$>Zj-f(!s-wh`>eB5ws!><izHXOC^Uc-3}w>2Uy;IxLj8s2K727M}*UIec+4#jaX!=(&oGTg}Ubi+Lj$25G#@DIa13{Ntg!|)BmdyJFm$tO%lQ!Jx#>bQ)BUO&(j(p^W+ETfz(chku-%P5D+-E^qTGRjeM7v*inG8(~oQD;?3Ov@4))2ndy%nbrA5O76+8v-1<aX!Y;*mSa$-j_oy&a^nv;slFxe$MrEjvq&T@I={FhLvSyTG>{{jeiZcpQcYu74p97E6lMg{-t(K9f3iQooP~9lm?|eX--;`#-uH$rkr{if2ESCA!$dNkyfM;X~R??=n?5ls+Vn!sW+-Ssz0hjsz)lTFaCNK)OqQ>Y$J_*q_L4Sc9O;o5+kLNv6nP9lg4h+I`wxXnAuIsgw_cy6j~`*DyTkCeW3b4^?~XG)d#8%R3BV#$O1w2f$GC+^^}`_Kq1x+V*#*kocvV}(7opJOdhxAJXXVvhP58WLeKNkHy-c$KmExoGTR!ntufo$jaE(+X0|nETVu90W?N&nHD+65wl!v3W45)Go}pi9bE(NXWh_)i3>LI@vPh*5tR$@fAZX=i<!I$-<!I$-<!I$-<!I$-<!I$-<!I$-<!I%TRyb@K9$PkQDx(5+z>|_(V40L8R;8)O5oU8zl4zC2{@u8zV%$@)UXvjb+*2{)(-`+tj6J+@4aGRl7SIsV2<o3~>Wy8!v8}g3(<v1U`jraQ@41JA&Ti`T(wT5GM$8x+H3K8TMI0l9jIqHtcK8Mk6#x+e32I$J$Qbu>j4i(r6UU;`V5bT|Ne)bMaFPR*9HitxCFd2KS8%-oaLK_-9z@iLsS#C^#y~^z)kwajT5U+aA^C>nYxwu^&(rVT3v|{&d1p>%ax8K04qnPo@6M&mzngidH;8x#LDLs55%11C-OzkP^9{{6G~dvCL-YSEnt!>z`u6kZFW(ZN|LySY>ihNk&F<sX_1l*O`oDhu^_QQnzDu@wxz3i{{qW)TZvXVSeV!HfeDQbd-NR=0w0~Id-f!j-H+hNs+s*!Vw^={zpYFDgGl^AR;&Jo5zg@rE%*ytKviF}pt<P&7@}-|1w%g~s`^~w)m>2lC-hJ3VJ#Tk&W#YW_Q4?mHezjD*uKi~+Wg`Sz5UaPuOe|}Tuq04rlbJ}_FiV<MS?8p&as~?Ff<)d%p<5QK9xG>(3ziy|C3D6KenBR0uhp_tt;x;=^HvM<COK(=ieYswQ5abl#1^b}L9%*$ok?bg*d>{~)y8G1TC1H27OXaxI=n~LUS2F)7+-x!U||9rN;1{fIg-harm<MMX5Kq0Uu%mak%ATGGv1LNIXiT=P`YZ&%tXqz=t5blY>lofOI3`MdHJkGIxkBUZPMM8h4NPE=8Da0u9(oy1oE~yY2{O?>gnuMuzI|l3RW2DOr&N7Iu$AppJ7q1V6FLNb5tmc7~(?df}Iu~urP*(ved#DT6k8@Jd5S?V`wZ(l*iD*V~S(w$Cuv$UEG@c')))\n_ROUTES={int(k):[_R108_DATA['actions'][i] for i in ids] for k,ids in _R108_DATA['routes'].items()}\n_R108_SHOP_ROUTES={tuple(r['shops']):r['route'] for r in _R108_DATA['shops']}\ndel _R108_DATA\n_SETTINGS={'hand_align': True, 'weed_repair': True, 'sell_lead': True, 'budget_guard': False, 'room_guard': False, 'clamp_sells': False, 'dead_stock': False, 'terminal_liquidation': False, 'front_run': False}\n\n_R110_OLD_SHOPS={('BAKERY', 'BAKERY'): 0, ('BAKERY', 'BRUNCH_SPOT'): 0, ('BAKERY', 'FARMERS_MARKET'): 0, ('BAKERY', 'ICE_CREAM_SHOP'): 0, ('BAKERY', 'PET_CAFE'): 0, ('BAKERY', 'PIZZA_SHOP'): 0, ('BAKERY', 'SMOOTHIE_SHOP'): 0, ('BAKERY', 'YARN_STORE'): 3, ('BRUNCH_SPOT', 'BAKERY'): 0, ('BRUNCH_SPOT', 'BRUNCH_SPOT'): 0, ('BRUNCH_SPOT', 'FARMERS_MARKET'): 0, ('BRUNCH_SPOT', 'ICE_CREAM_SHOP'): 0, ('BRUNCH_SPOT', 'PET_CAFE'): 0, ('BRUNCH_SPOT', 'PIZZA_SHOP'): 0, ('BRUNCH_SPOT', 'SMOOTHIE_SHOP'): 0, ('BRUNCH_SPOT', 'YARN_STORE'): 4, ('FARMERS_MARKET', 'BAKERY'): 0, ('FARMERS_MARKET', 'BRUNCH_SPOT'): 0, ('FARMERS_MARKET', 'FARMERS_MA",
    b'RKET\'): 0, (\'FARMERS_MARKET\', \'ICE_CREAM_SHOP\'): 0, (\'FARMERS_MARKET\', \'PET_CAFE\'): 0, (\'FARMERS_MARKET\', \'PIZZA_SHOP\'): 0, (\'FARMERS_MARKET\', \'SMOOTHIE_SHOP\'): 0, (\'FARMERS_MARKET\', \'YARN_STORE\'): 5, (\'ICE_CREAM_SHOP\', \'BAKERY\'): 0, (\'ICE_CREAM_SHOP\', \'BRUNCH_SPOT\'): 0, (\'ICE_CREAM_SHOP\', \'FARMERS_MARKET\'): 0, (\'ICE_CREAM_SHOP\', \'ICE_CREAM_SHOP\'): 0, (\'ICE_CREAM_SHOP\', \'PET_CAFE\'): 0, (\'ICE_CREAM_SHOP\', \'PIZZA_SHOP\'): 0, (\'ICE_CREAM_SHOP\', \'SMOOTHIE_SHOP\'): 0, (\'ICE_CREAM_SHOP\', \'YARN_STORE\'): 6, (\'PET_CAFE\', \'BAKERY\'): 0, (\'PET_CAFE\', \'BRUNCH_SPOT\'): 0, (\'PET_CAFE\', \'FARMERS_MARKET\'): 0, (\'PET_CAFE\', \'ICE_CREAM_SHOP\'): 0, (\'PET_CAFE\', \'PET_CAFE\'): 0, (\'PET_CAFE\', \'PIZZA_SHOP\'): 0, (\'PET_CAFE\', \'SMOOTHIE_SHOP\'): 0, (\'PET_CAFE\', \'YARN_STORE\'): 5, (\'PIZZA_SHOP\', \'BAKERY\'): 0, (\'PIZZA_SHOP\', \'BRUNCH_SPOT\'): 0, (\'PIZZA_SHOP\', \'FARMERS_MARKET\'): 0, (\'PIZZA_SHOP\', \'ICE_CREAM_SHOP\'): 0, (\'PIZZA_SHOP\', \'PET_CAFE\'): 0, (\'PIZZA_SHOP\', \'PIZZA_SHOP\'): 0, (\'PIZZA_SHOP\', \'SMOOTHIE_SHOP\'): 0, (\'PIZZA_SHOP\', \'YARN_STORE\'): 7, (\'SMOOTHIE_SHOP\', \'BAKERY\'): 0, (\'SMOOTHIE_SHOP\', \'BRUNCH_SPOT\'): 0, (\'SMOOTHIE_SHOP\', \'FARMERS_MARKET\'): 0, (\'SMOOTHIE_SHOP\', \'ICE_CREAM_SHOP\'): 0, (\'SMOOTHIE_SHOP\', \'PET_CAFE\'): 0, (\'SMOOTHIE_SHOP\', \'PIZZA_SHOP\'): 0, (\'SMOOTHIE_SHOP\', \'SMOOTHIE_SHOP\'): 0, (\'SMOOTHIE_SHOP\', \'YARN_STORE\'): 8, (\'YARN_STORE\', \'BAKERY\'): 9, (\'YARN_STORE\', \'BRUNCH_SPOT\'): 9, (\'YARN_STORE\', \'FARMERS_MARKET\'): 1, (\'YARN_STORE\', \'ICE_CREAM_SHOP\'): 9, (\'YARN_STORE\', \'PET_CAFE\'): 10, (\'YARN_STORE\', \'PIZZA_SHOP\'): 6, (\'YARN_STORE\', \'SMOOTHIE_SHOP\'): 11, (\'YARN_STORE\', \'YARN_STORE\'): 12}\n\ndef _router(observation,step,state):\n    if step>=144 and not state.get(\'day6\'):\n        shops=tuple((_get(_get(observation,\'town\',{}),\'unlocked_shops\',[]) or [])[:2])\n        use_new=shops.count(\'YARN_STORE\')<=0\n        state[\'expert\']=\'EXP240\' if use_new else \'V39\'\n        state[\'route\']=_R108_SHOP_ROUTES.get(shops,100) if use_new else _R110_OLD_SHOPS.get(shops,0)\n        state[\'day6\']=True\n    if step>=648 and not state.get(\'day27\'):\n        state[\'route\']=2\n        state[\'day27\']=True\n    return state.get(\'route\',0)\n\n_R42_OPENING=[[\'BUY_PRODUCT\', \'WHEAT\', 5], [\'BUY_PRODUCT\', \'WHEAT\', 10], [\'SELL\', \'WHEAT\', 60]]\nfor _r42_tape in _ROUTES.values():\n    _r42_tape[0]=dict(_r42_tape[0],market=[list(o) for o in _R42_OPENING])\ndel _r42_tape\n_IMPL=make_agent(_ROUTES,router=_router,**_SETTINGS)\n_IMPL.chassis.diagnostics[\'terminal_rescue_errors\']=0\n\ndef agent(observation,configuration=None):\n    try:\n        action=_IMPL(observation,configuration)\n        pass\n        return action\n    except Exception:\n        return {\'farmer\':[\'PASS\'],\'hands\':[],\'market\':[]}\n\n_SHOP_PARENT=agent\ndel agent\n\ndef agent(observation,configuration=None):\n    action=_SHOP_PARENT(observation,configuration)\n    try:\n        if _step_of(observation)>=718:\n            view=_View(observation,_int(_get(observation,\'player\',0)),_IMPL.chassis.cfg)\n            units=[]\n            for i,pos in enumerate(view.positions):\n                units.append([\'DROP\'] if _shed_adjacent(pos,view.board) and view.inv(i) else [\'PASS\'])\n            action={\'farmer\':units[0],\'hands\':units[1:],\'market\':[]}\n            projected=_IMPL.chassis._projected_shed(action,view)\n            action[\'market\']=[[\'SELL\',item,projected.get(item,0)] for item in PRODUCTS if projected.get(item,0)>0]\n            action[\'market\'].sort(key=lambda o:-view.prices.get(o[1],0)*o[2])\n    except Exception:\n        _IMPL.chassis.diagnostics[\'terminal_rescue_errors\'] += 1\n    return action\n\n# EXP-154 modifications: Ahmed Berat Ozer; public capabilities credited below.\n# Dmitrii Gluzdov Seven Turn Rescue and Kaggle engine contributors, Apache-2.0.\n\n_UNIT_NS={"__name__":"v28_own_unit_model"}\nexec(\'# SPDX-License-Identifier: Apache-2.0\\n# Extracted Kaggle / kaggle-environments contributor code; see NOTICE.txt.\\n"""Exact deterministic unit/decay semantics extracted from kaggle-environments 1.32.7.\\nSource kaggriculture.py SHA256 bc8a54879ef02c7ea64b8b333d6a976f0ea65c4949149d01f463f23bccee653e.\\nNo interpreter, market RNG, policy controls, or re',
    b'play content is included.\\n"""\\n\\nENGINE_VERSION = "1.32.7"\\nSOURCE_SHA256 = "bc8a54879ef02c7ea64b8b333d6a976f0ea65c4949149d01f463f23bccee653e"\\n\\nCROPS = {\\n    "WHEAT":      {"seed": 10, "first_yield_day": 2, "max_yield_day": 4, "interval": 0, "max_yield": 6, "ongoing": False},\\n    "CARROT":     {"seed": 20, "first_yield_day": 2, "max_yield_day": 3, "interval": 0, "max_yield": 4, "ongoing": False},\\n    "TOMATO":     {"seed": 50, "first_yield_day": 8, "max_yield_day": 8, "interval": 1, "max_yield": 4, "ongoing": True},\\n    "STRAWBERRY": {"seed": 100, "first_yield_day": 10, "max_yield_day": 10, "interval": 2, "max_yield": 4, "ongoing": True},\\n    "MELON":      {"seed": 80, "first_yield_day": 10, "max_yield_day": 12, "interval": 0, "max_yield": 6, "ongoing": False},\\n}\\n\\nANIMALS = {\\n    "GOOSE": {"cost": 300, "structure": "COOP",    "first_yield_day": 4, "interval": 1, "max_held": 4, "product": "EGG"},\\n    "COW":   {"cost": 400, "structure": "PASTURE", "first_yield_day": 8, "interval": 2, "max_held": 6, "product": "MILK"},\\n    "SHEEP": {"cost": 500, "structure": "PASTURE", "first_yield_day": 6, "interval": 3, "max_held": 6, "product": "WOOL"},\\n}\\n\\nPRODUCTS = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL", "FERTILIZER"]\\n\\nFARMER_MOVES = {\\n    "NORTH": (0, -1),\\n    "SOUTH": (0, 1),\\n    "EAST":  (1, 0),\\n    "WEST":  (-1, 0),\\n}\\n\\ndef _shed_access_tiles(board_size):\\n    """Four inner-corner tiles around the shed, in NWSE order."""\\n    half = board_size // 2\\n    return [(half - 1, half - 1), (half, half - 1), (half - 1, half), (half, half)]\\n\\ndef _is_shed_adjacent(pos, board_size):\\n    return tuple(pos) in {(x, y) for (x, y) in _shed_access_tiles(board_size)}\\n\\ndef _new_plant(crop, day, turns_per_day):\\n    cd = CROPS[crop]\\n    return {\\n        "kind": "PLANT",\\n        "crop": crop,\\n        "planted_day": day,\\n        "watered_today": False,\\n        "consecutive_unwatered": 1,  # planting day counts as unwatered\\n        "yield_units": 0 if cd["ongoing"] else 1,\\n        "max_lifespan_step": (-1 if cd["ongoing"] else (day + cd["max_yield_day"] + 1) * turns_per_day),\\n        "fertilized_until_day": -1,\\n    }\\n\\ndef _new_animal(animal, day):\\n    a = ANIMALS[animal]\\n    return {\\n        "kind": a["structure"],\\n        "animal": animal,\\n        "placed_day": day,\\n        "yield_units": 0,\\n        "consecutive_unfed": 0,\\n        "fed_today": False,\\n        "cared_today": False,\\n        "fertilizer_available": False,\\n        "pending_care_bonus": 0,\\n    }\\n\\ndef _farmer_position(farm, idx):\\n    """idx 0 = main farmer, 1+ = hand index."""\\n    if idx == 0:\\n        return farm["farmer"]\\n    return farm["hands"][idx - 1] if idx - 1 < len(farm["hands"]) else None\\n\\ndef _set_farmer_position(farm, idx, pos):\\n    if idx == 0:\\n        farm["farmer"] = list(pos)\\n    else:\\n        farm["hands"][idx - 1] = list(pos)\\n\\ndef _farmer_inventory(private, idx):\\n    """Inventories list is [main_farmer, *hands]; grow it if idx is past the end."""\\n    while len(private["inventories"]) <= idx:\\n        private["inventories"].append({})\\n    return private["inventories"][idx]\\n\\ndef _inv_add(inv, item, n=1):\\n    inv[item] = inv.get(item, 0) + n\\n\\ndef _inv_take(inv, item, n=1):\\n    if inv.get(item, 0) < n:\\n        return False\\n    inv[item] -= n\\n    if inv[item] == 0:\\n        del inv[item]\\n    return True\\n\\ndef _apply_unit_action(farm, private, idx, action, board_size, day, turns_per_day, shed_capacity=100):\\n    """Process one farmer/hand\\\'s action. Invalid / illegal actions are silent no-ops."""\\n    if not isinstance(action, list) or not action:\\n        return\\n    op = action[0]\\n    pos = _farmer_position(farm, idx)\\n    if pos is None:\\n        return\\n    fx, fy = pos[0], pos[1]\\n    inv = _farmer_inventory(private, idx)\\n\\n    if op in FARMER_MOVES:\\n        dx, dy = FARMER_MOVES[op]\\n        nx, ny = fx + dx, fy + dy\\n        if not (0 <= nx < board_size and 0 <= ny < board_size):\\n            return\\n        # Movement onto LOCKED tiles is allowed: a hand c',
    b'an spawn on a locked\\n        # shed-access tile, and blocking movement would strand it there forever.\\n        # Tile operations (PLANT, WATER, etc.) still no-op on LOCKED tiles.\\n        _set_farmer_position(farm, idx, (nx, ny))\\n        return\\n\\n    if op == "PASS":\\n        return\\n\\n    tile = farm["tiles"][fy][fx]\\n\\n    # Shed operations resolve before the LOCKED guard. They use the tile only as\\n    # a standing position -- the shed itself is always owned -- and three of the\\n    # four shed-access tiles start LOCKED, so guarding them first would make the\\n    # shed unreachable from those tiles.\\n    if op == "DROP":\\n        if not _is_shed_adjacent((fx, fy), board_size):\\n            return\\n        shed = private["shed"]\\n        for item, n in list(inv.items()):\\n            if n <= 0:\\n                del inv[item]\\n                continue\\n            room = max(0, shed_capacity - sum(shed.values()))\\n            take = min(n, room)\\n            if take > 0:\\n                shed[item] = shed.get(item, 0) + take\\n            del inv[item]\\n        return\\n\\n    if op == "PICKUP":\\n        if not _is_shed_adjacent((fx, fy), board_size):\\n            return\\n        if len(action) < 2:\\n            return\\n        item = action[1]\\n        n = int(action[2]) if len(action) >= 3 else 1\\n        if n <= 0:\\n            return\\n        # Seeds live in private["seeds"] and are consumed directly by PLANT;\\n        # they never pass through farmer inventory or the shed.\\n        available = private["shed"].get(item, 0)\\n        n = min(n, available)\\n        if n <= 0:\\n            return\\n        private["shed"][item] -= n\\n        _inv_add(inv, item, n)\\n        return\\n\\n    if op == "PLACE":\\n        if len(action) < 2:\\n            return\\n        item = action[1]\\n        # Animal placement: standing on a matching unoccupied structure. A LOCKED\\n        # tile is the string "LOCKED", never a dict, so this branch cannot match\\n        # there and PLACE falls through to the shed path below.\\n        if (\\n            item in ANIMALS\\n            and isinstance(tile, dict)\\n            and tile.get("kind") == ANIMALS[item]["structure"]\\n            and "animal" not in tile\\n        ):\\n            if _inv_take(inv, item, 1):\\n                farm["tiles"][fy][fx] = _new_animal(item, day)\\n            return\\n        # Shed drop: orthogonally adjacent to the shed; obeys shedCapacity.\\n        if _is_shed_adjacent((fx, fy), board_size):\\n            n = int(action[2]) if len(action) >= 3 else 1\\n            if n <= 0:\\n                return\\n            n = min(n, inv.get(item, 0))\\n            if n <= 0:\\n                return\\n            current = sum(private["shed"].values())\\n            room = max(0, shed_capacity - current)\\n            n = min(n, room)\\n            if n <= 0:\\n                return\\n            inv[item] -= n\\n            if inv[item] == 0:\\n                del inv[item]\\n            private["shed"][item] = private["shed"].get(item, 0) + n\\n        return\\n\\n    # Everything below mutates the tile the unit stands on, so it requires that\\n    # tile to be owned.\\n    if tile == "LOCKED":\\n        return\\n\\n    if op == "PLANT":\\n        if len(action) < 2:\\n            return\\n        crop = action[1]\\n        if crop not in CROPS:\\n            return\\n        if tile is not None:\\n            return\\n        if private["seeds"].get(crop, 0) <= 0:\\n            return\\n        private["seeds"][crop] -= 1\\n        farm["tiles"][fy][fx] = _new_plant(crop, day, turns_per_day)\\n        return\\n\\n    if op == "WATER":\\n        if not (isinstance(tile, dict) and tile.get("kind") == "PLANT"):\\n            return\\n        if tile["watered_today"]:\\n            return\\n        tile["watered_today"] = True\\n        crop_data = CROPS[tile["crop"]]\\n        if not crop_data["ongoing"]:\\n            age_days = day - tile["planted_day"]\\n            window_start = (crop_data["max_yield_day"] + 1) // 2\\n            if window_start <= age_days <= crop_data["max_yield_day"]:\\n                bonus = 2 ',
    b'if tile["fertilized_until_day"] >= day else 1\\n                tile["yield_units"] = min(crop_data["max_yield"], tile["yield_units"] + bonus)\\n        return\\n\\n    if op == "HARVEST":\\n        if not isinstance(tile, dict):\\n            return\\n        if tile.get("yield_units", 0) <= 0:\\n            return\\n        if tile.get("kind") == "PLANT":\\n            crop_data = CROPS[tile["crop"]]\\n            if day - tile["planted_day"] < crop_data["first_yield_day"]:\\n                # Ongoing crops only accumulate yield_units after first_yield_day,\\n                # so reaching here with yield_units > 0 indicates a bug.\\n                if crop_data["ongoing"]:\\n                    print(\\n                        f"WARNING: HARVEST on immature ongoing {tile[\\\'crop\\\']} "\\n                        f"(planted day {tile[\\\'planted_day\\\']}, current day {day}, "\\n                        f"first_yield_day {crop_data[\\\'first_yield_day\\\']}, "\\n                        f"yield_units {tile[\\\'yield_units\\\']}); should never happen"\\n                    )\\n                return\\n            units = tile["yield_units"]\\n            tile["yield_units"] = 0\\n            _inv_add(inv, tile["crop"], units)\\n            if not crop_data["ongoing"]:\\n                farm["tiles"][fy][fx] = None\\n        elif "animal" in tile:\\n            units = tile["yield_units"]\\n            tile["yield_units"] = 0\\n            _inv_add(inv, ANIMALS[tile["animal"]]["product"], units)\\n        return\\n\\n    if op == "FERTILIZE":\\n        if not (isinstance(tile, dict) and tile.get("kind") == "PLANT"):\\n            return\\n        if not _inv_take(inv, "FERTILIZER", 1):\\n            return\\n        # Active for `day`, `day+1`, `day+2` (3 days inclusive).\\n        tile["fertilized_until_day"] = max(tile.get("fertilized_until_day", -1), day + 2)\\n        return\\n\\n    if op == "DIG":\\n        if tile is None:\\n            return\\n        # Removes plants, weeds, empty coop/pasture. Does NOT remove a placed animal.\\n        if isinstance(tile, dict) and "animal" in tile:\\n            return\\n        farm["tiles"][fy][fx] = None\\n        return\\n\\n    if op == "BUILD_COOP":\\n        if tile is not None:\\n            return\\n        farm["tiles"][fy][fx] = {"kind": "COOP"}\\n        return\\n\\n    if op == "BUILD_PASTURE":\\n        if tile is not None:\\n            return\\n        farm["tiles"][fy][fx] = {"kind": "PASTURE"}\\n        return\\n\\n    if op == "FEED":\\n        if not (isinstance(tile, dict) and "animal" in tile):\\n            return\\n        if tile["fed_today"]:\\n            return\\n        if not _inv_take(inv, "WHEAT", 1):\\n            return\\n        tile["fed_today"] = True\\n        return\\n\\n    if op == "COLLECT_FERTILIZER":\\n        if not (isinstance(tile, dict) and "animal" in tile):\\n            return\\n        if not tile["fertilizer_available"]:\\n            return\\n        tile["fertilizer_available"] = False\\n        _inv_add(inv, "FERTILIZER", 1)\\n        return\\n\\n    if op == "CARE":\\n        if not (isinstance(tile, dict) and "animal" in tile):\\n            return\\n        if tile["cared_today"]:\\n            return\\n        tile["cared_today"] = True\\n        return\\n\\ndef _decay_plants(farm, step):\\n    board_size = len(farm["tiles"])\\n    for y in range(board_size):\\n        for x in range(board_size):\\n            tile = farm["tiles"][y][x]\\n            if not isinstance(tile, dict) or tile.get("kind") != "PLANT":\\n                continue\\n            mls = tile["max_lifespan_step"]\\n            if mls < 0 or step < mls:\\n                continue\\n            if (step - mls) % 2 != 0:\\n                continue\\n            tile["yield_units"] -= 1\\n            if tile["yield_units"] <= 0:\\n                farm["tiles"][y][x] = {"kind": "WEED"}\\n\\n\',_UNIT_NS)\n_PLANNER_NS=dict(_UNIT_NS)\nexec(\'"""E182 modification: Shop0909 last-seven-turn physical closure planner.\\n\\nNo engine imports, policy tapes, replay fixtures, RNG or remote calls.\\nThe only supported market continuation is SELL; unknown execution abstains.\\n"""\\nfrom copy imp',
    b'ort deepcopy\\nfrom time import perf_counter\\nSTART, FINAL = (712, 718)\\nOPS = set(FARMER_MOVES) | {\\\'PASS\\\', \\\'DROP\\\', \\\'PICKUP\\\', \\\'PLACE\\\', \\\'PLANT\\\', \\\'WATER\\\', \\\'HARVEST\\\', \\\'FERTILIZE\\\', \\\'DIG\\\', \\\'BUILD_COOP\\\', \\\'BUILD_PASTURE\\\', \\\'FEED\\\', \\\'CARE\\\', \\\'COLLECT_FERTILIZER\\\'}\\nITEMS = tuple(PRODUCTS) + tuple(ANIMALS)\\n\\nclass Unsupported(ValueError):\\n    pass\\n\\ndef _get(obj, key, default=None):\\n    return obj.get(key, default) if isinstance(obj, dict) else getattr(obj, key, default)\\n\\ndef _settings(config):\\n    size, turns, last = (_get(config, k, d) for k, d in [(\\\'boardSize\\\', 10), (\\\'turnsPerDay\\\', 24), (\\\'episodeSteps\\\', 720)])\\n    if (size, turns, last) != (10, 24, 720):\\n        raise Unsupported(\\\'requires pinned 10x10/24/720 terminal window\\\')\\n    cap = int(_get(config, \\\'shedCapacity\\\', 100))\\n    orders = min(10, int(_get(config, \\\'maxMarketOrdersPerTurn\\\', 10)))\\n    if cap < 1 or orders < 1:\\n        raise Unsupported(\\\'invalid capacity/order limit\\\')\\n    return (size, turns, cap, orders)\\n\\ndef physical_state(obs):\\n    """Comparable own physical state; market prices and bank are intentionally excluded."""\\n    seat = int(_get(obs, \\\'player\\\', 0))\\n    farm = _get(obs, \\\'farms\\\')[seat]\\n    return ({k: v for k, v in farm.items() if k != \\\'money\\\'}, _get(obs, \\\'private\\\'))\\n\\ndef _commands(action, n):\\n    return [action.get(\\\'farmer\\\', [\\\'PASS\\\']), *action.get(\\\'hands\\\', [])][:n] + [[\\\'PASS\\\'] for _ in range(max(0, n - 1 - len(action.get(\\\'hands\\\', []))))]\\n\\ndef _clone_state(farm, private):\\n    f = dict(farm)\\n    f[\\\'tiles\\\'] = [[dict(tile) if isinstance(tile, dict) else tile for tile in row] for row in farm[\\\'tiles\\\']]\\n    f[\\\'farmer\\\'] = list(farm[\\\'farmer\\\'])\\n    f[\\\'hands\\\'] = [list(pos) for pos in farm[\\\'hands\\\']]\\n    f[\\\'unlocked_quadrants\\\'] = list(farm[\\\'unlocked_quadrants\\\'])\\n    pr = dict(private)\\n    pr[\\\'shed\\\'], pr[\\\'seeds\\\'] = (dict(private[\\\'shed\\\']), dict(private[\\\'seeds\\\']))\\n    pr[\\\'inventories\\\'] = [dict(inv) for inv in private[\\\'inventories\\\']]\\n    return (f, pr)\\n\\ndef _clone_schedule(schedule):\\n    result = []\\n    for action in schedule:\\n        value = dict(action)\\n        if \\\'farmer\\\' in action:\\n            value[\\\'farmer\\\'] = list(action[\\\'farmer\\\'])\\n        for key in [\\\'hands\\\', \\\'market\\\']:\\n            if key in action:\\n                value[key] = [list(command) for command in action[key]]\\n        result.append(value)\\n    return result\\n\\ndef _validate(schedule, n, orders):\\n    for action in schedule:\\n        if not isinstance(action, dict) or set(action) - {\\\'farmer\\\', \\\'hands\\\', \\\'market\\\'}:\\n            raise Unsupported(\\\'unknown action shape\\\')\\n        if not isinstance(action.get(\\\'hands\\\', []), list):\\n            raise Unsupported(\\\'hands must be a list\\\')\\n        for command in [action.get(\\\'farmer\\\', [\\\'PASS\\\']), *action.get(\\\'hands\\\', [])]:\\n            if not isinstance(command, list) or not command or command[0] not in OPS:\\n                raise Unsupported(\\\'unknown/malformed unit operation\\\')\\n            if command[0] in {\\\'PICKUP\\\', \\\'PLACE\\\', \\\'PLANT\\\'}:\\n                if len(command) < 2 or command[1] not in ITEMS:\\n                    raise Unsupported(\\\'unknown unit item\\\')\\n                if len(command) > 2 and (not isinstance(command[2], int)):\\n                    raise Unsupported(\\\'noninteger unit quantity\\\')\\n        market = action.get(\\\'market\\\', [])\\n        if not isinstance(market, list) or len(market) > orders:\\n            raise Unsupported(\\\'market order shape/cap\\\')\\n        for order in market:\\n            if not isinstance(order, list) or len(order) != 3 or order[0] != \\\'SELL\\\' or (order[1] not in PRODUCTS) or (not isinstance(order[2], int)) or (order[2] <= 0):\\n                raise Unsupported(\\\'baseline market must contain positive integer SELL only\\\')\\n\\ndef liquidation(shed, inherited_market, max_orders=10):\\n    """Use actual post-unit stock; retain first parent item ordering, then stable product order."""\\n    items = []\\n    for order in inherited_market:\\n        if order[1] not in items:\\n  ',
    b'          items.append(order[1])\\n    items += [item for item in PRODUCTS if item not in items]\\n    orders = [[\\\'SELL\\\', item, int(shed.get(item, 0))] for item in items if shed.get(item, 0) > 0]\\n    if len(orders) > min(10, max_orders):\\n        raise Unsupported(\\\'actual final stock exceeds order slots\\\')\\n    return orders\\n\\ndef shop_liquidation(farm, private, prices):\\n    """Exact original final worker/drop and market rule, with current stock/prices."""\\n    return liquidate(FarmView({\\\'player\\\': 0, \\\'farms\\\': [farm], \\\'private\\\': private, \\\'market\\\': {\\\'prices\\\': prices}}))\\n\\ndef simulate(obs, config, schedule, *, final_liquidate=False, detailed=False, preserve_final_commands=False):\\n    """Exact own unit/decay and SELL-stock transitions. No claim to simulate shared prices."""\\n    size, turns, cap, order_cap = _settings(config)\\n    step = int(_get(obs, \\\'step\\\', -1))\\n    if step < START or step + len(schedule) - 1 > FINAL or (not schedule):\\n        raise Unsupported(\\\'outside 712..718; no day boundary or terminal auto-drop\\\')\\n    if any(((t + 1) % turns == 0 for t in range(step, step + len(schedule)))):\\n        raise Unsupported(\\\'day boundary\\\')\\n    farm0, private0 = physical_state(obs)\\n    farm, private = _clone_state(farm0, private0)\\n    n = 1 + len(farm[\\\'hands\\\'])\\n    if len(private[\\\'inventories\\\']) != n or n > 32:\\n        raise Unsupported(\\\'invalid/unbounded worker inventory shape\\\')\\n    _validate(schedule, n, order_cap)\\n    deposited = [dict() for _ in range(n)]\\n    sold = {}\\n    snapshots, rows, events = ([], [], [])\\n    executed = _clone_schedule(schedule)\\n    overflow = 0\\n    for offset, action in enumerate(executed):\\n        t = step + offset\\n        if detailed:\\n            snapshots.append(_clone_state(farm, private))\\n        if t == FINAL and (not preserve_final_commands):\\n            action = shop_liquidation(farm, private, _get(obs, \\\'market\\\')[\\\'prices\\\'])\\n            executed[offset] = action\\n        all_commands = [action.get(\\\'farmer\\\', [\\\'PASS\\\']), *action.get(\\\'hands\\\', [])]\\n        demand = {}\\n        for command in all_commands:\\n            if command[0] == \\\'PLANT\\\':\\n                demand[command[1]] = demand.get(command[1], 0) + 1\\n        blocked = {item for item, count in demand.items() if count > private[\\\'seeds\\\'].get(item, 0)}\\n        for actor, command in enumerate(_commands(action, n)):\\n            if command[0] == \\\'PLANT\\\' and command[1] in blocked:\\n                command = [\\\'PASS\\\']\\n            pos = farm[\\\'farmer\\\'] if actor == 0 else farm[\\\'hands\\\'][actor - 1]\\n            xy = tuple(pos)\\n            inv = private[\\\'inventories\\\'][actor]\\n            before_inv = dict(inv) if command[0] in {\\\'DROP\\\', \\\'HARVEST\\\', \\\'COLLECT_FERTILIZER\\\'} else None\\n            before_shed = dict(private[\\\'shed\\\']) if command[0] in {\\\'DROP\\\', \\\'PLACE\\\'} else None\\n            _apply_unit_action(farm, private, actor, command, size, t // turns, turns, cap)\\n            if before_shed is not None:\\n                delta = {item: amount - before_shed.get(item, 0) for item, amount in private[\\\'shed\\\'].items() if amount > before_shed.get(item, 0)}\\n                for item, amount in delta.items():\\n                    deposited[actor][item] = deposited[actor].get(item, 0) + amount\\n                if delta:\\n                    events.append({\\\'offset\\\': offset, \\\'actor\\\': actor, \\\'op\\\': command[0], \\\'xy\\\': xy, \\\'deposited\\\': delta})\\n                if command[0] == \\\'DROP\\\':\\n                    overflow += sum((max(0, amount - inv.get(item, 0) - delta.get(item, 0)) for item, amount in before_inv.items()))\\n            if command[0] in {\\\'HARVEST\\\', \\\'COLLECT_FERTILIZER\\\'}:\\n                delta = {item: amount - before_inv.get(item, 0) for item, amount in inv.items() if amount > before_inv.get(item, 0)}\\n                if delta:\\n                    events.append({\\\'offset\\\': offset, \\\'actor\\\': actor, \\\'op\\\': command[0], \\\'xy\\\': xy, \\\'acquired\\\': delta})\\n        pre_market = dict(private[\\\'shed\\\'])\\n        if t == FINAL and preserve_final_commands and ',
    b'final_liquidate:\\n            action[\\\'market\\\'] = liquidation(pre_market, [], order_cap)\\n            prices = _get(obs, \\\'market\\\')[\\\'prices\\\']\\n            action[\\\'market\\\'].sort(key=lambda order: -int(prices.get(order[1], 0)) * order[2])\\n        for _, item, requested in action.get(\\\'market\\\', []):\\n            quantity = min(requested, private[\\\'shed\\\'].get(item, 0), 99999)\\n            if quantity > 0:\\n                private[\\\'shed\\\'][item] -= quantity\\n                sold[item] = sold.get(item, 0) + quantity\\n        _decay_plants(farm, t)\\n        rows.append({\\\'pre_market_shed\\\': pre_market, \\\'post_market_shed\\\': dict(private[\\\'shed\\\']), \\\'deposited_by_actor\\\': [dict(v) for v in deposited], \\\'sold\\\': dict(sold)})\\n    if detailed:\\n        snapshots.append(_clone_state(farm, private))\\n    return {\\\'rows\\\': rows, \\\'states\\\': snapshots, \\\'events\\\': events, \\\'actions\\\': executed, \\\'overflow_units\\\': overflow, \\\'farm\\\': farm, \\\'private\\\': private, \\\'sold\\\': sold}\\n\\ndef _ge(left, right):\\n    return all((left.get(item, 0) >= value for item, value in right.items()))\\n\\ndef dominates(candidate, baseline):\\n    """Preserve every baseline worker\\\'s actual deposit prefixes and shed availability."""\\n    if candidate[\\\'overflow_units\\\']:\\n        return False\\n    for new, old in zip(candidate[\\\'rows\\\'], baseline[\\\'rows\\\']):\\n        if not _ge(new[\\\'pre_market_shed\\\'], old[\\\'pre_market_shed\\\']):\\n            return False\\n        if not _ge(new[\\\'sold\\\'], old[\\\'sold\\\']):\\n            return False\\n        if any((not _ge(a, b) for a, b in zip(new[\\\'deposited_by_actor\\\'], old[\\\'deposited_by_actor\\\']))):\\n            return False\\n    return True\\n\\ndef _value(run, prices):\\n    shed = run[\\\'private\\\'][\\\'shed\\\']\\n    return sum(((run[\\\'sold\\\'].get(item, 0) + shed.get(item, 0)) * prices[item] for item in PRODUCTS))\\n\\ndef _walk(start, end):\\n    x, y = start\\n    tx, ty = end\\n    return [[\\\'EAST\\\']] * max(0, tx - x) + [[\\\'WEST\\\']] * max(0, x - tx) + [[\\\'SOUTH\\\']] * max(0, ty - y) + [[\\\'NORTH\\\']] * max(0, y - ty)\\n\\ndef _return(pos):\\n    targets = _shed_access_tiles(10)\\n    target = min(targets, key=lambda xy: (abs(pos[0] - xy[0]) + abs(pos[1] - xy[1]), targets.index(xy)))\\n    return _walk(pos, target) + [[\\\'DROP\\\']]\\n\\ndef _proposals(run, actor, prices, max_per_actor):\\n    """One/two resource bundles plus direct carry closure, replacing a baseline suffix."""\\n    owners = {}\\n    for event in run[\\\'events\\\']:\\n        if \\\'acquired\\\' in event:\\n            owners.setdefault((tuple(event[\\\'xy\\\']), event[\\\'op\\\']), set()).add(event[\\\'actor\\\'])\\n    proposals = []\\n    seen = set()\\n    horizon = len(run[\\\'rows\\\'])\\n    for offset in range(horizon):\\n        farm, private = run[\\\'states\\\'][offset]\\n        pos = tuple(farm[\\\'farmer\\\'] if actor == 0 else farm[\\\'hands\\\'][actor - 1])\\n        inventory = private[\\\'inventories\\\'][actor]\\n        carried = sum((prices.get(item, 0) * count for item, count in inventory.items()))\\n        prefix_deposits = run[\\\'rows\\\'][offset - 1][\\\'deposited_by_actor\\\'][actor] if offset else {}\\n        future_deposits = run[\\\'rows\\\'][-1][\\\'deposited_by_actor\\\'][actor]\\n        obligation = sum((prices.get(item, 0) * (count - prefix_deposits.get(item, 0)) for item, count in future_deposits.items()))\\n        bundles = []\\n        for y, row in enumerate(farm[\\\'tiles\\\']):\\n            for x, tile in enumerate(row):\\n                if not isinstance(tile, dict):\\n                    continue\\n                xy, operations, value = ((x, y), [], 0)\\n                if tile.get(\\\'yield_units\\\', 0) > 0:\\n                    item = tile.get(\\\'crop\\\') if tile.get(\\\'kind\\\') == \\\'PLANT\\\' else ANIMALS.get(tile.get(\\\'animal\\\'), {}).get(\\\'product\\\')\\n                    mature = item and (\\\'animal\\\' in tile or (START + offset) // 24 - tile[\\\'planted_day\\\'] >= CROPS[item][\\\'first_yield_day\\\'])\\n                    if mature and (not owners.get((xy, \\\'HARVEST\\\'), set()) - {actor}):\\n                        operations.append([\\\'HARVEST\\\'])\\n                        value += prices[item] * tile[\\\'yield_un',
    b'its\\\']\\n                if tile.get(\\\'fertilizer_available\\\') and \\\'animal\\\' in tile and (not owners.get((xy, \\\'COLLECT_FERTILIZER\\\'), set()) - {actor}):\\n                    operations.append([\\\'COLLECT_FERTILIZER\\\'])\\n                    value += prices[\\\'FERTILIZER\\\']\\n                if operations:\\n                    distance = len(_walk(pos, xy)) + len(operations) + len(_return(xy))\\n                    if distance <= horizon - offset:\\n                        bundles.append((xy, operations, value, distance))\\n        bundles.sort(key=lambda b: (-b[2] / b[3], -b[2], b[0]))\\n        variants = [([], carried)] if carried else []\\n        for xy, ops, value, _ in bundles[:6]:\\n            variants.append(([(xy, ops)], carried + value))\\n        for first in bundles[:3]:\\n            for second in bundles[:3]:\\n                if first[0] != second[0]:\\n                    variants.append(([(first[0], first[1]), (second[0], second[1])], carried + first[2] + second[2]))\\n        for stops, value in variants:\\n            route, cursor = ([], pos)\\n            for xy, ops in stops:\\n                route += _walk(cursor, xy) + ops\\n                cursor = xy\\n            route += _return(cursor)\\n            if len(route) > horizon - offset:\\n                continue\\n            route += [[\\\'PASS\\\']] * (horizon - offset - len(route))\\n            key = (offset, tuple((tuple(c) for c in route)))\\n            if key not in seen:\\n                seen.add(key)\\n                proposals.append((value - obligation, offset, route, len(stops)))\\n    proposals.sort(key=lambda p: (-p[0], p[1], p[2]))\\n    direct = [p for p in proposals if p[3] == 0 and p[0] > 0][:2]\\n    chosen = direct + [p for p in proposals if p not in direct]\\n    return chosen[:max_per_actor]\\n\\ndef plan_terminal(obs, config, baseline_remaining, *, max_simulations=64, passes=1, proposals_per_actor=4):\\n    """At 712 accept seven actions; positive physical delivery is mandatory."""\\n    begun = perf_counter()\\n    fallback = {\\\'accepted\\\': False, \\\'reason\\\': \\\'\\\', \\\'actions\\\': None, \\\'simulations\\\': 0}\\n    try:\\n        if int(_get(obs, \\\'step\\\', -1)) != START or len(baseline_remaining) != FINAL - START + 1:\\n            raise Unsupported(\\\'planning requires step 712 and exactly seven actions through 718\\\')\\n        max_simulations = min(256, max(1, int(max_simulations)))\\n        passes = min(2, max(1, int(passes)))\\n        proposals_per_actor = min(16, max(1, int(proposals_per_actor)))\\n        baseline = simulate(obs, config, baseline_remaining, detailed=True)\\n        prices = {item: max(1, float(_get(obs, \\\'market\\\', {}).get(\\\'prices\\\', {}).get(item, 1))) for item in PRODUCTS}\\n        current, best = (_clone_schedule(baseline_remaining), baseline)\\n        baseline_value = best_value = _value(baseline, prices)\\n        changes, simulations = ([], 0)\\n        n = len(baseline[\\\'private\\\'][\\\'inventories\\\'])\\n        for sweep in range(passes):\\n            improved = False\\n            for actor in range(n):\\n                winner = None\\n                for _, offset, route, bundle_count in _proposals(best, actor, prices, proposals_per_actor):\\n                    if simulations >= max_simulations:\\n                        break\\n                    trial = _clone_schedule(current)\\n                    for i, command in enumerate(route, offset):\\n                        if actor == 0:\\n                            trial[i][\\\'farmer\\\'] = command\\n                        else:\\n                            trial[i].setdefault(\\\'hands\\\', [])\\n                            while len(trial[i][\\\'hands\\\']) < n - 1:\\n                                trial[i][\\\'hands\\\'].append([\\\'PASS\\\'])\\n                            trial[i][\\\'hands\\\'][actor - 1] = command\\n                    evaluated = simulate(obs, config, trial)\\n                    simulations += 1\\n                    score = _value(evaluated, prices)\\n                    if score > best_value and dominates(evaluated, baseline):\\n                        required = {(tuple(e[\\\'xy\\\']), e[\\\'op\\\'], e[\\\'acto',
    b'r\\\']): e[\\\'acquired\\\'] for e in best[\\\'events\\\'] if \\\'acquired\\\' in e and e[\\\'actor\\\'] != actor}\\n                        acquired = {}\\n                        for e in evaluated[\\\'events\\\']:\\n                            if \\\'acquired\\\' in e:\\n                                key = (tuple(e[\\\'xy\\\']), e[\\\'op\\\'], e[\\\'actor\\\'])\\n                                dst = acquired.setdefault(key, {})\\n                                for item, amount in e[\\\'acquired\\\'].items():\\n                                    dst[item] = dst.get(item, 0) + amount\\n                        if all((_ge(acquired.get(k, {}), v) for k, v in required.items())):\\n                            winner, best_value = ((trial, offset, bundle_count), score)\\n                if winner:\\n                    current, offset, bundle_count = winner\\n                    best = simulate(obs, config, current, detailed=True)\\n                    changes.append({\\\'pass\\\': sweep, \\\'actor\\\': actor, \\\'from_step\\\': START + offset, \\\'resource_bundles\\\': bundle_count, \\\'estimated_stock_value\\\': best_value})\\n                    improved = True\\n                if simulations >= max_simulations:\\n                    break\\n            if not improved or simulations >= max_simulations:\\n                break\\n        if not changes or best_value <= baseline_value:\\n            return {**fallback, \\\'reason\\\': \\\'no positive physical delivery gain\\\', \\\'simulations\\\': simulations, \\\'changed_workers\\\': [], \\\'changes\\\': [], \\\'certificate\\\': {\\\'stock_value_gain_at_initial_prices\\\': 0, \\\'sold_unit_delta\\\': dict.fromkeys(PRODUCTS, 0)}, \\\'planning_ms\\\': (perf_counter() - begun) * 1000}\\n        final = simulate(obs, config, current, final_liquidate=True, detailed=True)\\n        physical = simulate(obs, config, current)\\n        if not dominates(physical, baseline):\\n            raise Unsupported(\\\'no zero-overflow dominating continuation\\\')\\n        delta = {item: final[\\\'sold\\\'].get(item, 0) - baseline[\\\'sold\\\'].get(item, 0) for item in PRODUCTS}\\n        deposited_gain = any((final[\\\'rows\\\'][-1][\\\'deposited_by_actor\\\'][actor].get(item, 0) > baseline[\\\'rows\\\'][-1][\\\'deposited_by_actor\\\'][actor].get(item, 0) for actor in range(n) for item in PRODUCTS))\\n        worker_change = any((_commands(new, n) != _commands(old, n) for new, old in zip(final[\\\'actions\\\'], baseline[\\\'actions\\\'])))\\n        accepted = worker_change and deposited_gain and any((v > 0 for v in delta.values())) and all((v >= 0 for v in delta.values()))\\n        plan = {\\\'accepted\\\': accepted, \\\'reason\\\': \\\'joint physical dominance\\\' if accepted else \\\'no improvement\\\', \\\'baseline\\\': _clone_schedule(baseline_remaining), \\\'actions\\\': final[\\\'actions\\\'], \\\'expected_states\\\': final[\\\'states\\\'][:-1], \\\'simulations\\\': simulations, \\\'changes\\\': changes, \\\'abandoned\\\': False, \\\'changed_workers\\\': sorted({c[\\\'actor\\\'] for c in changes}), \\\'certificate\\\': {\\\'baseline_rows\\\': baseline[\\\'rows\\\'], \\\'physical_rows\\\': physical[\\\'rows\\\'], \\\'baseline_overflow\\\': baseline[\\\'overflow_units\\\'], \\\'candidate_overflow\\\': final[\\\'overflow_units\\\'], \\\'sold_unit_delta\\\': delta, \\\'stock_value_gain_at_initial_prices\\\': best_value - baseline_value, \\\'baseline_final_shed\\\': baseline[\\\'private\\\'][\\\'shed\\\'], \\\'final_shed\\\': final[\\\'private\\\'][\\\'shed\\\'], \\\'positive_physical_deposit_gain\\\': deposited_gain, \\\'markets_712_717_unchanged\\\': all((final[\\\'actions\\\'][i].get(\\\'market\\\', []) == baseline_remaining[i].get(\\\'market\\\', []) for i in range(FINAL - START)))}}\\n    except (Unsupported, KeyError, TypeError, ValueError, IndexError) as exc:\\n        plan = {**fallback, \\\'reason\\\': str(exc)}\\n    plan[\\\'planning_ms\\\'] = (perf_counter() - begun) * 1000\\n    return plan\\n\\ndef _effective_action(action, n):\\n    return (_commands(action, n), action.get(\\\'market\\\', []))\\n\\ndef _recover_observed(obs, config, parent_action, plan):\\n    """Bounded cargo salvage after deviation; never resume old positional commands."""\\n    farm, private = physical_state(obs)\\n    positions = [farm[\\\'farmer\\\'], *farm[\\\'hands\\\']]\\n    remaining = FINAL - int(_get(obs, \\\'step\\\')) + 1\\',
    b'n    room = max(0, int(_get(config, \\\'shedCapacity\\\', 100)) - sum(private[\\\'shed\\\'].values()))\\n    commands = []\\n    problems = []\\n    prices = _get(obs, \\\'market\\\', {}).get(\\\'prices\\\', {})\\n    for actor, (pos, inv) in enumerate(zip(positions, private[\\\'inventories\\\'])):\\n        command = [\\\'PASS\\\']\\n        if any((v > 0 for v in inv.values())):\\n            route = _return(pos)\\n            if len(route) > remaining:\\n                problems.append({\\\'actor\\\': actor, \\\'reason\\\': \\\'unreachable cargo\\\'})\\n            elif len(route) > 1:\\n                command = route[0]\\n            elif sum((max(0, q) for q in inv.values())) <= room:\\n                command = [\\\'DROP\\\']\\n                room -= sum((max(0, q) for q in inv.values()))\\n            else:\\n                items = [item for item in PRODUCTS if inv.get(item, 0) > 0]\\n                if room and items:\\n                    item = max(items, key=lambda i: (prices.get(i, 1) * min(inv[i], room), -PRODUCTS.index(i)))\\n                    quantity = min(inv[item], room)\\n                    command = [\\\'PLACE\\\', item, quantity]\\n                    room -= quantity\\n                else:\\n                    problems.append({\\\'actor\\\': actor, \\\'reason\\\': \\\'no shed capacity\\\'})\\n        commands.append(command)\\n    action = {\\\'farmer\\\': commands[0], \\\'hands\\\': commands[1:], \\\'market\\\': deepcopy(parent_action.get(\\\'market\\\', []))}\\n    if int(_get(obs, \\\'step\\\')) == FINAL:\\n        action[\\\'market\\\'] = []\\n        action = simulate(obs, config, [action], final_liquidate=True, preserve_final_commands=True)[\\\'actions\\\'][0]\\n    plan[\\\'recovery_steps\\\'] = plan.get(\\\'recovery_steps\\\', 0) + 1\\n    if problems:\\n        plan.setdefault(\\\'recovery_failures\\\', []).append({\\\'step\\\': int(_get(obs, \\\'step\\\')), \\\'problems\\\': problems})\\n    return action\\n\\ndef terminal_action(obs, config, parent_action, plan):\\n    """Canonical guard, pre-deviation abstention, observed recovery after deviation."""\\n    step = int(_get(obs, \\\'step\\\', -1))\\n    if not plan or not plan.get(\\\'accepted\\\') or (not START <= step <= FINAL):\\n        return parent_action\\n    if plan.get(\\\'abandoned\\\'):\\n        return _recover_observed(obs, config, parent_action, plan) if plan.get(\\\'deviated\\\') else parent_action\\n    index = step - START\\n    n = 1 + len(physical_state(obs)[0][\\\'hands\\\'])\\n    mismatch = physical_state(obs) != plan[\\\'expected_states\\\'][index] or _effective_action(parent_action, n) != _effective_action(plan[\\\'baseline\\\'][index], n)\\n    if mismatch:\\n        plan[\\\'abandoned\\\'] = True\\n        plan[\\\'abandon_step\\\'] = step\\n        plan[\\\'reason\\\'] = \\\'physical observation or effective baseline action diverged\\\'\\n        plan[\\\'safety_failure\\\'] = True\\n        return _recover_observed(obs, config, parent_action, plan) if plan.get(\\\'deviated\\\') else parent_action\\n    result = deepcopy(plan[\\\'actions\\\'][index])\\n    if step == FINAL:\\n        farm, private = physical_state(obs)\\n        result = shop_liquidation(farm, private, _get(obs, \\\'market\\\')[\\\'prices\\\'])\\n    if _commands(result, n) != _commands(parent_action, n):\\n        plan[\\\'deviated\\\'] = True\\n    return result\',_PLANNER_NS)\n# EXP-154 integration by Ahmed Berat Ozer, derived from Dmitrii Gluzdov E182.\n# The preserved v27 parent is simulated on a private shadow only at step 712.\n_PRE_TERMINAL_AGENT=agent\ndel agent\n_TERMINAL_PLANS={}\n_TERMINAL_PREVIOUS={}\n_UPGRADE_STATS={\'planning_calls\':0,\'accepted\':0,\'changed_steps\':0,\'aborted\':0,\'shadow_declines\':0,\'errors\':0,\'max_planning_ms\':0.0}\n\ndef _parent_liquidate(farm, private, prices):\n    # Exactly v27\'s final projected DROP ordering, in the planner\'s private state.\n    view=_View({\'player\':0,\'farms\':[farm],\'private\':private,\'market\':{\'prices\':prices}},0,_IMPL.chassis.cfg)\n    commands=[[\'DROP\'] if _shed_adjacent(pos,view.board) and view.inv(i) else [\'PASS\'] for i,pos in enumerate(view.positions)]\n    action={\'farmer\':commands[0],\'hands\':commands[1:],\'market\':[]}\n    stock=_IMPL.chassis._projected_shed(action,view)\n    action[\'market\']=[[\'SELL\',item,stock.get(item,0)]',
    b" for item in PRODUCTS if stock.get(item,0)>0]\n    action['market'].sort(key=lambda o:-view.prices.get(o[1],0)*o[2])\n    return action\n\n_PLANNER_NS['shop_liquidation']=_parent_liquidate\n\ndef _shadow_terminal(obs,config):\n    seat=int(obs['player']);chassis=_IMPL.chassis\n    state=chassis.players.get(seat)\n    if not state or state.get('last_step')!=711 or state.get('route')!=2:\n        return None\n    # No delayed weed/structure intervention may depend on an unmodeled future.\n    if state.get('pending'):\n        return None\n    shadow=copy.copy(chassis);shadow.players=copy.deepcopy(chassis.players)\n    shadow.diagnostics={k:0 for k in chassis.diagnostics}\n    projected=copy.deepcopy(obs);baseline=[];states=[]\n    for step in range(712,719):\n        projected['step']=step;projected['day']=step//24;projected['hour']=step%24\n        states.append(copy.deepcopy(shadow.players[seat]))\n        action=shadow.act(projected,config)\n        if step==718:\n            action=_parent_liquidate(projected['farms'][seat],projected['private'],projected['market']['prices'])\n        else:\n            market=action.get('market',[])\n            if len(market)!=9 or {o[1] for o in market}!=set(PRODUCTS) or any(o[0]!='SELL' or len(o)!=3 or type(o[2]) is not int or o[2]<100 for o in market):\n                return None\n        if any(shadow.diagnostics.values()):return None\n        run=_PLANNER_NS['simulate'](projected,config,[action])\n        if run['actions'][0]!=action:return None\n        baseline.append(action)\n        run['farm']['money']=projected['farms'][seat]['money']\n        projected['farms'][seat]=run['farm'];projected['private']=run['private']\n    return baseline,states\n\ndef agent(observation,configuration=None):\n    try:\n        step=int(observation['step']);seat=int(observation['player'])\n    except Exception:\n        return _PRE_TERMINAL_AGENT(observation,configuration)\n    previous=_TERMINAL_PREVIOUS.get(seat)\n    if step==0 or (previous is not None and step<=previous):_TERMINAL_PLANS.pop(seat,None)\n    _TERMINAL_PREVIOUS[seat]=step\n    plan=_TERMINAL_PLANS.get(seat)\n    if plan and plan.get('accepted') and 712<=step<=718:\n        if previous!=step-1:plan.update(abandoned=True,reason='nonconsecutive callback')\n        try:\n            result=_PLANNER_NS['terminal_action'](observation,configuration,plan['baseline'][step-712],plan)\n            if plan.get('abandoned'):\n                if not plan.get('abort_counted'):\n                    plan['abort_counted']=True;_UPGRADE_STATS['aborted']+=1\n                if not plan.get('deviated'):\n                    _IMPL.chassis.players[seat]=copy.deepcopy(plan['parent_states_before'][step-712])\n                    _TERMINAL_PLANS.pop(seat,None)\n                    return _PRE_TERMINAL_AGENT(observation,configuration)\n            _UPGRADE_STATS['changed_steps']+=int(result!=plan['baseline'][step-712])\n            return result\n        except Exception:\n            _UPGRADE_STATS['errors']+=1\n            if plan.get('deviated'):\n                try:return _PLANNER_NS['_recover_observed'](observation,configuration,plan['baseline'][step-712],plan)\n                except Exception:return _parent_liquidate(observation['farms'][seat],observation['private'],observation['market']['prices'])\n    if step!=712:return _PRE_TERMINAL_AGENT(observation,configuration)\n    _UPGRADE_STATS['planning_calls']+=1\n    try:shadow=_shadow_terminal(observation,configuration)\n    except (ValueError,KeyError,TypeError,IndexError):shadow=None\n    if shadow is None:\n        _UPGRADE_STATS['shadow_declines']+=1\n        return _PRE_TERMINAL_AGENT(observation,configuration)\n    baseline,states=shadow\n    actual=_PRE_TERMINAL_AGENT(observation,configuration)\n    if actual!=baseline[0]:\n        _UPGRADE_STATS['shadow_declines']+=1;return actual\n    try:\n        plan=_PLANNER_NS['plan_terminal'](observation,configuration,baseline,max_simulations=64,passes=1,proposals_per_actor=4)\n        _UPGRADE_STATS['max_planning_ms']=max(_UPGRADE_STATS['max_planning_ms'],plan.get('planning_ms',0.0))\n        if not plan.get('accepted'):re",
    b"turn actual\n        plan['parent_states_before']=states;_TERMINAL_PLANS[seat]=plan\n        _UPGRADE_STATS['accepted']+=1\n        result=_PLANNER_NS['terminal_action'](observation,configuration,actual,plan)\n        _UPGRADE_STATS['changed_steps']+=int(result!=actual)\n        return result\n    except Exception:\n        _UPGRADE_STATS['errors']+=1;return actual\n\nagent.telemetry=_UPGRADE_STATS\n\n# EXP-154: aurax7 Reactive v2 day-end storage guard, adapted to our v27 view.\n_PRE_ROOM_AGENT=agent\ndel agent\n_ROOM_STATS={'changed_turns':0,'added_units':0,'errors':0}\ndef agent(observation,configuration=None):\n    action=_PRE_ROOM_AGENT(observation,configuration)\n    try:\n        step=_step_of(observation)\n        if step%24!=23:return action\n        view=_View(observation,_int(_get(observation,'player',0)),_IMPL.chassis.cfg)\n        carried=sum(max(0,int(n)) for inv in view.invs for n in inv.values())\n        needed=sum(view.shed.values())+carried-99\n        if needed<=0:return action\n        planned={}\n        for o in action.get('market',[]):\n            if o and o[0]=='SELL' and len(o)>=3:planned[o[1]]=planned.get(o[1],0)+max(0,int(o[2]))\n        result=copy.deepcopy(action);added=0\n        for item in sorted(PRODUCTS,key=lambda it:-int(view.prices.get(it,0))):\n            qty=min(needed,max(0,view.shed.get(item,0)-planned.get(item,0)))\n            if qty<=0:continue\n            if len(result['market'])>=10:break\n            result['market'].append(['SELL',item,qty]);needed-=qty;added+=qty\n            if needed<=0:break\n        if added:_ROOM_STATS['changed_turns']+=1;_ROOM_STATS['added_units']+=added\n        return result\n    except Exception:\n        _ROOM_STATS['errors']+=1;return action\n\nagent.telemetry=_ROOM_STATS\n\n# Incorporated upstream attribution and change notice:\n# E182 Shop0909 + terminal physical closure (modified 2026-09-09)\n# \n# The active public parent is Yusuke Hayashi's yhay81/shop-router-0909 v3.\n# router_parent.py and actions.json are exact original bytes, not newly authored\n# routes. The parent credits aurax7's Reactive Router for sale timing and shed\n# projection; that attribution remains in router_parent.py. Original payload\n# LICENSE.txt is preserved unchanged (Apache License 2.0 text); it contains no\n# named copyright grantor and no separate NOTICE was supplied. No additional\n# ownership, endorsement, or upstream replay-data rights claim is made.\n# \n# Local changes: separate main.py/policy.py adapter; bounded start712 planner\n# copied from frozen E180/S78 and modified for seven callbacks, exact Shop final\n# liquidation, strict positive physical delivery/sale gain, and observation guards.\n# unit_model.py is an unchanged frozen E180 copy of Kaggle's extracted semantics.\n# The following original E180 notice is retained verbatim for attribution history.\n# Its references to Thomas files describe E180, not files supplied in this Shop\n# package: no Thomas tapes, trees or policy are included here.\n# \n# ----- Original E180 notice -----\n# Kaggriculture: Last-Mile Harvest Planner\n# Attribution and change notice\n# \n# Thomas Tschinkel is the author of the parent public state-router policy and its\n# published decision trees and action-route data. Source: Kaggriculture: 93.8% Win\n# Rate Public State Router, notebook version 3, scriptVersionId 347936183:\n# https://www.kaggle.com/code/thomastschinkel/kaggriculture-93-8-win-rate-public-state-router?scriptVersionId=347936183\n# The public notebook identifies its license as Apache License, Version 2.0.\n# Original published main.py SHA-256:\n# b87a27ed614a33329be85f1b662e51cf4078a019fee937afcebbbbf2f51f8522\n# \n# Changes to that source for this distribution: compressed route/tree literals\n# were decoded into readable tapes.json and trees.json; a read-only planned_action\n# helper was added; descriptive headers and local data loading were adapted.\n# The original parent feature extraction, tree traversal and agent behavior are\n# retained. These public routes are not claimed as newly authored or trained by\n# the notebook distributor.\n# \n# unit_model.py contains deterministic unit-a",
    b"ction and crop-decay definitions\n# extracted from Kaggle's kaggle-environments 1.32.7 Kaggriculture engine, licensed\n# under Apache License, Version 2.0. Credit: Kaggle and the kaggle-environments\n# contributors. Project: https://github.com/Kaggle/kaggle-environments\n# Source file: kaggle_environments/envs/kaggriculture/kaggriculture.py\n# Source SHA-256:\n# bc8a54879ef02c7ea64b8b333d6a976f0ea65c4949149d01f463f23bccee653e\n# The extracted unit/decay definitions are not a newly authored game engine;\n# market price dynamics and the full interpreter are not part of this module.\n# \n# Additional work in this distribution: a bounded last-nine-action collection\n# and delivery planner, observation guards and recovery, a settings-consuming\n# factory and entry point, standalone examples, and deterministic packaging.\n# The full Apache License, Version 2.0 is included as LICENSE.txt.\n# No endorsement by Thomas Tschinkel or Kaggle is implied.\n# \n# Data provenance limitation: Thomas's source refers to public replay data and\n# an upstream provenance.json. That original episode-level manifest, replay IDs\n# and individual replay-author identities were not supplied with the public\n# notebook/output used here. No names or episode lineage have been invented.\n# Notebook-level licensing does not independently establish the missing underlying\n# replay-data rights chain. The package supplies usable readable routes, not a\n# reproducible reconstruction of their original collection or training process.\n# \n# Packaging note: source inputs described as byte-exact above are\n# normalized to UTF-8/LF text with a final newline in this standalone\n# notebook package. Route JSON values and parent policy behavior are unchanged.\n\n# Final public-entry guard; measured separately and compared on captured observations.\n_V28_CORE=agent\ndel agent\n_IMPL.chassis.diagnostics['v28_entry_errors']=0\ndef agent(observation,configuration=None):\n    try:\n        return _V28_CORE(observation,configuration)\n    except Exception:\n        _IMPL.chassis.diagnostics['v28_entry_errors']+=1\n        return {'farmer':['PASS'],'hands':[],'market':[]}\nagent.telemetry=_ROOM_STATS\n\n# EXP-155: prvsiyan V221B finite tomato investment, adapted by Ahmed Berat Ozer.\n# Original public source is retained under research24/public; Apache-2.0.\nMAX_ORDERS=10\nclass FarmView(_View):\n    def __init__(self,obs):super().__init__(obs,int(obs['player']),_IMPL.chassis.cfg)\n    def inventory(self,actor):return self.inv(actor)\ndef projected_shed(action,view):return _IMPL.chassis._projected_shed(action,view)\n\nCROP_MIN_PRICE=70\n\n# V219: a finite late tomato investment with dedicated, observed workers.\n_V219_PARENT = agent\ndel agent\n_V219_FERTILIZE = True  # Builder changes only this flag for the ablation.\n_V219_STATES = {}\n_V219_REPORT = {'commitments': 0, 'hire_requests': 0, 'confirmed_workers': 0,\n                'hire_shortfalls': 0, 'plant_requests': 0, 'confirmed_plants': 0,\n                'water_requests': 0, 'fertilize_requests': 0, 'harvest_requests': 0,\n                'confirmed_harvest_units': 0, 'drop_requests': 0,\n                'tomato_sale_requests': 0, 'budget_declines': 0, 'lost_plants': 0}\n\n\ndef _v219_fib(n):\n    a, b = 1, 1\n    for _ in range(n): a, b = b, a+b\n    return a\n\n\ndef _v219_native_day(native, day):\n    tape = _IMPL.chassis.routes[native['route']]\n    return tape[day*24:min((day+1)*24,719)]\n\n\ndef _v219_qualifies(obs, native):\n    farm=obs['farms'][obs['player']]\n    if len(farm['tiles']) != 10 or set(farm['unlocked_quadrants']) != {'NW','NE','SW'}:\n        return False\n    if farm['money'] < 12000 or obs['market']['prices']['TOMATO'] < CROP_MIN_PRICE:\n        return False\n    if sum(s in ('PIZZA_SHOP','FARMERS_MARKET') for s in obs['town']['unlocked_shops']) < 3:\n        return False\n    if any(farm['tiles'][y][x] != 'LOCKED' for y in (5,6) for x in range(5,10)):\n        return False\n    if obs['private']['seeds'].get('TOMATO',0) or obs['private']['shed'].get('TOMATO',0):\n        return False\n    if any(isinstance(t,dict) and t.get('crop')=='TOMATO' for row in farm['tiles'] for t in ",
    b"row):\n        return False\n    # The investment uses spare land and new worker indices. Avoid taking over\n    # any native tomato or land purchase obligation on the known own schedule.\n    for tape in _IMPL.chassis.routes.values():\n        for a in tape[432:719]:\n            if any(o and o[0]=='BUY_LAND' for o in a.get('market',[])):return False\n            if any(c==['PLANT','TOMATO'] for c in [a.get('farmer')]+a.get('hands',[])):return False\n    return True\n\n\ndef _v219_walk(pos, target):\n    x,y=pos;tx,ty=target\n    if x != tx:return ['EAST' if x < tx else 'WEST']\n    if y != ty:return ['SOUTH' if y < ty else 'NORTH']\n    return None\n\n\ndef _v219_home(pos):\n    return min(((4,4),(5,4),(4,5),(5,5)),key=lambda p:abs(pos[0]-p[0])+abs(pos[1]-p[1]))\n\n\ndef _v219_request(obs, action, state, native):\n    step=int(obs['step']);day=step//24;offset=step%24\n    farm=obs['farms'][obs['player']];private=obs['private']\n    # If the planting-day transaction could not complete, abandon investment.\n    # Later purchases would miss the finite day26..29 production window.\n    if not state.get('committed') and day!=18:return action\n    if state.get('requested_day')==day:return action\n    planned=_v219_native_day(native,day)\n    # EXP240: committed crops must wait for the native worker indices.\n    # New schedules finish native hiring at hour4 or6. The existing two/three\n    # crop-worker groups can still water/harvest their ten cells by midnight.\n    # Initial investment stays within hour3; ordinary schedules are unchanged.\n    latest_hire=max((i for i,a in enumerate(planned) if any(o and o[0]=='HIRE' for o in a.get('market',[]))),default=-1)\n    deadline=6 if state.get('committed') and 3<latest_hire<=6 else 3\n    if offset>deadline:return action\n    remaining=planned[offset+1:]\n    if any(o and o[0]=='HIRE' for a in remaining for o in a.get('market',[])):\n        return action\n    parent_hires=sum(bool(o) and o[0]=='HIRE' for o in action['market'])\n    expected=max(len(a.get('hands',[])) for a in planned)\n    if len(farm['hands'])+parent_hires != expected:return action\n    fertilizer=bool(_V219_FERTILIZE and day in (24,27) and _r79_tomato_fertilizer_worthwhile(obs,action))\n    # One watering tour: at most 2 entry moves + 9 between tiles + 10 waters.\n    # A hire request by hour2 leaves at least21 callbacks after confirmation.\n    crop_workers=1 if day in (19,20,21,22,23,25) and offset<=2 else (3 if 26<=day<=28 else 2)\n    labor=_r53_labor_assignment(obs,action,fertilizer)\n    if labor is not None:crop_workers=labor['workers']\n    count=crop_workers+int(fertilizer and day==27 and labor is None)\n    extra=[]\n    if not state.get('committed'):\n        extra += [['BUY_LAND'],['BUY_SEED','TOMATO',10]]\n    fertilizer_quantity=_r70_parent_fert_qty(obs,action,planned,offset) if fertilizer else 0\n    if fertilizer:extra.append(['BUY_PRODUCT','FERTILIZER',fertilizer_quantity])\n    extra += [['HIRE'] for _ in range(count)]\n    if len(action['market'])+len(extra)>MAX_ORDERS:return action\n    # No assumed sale proceeds. Reserve 3,000 for parent obligations and price\n    # movement; the qualification separately requires 12,000 initial liquidity.\n    budget=sum(_v219_fib(n) for n in range(farm['hires_today'],farm['hires_today']+parent_hires+count))\n    if not state.get('committed'):budget+=4500\n    if fertilizer:budget+=fertilizer_quantity*(obs['market']['prices']['FERTILIZER']+5)\n    for order in action['market']:\n        if not order:continue\n        if order[0]=='BUY_PRODUCT':budget+=int(order[2])*(int(obs['market']['prices'][order[1]])+10)\n        elif order[0]=='BUY_ANIMAL':budget+=int(order[2])*{'COW':400,'SHEEP':500,'GOOSE':300}[order[1]]\n        elif order[0]=='BUY_SEED':budget+=int(order[2])*{'WHEAT':10,'CARROT':20,'TOMATO':50,'STRAWBERRY':100,'MELON':80}[order[1]]\n    if farm['money']<budget+3000:\n        _V219_REPORT['budget_declines']+=1;return action\n    state['pending']={'step':step,'first_actor':expected+1,'count':count,'crop_workers':crop_workers,'fertilizer':fertilizer,'labor':labor}\n    if labor is not None:\n        _R53_LABOR_REPORT['l",
    b"abor_requests']+=1;_R53_LABOR_REPORT['labor_hires_avoided']+=1;_R53_LABOR_REPORT['labor_day'+str(day)]+=1\n    state['requested_day']=day\n    _V219_REPORT['hire_requests']+=count\n    if not state.get('committed'):\n        state['committed']=True;_V219_REPORT['commitments']+=1\n    changed=copy.deepcopy(action);changed['market']+=extra\n    return changed\n\n\ndef _v219_worker(obs, state, actor, role):\n    day=int(obs['step'])//24;step=int(obs['step']);view=FarmView(obs)\n    pos=tuple(view.positions[actor]);inv=view.inventory(actor)\n    targets=role['targets']\n    # Actual cargo differences, observed on the next callback, verify harvests.\n    previous=state['last_work'].get(actor)\n    if previous and previous['step']==step-1 and previous['command']==['HARVEST']:\n        _V219_REPORT['confirmed_harvest_units']+=max(0,int(inv.get('TOMATO',0))-previous['tomatoes'])\n    if role.get('needs_fertilizer') and not role.get('loaded'):\n        home=_v219_home(pos)\n        walk=_v219_walk(pos,home)\n        if walk:return walk\n        desired=role.get('fertilizer_quantity',10 if role['kind']=='fertilizer' else 5)\n        if inv.get('FERTILIZER',0)>=desired:role['loaded']=True\n        elif role.get('pickup_requested'):\n            # Never spend repeated turns waiting for stock that was not bought.\n            role['loaded']=True;role['fertilizer_available']=int(inv.get('FERTILIZER',0))\n        elif view.shed.get('FERTILIZER',0)>=desired:\n            role['pickup_requested']=True;return ['PICKUP','FERTILIZER',desired]\n        else:role['loaded']=True\n    todo=[]\n    for target in targets:\n        x,y=target;tile=view.tiles[y][x]\n        tomato=isinstance(tile,dict) and tile.get('crop')=='TOMATO'\n        if tomato and target not in state['seen_plants']:\n            state['seen_plants'].add(target);_V219_REPORT['confirmed_plants']+=1\n        if target in state['seen_plants'] and not tomato and target not in state['lost']:\n            state['lost'].add(target);_V219_REPORT['lost_plants']+=1\n        command=None\n        if role['kind']=='fertilizer':\n            if tomato and tile.get('fertilized_until_day',-1)<day+2 and inv.get('FERTILIZER',0)>0:\n                command=['FERTILIZE']\n        elif day==18 and not tomato:\n            if tile is None and obs['private']['seeds'].get('TOMATO',0)>0:command=['PLANT','TOMATO']\n            elif isinstance(tile,dict) and tile.get('kind')=='WEED':command=['DIG']\n        elif tomato:\n            # No later production follows the final day, so watering then would\n            # consume time needed to harvest and deliver the final cargo.\n            if day<29 and not tile.get('watered_today'):command=['WATER']\n            elif role.get('needs_fertilizer') and tile.get('fertilized_until_day',-1)<day+2 and inv.get('FERTILIZER',0)>0:\n                command=['FERTILIZE']\n            elif tile.get('yield_units',0)>0:command=['HARVEST']\n        if command:todo.append((target,command))\n    # Final return has priority once only the exact distance plus DROP remains.\n    home=_v219_home(pos);distance=abs(pos[0]-home[0])+abs(pos[1]-home[1])\n    if step>=718-distance and inv.get('TOMATO',0):\n        return _v219_walk(pos,home) or ['PLACE','TOMATO',int(inv.get('TOMATO',0))]\n    if todo:\n        target,command=min(todo,key=lambda v:(abs(pos[0]-v[0][0])+abs(pos[1]-v[0][1]),targets.index(v[0])))\n        return _v219_walk(pos,target) or command\n    if inv.get('TOMATO',0):return _v219_walk(pos,home) or ['PLACE','TOMATO',int(inv['TOMATO'])]\n    if any(inv.values()):return _v219_walk(pos,home) or ['DROP']\n    return ['PASS']\n\n\ndef agent(observation, configuration=None):\n    action=_V219_PARENT(observation,configuration)\n    step=int(observation['step']);player=int(observation['player']);day=step//24\n    state=_V219_STATES.get(player)\n    if state is None or step<=state['last_step']:\n        state={'last_step':step,'day':-1,'workers':{},'last_work':{},'seen_plants':set(),'lost':set(),\n               'targets':[(x,y) for y in (5,6) for x in range(5,10)]}\n        _V219_STATES[player]=state\n    state['last_step']=step\n    native=_IM",
    b"PL.chassis.players[player]\n    if step==432:state['eligible']=_v219_qualifies(observation,native)\n    if not state.get('eligible') or day<18:return action\n    if state['day']!=day:\n        state['day']=day;state['workers']={};state['last_work']={}\n    farm=observation['farms'][player]\n    pending=state.pop('pending',None)\n    if pending:\n        if len(farm['hands'])+1 >= pending['first_actor']+pending['count'] and 'SE' in farm['unlocked_quadrants']:\n            for index in range(pending['count']):\n                fertilizer_worker=index==pending['crop_workers']\n                if fertilizer_worker:targets=state['targets']\n                elif pending['crop_workers']==1:targets=state['targets']\n                elif pending['crop_workers']==2:targets=state['targets'][index*5:index*5+5]\n                else:targets=[[(5,5),(6,5),(7,5)],[(8,5),(9,5),(9,6),(8,6)],[(5,6),(6,6),(7,6)]][index]\n                state['workers'][pending['first_actor']+index]={'kind':'fertilizer' if fertilizer_worker else 'crop','targets':targets,\n                    'needs_fertilizer':pending['fertilizer'] and (day==24 or fertilizer_worker)}\n                if pending.get('labor') is not None:\n                    role=state['workers'][pending['first_actor']+index]\n                    role['targets']=[tuple(p) for p in pending['labor']['paths'][index]]\n                    role['needs_fertilizer']=pending['labor']['fertilizer'];role['fertilizer_quantity']=len(role['targets'])\n                    if tuple(farm['hands'][pending['first_actor']+index-1])!=tuple(pending['labor']['spawns'][index]):_R53_LABOR_REPORT['labor_spawn_errors']+=1\n                    if index==0:_R53_LABOR_REPORT['labor_confirmed']+=1\n            _V219_REPORT['confirmed_workers']+=pending['count']\n        else:_V219_REPORT['hire_shortfalls']+=pending['count']\n    action=_v219_request(observation,action,state,native)\n    if state['workers']:\n        commands=[action.get('farmer') or ['PASS']]+list(action.get('hands') or [])\n        commands += [['PASS'] for _ in range(len(farm['hands'])+1-len(commands))]\n        for actor,role in state['workers'].items():\n            if actor>=len(commands):continue\n            command=_v219_worker(observation,state,actor,role)\n            commands[actor]=command\n            name={'PLANT':'plant_requests','WATER':'water_requests','FERTILIZE':'fertilize_requests',\n                  'HARVEST':'harvest_requests','DROP':'drop_requests'}.get(command[0])\n            if name:_V219_REPORT[name]+=1\n            state['last_work'][actor]={'step':step,'command':command,'tomatoes':observation['private']['inventories'][actor].get('TOMATO',0)}\n        action=copy.deepcopy(action);action['farmer'],action['hands']=commands[0],commands[1:]\n    if state.get('committed') and len(action['market'])<MAX_ORDERS and not any(o[:2]==['SELL','TOMATO'] for o in action['market']):\n        quantity=projected_shed(action,FarmView(observation)).get('TOMATO',0)\n        if quantity>0:\n            action=copy.deepcopy(action);action['market'].append(['SELL','TOMATO',quantity])\n            _V219_REPORT['tomato_sale_requests']+=quantity\n    return action\n\n\nagent.telemetry=_V219_REPORT\n\n# V221B: labor-only ablation of frozen V219G; not yet publicly scored.\n\n\n# Crop workers own their final routes after commitment. A private parent shadow\n# does not contain these obligations, so terminal rescue must abstain there.\n_ORIGINAL_SHADOW_TERMINAL=_shadow_terminal\ndef _shadow_terminal(obs,config):\n    if _V219_STATES.get(int(obs['player']),{}).get('committed'):return None\n    return _ORIGINAL_SHADOW_TERMINAL(obs,config)\n\nAPPLY_TIMING=False\n\n_EXPERIMENT_PARENT=agent\ndel agent\n_V219_REPORT['extra_fertilizer_days']=0\n_V219_REPORT['reordered_market_turns']=0\n_V219_REPORT['errors']=0\ndef agent(observation,configuration=None):\n    try:\n        action=_EXPERIMENT_PARENT(observation,configuration)\n        if APPLY_TIMING and int(observation['step'])>=144:action=_v224_sales_first(action)\n        return action\n    except Exception:\n        _V219_REPORT['errors']+=1\n        return {'farmer':['PASS'],'hand",
    b's\':[],\'market\':[]}\nagent.telemetry=_V219_REPORT\n\ndef _v224_sales_first(action):\n    original=action.get(\'market\',[])[:MAX_ORDERS]\n    orders=[list(o) for o in original if o and (o[0] in (\'HIRE\',\'BUY_LAND\') or (len(o)>=3 and int(o[2])>0))]\n    for index in range(len(orders)):\n        order=orders[index]\n        if order[0]!=\'SELL\':continue\n        cursor=index\n        while cursor>0:\n            previous=orders[cursor-1]\n            if previous[0]==\'SELL\':break\n            if previous[0] in (\'BUY_PRODUCT\',\'BUY_ANIMAL\') and previous[1]==order[1]:break\n            orders[cursor-1],orders[cursor]=orders[cursor],orders[cursor-1]\n            cursor-=1\n    if orders==original:return action\n    _V219_REPORT[\'reordered_market_turns\']+=1\n    changed=copy.deepcopy(action);changed[\'market\']=orders\n    return changed\n_ORDER_PARENT=agent\ndel agent\n\ndef agent(observation,configuration=None):\n    try:\n        action=_ORDER_PARENT(observation,configuration)\n        if int(observation["step"])>=144:action=_v224_sales_first(action)\n        return action\n    except Exception:\n        _V219_REPORT["errors"]+=1\n        return {"farmer":["PASS"],"hands":[],"market":[]}\nagent.telemetry=_V219_REPORT\n\n_V31_CORE=agent\ndel agent\n_IMPL.chassis.diagnostics[\'production_errors\']=0\n_IMPL.chassis.diagnostics[\'v31_entry_errors\']=0\ndef agent(observation,configuration=None):\n    before=_V219_REPORT[\'errors\']\n    try:\n        action=_V31_CORE(observation,configuration)\n        _IMPL.chassis.diagnostics[\'production_errors\']+=_V219_REPORT[\'errors\']-before\n        return action\n    except Exception:\n        _IMPL.chassis.diagnostics[\'v31_entry_errors\']+=1\n        return {\'farmer\':[\'PASS\'],\'hands\':[],\'market\':[]}\nagent.telemetry=_V219_REPORT\n\n# Apache-2.0; later cattle transfer from prvsiyan, Moon (2026-09-10).\n# Bounded livestock substitution; confirm owned animals before redirecting workers.\n_V231_PARENT=agent\n_V231_CAP=4\n_V231_STATES={}\n_V231_REPORT={}\n\ndef _v231_new_state():\n    return {\'last\':-1,\'confirmed\':0,\'reserved\':0,\'pending_buy\':None,\n            \'carrying\':{},\'pending_places\':[],\'sites\':{},\'milk_credit\':0,\n            \'requested\':0,\'failed_purchase_units\':0,\'picked\':0,\'placed\':0,\n            \'failed_placements\':0,\'extra_milk_harvested\':0,\'extra_milk_sale_requests\':0}\n\ndef _v231_controller(obs,action,state,cap):\n    step=int(obs[\'step\']);seat=int(obs[\'player\']);farm=obs[\'farms\'][seat]\n    private=obs[\'private\'];shed=private[\'shed\'];inventories=private[\'inventories\']\n    positions=[farm[\'farmer\'],*farm[\'hands\']]\n    pending=state[\'pending_buy\']\n    if pending is not None:\n        gained=max(0,int(shed.get(\'COW\',0))-pending[\'before\'])\n        confirmed=min(pending[\'quantity\'],gained)\n        state[\'confirmed\']+=confirmed;state[\'reserved\']+=confirmed\n        state[\'failed_purchase_units\']+=pending[\'quantity\']-confirmed\n        state[\'pending_buy\']=None\n    for pending in state[\'pending_places\']:\n        x,y=pending[\'site\'];tile=farm[\'tiles\'][y][x]\n        if (isinstance(tile,dict) and tile.get(\'animal\')==\'COW\'\n                and tile.get(\'placed_day\')==pending[\'day\']):\n            state[\'sites\'][(x,y)]=pending[\'day\'];state[\'placed\']+=1\n            actor=pending[\'actor\'];state[\'carrying\'][actor]=max(0,state[\'carrying\'].get(actor,0)-1)\n        else:state[\'failed_placements\']+=1\n    state[\'pending_places\']=[]\n    state[\'last\']=step\n    result=copy.deepcopy(action)\n    workers=[result.get(\'farmer\') or [\'PASS\'],*(result.get(\'hands\') or [])]\n    seen_harvest=set();cow_available=int(shed.get(\'COW\',0));occupied=set()\n    for actor,work in enumerate(workers[:len(positions)]):\n        inventory=inventories[actor] if actor<len(inventories) else {}\n        x,y=positions[actor];tile=farm[\'tiles\'][y][x];site=(x,y)\n        if (work==[\'HARVEST\'] and site in state[\'sites\'] and site not in seen_harvest\n                and isinstance(tile,dict) and tile.get(\'animal\')==\'COW\'\n                and tile.get(\'placed_day\')==state[\'sites\'][site]):\n            units=max(0,int(tile.get(\'yield_units\',0)))\n            state[\'milk_credit\']+=units;state[\'extra_milk_harvested\']+=units\n',
    b"            seen_harvest.add(site)\n        if len(work)>=2 and work[:2]==['PICKUP','SHEEP']:\n            quantity=max(0,int(work[2]) if len(work)>2 else 1)\n            center=len(farm['tiles'])//2\n            if (quantity and state['reserved']>=quantity and cow_available>=quantity\n                    and x in (center-1,center) and y in (center-1,center)\n                    and not any(inventory.get(a,0) for a in ('COW','SHEEP','GOOSE'))):\n                work[1]='COW';state['reserved']-=quantity;cow_available-=quantity\n                state['carrying'][actor]=state['carrying'].get(actor,0)+quantity\n                state['picked']+=quantity\n        if (len(work)>=2 and work[:2]==['PLACE','SHEEP']\n                and state['carrying'].get(actor,0)>0 and inventory.get('COW',0)>0\n                and isinstance(tile,dict) and tile.get('kind')=='PASTURE'\n                and 'animal' not in tile and site not in occupied):\n            work[1]='COW'\n            state['pending_places'].append({'actor':actor,'site':site,'day':step//24})\n        if (len(work)>=2 and work[0]=='PLACE' and work[1] in ('COW','SHEEP','GOOSE')\n                and inventory.get(work[1],0)>0):occupied.add(site)\n    result['farmer'],result['hands']=workers[0],workers[1:]\n    market=result.get('market',[])\n    animal_orders=[o for o in market if len(o)>=3 and o[0]=='BUY_ANIMAL']\n    shops=obs['town']['unlocked_shops'];prices=obs['market']['prices']\n    counts={'COW':0,'SHEEP':0}\n    for line in farm['tiles']:\n        for tile in line:\n            if isinstance(tile,dict) and tile.get('animal') in counts:counts[tile['animal']]+=1\n    cargo=sum(int(inv.get(a,0)) for inv in inventories for a in ('COW','SHEEP','GOOSE'))\n    stock_animals=sum(int(shed.get(a,0)) for a in ('COW','SHEEP','GOOSE'))\n    milk_shops=sum(shop in ('PIZZA_SHOP','ICE_CREAM_SHOP','SMOOTHIE_SHOP') for shop in shops)\n    if (216<=step<=227 and len(shops)>=3 and state['confirmed']<cap and not state['reserved']\n            and not any(state['carrying'].values()) and not state['pending_places']\n            and not cargo and not stock_animals and len(animal_orders)==1\n            and animal_orders[0][1]=='SHEEP' and milk_shops>=2 and 'YARN_STORE' not in shops\n            and int(prices.get('MILK',0))>=int(prices.get('WOOL',0))\n            and counts['COW']>=4 and counts['SHEEP']>=2):\n        order=animal_orders[0];quantity=int(order[2])\n        if 1<=quantity<=2 and quantity<=cap-state['confirmed']:\n            order[1]='COW';state['requested']+=quantity\n            state['pending_buy']={'before':int(shed.get('COW',0)),'quantity':quantity}\n    # Sell only additional physically harvested production at an existing sale slot.\n    if state['milk_credit']>0:\n        stock=projected_shed(result,FarmView(obs))\n        total_planned=sum(max(0,int(o[2])) for o in market if len(o)>=3 and o[:2]==['SELL','MILK'])\n        extra=min(state['milk_credit'],max(0,int(stock.get('MILK',0))-total_planned))\n        if extra:\n            for order in market:\n                if len(order)>=3 and order[:2]==['SELL','MILK'] and int(order[2])>0:\n                    order[2]=int(order[2])+extra\n                    state['milk_credit']-=extra;state['extra_milk_sale_requests']+=extra\n                    break\n    result['market']=market\n    return result\n\ndef agent(observation,configuration=None):\n    step=int(observation['step']);seat=int(observation['player'])\n    state=_V231_STATES.get(seat)\n    if state is None or step<=state['last']:\n        state=_V231_STATES[seat]=_v231_new_state()\n    action=_V231_PARENT(observation,configuration)\n    action=_v231_controller(observation,action,state,_V231_CAP)\n    _V231_REPORT.clear();_V231_REPORT.update(_V231_PARENT.telemetry)\n    for name in ('confirmed','reserved','requested','failed_purchase_units','picked','placed',\n                 'failed_placements','extra_milk_harvested','extra_milk_sale_requests','milk_credit'):\n        _V231_REPORT['cattle_'+name]=state[name]\n    _V231_REPORT['cattle_carried_pending']=sum(state['carrying'].values())\n    return action\n\nagent.telemetry=_V231_RE",
    b"PORT\n\n\n# EXP-167, adapted from Dmitrii Gluzdov's Two Coins, One Sheep (Apache-2.0).\n# Reserve only physically available stock after the final parent worker actions.\n_R36_SALE_PARENT=agent\n_R36_NATIVE_LEAD=Chassis._sell_lead\n_R36_NATIVE_SUPPRESS=Chassis._apply_suppression\n_R36_SALE_REPORT={}\n\ndef _r36_native_lead(self,action,view,projected,route,step,next_sup):\n    if step<216 or step>=696:\n        return _R36_NATIVE_LEAD(self,action,view,projected,route,step,next_sup)\n\ndef _r36_suppress(action,state,step):\n    _R36_NATIVE_SUPPRESS(action,state,step)\n    due=state.get('r36_debts',{}).pop(step,{})\n    for order in action.get('market',[]):\n        if len(order)>=3 and order[0]=='SELL':\n            removed=min(max(0,int(order[2])),due.get(order[1],0))\n            order[2]-=removed\n            due[order[1]]=due.get(order[1],0)-removed\n\nChassis._sell_lead=_r36_native_lead\nChassis._apply_suppression=staticmethod(_r36_suppress)\n\ndef _r36_reserve(obs,action):\n    step=int(obs['step'])\n    # The final planner forecasts its own parent, so keep its full window native.\n    if not 216<=step<696:return action\n    native=_IMPL.chassis.players[int(obs['player'])]\n    tape=_IMPL.chassis.routes[native['route']]\n    end=min(695,step+_R37_HORIZONS.get(int(obs['player']),2),(step//72+1)*72-1)\n    if end<=step:return action\n    commands=[action.get('farmer') or ['PASS'],*(action.get('hands') or [])]\n    view=FarmView(obs)\n    # This projection intentionally abstains on ambiguous animal depot returns.\n    if any(len(c)>1 and c[0]=='PLACE' and c[1] in ANIMAL_STRUCTURE\n           and view.inv(i).get(c[1],0)>0 for i,c in enumerate(commands[:len(view.positions)])):\n        return action\n    stock=projected_shed(action,view)\n    market=action.get('market',[])\n    blocked={o[1] for o in market if len(o)>1 and o[0] in ('SELL','BUY_PRODUCT')}\n    blocked.update(c[1] for c in commands if len(c)>1 and c[0]=='PICKUP')\n    blocked.update(c[1] for queue in native['pending'].values() for pos,c in queue\n                   if len(c)>1 and c[0]=='PICKUP')\n    debts=native['sell_state'].setdefault('r36_debts',{})\n    for item in PRODUCTS:\n        if item in ('WHEAT','FERTILIZER') or item in blocked or view.prices.get(item,0)<2:continue\n        available=max(0,int(stock.get(item,0)))\n        if not available or len(market)>=10:continue\n        reservations=[]\n        for due_step in range(step+1,end+1):\n            future=tape[due_step]\n            work=[future.get('farmer') or ['PASS'],*(future.get('hands') or [])]\n            if any(len(c)>1 and c[:2]==['PICKUP',item] for c in work):break\n            if any(len(o)>1 and o[:2]==['BUY_PRODUCT',item] for o in future.get('market',[])):break\n            planned=sum(max(0,int(o[2])) for o in future.get('market',[]) if len(o)>=3 and o[:2]==['SELL',item])\n            amount=min(available,max(0,planned-debts.get(due_step,{}).get(item,0)))\n            if amount:\n                reservations.append((due_step,amount));available-=amount\n            if not available:break\n        qty=sum(q for _,q in reservations)\n        if qty:\n            market.append(['SELL',item,qty])\n            for due,q in reservations:\n                debt=debts.setdefault(due,{})\n                debt[item]=debt.get(item,0)+q\n            _R36_SALE_REPORT['sale_reserved_units']+=qty\n            _R36_SALE_REPORT['sale_reservations']+=1\n    return action\n\ndef agent(observation,configuration=None):\n    if int(observation.get('step',0))==0:\n        _R36_SALE_REPORT.update(sale_reserved_units=0,sale_reservations=0,sale_errors=0)\n    action=_R36_SALE_PARENT(observation,configuration)\n    try:\n        if configuration is None or all(configuration.get(k,v)==v for k,v in\n            [('boardSize',10),('turnsPerDay',24),('shedCapacity',100),('maxMarketOrdersPerTurn',10)]):\n            action=_r36_reserve(observation,action)\n            if int(observation['step'])>=288:action=_v224_sales_first(action)\n    except Exception:\n        _R36_SALE_REPORT['sale_errors']=_R36_SALE_REPORT.get('sale_errors',0)+1\n    _R36_SALE_REPORT.update(_R36_SALE_PARENT.telemetry)\n   ",
    b' return action\n\nagent.telemetry=_R36_SALE_REPORT\n\n# Ensure the Kaggle-selected final callable is the exported policy.\nagent = globals().pop("agent")\n\n\n# Public capability transfer: lucifer19; Flexon is the same Two Coins asset set.\n# Apache-2.0; exact functions from Kaggle kaggle-environments 1.32.7.\n# https://github.com/Kaggle/kaggle-environments/tree/master/kaggle_environments/envs/kaggriculture\nimport math\n_R37_MARKET_PARAMS = {\'WHEAT\': {\'base\': 25, \'I0\': 10000, \'T\': 400, \'below_func\': \'sqrt\', \'below_target\': 0.8, \'above_func\': \'log\', \'above_target\': 0.2}, \'CARROT\': {\'base\': 35, \'I0\': 10000, \'T\': 450, \'below_func\': \'hinge\', \'below_target\': 1.0, \'above_func\': \'sqrt\', \'above_target\': 0.7}, \'TOMATO\': {\'base\': 60, \'I0\': 10000, \'T\': 200, \'below_func\': \'hinge\', \'below_target\': 0.4, \'above_func\': \'sqrt\', \'above_target\': 0.6}, \'STRAWBERRY\': {\'base\': 120, \'I0\': 10000, \'T\': 100, \'below_func\': \'sqrt\', \'below_target\': 0.7, \'above_func\': \'linear\', \'above_target\': 1.6}, \'MELON\': {\'base\': 250, \'I0\': 10000, \'T\': 300, \'below_func\': \'log\', \'below_target\': 0.2, \'above_func\': \'sq\', \'above_target\': 3.6}, \'EGG\': {\'base\': 50, \'I0\': 10000, \'T\': 332, \'below_func\': \'hinge\', \'below_target\': 0.4, \'above_func\': \'log\', \'above_target\': 0.2}, \'MILK\': {\'base\': 160, \'I0\': 10000, \'T\': 122, \'below_func\': \'sqrt\', \'below_target\': 0.6, \'above_func\': \'linear\', \'above_target\': 1.6}, \'WOOL\': {\'base\': 200, \'I0\': 10000, \'T\': 105, \'below_func\': \'log\', \'below_target\': 0.2, \'above_func\': \'sq\', \'above_target\': 3.2}, \'FERTILIZER\': {\'base\': 100, \'I0\': 10000, \'T\': 200, \'below_func\': \'linear\', \'below_target\': 0.4, \'above_func\': \'linear\', \'above_target\': 0.4}}\n_R37_PRICE_FLOOR = 1\n_R37_HINGE_GAIN = 8.0\ndef _r37_shape(func, x, T=None):\n    x = max(0.0, x)\n    if func == "linear": return x\n    if func == "sq":     return x * x\n    if func == "sqrt":   return math.sqrt(x)\n    if func == "log":    return math.log(1.0 + x)\n    if func == "log10":  return math.log10(1.0 + x)\n    if func == "hinge":\n        # Degenerates to linear if T is missing or non-positive.\n        if not T or T <= 0:\n            return x\n        u = x / T\n        return u + _R37_HINGE_GAIN * max(0.0, u - 1.0) ** 2\n    return x\n\ndef _r37_market_price(item, inventory, params=None):\n    """Floor at _R37_PRICE_FLOOR."""\n    p = (params or _R37_MARKET_PARAMS)[item]\n    base = p["base"]\n    I0 = p["I0"]\n    T = p["T"]\n    if inventory < I0:\n        f = p["below_func"]\n        amp = p["below_target"] * base / _r37_shape(f, T, T)\n        price = base + amp * _r37_shape(f, I0 - inventory, T)\n    else:\n        f = p["above_func"]\n        amp = p["above_target"] * base / _r37_shape(f, T, T)\n        price = base - amp * _r37_shape(f, inventory - I0, T)\n    return max(_R37_PRICE_FLOOR, int(round(price)))\n\ndef _r37_similarity(observation):\n    """Empty tiles cannot make two unrelated production layouts look alike."""\n    farms = observation[\'farms\']\n    own, rival = farms[observation[\'player\']], farms[1-observation[\'player\']]\n    if own[\'unlocked_quadrants\'] != rival[\'unlocked_quadrants\']:\n        return 0.0\n    matches = total = 0\n    for a, b in zip([t for row in own[\'tiles\'] for t in row],\n                    [t for row in rival[\'tiles\'] for t in row]):\n        sa = (a.get(\'crop\'), a.get(\'animal\')) if isinstance(a, dict) else (None, None)\n        sb = (b.get(\'crop\'), b.get(\'animal\')) if isinstance(b, dict) else (None, None)\n        if sa != (None, None) or sb != (None, None):\n            total += 1\n            matches += sa == sb\n    return matches / total if total >= 8 else 0.0\n\n\ndef _r37_quote_priority(observation, order, stock):\n    """Revenue exposed to a small rival batch, not nominal headline revenue."""\n    item = order[1]\n    quantity = min(max(0, int(order[2])), stock.get(item, 0))\n    if not quantity or item not in _R37_MARKET_PARAMS:\n        return 0.0\n    inventory = observation[\'market\'][\'inventory\'][item]\n    params = {k: dict(v) for k, v in _R37_MARKET_PARAMS.items()}\n    for k, patch in observation[\'market\'].get(\'params\', {}).items():\n        if k in params:\n            params[k].update(patch)\n    ',
    b'rival = observation[\'farms\'][1-observation[\'player\']]\n    crop_item = item if item in (\'WHEAT\',\'CARROT\',\'TOMATO\',\'STRAWBERRY\',\'MELON\') else None\n    animal = {\'EGG\':\'GOOSE\',\'MILK\':\'COW\',\'WOOL\':\'SHEEP\'}.get(item)\n    standing = sum(max(0, int(t.get(\'yield_units\', 0))) for row in rival[\'tiles\'] for t in row\n                   if isinstance(t, dict) and\n                   ((crop_item is not None and t.get(\'crop\') == crop_item) or\n                    (animal is not None and t.get(\'animal\') == animal)))\n    # Public fields do not reveal the rival shed. Eight units are a scenario,\n    # not a recovered hidden quantity; visible ripe yield increases the stress.\n    batch = min(24, max(8, standing))\n    now = sum(_r37_market_price(item, inventory+j, params) for j in range(quantity))\n    later = sum(_r37_market_price(item, inventory+batch+j, params) for j in range(quantity))\n    return now-later\n\n\ndef _r37_reorder_sales(observation, action):\n    """Keep quantities and purchase barriers; rank distinct contiguous sales."""\n    stock = projected_shed(action, FarmView(observation))\n    orders = [list(o) for o in action[\'market\']]\n    start = 0\n    while start < len(orders):\n        if orders[start][0] != \'SELL\':\n            start += 1\n            continue\n        end = start\n        while end < len(orders) and orders[end][0] == \'SELL\':\n            end += 1\n        block = orders[start:end]\n        if len({o[1] for o in block}) == len(block):\n            orders[start:end] = sorted(block, key=lambda o: _r37_quote_priority(observation, o, stock), reverse=True)\n        start = end\n    if orders != action[\'market\']:\n        _R37_STATS[\'quote_reordered_turns\'] += 1\n        action = dict(action, market=orders)\n    return action\n\n\n\n# EXP175: bounded public cash-response probe inspired by leoprovorov,\n# Two Coins Mirror Counter v1 (Apache-2.0). No hidden rival inventory.\n_R44_PROBES={}\n_R44_REPORT=dict(probe_matches=0,probe_four_turn_calls=0,probe_errors=0)\n\ndef _r44_before(obs):\n    player=int(obs[\'player\']);step=int(obs[\'step\'])\n    st=_R44_PROBES.get(player)\n    if st is None or step<=st[\'step\']:\n        st=_R44_PROBES[player]={\'step\':-1,\'money\':None,\'probe\':0,\'matched\':False}\n    if step==0:_R44_REPORT.update(probe_matches=0,probe_four_turn_calls=0,probe_errors=0)\n    money=tuple(float(obs[\'farms\'][i][\'money\']) for i in (player,1-player))\n    if st[\'money\'] is not None and st[\'probe\']>=100 and _r37_similarity(obs)>=.90:\n        own=money[0]-st[\'money\'][0];rival=money[1]-st[\'money\'][1]\n        if own>0 and rival>0 and abs(own-rival)<=max(5.0,.05*st[\'probe\']):\n            if not st[\'matched\']:_R44_REPORT[\'probe_matches\']+=1\n            st[\'matched\']=True\n    st.update(step=step,money=money,probe=0)\n    return st\n\ndef _r44_after(obs,action,st):\n    step=int(obs[\'step\']);player=int(obs[\'player\'])\n    if not 336<=step<648 or st[\'matched\']:return\n    # Positive all-sale probes avoid mistaking equal spending for preemption.\n    if not action[\'market\'] or any(o and o[0]!=\'SELL\' for o in action[\'market\']):return\n    debts=_IMPL.chassis.players[player][\'sell_state\'].get(\'r36_debts\',{})\n    own=debts.get(step+3,{})\n    if own:st[\'probe\']=sum(max(0,int(n))*int(obs[\'market\'][\'prices\'].get(item,0)) for item,n in own.items())\n\n_R37_ADAPTIVE = True\n_R37_QUOTE = True\n# EXP-168: adapted from lucifer19 / Harvest Nocturne, Apache-2.0.\n# All rivalry features use public occupied tiles; no private rival inventory.\n_R37_PARENT = agent\n_R37_PLAYERS = {}\n_R37_HORIZONS = {}\n_R37_REPORT = {}\n_R37_STATS = dict(quote_reordered_turns=0, three_turn_calls=0, nocturne_errors=0)\ndel agent\n\ndef agent(observation, configuration=None):\n    player, step = int(observation[\'player\']), int(observation[\'step\'])\n    state = _R37_PLAYERS.get(player)\n    if state is None or step <= state[\'step\']:\n        state = _R37_PLAYERS[player] = {\'step\': -1, \'streak\': 0}\n    if step == 0:\n        _R37_STATS.update(quote_reordered_turns=0, three_turn_calls=0, nocturne_errors=0)\n    state[\'step\'] = step\n    _R37_HORIZONS[player] = 2\n    probe_state=_r44_before(observation)\n    try:\n        if _R37',
    b"_ADAPTIVE and step < 648:\n            state['streak'] = state['streak'] + 1 if _r37_similarity(observation) >= .90 else 0\n            if 336 <= step < 648 and state['streak'] >= 6:\n                _R37_HORIZONS[player] = 3\n                _R37_STATS['three_turn_calls'] += 1\n    except Exception:\n        _R37_STATS['nocturne_errors'] += 1\n    if _R37_HORIZONS[player]==3 and probe_state['matched']:\n        _R37_HORIZONS[player]=4\n        _R44_REPORT['probe_four_turn_calls']+=1\n    # EXP179: four-turn reservation; retain stock, debt and purchase barriers.\n    if 288 <= step < 696:_R37_HORIZONS[player] = 4\n    action = _R37_PARENT(observation, configuration)\n    _r44_after(observation,action,probe_state)\n    if _R37_QUOTE and step >= 288:\n        try:\n            action = _r37_reorder_sales(observation, action)\n        except Exception:\n            _R37_STATS['nocturne_errors'] += 1\n    _R37_REPORT.update(getattr(_R37_PARENT, 'telemetry', {}))\n    _R37_REPORT.update(_R37_STATS)\n    _R37_REPORT.update(_R44_REPORT)\n    return action\n\nagent.telemetry = _R37_REPORT\n\n# Export guard: normal decisions stay identical to the frozen screened policy.\n_RELEASE_PARENT=agent\n_RELEASE_REPORT={}\n_RELEASE_ERRORS=0\ndel agent\n\ndef agent(observation,configuration=None):\n    global _RELEASE_ERRORS\n    try:\n        result=_RELEASE_PARENT(observation,configuration)\n    except Exception:\n        _RELEASE_ERRORS+=1\n        count=0\n        try:\n            count=min(64,len(observation['farms'][int(observation['player'])]['hands']))\n        except Exception:\n            pass\n        result={'farmer':['PASS'],'hands':[['PASS'] for _ in range(count)],'market':[]}\n    _RELEASE_REPORT.update(getattr(_RELEASE_PARENT,'telemetry',{}))\n    _RELEASE_REPORT['release_errors']=_RELEASE_ERRORS\n    return result\n\nagent.telemetry=_RELEASE_REPORT\nagent=globals().pop('agent')\n\n# Adapted from prvsiyan / The Soil Remembers Rain, Apache-2.0.\n# V233: bounded, financed six-sheep SE discovery investment.\n_V233_PARENT=agent\ndel agent\n_V233_STATES={}\n_V233_REPORT=dict(sheep_commit_requests=0,sheep_committed=0,sheep_hire_requests=0,\n    sheep_workers_confirmed=0,sheep_hire_shortfalls=0,sheep_budget_declines=0,\n    sheep_capacity_declines=0,sheep_purchase_shortfalls=0,sheep_feed_buy_requests=0,\n    sheep_wool_harvested=0,sheep_fert_collected=0,sheep_extra_wool_sales=0,\n    sheep_extra_fert_sales=0,sheep_rescue_feed_requests=0)\n\ndef _v233_eligible(obs,native):\n    farm=obs['farms'][obs['player']];prices=obs['market']['prices']\n    if len(farm['tiles'])!=10 or set(farm['unlocked_quadrants'])!={'NW','NE','SW'}:return False\n    if obs['town']['unlocked_shops'].count('YARN_STORE')<2 or prices['WOOL']<220 or prices['WHEAT']>45:return False\n    if any(farm['tiles'][y][x]!='LOCKED' for y in (5,6) for x in range(5,8)):return False\n    if obs['private']['shed'].get('SHEEP',0) or any(i.get('SHEEP',0) for i in obs['private']['inventories']):return False\n    for day in range(12,30):\n        for a in _v219_native_day(native,day):\n            if any(o and (o[0]=='BUY_LAND' or o[:2]==['BUY_ANIMAL','SHEEP']) for o in a.get('market',[])):return False\n            if any(c and c[0] in ('PICKUP','PLACE') and len(c)>1 and c[1]=='SHEEP' for c in [a.get('farmer')]+a.get('hands',[])):return False\n    return True\n\ndef _v233_request(obs,action,state,native):\n    step=int(obs['step']);day=step//24;hour=step%24\n    # EXP242: preserve native indices while servicing already committed sheep.\n    if state.get('requested_day')==day:return action\n    committed=state.get('committed')\n    if not committed and (hour>1 or day!=12 or not _v233_eligible(obs,native)):return action\n    planned=_v219_native_day(native,day)\n    deadline=2 if committed else 1\n    if committed:\n        last_native_hire=max((h for h,a in enumerate(planned) if any(o and o[0]=='HIRE' for o in a.get('market',[]))),default=0)\n        if 2<last_native_hire<=6:deadline=6\n    if hour>deadline:return action\n    if any(o and o[0]=='HIRE' for a in planned[hour+1:] for o in a.get('market',[])):return action\n    farm=obs['farms'][obs['player']];market=a",
    b"ction.get('market',[])\n    parent_hires=sum(bool(o) and o[0]=='HIRE' for o in market)\n    expected=max(len(a.get('hands',[])) for a in planned)\n    if len(farm['hands'])+parent_hires!=expected:return action\n    initial=not state.get('committed')\n    extra=([['BUY_LAND'],['BUY_ANIMAL','SHEEP',6]] if initial else [])+[['BUY_PRODUCT','WHEAT',6],['HIRE'],['HIRE']]\n    if len(market)+len(extra)>MAX_ORDERS:return action\n    stock=projected_shed(action,FarmView(obs))\n    incoming=6+6*initial\n    budget=7000*initial+6*(int(obs['market']['prices']['WHEAT'])+10)\n    budget+=sum(_v219_fib(n) for n in range(farm['hires_today'],farm['hires_today']+parent_hires+2))\n    for o in market:\n        if not o:continue\n        if o[0]=='BUY_LAND':return action\n        if o[0]=='BUY_PRODUCT':\n            incoming+=int(o[2]);budget+=int(o[2])*(int(obs['market']['prices'][o[1]])+10)\n        elif o[0]=='BUY_ANIMAL':\n            incoming+=int(o[2]);budget+=int(o[2])*{'SHEEP':500,'COW':400,'GOOSE':300}[o[1]]\n        elif o[0]=='BUY_SEED':budget+=int(o[2])*{'WHEAT':10,'CARROT':20,'TOMATO':50,'STRAWBERRY':100,'MELON':80}[o[1]]\n    if sum(stock.values())+incoming>100:\n        _V233_REPORT['sheep_capacity_declines']+=1;return action\n    if farm['money']<budget+(3000 if initial else 1000):\n        _V233_REPORT['sheep_budget_declines']+=1;return action\n    state['requested_day']=day\n    state['pending']={'first':expected+1,'initial':initial}\n    _V233_REPORT['sheep_hire_requests']+=2;_V233_REPORT['sheep_feed_buy_requests']+=6\n    if initial:_V233_REPORT['sheep_commit_requests']+=1\n    result=copy.deepcopy(action);result['market']=market+extra\n    return result\n\ndef _v233_worker(obs,actor,targets):\n    farm=obs['farms'][obs['player']];private=obs['private'];step=int(obs['step'])\n    pos=tuple(farm['hands'][actor-1]);inv=private['inventories'][actor]\n    access=((4,4),(5,4),(4,5),(5,5))\n    home=min(access,key=lambda p:(abs(pos[0]-p[0])+abs(pos[1]-p[1]),p))\n    distance=abs(pos[0]-home[0])+abs(pos[1]-home[1])\n    cargo=[item for item in ('WOOL','FERTILIZER') if inv.get(item,0)]\n    if cargo and step%24 >= (22 if step//24==29 else 23)-distance:\n        return _v219_walk(pos,home) or ['PLACE',cargo[0],inv[cargo[0]]]\n    missing=sum(not(isinstance(farm['tiles'][y][x],dict) and farm['tiles'][y][x].get('animal')=='SHEEP') for x,y in targets)\n    if missing and not inv.get('SHEEP',0) and private['shed'].get('SHEEP',0):\n        return _v219_walk(pos,home) or ['PICKUP','SHEEP',min(missing,private['shed']['SHEEP'])]\n    hungry=sum(not(isinstance(farm['tiles'][y][x],dict) and farm['tiles'][y][x].get('fed_today')) for x,y in targets)\n    if hungry and not inv.get('WHEAT',0) and private['shed'].get('WHEAT',0):\n        return _v219_walk(pos,home) or ['PICKUP','WHEAT',min(hungry,private['shed']['WHEAT'])]\n    tasks=[]\n    for target in targets:\n        x,y=target;tile=farm['tiles'][y][x];command=None\n        if tile is None:command=['BUILD_PASTURE']\n        elif isinstance(tile,dict) and tile.get('kind')=='WEED':command=['DIG']\n        elif isinstance(tile,dict) and tile.get('kind')=='PASTURE' and not tile.get('animal'):\n            if inv.get('SHEEP',0):command=['PLACE','SHEEP']\n        elif isinstance(tile,dict) and tile.get('animal')=='SHEEP':\n            if not tile['fed_today'] and inv.get('WHEAT',0):command=['FEED']\n            elif not tile['cared_today']:command=['CARE']\n            elif tile['yield_units']:command=['HARVEST']\n            elif tile['fertilizer_available']:command=['COLLECT_FERTILIZER']\n        if command:tasks.append((abs(pos[0]-x)+abs(pos[1]-y),targets.index(target),target,command))\n    if tasks:\n        _,_,target,command=min(tasks);return _v219_walk(pos,target) or command\n    if cargo:return _v219_walk(pos,home) or ['PLACE',cargo[0],inv[cargo[0]]]\n    return ['PASS']\n\ndef _v234_rescue(obs,action,state):\n    if not state['workers'] or int(obs['step'])%24>14:return action\n    orders=action.get('market',[])\n    if len(orders)>=MAX_ORDERS:return action\n    if any(o and (o[0] in ('HIRE','BUY_LAND','BUY_ANIMAL','BUY_PRODUCT','BUY_SEED') or (len(o)>",
    b"1 and o[1]=='WHEAT')) for o in orders):return action\n    farm=obs['farms'][obs['player']];private=obs['private'];hungry=carried=0\n    commands=[action.get('farmer') or ['PASS']]+list(action.get('hands') or [])\n    for actor,targets in state['workers'].items():\n        command=commands[actor]\n        if command==['FEED'] or command[:2]==['PICKUP','WHEAT']:return action\n        carried+=private['inventories'][actor].get('WHEAT',0)\n        hungry+=sum(isinstance(farm['tiles'][y][x],dict) and farm['tiles'][y][x].get('animal')=='SHEEP' and not farm['tiles'][y][x].get('fed_today') for x,y in targets)\n    stock=projected_shed(action,FarmView(obs))\n    shortage=hungry-carried-stock.get('WHEAT',0)\n    if not 0<shortage<=6 or state.get('rescue_today',0)+shortage>6:return action\n    quote=int(obs['market']['prices']['WHEAT'])\n    if quote<1 or farm['money']<1000+shortage*(quote+10) or sum(stock.values())+shortage>100:return action\n    result=copy.deepcopy(action);result['market'].append(['BUY_PRODUCT','WHEAT',shortage])\n    state['rescue_today']=state.get('rescue_today',0)+shortage\n    _V233_REPORT['sheep_rescue_feed_requests']+=shortage\n    return result\n\ndef agent(observation,configuration=None):\n    action=_V233_PARENT(observation,configuration)\n    step=int(observation['step']);player=int(observation['player']);day=step//24\n    state=_V233_STATES.get(player)\n    if state is None or step<=state['last_step']:\n        state={'last_step':step,'day':-1,'workers':{},'work':{},'credit':{'WOOL':0,'FERTILIZER':0}}\n        _V233_STATES[player]=state\n    state['last_step']=step\n    if configuration is not None and any(configuration.get(k,v)!=v for k,v in\n        (('boardSize',10),('turnsPerDay',24),('shedCapacity',100),('maxMarketOrdersPerTurn',10))):return action\n    if day<12:return action\n    farm=observation['farms'][player];private=observation['private']\n    if state['day']!=day:state['day']=day;state['workers']={};state['work']={};state['rescue_today']=0\n    for actor,previous in state['work'].items():\n        if previous['step']!=step-1 or actor>=len(private['inventories']):continue\n        item={'HARVEST':'WOOL','COLLECT_FERTILIZER':'FERTILIZER'}.get(previous['command'][0])\n        if item:\n            gained=max(0,private['inventories'][actor].get(item,0)-previous['inventory'].get(item,0))\n            state['credit'][item]+=gained\n            _V233_REPORT['sheep_wool_harvested' if item=='WOOL' else 'sheep_fert_collected']+=gained\n    pending=state.pop('pending',None)\n    if pending:\n        funded='SE' in farm['unlocked_quadrants'] and (not pending['initial'] or private['shed'].get('SHEEP',0)>=6)\n        if not funded:_V233_REPORT['sheep_purchase_shortfalls']+=1\n        elif len(farm['hands'])<pending['first']+1:_V233_REPORT['sheep_hire_shortfalls']+=1\n        else:\n            for i in range(2):state['workers'][pending['first']+i]=[(x,5+i) for x in range(5,8)]\n            _V233_REPORT['sheep_workers_confirmed']+=2\n            if pending['initial']:state['committed']=True;_V233_REPORT['sheep_committed']+=1\n    action=_v233_request(observation,action,state,_IMPL.chassis.players[player])\n    if not state.get('committed'):return action\n    result=copy.deepcopy(action)\n    commands=[result.get('farmer') or ['PASS']]+list(result.get('hands') or [])\n    commands += [['PASS'] for _ in range(len(farm['hands'])+1-len(commands))]\n    state['work']={}\n    for actor,targets in state['workers'].items():\n        command=_v233_worker(observation,actor,targets);commands[actor]=command\n        state['work'][actor]={'step':step,'command':command,'inventory':dict(private['inventories'][actor])}\n    result['farmer'],result['hands']=commands[0],commands[1:]\n    result=_v234_rescue(observation,result,state)\n    stock=projected_shed(result,FarmView(observation))\n    for item in ('WOOL','FERTILIZER'):\n        scheduled=sum(int(o[2]) for o in result['market'] if o[:2]==['SELL',item])\n        count=min(state['credit'][item],max(0,stock.get(item,0)-scheduled))\n        if count and len(result['market'])<MAX_ORDERS:\n            result['market'].append(['SELL",
    b"',item,count]);state['credit'][item]-=count\n            _V233_REPORT['sheep_extra_wool_sales' if item=='WOOL' else 'sheep_extra_fert_sales']+=count\n    return result\n\n_R46_SHEEP_AGENT=agent\n_R46_SHADOW_PARENT=_shadow_terminal\n_R46_REPORT={}\ndef _shadow_terminal(obs,config):\n    if _V233_STATES.get(int(obs['player']),{}).get('committed'):return None\n    return _R46_SHADOW_PARENT(obs,config)\ndel agent\ndef agent(observation,configuration=None):\n    try:\n        if int(observation.get('step',-1))==0:\n            for k in _V233_REPORT:_V233_REPORT[k]=0\n        result=_R46_SHEEP_AGENT(observation,configuration)\n    except Exception:\n        _R46_REPORT['sheep_overlay_errors']=_R46_REPORT.get('sheep_overlay_errors',0)+1\n        result={'farmer':['PASS'],'hands':[],'market':[]}\n    _R46_REPORT.update(getattr(_V233_PARENT,'telemetry',{}))\n    _R46_REPORT.update(_V233_REPORT)\n    return result\nagent.telemetry=_R46_REPORT\nagent=globals().pop('agent')\n\n# EXP182: finite-harvest wheat/carrot input planner; original adaptation.\n_R51_INPUT_PARENT=agent\n_R51_INPUT_STATES={}\n_R51_INPUT_REPORT={}\n_R51_INPUT_MAX_WORKERS=2\n_R51_INPUT_CROPS={'WHEAT':(2,4,6),'CARROT':(2,3,4)}\n\ndef _r51_input_forecast(obs,route,expected):\n    step=int(obs['step']);day=step//24;farm=obs['farms'][obs['player']]\n    pos=[list(farm['farmer'])]+[list(p) for p in farm['hands'][:expected]];targets={}\n    for y,line in enumerate(farm['tiles']):\n        for x,tile in enumerate(line):\n            if not isinstance(tile,dict) or tile.get('crop') not in _R51_INPUT_CROPS:continue\n            item=tile['crop'];first,last,cap=_R51_INPUT_CROPS[item]\n            if 1<=day-tile['planted_day']<last:\n                targets[(x,y)]={'crop':item,'birth':tile['planted_day'],'yield':tile['yield_units'],\n                    'until':tile.get('fertilized_until_day',-1),'watered':tile.get('watered_today',False),'water':[],'harvest':None,'first':first,'last':last,'cap':cap}\n    access=((4,4),(5,4),(4,5),(5,5));seen=set()\n    # Native continuation ends before the reactive terminal closure planner.\n    for t in range(step,min(712,(day+4)*24)):\n        tape=_IMPL.chassis.routes[2 if t>=648 else route];a=tape[t]\n        for actor,c in enumerate([a.get('farmer') or ['PASS'],*(a.get('hands') or [])][:len(pos)]):\n            if not c:continue\n            xy=tuple(pos[actor]);target=targets.get(xy)\n            if target is not None and target['harvest'] is None:\n                if c[0]=='WATER' and (t//24,xy) not in seen:\n                    seen.add((t//24,xy))\n                    if not(t//24==day and target['watered']) and target['first']<=t//24-target['birth']<=target['last']:target['water'].append(t)\n                if c[0]=='HARVEST':target['harvest']=t\n            if c[0] in MOVES:\n                dx,dy=MOVES[c[0]];pos[actor]=[max(0,min(9,pos[actor][0]+dx)),max(0,min(9,pos[actor][1]+dy))]\n        for o in a.get('market',[]):\n            if o and o[0]=='HIRE':\n                counts={p:sum(tuple(q)==p for q in pos) for p in access}\n                pos.append(list(min(access,key=lambda p:(counts[p],access.index(p)))))\n        if (t+1)%24==0:pos=[[4,4]]\n    return targets\n\ndef _r51_input_gain(target,arrival,day):\n    if target['harvest'] is None or target['harvest']<=arrival:return 0\n    extra=sum(arrival<t<=target['harvest'] and day<=t//24<=day+2 and t//24>target['until'] for t in target['water'])\n    baseline=target['yield']+sum(2 if t//24<=target['until'] else 1 for t in target['water'])\n    return max(0,min(extra,target['cap']-baseline))\n\ndef _r51_input_path(obs,targets,action,index):\n    step=int(obs['step']);day=step//24\n    ready,start=_r62_input_start(obs,action,index)\n    prices={p:max(1,int(obs['market']['prices'][p])-2) for p in ('WHEAT','CARROT')}\n    fertilizer=max(1,_r37_market_price('FERTILIZER',obs['market']['inventory']['FERTILIZER']-16)+2)\n    # Tuple: penalized value, gross value, next free turn, position, path, used,\n    # wheat units, carrot units. No state reads from the rival's private farm.\n    beam=[(0,0,ready,start,(),frozenset(),0,0)]\n    best=None\n    for depth in r",
    b"ange(8):\n        expanded=[]\n        for score,gross,now,pos,path,used,wheat,carrot in beam:\n            for xy,target in targets.items():\n                if xy in used:continue\n                arrival=now+abs(pos[0]-xy[0])+abs(pos[1]-xy[1])\n                if arrival>=day*24+23:continue\n                gain=_r51_input_gain(target,arrival,day)\n                if not gain:continue\n                item=target['crop'];new_gross=gross+gain*prices[item]\n                new_path=path+((xy[0],xy[1],item,target['birth']),)\n                expanded.append((new_gross-1.5*fertilizer*len(new_path),new_gross,arrival+1,xy,new_path,\n                                 used|{xy},wheat+(gain if item=='WHEAT' else 0),carrot+(gain if item=='CARROT' else 0)))\n        if not expanded:break\n        expanded.sort(key=lambda s:(-s[0],-s[1],s[2],s[4]))\n        beam=expanded[:8]\n        if depth>=2:\n            candidate=beam[0]\n            if best is None or (-candidate[0],-candidate[1],candidate[2],candidate[4])<(-best[0],-best[1],best[2],best[4]):best=candidate\n    if best is None:return [],{'WHEAT':0,'CARROT':0}\n    return list(best[4]),{'WHEAT':best[6],'CARROT':best[7]}\n\ndef _r51_input_control(obs,action,state):\n    step=int(obs['step']);day=step//24;hour=step%24;player=int(obs['player']);farm=obs['farms'][player];private=obs['private']\n    native=_IMPL.chassis.players[player]\n    if state.get('day')!=day:state.update(day=day,workers={},pending=None,placed=[])\n    for x,y in state['placed']:\n        tile=farm['tiles'][y][x]\n        if isinstance(tile,dict) and tile.get('fertilized_until_day',-1)>=day+2:_R51_INPUT_REPORT['input_confirmed_applications']+=1\n        else:_R51_INPUT_REPORT['input_application_errors']+=1\n    state['placed']=[]\n    if state.get('pending'):\n        pending=state.pop('pending')\n        for actor,plan in pending.items():\n            if len(farm['hands'])>=actor:state['workers'][actor]=plan;_R51_INPUT_REPORT['input_confirmed_hires']+=1\n            else:_R51_INPUT_REPORT['input_hire_errors']+=1\n    if state['workers']:\n        changed=copy.deepcopy(action)\n        for actor,plan in state['workers'].items():\n            inv=private['inventories'][actor];pos=tuple(farm['hands'][actor-1]);cmd=['PASS']\n            if not plan['loaded']:\n                stock=projected_shed(changed,FarmView(obs));q=min(plan['quantity'],max(0,stock.get('FERTILIZER',0)))\n                if q and _shed_adjacent(pos,10):\n                    cmd=['PICKUP','FERTILIZER',q];plan['loaded']=True;_R51_INPUT_REPORT['input_loaded_units']+=q\n                    if q<plan['quantity']:_R51_INPUT_REPORT['input_stock_shortfalls']+=plan['quantity']-q\n            elif inv.get('FERTILIZER',0):\n                while plan['path']:\n                    x,y,crop,birth=plan['path'][0];tile=farm['tiles'][y][x]\n                    if not isinstance(tile,dict) or tile.get('crop')!=crop or tile.get('planted_day')!=birth or tile.get('fertilized_until_day',-1)>=day+2:\n                        plan['path'].pop(0);continue\n                    cmd=_v219_walk(pos,(x,y)) or ['FERTILIZE']\n                    if cmd==['FERTILIZE']:state['placed'].append((x,y));plan['path'].pop(0);_R51_INPUT_REPORT['input_application_requests']+=1\n                    break\n            changed['hands'][actor-1]=cmd\n        return changed\n    if hour not in (1,2,3) or not 12<=day<=28:return action\n    planned=_v219_native_day(native,day);expected=max(len(a.get('hands',[])) for a in planned)\n    if any(o and o[0]=='HIRE' for a in planned[hour:] for o in a.get('market',[])) or native['pending']:return action\n    parents=[_V219_STATES.get(player,{}),_V233_STATES.get(player,{})]\n    # A parent may retry after a full market queue; its headcount must remain native.\n    if day in (12,18) or any(p.get('committed') and p.get('requested_day')!=day for p in parents):return action\n    if any(p.get('pending') for p in parents) or any(o and o[0]=='HIRE' for o in action.get('market',[])):return action\n    owned=set(range(1,expected+1))\n    for p in parents:\n        actors=set(p.get('workers',{}))\n        if owned&actor",
    b"s:return action\n        owned|=actors\n    if owned!=set(range(1,len(farm['hands'])+1)):return action\n    targets=_r51_input_forecast(obs,native['route'],expected);plans=[];total_q=0;total_cost=0;all_units={'WHEAT':0,'CARROT':0}\n    stock=projected_shed(action,FarmView(obs));purchases=sum(max(0,int(o[2])) for o in action.get('market',[]) if len(o)>2 and o[0] in ('BUY_PRODUCT','BUY_ANIMAL'))\n    # Units act before market orders. Preserve the native next-turn pickup,\n    # after the current parent's actual sales/purchases, before buying tour inputs.\n    available=max(0,stock.get('FERTILIZER',0))\n    for o in action.get('market',[]):\n        if len(o)>=3 and o[:2]==['SELL','FERTILIZER']:available=max(0,available-max(0,int(o[2])))\n        elif len(o)>=3 and o[:2]==['BUY_PRODUCT','FERTILIZER']:available+=max(0,int(o[2]))\n    next_native=planned[hour+1];native_pickups=sum(max(0,int(c[2]) if len(c)>2 else 1) for c in [next_native.get('farmer') or ['PASS'],*(next_native.get('hands') or [])] if len(c)>1 and c[:2]==['PICKUP','FERTILIZER'])\n    topup=max(0,native_pickups-available)\n    plans,total_q,total_cost,all_units=_r68_joint_plans(obs,action,targets,stock,purchases,topup)\n    if not plans:return action\n    state['pending']={len(farm['hands'])+1+i:plan for i,plan in enumerate(plans)}\n    _R51_INPUT_REPORT['input_hire_requests']+=len(plans);_R51_INPUT_REPORT['input_purchase_requests']+=total_q+topup\n    _R51_INPUT_REPORT['input_forecast_wheat']+=all_units['WHEAT'];_R51_INPUT_REPORT['input_forecast_carrot']+=all_units['CARROT']\n    changed=copy.deepcopy(action);changed['market'] += [['BUY_PRODUCT','FERTILIZER',total_q+topup]]+[['HIRE'] for _ in plans];return changed\n\ndef agent(observation,configuration=None):\n    try:\n        step=int(observation['step']);player=int(observation['player']);state=_R51_INPUT_STATES.get(player)\n        if state is None or step<=state['step']:\n            state=_R51_INPUT_STATES[player]={'step':-1}\n            _R51_INPUT_REPORT.update(input_hire_requests=0,input_confirmed_hires=0,input_hire_errors=0,input_purchase_requests=0,\n                input_loaded_units=0,input_stock_shortfalls=0,input_application_requests=0,input_confirmed_applications=0,\n                input_application_errors=0,input_errors=0,input_forecast_wheat=0,input_forecast_carrot=0)\n        state['step']=step;action=_R51_INPUT_PARENT(observation,configuration)\n        if configuration is None or all(configuration.get(k,v)==v for k,v in [('boardSize',10),('turnsPerDay',24),('shedCapacity',100),('maxMarketOrdersPerTurn',10)]):\n            action=_r51_input_control(observation,action,state)\n        _R51_INPUT_REPORT.update(getattr(_R51_INPUT_PARENT,'telemetry',{}));return action\n    except Exception:\n        _R51_INPUT_REPORT['input_errors']=_R51_INPUT_REPORT.get('input_errors',0)+1\n        return {'farmer':['PASS'],'hands':[],'market':[]}\nagent.telemetry=_R51_INPUT_REPORT\nagent=globals().pop('agent')\n\n# EXP182: project the final hour's actual worker actions before automatic deposit.\n_R51_WAREHOUSE_PARENT=agent\n_R51_WAREHOUSE_REPORT={}\n\ndef _r51_close_warehouse(obs,action):\n    step=int(obs['step']);day=step//24\n    if step%24!=23 or not 12<=day<=28:return action\n    # No speculative product purchase/worker count model: these hours abstain.\n    if any(o and o[0] not in ('SELL',) for o in action.get('market',[])):return action\n    farm,private=_PLANNER_NS['_clone_state'](obs['farms'][obs['player']],obs['private'])\n    commands=[action.get('farmer') or ['PASS'],*(action.get('hands') or [])]\n    demand={}\n    for c in commands:\n        if len(c)>1 and c[0]=='PLANT':demand[c[1]]=demand.get(c[1],0)+1\n    blocked={k for k,q in demand.items() if q>private['seeds'].get(k,0)}\n    for actor,c in enumerate(commands[:len(private['inventories'])]):\n        if len(c)>1 and c[0]=='PLANT' and c[1] in blocked:c=['PASS']\n        _PLANNER_NS['_apply_unit_action'](farm,private,actor,c,10,day,24,100)\n    post=dict(private['shed'])\n    for o in action.get('market',[]):\n        if len(o)>=3 and o[0]=='SELL':post[o[1]]=max(0,post.get(o[1],0)-max(0,int(o[2])))\n",
    b"    needed=sum(post.values())+sum(max(0,q) for inv in private['inventories'] for q in inv.values())-100\n    if needed<=0:return action\n    result=copy.deepcopy(action);orders=result['market']\n    # Grain and fertilizer have native input obligations; other products do not.\n    # Additional commodity sales are bounded by actual post-action physical stock.\n    for item in sorted((p for p in PRODUCTS if p not in ('WHEAT','FERTILIZER')),key=lambda p:-obs['market']['prices'].get(p,0)):\n        qty=min(needed,post.get(item,0))\n        if not qty:continue\n        existing=next((o for o in orders if len(o)>=3 and o[:2]==['SELL',item]),None)\n        if existing is not None:existing[2]=max(0,int(existing[2]))+qty\n        elif len(orders)<10:orders.append(['SELL',item,qty])\n        else:continue\n        needed-=qty;post[item]-=qty;_R51_WAREHOUSE_REPORT['warehouse_extra_sales']+=qty\n        if needed<=0:break\n    if needed>0:\n        native=_IMPL.chassis.players[int(obs['player'])];reserve=0\n        for t in range(step+1,719):\n            future=_IMPL.chassis.routes[2 if t>=648 else native['route']][t]\n            for c in [future.get('farmer') or ['PASS'],*(future.get('hands') or [])]:\n                if len(c)>1 and c[:2]==['PICKUP','WHEAT']:reserve+=max(0,int(c[2]) if len(c)>2 else 1)\n            if any(len(o)>1 and o[:2]==['BUY_PRODUCT','WHEAT'] for o in future.get('market',[])):break\n        incoming=sum(max(0,inv.get('WHEAT',0)) for inv in private['inventories'])\n        others=sum(q for p,q in post.items() if p!='WHEAT')+sum(max(0,q) for inv in private['inventories'] for p,q in inv.items() if p!='WHEAT')\n        # Even if every other carried item deposits first, this grain reserve fits.\n        qty=min(needed,post.get('WHEAT',0),max(0,post.get('WHEAT',0)+incoming-reserve)) if 100-others>=reserve else 0\n        existing=next((o for o in orders if len(o)>=3 and o[:2]==['SELL','WHEAT']),None)\n        if qty and (existing is not None or len(orders)<10):\n            if existing is not None:existing[2]=max(0,int(existing[2]))+qty\n            else:orders.append(['SELL','WHEAT',qty])\n            needed-=qty;_R51_WAREHOUSE_REPORT['warehouse_extra_sales']+=qty\n    _R51_WAREHOUSE_REPORT['warehouse_projected_unresolved']+=max(0,needed)\n    if result!=action:_R51_WAREHOUSE_REPORT['warehouse_changed_turns']+=1\n    return result\n\ndef agent(observation,configuration=None):\n    result=_R51_WAREHOUSE_PARENT(observation,configuration)\n    try:\n        if int(observation['step'])==0:_R51_WAREHOUSE_REPORT.update(warehouse_changed_turns=0,warehouse_extra_sales=0,warehouse_projected_unresolved=0,warehouse_errors=0)\n        if configuration is None or all(configuration.get(k,v)==v for k,v in [('boardSize',10),('turnsPerDay',24),('shedCapacity',100),('maxMarketOrdersPerTurn',10)]):result=_r51_close_warehouse(observation,result)\n    except Exception:_R51_WAREHOUSE_REPORT['warehouse_errors']=_R51_WAREHOUSE_REPORT.get('warehouse_errors',0)+1\n    _R51_WAREHOUSE_REPORT.update(getattr(_R51_WAREHOUSE_PARENT,'telemetry',{}));return result\nagent.telemetry=_R51_WAREHOUSE_REPORT\nagent=globals().pop('agent')\n\nfrom itertools import permutations as _r53_permutations\n_R53_LABOR_REPORT=dict(labor_requests=0,labor_hires_avoided=0,labor_spawn_errors=0,labor_confirmed=0,labor_day26=0,labor_day27=0,labor_day28=0)\n\ndef _r53_labor_assignment(obs,action,fertilizer):\n    step=int(obs['step']);day=step//24;farm=obs['farms'][obs['player']]\n    if day not in (26,27,28) or step%24>2:return None\n    # Do not preempt a later price-gated fertilizer request with a smaller unfertilized team.\n    if day==27 and not fertilizer:return None\n    count=3 if fertilizer else 2\n    positions=[list(farm['farmer'])]+[list(p) for p in farm['hands']]\n    for i,c in enumerate([action.get('farmer') or ['PASS'],*(action.get('hands') or [])][:len(positions)]):\n        if c and c[0] in MOVES:\n            dx,dy=MOVES[c[0]];positions[i]=[max(0,min(9,positions[i][0]+dx)),max(0,min(9,positions[i][1]+dy))]\n    access=((4,4),(5,4),(4,5),(5,5));spawns=[]\n    native_hires=sum(bool(o) and o[0]=='HIRE' for o in",
    b" action.get('market',[]))\n    for i in range(native_hires+count):\n        chosen=min(access,key=lambda p:(sum(tuple(q)==p for q in positions),access.index(p)));positions.append(list(chosen))\n        if i>=native_hires:spawns.append(chosen)\n    groups=(((5,5),(6,5),(7,5),(8,5)),((9,5),(9,6),(8,6)),((5,6),(6,6),(7,6))) if fertilizer else (tuple((x,5) for x in range(5,10)),tuple((x,6) for x in range(5,10)))\n    choices=[];remaining=23-step%24\n    for assignment in _r53_permutations(groups):\n        costs=[]\n        for start,path in zip(spawns,assignment):\n            distance=abs(start[0]-path[0][0])+abs(start[1]-path[0][1])\n            distance+=sum(abs(a[0]-b[0])+abs(a[1]-b[1]) for a,b in zip(path,path[1:]))\n            distance+=min(abs(path[-1][0]-x)+abs(path[-1][1]-y) for x,y in access)\n            costs.append(distance+(3 if fertilizer else 2)*len(path)+1+int(fertilizer))\n        if max(costs)<=remaining:choices.append((max(costs),sum(costs),assignment))\n    if not choices:return None\n    _,_,assignment=min(choices)\n    return dict(paths=assignment,spawns=spawns,remaining=remaining,workers=count,fertilizer=fertilizer)\n\n_R53_LABOR_PARENT=agent\ndef agent(observation,configuration=None):\n    if isinstance(observation,dict) and observation.get('step')==0:\n        for k in _R53_LABOR_REPORT:_R53_LABOR_REPORT[k]=0\n    result=_R53_LABOR_PARENT(observation,configuration)\n    _R53_LABOR_COMBINED.update(getattr(_R53_LABOR_PARENT,'telemetry',{}));_R53_LABOR_COMBINED.update(_R53_LABOR_REPORT)\n    return result\n_R53_LABOR_COMBINED={}\nagent.telemetry=_R53_LABOR_COMBINED\nagent=globals().pop('agent')\n\n# EXP193: deterministic HIRE spawn after native unit actions, then next-turn pickup.\ndef _r62_input_start(obs,action,index):\n    farm=obs['farms'][obs['player']]\n    positions=[list(farm['farmer'])]+[list(p) for p in farm['hands']]\n    for actor,c in enumerate([action.get('farmer') or ['PASS'],*(action.get('hands') or [])][:len(positions)]):\n        if c and c[0] in MOVES:\n            dx,dy=MOVES[c[0]]\n            positions[actor]=[max(0,min(9,positions[actor][0]+dx)),max(0,min(9,positions[actor][1]+dy))]\n    access=((4,4),(5,4),(4,5),(5,5))\n    native_hires=sum(bool(o) and o[0]=='HIRE' for o in action.get('market',[]))\n    for _ in range(native_hires+index+1):\n        chosen=min(access,key=lambda p:(sum(tuple(q)==p for q in positions),access.index(p)))\n        positions.append(list(chosen))\n    return int(obs['step'])+2,chosen\n\nagent=globals().pop('agent')\n\ndef _r68_joint_plans(obs,action,targets,stock,purchases,topup):\n    farm=obs['farms'][obs['player']];choices=[]\n    for mode,first_crop in enumerate((None,'WHEAT','CARROT')):\n        remaining=dict(targets);plans=[];total_q=0;total_cost=0;total_value=0\n        all_units={'WHEAT':0,'CARROT':0}\n        for i in range(_R51_INPUT_MAX_WORKERS):\n            subset={xy:t for xy,t in remaining.items() if t['crop']==first_crop} if i==0 and first_crop else remaining\n            path,units=_r51_input_path(obs,subset,action,i);q=len(path)\n            if q<3 or len(action.get('market',[]))+2+i>10 or sum(stock.values())+purchases+total_q+q+topup>95:break\n            quote=_r37_market_price('FERTILIZER',obs['market']['inventory']['FERTILIZER']-total_q-q-topup)\n            cost=(q+(topup if i==0 else 0))*(quote+2)+_v219_fib(int(farm['hires_today'])+i)\n            value=sum(n*max(1,_r37_market_price(item,obs['market']['inventory'][item]+all_units[item]+n)-2) for item,n in units.items())\n            if value<1.5*cost+50 or farm['money']<total_cost+cost+3000:break\n            plans.append({'path':path,'quantity':q,'loaded':False});total_q+=q;total_cost+=cost;total_value+=value\n            for item,n in units.items():all_units[item]+=n\n            for x,y,_,_ in path:remaining.pop((x,y),None)\n        score=(total_value-total_cost,total_value,-total_cost,-len(plans),-mode)\n        choices.append((score,plans,total_q,total_cost,all_units))\n    _,plans,total_q,total_cost,all_units=max(choices,key=lambda v:v[0])\n    return plans,total_q,total_cost,all_units\n\nagent=globals().pop('agent')\n\n_R70_STATES={}\n_R",
    b"70_REPORT={}\n\ndef _r70_parent_fert_qty(obs,action,planned,offset):\n    stock=dict(projected_shed(action,FarmView(obs)))\n    for order in action.get('market',[]):\n        if len(order)<3:continue\n        op,item,quantity=order[:3];quantity=max(0,int(quantity))\n        if op=='SELL':stock[item]=max(0,stock.get(item,0)-quantity)\n        elif op in ('BUY_PRODUCT','BUY_ANIMAL'):\n            stock[item]=stock.get(item,0)+min(quantity,max(0,100-sum(stock.values())))\n    next_action=planned[offset+1] if offset+1<len(planned) else {}\n    commands=[next_action.get('farmer') or ['PASS'],*(next_action.get('hands') or [])]\n    native_need=sum(max(0,int(c[2]) if len(c)>2 else 1) for c in commands if len(c)>1 and c[:2]==['PICKUP','FERTILIZER'])\n    quantity=max(10,10+native_need-max(0,stock.get('FERTILIZER',0)))\n    if quantity>max(0,100-sum(stock.values())):\n        _R70_REPORT['parent_input_capacity_declines']+=1\n        return 10\n    if quantity>10:\n        _R70_REPORT['parent_input_guard_turns']+=1\n        _R70_REPORT['parent_input_guard_extra_units']+=quantity-10\n    return quantity\n\ndef _r70_before(obs):\n    player=int(obs['player']);step=int(obs['step']);state=_R70_STATES.get(player)\n    if state is None or step<=state['step']:\n        state=_R70_STATES[player]={'step':-1,'pending':[],'roles':set()}\n        _R70_REPORT.update(parent_input_guard_turns=0,parent_input_guard_extra_units=0,\n            parent_input_requests=0,parent_input_confirmed=0,parent_input_shortfalls=0,\n            parent_input_errors=0,parent_input_capacity_declines=0)\n    state['step']=step\n    for request in state['pending']:\n        actor,quantity,old=request\n        actual=max(0,int(obs['private']['inventories'][actor].get('FERTILIZER',0))-old)\n        _R70_REPORT['parent_input_confirmed']+=min(quantity,actual)\n        _R70_REPORT['parent_input_shortfalls']+=max(0,quantity-actual)\n    state['pending']=[]\n    return state\n\ndef _r70_after(obs,action,state):\n    player=int(obs['player']);day=int(obs['step'])//24\n    for actor,role in _V219_STATES.get(player,{}).get('workers',{}).items():\n        if not role.get('needs_fertilizer'):continue\n        key=(day,actor);inv=obs['private']['inventories'][actor].get('FERTILIZER',0)\n        if key not in state['roles']:\n            state['roles'].add(key)\n            desired=role.get('fertilizer_quantity',10 if role['kind']=='fertilizer' else 5)\n            if role.get('loaded') and not role.get('pickup_requested') and inv<desired:\n                _R70_REPORT['parent_input_shortfalls']+=desired-inv\n        command=action.get('hands',[])[actor-1] if actor<=len(action.get('hands',[])) else ['PASS']\n        if len(command)>1 and command[:2]==['PICKUP','FERTILIZER']:\n            quantity=max(0,int(command[2]) if len(command)>2 else 1)\n            _R70_REPORT['parent_input_requests']+=quantity\n            state['pending'].append((actor,quantity,int(inv)))\n\n_R70_PARENT=agent\n\ndef agent(observation,configuration=None):\n    state=None\n    try:state=_r70_before(observation)\n    except Exception:_R70_REPORT['parent_input_errors']=_R70_REPORT.get('parent_input_errors',0)+1\n    result=_R70_PARENT(observation,configuration)\n    try:\n        if state is not None:_r70_after(observation,result,state)\n    except Exception:_R70_REPORT['parent_input_errors']=_R70_REPORT.get('parent_input_errors',0)+1\n    _R70_REPORT.update(getattr(_R70_PARENT,'telemetry',{}))\n    return result\n\nagent.telemetry=_R70_REPORT\nagent=globals().pop('agent')\n\ndef _r79_tomato_fertilizer_worthwhile(obs,action):\n    if obs['market']['prices']['FERTILIZER']<=30:return True\n    farm=obs['farms'][obs['player']];day=int(obs['step'])//24;bonus=0\n    for y in (5,6):\n        for x in range(5,10):\n            tile=farm['tiles'][y][x]\n            if not isinstance(tile,dict) or tile.get('crop')!='TOMATO':continue\n            birth=tile['planted_day'];until=tile.get('fertilized_until_day',-1)\n            bonus+=sum(until<d and 8<=d+1-birth<=11 for d in range(day,day+3))\n    if not bonus:return False\n    inventory=obs['market']['inventory']\n    price=max(1,_r37_market_price(",
    b'\'TOMATO\',inventory[\'TOMATO\']+bonus+10)-2)\n    fertilizer=max(1,_r37_market_price(\'FERTILIZER\',inventory[\'FERTILIZER\']-10)+2)\n    native_hires=sum(bool(o) and o[0]==\'HIRE\' for o in action.get(\'market\',[]))\n    extra_labor=_v219_fib(int(farm[\'hires_today\'])+native_hires+3)\n    return bonus*price>=2*(10*fertilizer+extra_labor)+100\n\nagent=globals().pop(\'agent\')\n\n# EXP216: original adaptation of economic feed and fertilizer-sale concepts.\n# Conceptual credit: Steven Lee Hans, "Lord Momo Returns", September12 snapshot.\n_R85_FEED = True\n_R85_FERT = True\n_R85_PARENT = agent\n_R85_STATES = {}\n_R85_REPORT = {}\n\ndef _r85_feed(obs, action):\n    step=int(obs[\'step\']);day=step//24\n    if not 10<=day<=28 or step%24>21:return action\n    player=int(obs[\'player\']);native=_IMPL.chassis.players[player]\n    tape=_v219_native_day(native,day)\n    expected=max(len(a.get(\'hands\',[])) for a in tape)\n    farm=obs[\'farms\'][player];positions=[farm[\'farmer\'],*farm[\'hands\']]\n    commands=[action.get(\'farmer\') or [\'PASS\'],*(action.get(\'hands\') or [])]\n    prices=obs[\'market\'][\'prices\'];changed=False\n    for actor,command in enumerate(commands[:expected+1]):\n        if command!=[\'FEED\'] or actor>=len(positions):continue\n        tile=_tile_at(farm[\'tiles\'],positions[actor])\n        if not isinstance(tile,dict) or tile.get(\'animal\') not in (\'GOOSE\',\'COW\',\'SHEEP\'):continue\n        if tile.get(\'fed_today\') or int(tile.get(\'consecutive_unfed\',0))!=0:continue\n        if int(obs[\'private\'][\'inventories\'][actor].get(\'WHEAT\',0))<=0:continue\n        item={\'GOOSE\':\'EGG\',\'COW\':\'MILK\',\'SHEEP\':\'WOOL\'}[tile[\'animal\']]\n        bonus=_r88_feed_bonus_cost(tile,day)\n        if bonus*(float(prices[item])+5)*1.25>=float(prices[\'WHEAT\']):continue\n        if not _r86_next_feed(obs,positions[actor]):continue\n        commands[actor]=[\'PASS\'];changed=True\n        _R85_REPORT[\'feed_skips\']+=1\n    if not changed:return action\n    result=copy.deepcopy(action);result[\'farmer\'],result[\'hands\']=commands[0],commands[1:]\n    return result\n\ndef _r85_reserve(obs, state):\n    step=int(obs[\'step\']);player=int(obs[\'player\']);native=_IMPL.chassis.players[player]\n    route=native[\'route\'];key=(route,step)\n    cache=state.setdefault(\'native_reserves\',{})\n    if route not in cache:\n        # Backward recurrence preserves field-before-market order within a turn.\n        reserve=[0]*720\n        for t in range(718,-1,-1):\n            a=_IMPL.chassis.routes[2 if t>=648 else route][t]\n            pickup=sum(max(0,int(c[2]) if len(c)>2 else 1) for c in [a.get(\'farmer\') or [\'PASS\'],*(a.get(\'hands\') or [])] if len(c)>1 and c[:2]==[\'PICKUP\',\'FERTILIZER\'])\n            purchase=sum(max(0,int(o[2])) for o in a.get(\'market\',[]) if len(o)>2 and o[:2]==[\'BUY_PRODUCT\',\'FERTILIZER\'])\n            reserve[t]=pickup+max(0,reserve[t+1]-purchase)\n        cache[route]=reserve\n    dedicated=0\n    for parent in (_V219_STATES.get(player,{}),_V233_STATES.get(player,{})):\n        for actor,role in parent.get(\'workers\',{}).items():\n            if not isinstance(role,dict) or not role.get(\'needs_fertilizer\') or role.get(\'loaded\'):continue\n            desired=role.get(\'fertilizer_quantity\',10 if role.get(\'kind\')==\'fertilizer\' else 5)\n            carried=obs[\'private\'][\'inventories\'][actor].get(\'FERTILIZER\',0)\n            dedicated+=max(0,desired-carried)\n        pending=parent.get(\'pending\') or {}\n        if pending.get(\'fertilizer\'):dedicated+=10\n    inputs=_R51_INPUT_STATES.get(player,{})\n    for actor,plan in {**inputs.get(\'workers\',{}),**(inputs.get(\'pending\') or {})}.items():\n        if not plan.get(\'loaded\'):dedicated+=max(0,int(plan[\'quantity\']))\n    return max(14,cache[route][min(719,step+1)]+dedicated)\n\ndef _r85_fertilizer(obs, action, state):\n    step=int(obs[\'step\']);day=step//24\n    if not 6<=day<=28:return action\n    market=action.get(\'market\',[])\n    if len(market)>=MAX_ORDERS or any(o and o[0]!=\'SELL\' for o in market):return action\n    stock=projected_shed(action,FarmView(obs))\n    held=max(0,int(stock.get(\'FERTILIZER\',0)))\n    sold=sum(max(0,int(o[2])) for o in market if len(o)>2 and o[:2]==[\'SELL\',\'FERTILIZER\'',
    b"])\n    extra=held-sold-_r85_reserve(obs,state)\n    if extra<=0:return action\n    result=copy.deepcopy(action);result['market'].append(['SELL','FERTILIZER',extra])\n    _R85_REPORT['fert_sale_turns']+=1;_R85_REPORT['fert_sale_units']+=extra\n    return result\n\ndef agent(observation, configuration=None):\n    result=_R85_PARENT(observation,configuration)\n    try:\n        step=int(observation['step']);player=int(observation['player'])\n        state=_R85_STATES.get(player)\n        if state is None or step<=state['step']:\n            state=_R85_STATES[player]={'step':-1}\n            _R85_REPORT.update(feed_skips=0,fert_sale_turns=0,fert_sale_units=0,economic_overlay_errors=0)\n        state['step']=step\n        if configuration is not None and any(configuration.get(k,v)!=v for k,v in [('boardSize',10),('turnsPerDay',24),('shedCapacity',100),('maxMarketOrdersPerTurn',10)]):return result\n        if _R85_FEED:result=_r85_feed(observation,result)\n        if _R85_FERT:result=_r85_fertilizer(observation,result,state)\n        if step%24==23:result=_r51_close_warehouse(observation,result)\n    except Exception:\n        _R85_REPORT['economic_overlay_errors']=_R85_REPORT.get('economic_overlay_errors',0)+1\n    _R85_REPORT.update(getattr(_R85_PARENT,'telemetry',{}))\n    return result\n\nagent.telemetry=_R85_REPORT\nagent=globals().pop('agent')\n\n# EXP217: planned next-day service is required before discretionary feed cuts.\n_R86_FEED_CACHE = {}\n\ndef _r86_next_feed(obs, target):\n    step=int(obs['step']);day=step//24\n    if day==28:return True  # No second dawn follows before game termination.\n    player=int(obs['player']);native=_IMPL.chassis.players[player]\n    tomorrow=day+1;route=2 if tomorrow>=27 else native['route'];key=(route,tomorrow)\n    if key not in _R86_FEED_CACHE:\n        positions=[(4,4)];wheat=[0];access=((4,4),(5,4),(4,5),(5,5));feeds=set()\n        for hour in range(24):\n            a=_IMPL.chassis.routes[route][tomorrow*24+hour]\n            commands=[a.get('farmer') or ['PASS'],*(a.get('hands') or [])]\n            for actor,command in enumerate(commands[:len(positions)]):\n                if not command:continue\n                pos=positions[actor];op=command[0]\n                if op in MOVES:\n                    dx,dy=MOVES[op];positions[actor]=(max(0,min(9,pos[0]+dx)),max(0,min(9,pos[1]+dy)))\n                elif command[:2]==['PICKUP','WHEAT'] and pos in access:\n                    wheat[actor]+=max(0,int(command[2]) if len(command)>2 else 1)\n                elif op=='FEED' and wheat[actor]>0:\n                    wheat[actor]-=1\n                    if hour<=21:feeds.add(pos)\n                elif op=='DROP' and pos in access:wheat[actor]=0\n                elif command[:2]==['PLACE','WHEAT'] and pos in access:\n                    wheat[actor]=max(0,wheat[actor]-max(0,int(command[2]) if len(command)>2 else 1))\n            for order in a.get('market',[]):\n                if order and order[0]=='HIRE':\n                    chosen=min(access,key=lambda p:(positions.count(p),access.index(p)))\n                    positions.append(chosen);wheat.append(0)\n        _R86_FEED_CACHE[key]=frozenset(feeds)\n    return tuple(target) in _R86_FEED_CACHE[key]\n\nagent=globals().pop('agent')\n\n# EXP219: charge care credits only when this feeding decision can affect them.\n_R88_PHASE = True\n_R88_HORIZON = True\n_R88_ANIMAL_DAYS = {'GOOSE': (4, 1), 'COW': (8, 2), 'SHEEP': (6, 3)}\n\n\ndef _r88_feed_bonus_cost(tile, day):\n    first, interval = _R88_ANIMAL_DAYS[tile['animal']]\n    first += int(tile['placed_day'])\n    tomorrow = day + 1\n    produces = tomorrow >= first and (tomorrow - first) % interval == 0\n    pending = max(0, int(tile.get('pending_care_bonus', 0)))\n    if _R88_PHASE and not produces:\n        pending = 0  # It remains banked on non-production dawns.\n    care = 1  # Conservative: charge one possible CARE even if not yet observed.\n    if _R88_HORIZON:\n        # Today's care is added AFTER tomorrow's production; its first possible\n        # payout is a later production dawn, which must occur before game end.\n        next_use = first\n        if next_u",
    b"se <= tomorrow:\n            next_use += ((tomorrow - next_use) // interval + 1) * interval\n        if next_use > 29:\n            care = 0\n    return pending + care\n\n\nagent = globals().pop('agent')\n\n# EXP226: retain physical grain for two complete days before trimming a buy.\n_R95_PARENT = agent\n_R95_REPORT = {}\n_R95_RESERVES = {}\n\ndef _r95_reserve(obs):\n    step=int(obs['step']);player=int(obs['player'])\n    native=_IMPL.chassis.players[player];route=native['route']\n    key=(route,step)\n    if key not in _R95_RESERVES:\n        demand=6  # Physical buffer beyond every scheduled pickup and sale.\n        for t in range(step+1,min(719,step+49)):\n            a=_IMPL.chassis.routes[2 if t>=648 else route][t]\n            for c in [a.get('farmer') or ['PASS'],*(a.get('hands') or [])]:\n                if c[:2]==['PICKUP','WHEAT']:\n                    demand+=max(0,int(c[2]) if len(c)>2 else 1)\n            for o in a.get('market',[]):\n                if len(o)>2 and o[:2]==['SELL','WHEAT']:\n                    demand+=max(0,int(o[2]))\n        _R95_RESERVES[key]=demand\n    demand=_R95_RESERVES[key]\n    # Reserve full feed for a possible southeast sheep commitment. Do not\n    # rely on its future discretionary buy, eligibility, or existing cargo.\n    if obs['town']['unlocked_shops'].count('YARN_STORE')>=2:\n        demand+=6*len({t//24 for t in range(step+1,step+49) if t//24>=12})\n    return demand\n\ndef _r95_replenish(obs,action):\n    step=int(obs['step'])\n    if not 10<=step//24<=11:return action\n    orders=action.get('market') or []\n    if not any(len(o)>2 and o[:2]==['BUY_PRODUCT','WHEAT'] and int(o[2])>0 for o in orders):return action\n    # Preserve all same-turn grain trading/arbitrage sequences unchanged.\n    if any(o[:2]==['SELL','WHEAT'] for o in orders):return action\n    farm,private=_PLANNER_NS['_clone_state'](obs['farms'][obs['player']],obs['private'])\n    commands=[action.get('farmer') or ['PASS'],*(action.get('hands') or [])]\n    for actor,c in enumerate(commands[:len(private['inventories'])]):\n        _PLANNER_NS['_apply_unit_action'](farm,private,actor,c,10,step//24,24,100)\n    held=max(0,int(private['shed'].get('WHEAT',0)))\n    reserve=_r95_reserve(obs);result=None;removed=0\n    for i,o in enumerate(orders):\n        if len(o)<3 or o[:2]!=['BUY_PRODUCT','WHEAT']:continue\n        quantity=max(0,int(o[2]));retained=min(quantity,max(0,reserve-held))\n        held+=retained\n        if retained<quantity:\n            if result is None:result=copy.deepcopy(action)\n            result['market'][i][2]=retained  # Zero keeps every later order slot.\n            removed+=quantity-retained\n    if result is None:return action\n    _R95_REPORT['replenishment_trim_turns']+=1\n    _R95_REPORT['replenishment_trim_units']+=removed\n    return result\n\ndef agent(observation,configuration=None):\n    result=_R95_PARENT(observation,configuration)\n    try:\n        if int(observation['step'])==0:\n            _R95_REPORT.update(replenishment_trim_turns=0,replenishment_trim_units=0,replenishment_errors=0)\n        if configuration is not None and any(configuration.get(k,v)!=v for k,v in [('boardSize',10),('turnsPerDay',24),('shedCapacity',100),('maxMarketOrdersPerTurn',10)]):return result\n        result=_r95_replenish(observation,result)\n    except Exception:\n        _R95_REPORT['replenishment_errors']=_R95_REPORT.get('replenishment_errors',0)+1\n    _R95_REPORT.update(getattr(_R95_PARENT,'telemetry',{}))\n    return result\n\nagent.telemetry=_R95_REPORT\nagent=globals().pop('agent')\n\n# EXP231: protect inputs using funded current orders without unassigned cash padding from observed physical resources.\n_R97_PARENT=agent\n_R97_REPORT={}\n_R97_LAST={}\n\ndef _r97_market_stock(shed,orders):\n    stock=dict(shed);buys={};sales={}\n    for index,order in enumerate(orders):\n        if len(order)<3:continue\n        op,item,n=order[:3];n=max(0,int(n))\n        if op=='SELL':\n            q=min(n,max(0,stock.get(item,0)));stock[item]=stock.get(item,0)-q;sales[index]=q\n        elif op in ('BUY_PRODUCT','BUY_ANIMAL'):\n            q=min(n,max(0,100-sum(stock.values())));stock[it",
    b"em]=stock.get(item,0)+q;buys[index]=q\n    return stock,buys,sales\n\ndef _r97_delivery(stock,private,night):\n    stock=dict(stock);lost={}\n    if night:\n        for inv in private['inventories']:\n            for item,q in inv.items():\n                q=max(0,int(q));take=min(q,max(0,100-sum(stock.values())))\n                stock[item]=stock.get(item,0)+take\n                if q>take:lost[item]=lost.get(item,0)+q-take\n    return stock,lost\n\ndef _r97_budget(obs,orders):\n    farm=obs['farms'][obs['player']];cost=0;hires=int(farm['hires_today'])\n    # At most ten 100-unit purchases per opponent turn. The additional 1000\n    # own units give an intentionally conservative upper bound on buy quotes.\n    prices={p:_r37_market_price(p,obs['market']['inventory'][p]-2000) for p in ('WHEAT','FERTILIZER')}\n    for order in orders:\n        if not order:continue\n        op=order[0]\n        if op=='HIRE':cost+=_v219_fib(hires);hires+=1\n        elif op=='BUY_LAND':cost+=4000\n        elif len(order)>2:\n            item=order[1];q=max(0,int(order[2]))\n            if op=='BUY_PRODUCT':cost+=q*prices[item]\n            elif op=='BUY_ANIMAL':cost+=q*{'GOOSE':300,'COW':400,'SHEEP':500}[item]\n            elif op=='BUY_SEED':cost+=q*{'WHEAT':10,'CARROT':20,'TOMATO':50,'STRAWBERRY':100,'MELON':80}[item]\n    return cost<=farm['money']  # No current sale proceeds are assumed.\n\ndef _r97_supply(obs,action):\n    step=int(obs['step']);player=int(obs['player']);day=step//24\n    if not 144<=step<695:return action\n    native=_IMPL.chassis.players[player]\n    future=_IMPL.chassis.routes[2 if step+1>=648 else native['route']][step+1]\n    commands=[future.get('farmer') or ['PASS'],*(future.get('hands') or [])]\n    following=_IMPL.chassis.routes[2 if step+2>=648 else native['route']][step+2]\n    next_orders=future.get('market') or []\n    prefund=0\n    if len(next_orders)==10 and not any(o[:2] in (['BUY_PRODUCT','WHEAT'],['SELL','WHEAT']) for o in next_orders):\n        later=[following.get('farmer') or ['PASS'],*(following.get('hands') or [])]\n        demand=lambda cs:sum(max(0,int(c[2]) if len(c)>2 else 1) for c in cs if c[:2]==['PICKUP','WHEAT'])\n        if demand(later):prefund=demand(commands)+demand(later)\n    if not prefund and not any(c[:2]==['PICKUP','WHEAT'] for c in commands):return action\n    orders=action.get('market') or []\n    if len(orders)>10 or not _r97_budget(obs,orders):\n        _R97_REPORT['supply_budget_declines']+=1;return action\n    farm,private=_PLANNER_NS['_clone_state'](obs['farms'][player],obs['private'])\n    for actor,c in enumerate([action.get('farmer') or ['PASS'],*(action.get('hands') or [])][:len(private['inventories'])]):\n        _PLANNER_NS['_apply_unit_action'](farm,private,actor,c,10,day,24,100)\n    positions=[tuple(farm['farmer']),*map(tuple,farm['hands'])];night=step%24==23\n    access=((4,4),(5,4),(4,5),(5,5))\n    if night:positions=[(4,4)]\n    else:\n        for order in orders:\n            if order and order[0]=='HIRE':positions.append(min(access,key=lambda p:(positions.count(p),access.index(p))))\n    need=sum(max(0,int(c[2]) if len(c)>2 else 1) for pos,c in zip(positions,commands) if pos in access and c[:2]==['PICKUP','WHEAT'])\n    need=max(need,prefund)\n    if not need:return action\n    original_stock,original_buys,_=_r97_market_stock(private['shed'],orders)\n    original_final,original_loss=_r97_delivery(original_stock,private,night)\n    if original_final.get('WHEAT',0)>=need:return action\n    result=copy.deepcopy(action);proposed=result['market'];blocked=False\n    def project(candidate):\n        stock,buys,sales=_r97_market_stock(private['shed'],candidate)\n        final,loss=_r97_delivery(stock,private,night)\n        safe=all(buys.get(i,0)>=q for i,q in original_buys.items()) and all(q<=original_loss.get(item,0) for item,q in loss.items())\n        return final,sales,safe\n    # Hold an existing grain sale first. Preserve all order indices and every\n    # originally funded buy; no extra overnight overflow may be introduced.\n    for index in range(len(proposed)-1,-1,-1):\n        if proposed[index][:2]!=['SELL','WHEAT']:cont",
    b'inue\n        final,sales,safe=project(proposed);shortage=max(0,need-final.get(\'WHEAT\',0))\n        if not shortage:break\n        sold=sales.get(index,0)\n        if not sold:continue\n        old=proposed[index][2];proposed[index][2]=max(0,sold-shortage)\n        after,_,safe=project(proposed)\n        if not safe or after.get(\'WHEAT\',0)<=final.get(\'WHEAT\',0):proposed[index][2]=old\n    final,_,safe=project(proposed);shortage=max(0,need-final.get(\'WHEAT\',0))\n    if shortage:\n        last_sale=max((i for i,o in enumerate(proposed) if o[:2]==[\'SELL\',\'WHEAT\']),default=-1)\n        index=next((i for i in range(len(proposed)-1,last_sale,-1) if proposed[i][:2]==[\'BUY_PRODUCT\',\'WHEAT\']),None)\n        if index is not None:proposed[index][2]=max(0,int(proposed[index][2]))+shortage\n        elif len(proposed)<10:proposed.append([\'BUY_PRODUCT\',\'WHEAT\',shortage])\n        else:_R97_REPORT[\'supply_slot_declines\']+=1;return action\n    final,_,safe=project(proposed)\n    if not safe or final.get(\'WHEAT\',0)<need:\n        _R97_REPORT[\'supply_capacity_declines\']+=1;return action\n    if not _r97_budget(obs,proposed):\n        _R97_REPORT[\'supply_budget_declines\']+=1;return action\n    if prefund:\n        _R97_REPORT[\'supply_prefund_changes\']+=1\n        _R97_REPORT[\'supply_prefund_units\']+=max(0,final.get(\'WHEAT\',0)-original_final.get(\'WHEAT\',0))\n    if step<288:\n        _R97_REPORT[\'supply_early_changes\']+=1\n        _R97_REPORT[\'supply_early_units\']+=max(0,final.get(\'WHEAT\',0)-original_final.get(\'WHEAT\',0))\n    _R97_REPORT[\'supply_guard_changes\']+=1\n    _R97_REPORT[\'supply_grain_protected\']+=final.get(\'WHEAT\',0)-original_final.get(\'WHEAT\',0)\n    _R97_REPORT[\'supply_buy_units\']+=sum(max(0,int(o[2])) for o in proposed if o[:2]==[\'BUY_PRODUCT\',\'WHEAT\'])-sum(max(0,int(o[2])) for o in orders if o[:2]==[\'BUY_PRODUCT\',\'WHEAT\'])\n    return result\n\ndef agent(observation,configuration=None):\n    result=_R97_PARENT(observation,configuration)\n    try:\n        player=int(observation[\'player\']);step=int(observation[\'step\'])\n        if player not in _R97_LAST or step<=_R97_LAST[player]:\n            _R97_REPORT.update(supply_guard_changes=0,supply_grain_protected=0,supply_buy_units=0,supply_early_changes=0,supply_early_units=0,supply_prefund_changes=0,supply_prefund_units=0,supply_slot_declines=0,supply_capacity_declines=0,supply_budget_declines=0,supply_errors=0)\n        _R97_LAST[player]=step\n        if configuration is None or all(configuration.get(k,v)==v for k,v in [(\'boardSize\',10),(\'turnsPerDay\',24),(\'shedCapacity\',100),(\'maxMarketOrdersPerTurn\',10),(\'farmHandCostMult\',1)]):result=_r97_supply(observation,result)\n    except Exception:_R97_REPORT[\'supply_errors\']=_R97_REPORT.get(\'supply_errors\',0)+1\n    _R97_REPORT.update(getattr(_R97_PARENT,\'telemetry\',{}))\n    return result\n\nagent.telemetry=_R97_REPORT\nagent=globals().pop(\'agent\')\n\n"""Original funded planting and first-dawn labor contract, Ahmed Berat Ozer."""\n_R124_PARENT=agent\n_R124_STATES={}\n_R124_REPORT={}\n\ndef _r124_labor_reserve(native):\n    n=sum(bool(o) and o[0]==\'HIRE\' for o in native[24].get(\'market\',[]))\n    return sum(_v219_fib(i) for i in range(n)),n\n\ndef _r124_seed_budget(obs,action,reserve):\n    if not any(o and o[0]==\'BUY_SEED\' for o in action.get(\'market\',[])):return action\n    player=int(obs[\'player\']);budget=dict(obs,farms=[dict(f) for f in obs[\'farms\']]);budget[\'farms\'][player][\'money\']-=reserve\n    if _r97_budget(budget,action.get(\'market\',[])):return action\n    result=copy.deepcopy(action)\n    for i in range(len(result[\'market\'])-1,-1,-1):\n        order=result[\'market\'][i]\n        if len(order)<3 or order[0]!=\'BUY_SEED\':continue\n        before=max(0,int(order[2]));order[2]=before\n        while order[2]>0 and not _r97_budget(budget,result[\'market\']):order[2]-=1\n        n=before-order[2]\n        if n:\n            _R124_REPORT[\'opening_seed_budget_units\']+=n\n            _R124_REPORT[\'opening_seed_budget_cost\']+=n*{\'WHEAT\':10,\'CARROT\':20,\'TOMATO\':50,\'STRAWBERRY\':100,\'MELON\':80}[order[1]]\n        if _r97_budget(budget,result[\'market\']):break\n    return result\n\ndef _r124_atomic(obs,action,state',
    b'):\n    commands=[action.get(\'farmer\') or [\'PASS\'],*(action.get(\'hands\') or [])]\n    demand={}\n    for c in commands:\n        if len(c)>1 and c[0]==\'PLANT\':demand[c[1]]=demand.get(c[1],0)+1\n    if not demand:return action\n    available=obs[\'private\'][\'seeds\'];blocked={p for p,n in demand.items() if n>available.get(p,0)}\n    farm,private=_PLANNER_NS[\'_clone_state\'](obs[\'farms\'][obs[\'player\']],obs[\'private\'])\n    changed=False;kept=[]\n    for actor,c in enumerate(commands):\n        if len(c)>1 and c[0]==\'PLANT\':\n            pos=None if actor>=len(private[\'inventories\']) else farm[\'farmer\'] if actor==0 else farm[\'hands\'][actor-1]\n            valid=pos is not None and farm[\'tiles\'][pos[1]][pos[0]] is None and private[\'seeds\'].get(c[1],0)>0\n            if not valid:\n                commands[actor]=[\'PASS\'];changed=True;_R124_REPORT[\'opening_atomic_dropped\']+=1\n            elif c[1] in blocked:\n                kept.append(dict(xy=list(pos),crop=c[1],birth=0));_R124_REPORT[\'opening_atomic_rescued_requests\']+=1\n        if actor<len(private[\'inventories\']):_PLANNER_NS[\'_apply_unit_action\'](farm,private,actor,commands[actor],10,0,24,100)\n    if kept:state[\'pending_plants\']=kept\n    if changed:\n        action=dict(action,farmer=commands[0],hands=commands[1:])\n    return action\n\ndef agent(observation,configuration=None):\n    result=_R124_PARENT(observation,configuration)\n    try:\n        player=int(observation[\'player\']);step=int(observation[\'step\']);state=_R124_STATES.get(player)\n        if state is None or step<=state[\'step\']:\n            state=_R124_STATES[player]={\'step\':-1,\'pending_plants\':[]}\n            _R124_REPORT.update(opening_seed_budget_units=0,opening_seed_budget_cost=0,opening_atomic_dropped=0,opening_atomic_rescued_requests=0,opening_atomic_rescued_confirmed=0,opening_atomic_plant_errors=0,opening_day1_cash=0,opening_day1_hires_requested=0,opening_day1_hires_confirmed=0,opening_day1_hire_shortfalls=0,opening_contract_errors=0)\n        state[\'step\']=step;farm=observation[\'farms\'][player]\n        for p in state.pop(\'pending_plants\',[]):\n            x,y=p[\'xy\'];t=farm[\'tiles\'][y][x]\n            if isinstance(t,dict) and t.get(\'crop\')==p[\'crop\'] and t.get(\'planted_day\')==p[\'birth\']:_R124_REPORT[\'opening_atomic_rescued_confirmed\']+=1\n            else:_R124_REPORT[\'opening_atomic_plant_errors\']+=1\n        standard=configuration is None or all(configuration.get(k,v)==v for k,v in [(\'boardSize\',10),(\'turnsPerDay\',24),(\'shedCapacity\',100),(\'maxMarketOrdersPerTurn\',10),(\'farmHandCostMult\',1)])\n        if standard and 0<=step<24:\n            native=_ROUTES[_IMPL.chassis.players[player][\'route\']];reserve,hires=_r124_labor_reserve(native)\n            result=_r124_seed_budget(observation,result,reserve);result=_r124_atomic(observation,result,state)\n        if step==24:\n            _R124_REPORT[\'opening_day1_cash\']=farm[\'money\'];state[\'hires\']=sum(bool(o) and o[0]==\'HIRE\' for o in result.get(\'market\',[]));_R124_REPORT[\'opening_day1_hires_requested\']=state[\'hires\']\n        if step==25:\n            actual=len(farm[\'hands\']);_R124_REPORT[\'opening_day1_hires_confirmed\']=actual;_R124_REPORT[\'opening_day1_hire_shortfalls\']=max(0,state.get(\'hires\',0)-actual)\n    except Exception:_R124_REPORT[\'opening_contract_errors\']=_R124_REPORT.get(\'opening_contract_errors\',0)+1\n    _R124_REPORT.update(getattr(_R124_PARENT,\'telemetry\',{}))\n    return result\nagent.telemetry=_R124_REPORT\nagent=globals().pop(\'agent\')\n\n"""Fund urgent grain at the first quote slot; avoid unwatered last-hour plants.\nOriginal safety contracts by Ahmed Berat Ozer, EXP258.\n"""\n_R127_PARENT=agent\n_R127_STATES={}\n_R127_REPORT={}\n\ndef _r127_fields(obs,action):\n    farm,private=_PLANNER_NS[\'_clone_state\'](obs[\'farms\'][obs[\'player\']],obs[\'private\'])\n    commands=[action.get(\'farmer\') or [\'PASS\'],*(action.get(\'hands\') or [])]\n    demand={}\n    for c in commands:\n        if len(c)>1 and c[0]==\'PLANT\':demand[c[1]]=demand.get(c[1],0)+1\n    blocked={p for p,n in demand.items() if n>private[\'seeds\'].get(p,0)}\n    for actor,c in enumerate(commands[:len(private[\'inventories\'])]):\n        if ',
    b"len(c)>1 and c[0]=='PLANT' and c[1] in blocked:continue\n        _PLANNER_NS['_apply_unit_action'](farm,private,actor,c,10,int(obs['step'])//24,24,100)\n    return farm,private\n\ndef _r127_last_hour(obs,action):\n    if int(obs['step'])%24!=23:return action\n    commands=[action.get('farmer') or ['PASS'],*(action.get('hands') or [])]\n    if not any(c and c[0]=='PLANT' for c in commands):return action\n    result=copy.deepcopy(action);changed=False\n    # Removing rejected requests can unblock the engine's atomic crop batch.\n    # Recompute until every retained request ends the turn with a watered crop.\n    for _ in range(len(commands)+1):\n        farm,_=_r127_fields(obs,result);original=obs['farms'][obs['player']]\n        positions=[original['farmer'],*original['hands']];drop=[]\n        for actor,c in enumerate(commands):\n            if not c or c[0]!='PLANT':continue\n            tile=None\n            if actor<len(positions):\n                x,y=positions[actor];tile=farm['tiles'][y][x]\n            if not (isinstance(tile,dict) and tile.get('kind')=='PLANT' and tile.get('crop')==c[1] and tile.get('planted_day')==int(obs['step'])//24 and tile.get('watered_today')):drop.append(actor)\n        if not drop:break\n        for actor in drop:commands[actor]=['PASS']\n        result['farmer']=commands[0];result['hands']=commands[1:];changed=True\n        _R127_REPORT['last_hour_plants_dropped']+=len(drop)\n    return result if changed else action\n\ndef _r127_prefix_bound(obs,quantity):\n    inventory=int(obs['market']['inventory']['WHEAT'])\n    # Both players quote one unit before either commits. Before own unit j,\n    # at most j-1 own and j-1 opponent wheat purchases have depleted inventory.\n    return sum(_r37_market_price('WHEAT',inventory-(2*j-1)) for j in range(1,quantity+1))\n\ndef _r127_priority(obs,action,state):\n    step=int(obs['step']);player=int(obs['player'])\n    if not 144<=step<695:return action\n    orders=action.get('market') or []\n    if len(orders)>9 or any(o[:2] in (['BUY_PRODUCT','WHEAT'],['SELL','WHEAT']) for o in orders):return action\n    native=_IMPL.chassis.players[player]\n    future=_IMPL.chassis.routes[2 if step+1>=648 else native['route']][step+1]\n    commands=[future.get('farmer') or ['PASS'],*(future.get('hands') or [])]\n    if not any(c[:2]==['PICKUP','WHEAT'] for c in commands):return action\n    if not _r97_budget(obs,orders):return action\n    farm,private=_r127_fields(obs,action);night=step%24==23\n    access=((4,4),(5,4),(4,5),(5,5));positions=[tuple(farm['farmer']),*map(tuple,farm['hands'])]\n    if night:positions=[(4,4)]\n    else:\n        for o in orders:\n            if o and o[0]=='HIRE':positions.append(min(access,key=lambda p:(positions.count(p),access.index(p))))\n    need=sum(max(0,int(c[2]) if len(c)>2 else 1) for pos,c in zip(positions,commands) if pos in access and c[:2]==['PICKUP','WHEAT'])\n    stock,buys,_=_r97_market_stock(private['shed'],orders);before,loss=_r97_delivery(stock,private,night)\n    shortage=max(0,need-before.get('WHEAT',0))\n    if not shortage or shortage>100-sum(private['shed'].values()):return action\n    cost=_r127_prefix_bound(obs,shortage)\n    budget=dict(obs,farms=[dict(f) for f in obs['farms']]);budget['farms'][player]['money']-=cost\n    if not _r97_budget(budget,orders):return action\n    proposed=[['BUY_PRODUCT','WHEAT',shortage],*copy.deepcopy(orders)]\n    stock,after_buys,_=_r97_market_stock(private['shed'],proposed);after,after_loss=_r97_delivery(stock,private,night)\n    if after.get('WHEAT',0)<need or any(after_buys.get(i+1,0)<q for i,q in buys.items()) or any(q>loss.get(p,0) for p,q in after_loss.items()):return action\n    state['pending_grain']=(step+1,after.get('WHEAT',0),shortage)\n    _R127_REPORT['priority_grain_orders']+=1;_R127_REPORT['priority_grain_units']+=shortage\n    return dict(action,market=proposed)\n\ndef agent(observation,configuration=None):\n    result=_R127_PARENT(observation,configuration)\n    try:\n        player=int(observation['player']);step=int(observation['step']);state=_R127_STATES.get(player)\n        if state is None or step<=state['step']:\n         ",
    b'   state=_R127_STATES[player]={\'step\':-1}\n            _R127_REPORT.update(last_hour_plants_dropped=0,priority_grain_orders=0,priority_grain_units=0,priority_grain_confirmed=0,priority_grain_shortfalls=0,priority_contract_errors=0)\n        pending=state.pop(\'pending_grain\',None)\n        if pending and step==pending[0]:\n            if observation[\'private\'][\'shed\'].get(\'WHEAT\',0)>=pending[1]:_R127_REPORT[\'priority_grain_confirmed\']+=pending[2]\n            else:_R127_REPORT[\'priority_grain_shortfalls\']+=1\n        state[\'step\']=step\n        standard=configuration is None or all(configuration.get(k,v)==v for k,v in [(\'boardSize\',10),(\'turnsPerDay\',24),(\'shedCapacity\',100),(\'maxMarketOrdersPerTurn\',10),(\'farmHandCostMult\',1)])\n        if standard:result=_r127_priority(observation,_r127_last_hour(observation,result),state)\n    except Exception:_R127_REPORT[\'priority_contract_errors\']=_R127_REPORT.get(\'priority_contract_errors\',0)+1\n    _R127_REPORT.update(getattr(_R127_PARENT,\'telemetry\',{}))\n    return result\nagent.telemetry=_R127_REPORT\nagent=globals().pop(\'agent\')\n\n"""Observed-input service order and guaranteed first-sale funding, Ahmed Berat Ozer."""\n_R128_PARENT=agent\n_R128_STATES={}\n_R128_REPORT={}\n\ndef _r128_commands(action):\n    return [action.get(\'farmer\') or [\'PASS\'],*(action.get(\'hands\') or [])]\n\ndef _r128_future(obs,offset=1):\n    step=int(obs[\'step\'])+offset;native=_IMPL.chassis.players[int(obs[\'player\'])]\n    return _IMPL.chassis.routes[2 if step>=648 else native[\'route\']][step]\n\ndef _r128_sale_credit(obs,action):\n    orders=action.get(\'market\') or []\n    if not orders or len(orders[0])<3 or orders[0][0]!=\'SELL\' or orders[0][1]==\'WHEAT\':return 0\n    item=orders[0][1]\n    if item not in obs[\'market\'][\'inventory\']:return 0\n    _,private=_r127_fields(obs,action)\n    q=min(max(0,int(orders[0][2])),max(0,int(private[\'shed\'].get(item,0))))\n    inventory=int(obs[\'market\'][\'inventory\'][item])\n    return sum(_r37_market_price(item,inventory+2*j) for j in range(q))\n\ndef _r128_next_need(obs,farm,orders,advanced):\n    step=int(obs[\'step\']);access=((4,4),(5,4),(4,5),(5,5))\n    positions=[tuple(farm[\'farmer\']),*map(tuple,farm[\'hands\'])]\n    if step%24==23:positions=[(4,4)]\n    else:\n        for order in orders:\n            if order and order[0]==\'HIRE\':positions.append(min(access,key=lambda p:(positions.count(p),access.index(p))))\n    return sum(max(0,int(c[2]) if len(c)>2 else 1) for actor,(p,c) in enumerate(zip(positions,_r128_commands(_r128_future(obs)))) if actor not in advanced and p in access and c[:2]==[\'PICKUP\',\'WHEAT\'])\n\ndef _r128_field_safe(obs,before,after,advanced):\n    orders=before.get(\'market\') or []\n    if not _r97_budget(obs,orders):return False,None\n    oldfarm,oldprivate=_r127_fields(obs,before);farm,private=_r127_fields(obs,after)\n    oldcommands=_r128_commands(before);commands=_r128_commands(after)\n    for i,c in enumerate(oldcommands[:len(private[\'inventories\'])]):\n        if c and c[0]==\'PICKUP\' and c==commands[i]:\n            item=c[1]\n            if private[\'inventories\'][i].get(item,0)<oldprivate[\'inventories\'][i].get(item,0):return False,None\n    oldstock,oldbuys,oldsales=_r97_market_stock(oldprivate[\'shed\'],orders)\n    stock,buys,sales=_r97_market_stock(private[\'shed\'],orders)\n    oldfinal,oldloss=_r97_delivery(oldstock,oldprivate,int(obs[\'step\'])%24==23)\n    final,loss=_r97_delivery(stock,private,int(obs[\'step\'])%24==23)\n    if any(buys.get(i,0)<q for i,q in oldbuys.items()) or any(sales.get(i,0)<q for i,q in oldsales.items()) or any(q>oldloss.get(p,0) for p,q in loss.items()):return False,None\n    need=_r128_next_need(obs,farm,orders,advanced)\n    pending=_R127_STATES.get(int(obs[\'player\']),{}).get(\'pending_grain\')\n    if pending and pending[0]==int(obs[\'step\'])+1:need=max(need,pending[1])\n    if final.get(\'WHEAT\',0)<min(need,oldfinal.get(\'WHEAT\',0)):return False,None\n    return True,private\n\ndef _r128_food_need(obs,farm,private,actor):\n    farm,private=copy.deepcopy(farm),copy.deepcopy(private);missing=0;day=int(obs[\'step\'])//24\n    for offset in range(1,min(5,24-int(obs[\'step\'])%24)):\n        cs=',
    b"_r128_commands(_r128_future(obs,offset));c=cs[actor] if actor<len(cs) else ['PASS']\n        if c[:2]==['PICKUP','WHEAT']:break\n        if c==['FEED']:\n            x,y=farm['farmer'] if actor==0 else farm['hands'][actor-1];tile=farm['tiles'][y][x]\n            if isinstance(tile,dict) and tile.get('animal') and not tile.get('fed_today') and private['inventories'][actor].get('WHEAT',0)<=0:\n                private['inventories'][actor]['WHEAT']=1;missing+=1\n        _PLANNER_NS['_apply_unit_action'](farm,private,actor,c,10,day,24,100)\n    return missing\n\ndef _r128_service(obs,action,state):\n    step=int(obs['step']);farm=obs['farms'][obs['player']];private=obs['private'];positions=[farm['farmer'],*farm['hands']]\n    for item in state.pop('arrivals',[]):\n        if item['step']!=step or item['actor']>=len(private['inventories']) or private['inventories'][item['actor']].get('WHEAT',0)<item['expected']:_R128_REPORT['service_arrival_errors']+=1\n        else:_R128_REPORT['service_confirmed_units']+=item['quantity']\n    old_pending=state.pop('swaps',{});result=action;cs=_r128_commands(result)\n    for actor,pending in old_pending.items():\n        if step!=pending['step'] or actor>=len(positions) or list(positions[actor])!=pending['xy'] or cs[actor]!=pending['pickup']:\n            _R128_REPORT['service_swap_errors']+=1;continue\n        x,y=positions[actor];tile=farm['tiles'][y][x]\n        if not isinstance(tile,dict) or tile.get('animal')!=pending['animal'] or tile.get('placed_day')!=pending['birth']:\n            _R128_REPORT['service_swap_errors']+=1;continue\n        if private['inventories'][actor].get('WHEAT',0)<pending['quantity']+1:\n            _R128_REPORT['service_swap_errors']+=1;continue\n        cs[actor]=['PASS'] if tile.get('fed_today') else ['FEED']\n        _R128_REPORT['service_swaps_completed']+=1\n    if old_pending:\n        proposed=dict(result,farmer=cs[0],hands=cs[1:]);safe,_=_r128_field_safe(obs,result,proposed,{})\n        if safe:result=proposed\n        else:_R128_REPORT['service_swap_errors']+=1\n    if not 144<=step<647:return result\n    cs=_r128_commands(result);future=_r128_commands(_r128_future(obs));proposed=copy.deepcopy(result);commands=_r128_commands(proposed)\n    swaps={};arrivals=[];prefetches=0\n    projected_farm,projected_private=_r127_fields(obs,result)\n    for actor,c in enumerate(cs[:len(private['inventories'])]):\n        x,y=positions[actor]\n        if (x,y) not in ((4,4),(5,4),(4,5),(5,5)):continue\n        held=max(0,int(private['inventories'][actor].get('WHEAT',0)));tile=farm['tiles'][y][x]\n        nxt=future[actor] if actor<len(future) else ['PASS']\n        if c==['FEED'] and held==0 and step%24<=21 and isinstance(tile,dict) and tile.get('animal') and not tile.get('fed_today') and nxt[:2]==['PICKUP','WHEAT']:\n            q=max(0,int(nxt[2]) if len(nxt)>2 else 1)\n            if not q:continue\n            commands[actor]=['PICKUP','WHEAT',q+1]\n            swaps[actor]=dict(step=step+1,xy=[x,y],animal=tile['animal'],birth=tile.get('placed_day'),quantity=q,pickup=copy.deepcopy(nxt))\n            arrivals.append(dict(step=step+1,actor=actor,expected=q+1,quantity=q+1))\n        elif c==['PASS']:\n            q=_r128_food_need(obs,projected_farm,projected_private,actor)\n            if q:\n                commands[actor]=['PICKUP','WHEAT',q];prefetches+=1\n                arrivals.append(dict(step=step+1,actor=actor,expected=held+q,quantity=q))\n    if not arrivals:return result\n    proposed['farmer']=commands[0];proposed['hands']=commands[1:]\n    safe,after=_r128_field_safe(obs,result,proposed,swaps)\n    if not safe or any(after['inventories'][p['actor']].get('WHEAT',0)<p['expected'] for p in arrivals):\n        _R128_REPORT['service_capacity_declines']+=1;return result\n    state['swaps']=swaps;state['arrivals']=arrivals\n    _R128_REPORT['service_swaps_started']+=len(swaps);_R128_REPORT['service_idle_preloads']+=prefetches\n    _R128_REPORT['service_requested_units']+=sum(p['quantity'] for p in arrivals)\n    return proposed\n\ndef _r128_credit_supply(obs,action,state):\n    step=int(obs['step'])\n    if not 144<=st",
    b'ep<695 or _r97_budget(obs,action.get(\'market\') or []):return action\n    credit=_r128_sale_credit(obs,action)\n    if not credit:return action\n    budget=dict(obs,farms=[dict(f) for f in obs[\'farms\']]);budget[\'farms\'][obs[\'player\']][\'money\']+=credit\n    if not _r97_budget(budget,action.get(\'market\') or []):return action\n    result=_r97_supply(budget,action)\n    if result==action:return action\n    assert result[\'market\'][0]==action[\'market\'][0]\n    _,private=_r127_fields(obs,result);stock,_,_=_r97_market_stock(private[\'shed\'],result[\'market\']);final,_=_r97_delivery(stock,private,step%24==23)\n    quantity=lambda a:sum(max(0,int(o[2])) for o in a.get(\'market\',[]) if len(o)>2 and o[:2]==[\'BUY_PRODUCT\',\'WHEAT\'])\n    extra=max(0,quantity(result)-quantity(action));state[\'credit_pending\']=(step+1,final.get(\'WHEAT\',0),extra)\n    _R128_REPORT[\'sale_credit_orders\']+=1;_R128_REPORT[\'sale_credit_lower_bound\']+=credit;_R128_REPORT[\'sale_credit_grain_units\']+=extra\n    return result\n\ndef agent(observation,configuration=None):\n    result=_R128_PARENT(observation,configuration)\n    try:\n        player=int(observation[\'player\']);step=int(observation[\'step\']);state=_R128_STATES.get(player)\n        if state is None or step<=state[\'step\']:\n            state=_R128_STATES[player]={\'step\':-1}\n            _R128_REPORT.update(service_requested_units=0,service_confirmed_units=0,service_idle_preloads=0,service_swaps_started=0,service_swaps_completed=0,service_capacity_declines=0,service_arrival_errors=0,service_swap_errors=0,sale_credit_orders=0,sale_credit_lower_bound=0,sale_credit_grain_units=0,sale_credit_confirmed_units=0,sale_credit_errors=0,service_errors=0)\n        state[\'step\']=step;pending=state.pop(\'credit_pending\',None)\n        if pending:\n            if pending[0]!=step or observation[\'private\'][\'shed\'].get(\'WHEAT\',0)<pending[1]:_R128_REPORT[\'sale_credit_errors\']+=1\n            else:_R128_REPORT[\'sale_credit_confirmed_units\']+=pending[2]\n        standard=configuration is None or all(configuration.get(k,v)==v for k,v in [(\'boardSize\',10),(\'turnsPerDay\',24),(\'shedCapacity\',100),(\'maxMarketOrdersPerTurn\',10),(\'farmHandCostMult\',1)])\n        if standard:result=_r128_credit_supply(observation,_r128_service(observation,result,state),state)\n    except Exception:_R128_REPORT[\'service_errors\']=_R128_REPORT.get(\'service_errors\',0)+1\n    _R128_REPORT.update(getattr(_R128_PARENT,\'telemetry\',{}));_R128_REPORT.update(_R97_REPORT)\n    return result\nagent.telemetry=_R128_REPORT\nagent=globals().pop(\'agent\')\n\n_R148_OVERFLOW=True\n_R148_SEEDS=False\n# Original targeted contracts, Ahmed Berat Ozer, EXP277.\n# Uses only current observations and the agent\'s own existing raw plan.\n_R148_PARENT=agent\n_R148_REPORT={}\n_R148_PENDING={}\n\n\ndef _r148_same_stock(a,b):\n    return all(int(a.get(p,0))==int(b.get(p,0)) for p in set(a)|set(b))\n\n\ndef _r148_overflow(obs,action):\n    """Sell only inventory replaced by otherwise destroyed dawn cargo.\n\n    All original orders/field jobs remain in place. The COMPLETE warehouse\n    vector after dawn must match the original funded action exactly.\n    """\n    if int(obs[\'step\'])%24!=23:return action\n    orders=action.get(\'market\') or []\n    if len(orders)>=10 or not _r97_budget(obs,orders):return action\n    _,private=_r127_fields(obs,action)\n    stock,_,_=_r97_market_stock(private[\'shed\'],orders)\n    original,loss=_r97_delivery(stock,private,True)\n    if not loss:return action\n    # The deposits are ordered. Recoverable cargo is the discarded suffix in\n    # that same order, not an unordered product total or future forecast.\n    remaining=max(0,100-sum(stock.values()));tail=[]\n    for bag in private[\'inventories\']:\n        for item,n in bag.items():\n            n=max(0,int(n));take=min(n,remaining);remaining-=take\n            if n>take:tail.extend([item]*(n-take))\n    released={};best=None\n    for item in tail:\n        released[item]=released.get(item,0)+1\n        if item not in obs[\'market\'][\'prices\'] or released[item]>stock.get(item,0):break\n        if len(orders)+len(released)>10:break\n        proposed=list(orders)+[[\'SELL\',p,n] for',
    b' p,n in released.items()]\n        after,_,_=_r97_market_stock(private[\'shed\'],proposed)\n        final,new_loss=_r97_delivery(after,private,True)\n        if _r148_same_stock(original,final):best=(proposed,dict(released),final,new_loss)\n    if best is None:return action\n    proposed,released,final,new_loss=best\n    _R148_REPORT[\'overflow_turns\']+=1\n    _R148_REPORT[\'overflow_units_reclaimed\']+=sum(released.values())\n    _R148_REPORT[\'overflow_quote_exposure\']+=sum(n*obs[\'market\'][\'prices\'][p] for p,n in released.items())\n    _R148_PENDING[int(obs[\'player\'])]=(int(obs[\'step\'])+1,dict(final))\n    return dict(action,market=proposed)\n\n\ndef _r148_atomic(obs,action):\n    """A shortage must not cancel every otherwise executable same-crop plant."""\n    commands=[list(c) for c in [action.get(\'farmer\') or [\'PASS\'],*(action.get(\'hands\') or [])]]\n    demand={}\n    for c in commands:\n        if len(c)>1 and c[0]==\'PLANT\':demand[c[1]]=demand.get(c[1],0)+1\n    blocked={p for p,n in demand.items() if n>obs[\'private\'][\'seeds\'].get(p,0)}\n    if not blocked:return action\n    farm,private=_PLANNER_NS[\'_clone_state\'](obs[\'farms\'][obs[\'player\']],obs[\'private\'])\n    kept=removed=0\n    for actor,c in enumerate(commands):\n        if len(c)>1 and c[0]==\'PLANT\' and c[1] in blocked:\n            pos=None if actor>=len(private[\'inventories\']) else farm[\'farmer\'] if actor==0 else farm[\'hands\'][actor-1]\n            valid=pos is not None and farm[\'tiles\'][pos[1]][pos[0]] is None and private[\'seeds\'].get(c[1],0)>0\n            if not valid:commands[actor]=[\'PASS\'];removed+=1\n            else:kept+=1\n        if actor<len(private[\'inventories\']):\n            _PLANNER_NS[\'_apply_unit_action\'](farm,private,actor,commands[actor],10,int(obs[\'step\'])//24,24,100)\n    if not removed:return action\n    result=dict(action,farmer=commands[0],hands=commands[1:])\n    # The inherited final-hour watering contract still governs any rescue.\n    result=_r127_last_hour(obs,result)\n    _R148_REPORT[\'atomic_turns\']+=1;_R148_REPORT[\'atomic_kept_requests\']+=kept;_R148_REPORT[\'atomic_removed_requests\']+=removed\n    return result\n\n\ndef _r148_seed_prefund(obs,action):\n    """Fund next-turn valid own-plan planting from already available cash.\n\n    No dawn/shop prediction, displaced purchases, new land or future opponent\n    observation. Future geometry is obtained from exact current unit effects.\n    """\n    step=int(obs[\'step\']);orders=action.get(\'market\') or []\n    if step<24 or step>=695 or step%24 in (22,23) or len(orders)>=10:return action\n    future=_r128_future(obs)\n    commands=[future.get(\'farmer\') or [\'PASS\'],*(future.get(\'hands\') or [])]\n    if not any(c and c[0]==\'PLANT\' for c in commands):return action\n    if not _r97_budget(obs,orders):return action\n    farm,private=_r127_fields(obs,action)\n    # Avoid predicting geometry changed by a land purchase in this callback.\n    if any(o and o[0]==\'BUY_LAND\' for o in orders):return action\n    access=((4,4),(5,4),(4,5),(5,5));positions=[tuple(farm[\'farmer\']),*map(tuple,farm[\'hands\'])]\n    for order in orders:\n        if order and order[0]==\'HIRE\':\n            pos=min(access,key=lambda p:(positions.count(p),access.index(p)))\n            positions.append(pos);farm[\'hands\'].append(list(pos));private[\'inventories\'].append({})\n        elif len(order)>2 and order[0]==\'BUY_SEED\':\n            private[\'seeds\'][order[1]]=private[\'seeds\'].get(order[1],0)+max(0,int(order[2]))\n    available=dict(private[\'seeds\']);intended={}\n    # Simulate the known next own commands with virtual seeds, solely to count\n    # physically valid births. The current atomic repair drops invalid requests.\n    for c in commands:\n        if len(c)>1 and c[0]==\'PLANT\':intended[c[1]]=intended.get(c[1],0)+1\n    for item,n in intended.items():private[\'seeds\'][item]=available.get(item,0)+n\n    before=dict(private[\'seeds\'])\n    for actor,c in enumerate(commands[:len(private[\'inventories\'])]):\n        _PLANNER_NS[\'_apply_unit_action\'](farm,private,actor,c,10,step//24,24,100)\n    short={item:max(0,before[item]-private[\'seeds\'].get(item,0)-available.get(item,0)) for item in inte',
    b'nded}\n    short={item:n for item,n in short.items() if n>0}\n    if not short or len(orders)+len(short)>10:return action\n    proposed=list(orders)+[[\'BUY_SEED\',item,n] for item,n in sorted(short.items())]\n    if not _r97_budget(obs,proposed):return action\n    _R148_REPORT[\'seed_prefund_turns\']+=1;_R148_REPORT[\'seed_prefund_units\']+=sum(short.values())\n    return dict(action,market=proposed)\n\n\ndef agent(observation,configuration=None):\n    action=_R148_PARENT(observation,configuration)\n    try:\n        player=int(observation[\'player\']);step=int(observation[\'step\'])\n        if step==0:\n            _R148_PENDING.pop(player,None);_R148_REPORT.clear()\n            _R148_REPORT.update(overflow_turns=0,overflow_units_reclaimed=0,overflow_quote_exposure=0,overflow_contract_checks=0,overflow_contract_errors=0,atomic_turns=0,atomic_kept_requests=0,atomic_removed_requests=0,seed_prefund_turns=0,seed_prefund_units=0,targeted_errors=0)\n        pending=_R148_PENDING.pop(player,None)\n        if pending:\n            if step!=pending[0] or not _r148_same_stock(observation[\'private\'][\'shed\'],pending[1]):_R148_REPORT[\'overflow_contract_errors\']+=1\n            else:_R148_REPORT[\'overflow_contract_checks\']+=1\n        standard=configuration is None or all(configuration.get(k,v)==v for k,v in [(\'boardSize\',10),(\'turnsPerDay\',24),(\'shedCapacity\',100),(\'maxMarketOrdersPerTurn\',10),(\'farmHandCostMult\',1)])\n        if standard:\n            if _R148_SEEDS:action=_r148_seed_prefund(observation,_r148_atomic(observation,action))\n            if _R148_OVERFLOW:action=_r148_overflow(observation,action)\n    except Exception:_R148_REPORT[\'targeted_errors\']=_R148_REPORT.get(\'targeted_errors\',0)+1\n    _R148_REPORT.update(getattr(_R148_PARENT,\'telemetry\',{}))\n    return action\nagent.telemetry=_R148_REPORT\nagent=globals().pop(\'agent\')\n\n# EXP278: preserve deepcopy semantics while specializing ordinary JSON containers.\n# Original performance implementation by Ahmed Berat Ozer\'s project.\n_R149_COPY=copy\n_R149_ATOMIC={str,int,float,bool,bytes,type(None)}\n_R149_MISSING=object()\n\ndef _r149_deepcopy(value,memo=None):\n    kind=type(value)\n    if kind in _R149_ATOMIC:return value\n    if kind not in (list,dict):return _R149_COPY.deepcopy(value,memo)\n    if memo is None:memo={}\n    ident=id(value);existing=memo.get(ident,_R149_MISSING)\n    if existing is not _R149_MISSING:return existing\n    if kind is list:\n        result=[];memo[ident]=result\n        result.extend(_r149_deepcopy(item,memo) for item in value)\n    else:\n        result={};memo[ident]=result\n        for key,item in value.items():\n            result[_r149_deepcopy(key,memo)]=_r149_deepcopy(item,memo)\n    # Match stdlib\'s memo lifetime semantics, including external memo reuse.\n    memo.setdefault(id(memo),[]).append(value)\n    return result\n\nclass _R149CopyProxy:\n    deepcopy=staticmethod(_r149_deepcopy)\n    def __getattr__(self,name):return getattr(_R149_COPY,name)\n\ncopy=_R149CopyProxy()\n_PLANNER_NS[\'deepcopy\']=_r149_deepcopy\nagent=globals().pop(\'agent\')\n\n# EXP279: original controller retained; exact route lengths without temporary movement lists.\n_PLANNER_NS[\'_R150_HOME_DISTANCE\']={(x,y):len(_PLANNER_NS[\'_return\']((x,y))) for y in range(10) for x in range(10)}\nexec(\'def _proposals(run, actor, prices, max_per_actor):\\n    """One/two resource bundles plus direct carry closure, replacing a baseline suffix."""\\n    owners = {}\\n    for event in run[\\\'events\\\']:\\n        if \\\'acquired\\\' in event:\\n            owners.setdefault((tuple(event[\\\'xy\\\']), event[\\\'op\\\']), set()).add(event[\\\'actor\\\'])\\n    proposals = []\\n    seen = set()\\n    horizon = len(run[\\\'rows\\\'])\\n    for offset in range(horizon):\\n        farm, private = run[\\\'states\\\'][offset]\\n        pos = tuple(farm[\\\'farmer\\\'] if actor == 0 else farm[\\\'hands\\\'][actor - 1])\\n        inventory = private[\\\'inventories\\\'][actor]\\n        carried = sum((prices.get(item, 0) * count for item, count in inventory.items()))\\n        prefix_deposits = run[\\\'rows\\\'][offset - 1][\\\'deposited_by_actor\\\'][actor] if offset else {}\\n        future_deposits = run[\\\'rows\\\'][-',
    b"1][\\'deposited_by_actor\\'][actor]\\n        obligation = sum((prices.get(item, 0) * (count - prefix_deposits.get(item, 0)) for item, count in future_deposits.items()))\\n        bundles = []\\n        for y, row in enumerate(farm[\\'tiles\\']):\\n            for x, tile in enumerate(row):\\n                if not isinstance(tile, dict):\\n                    continue\\n                xy, operations, value = ((x, y), [], 0)\\n                if tile.get(\\'yield_units\\', 0) > 0:\\n                    item = tile.get(\\'crop\\') if tile.get(\\'kind\\') == \\'PLANT\\' else ANIMALS.get(tile.get(\\'animal\\'), {}).get(\\'product\\')\\n                    mature = item and (\\'animal\\' in tile or (START + offset) // 24 - tile[\\'planted_day\\'] >= CROPS[item][\\'first_yield_day\\'])\\n                    if mature and (not owners.get((xy, \\'HARVEST\\'), set()) - {actor}):\\n                        operations.append([\\'HARVEST\\'])\\n                        value += prices[item] * tile[\\'yield_units\\']\\n                if tile.get(\\'fertilizer_available\\') and \\'animal\\' in tile and (not owners.get((xy, \\'COLLECT_FERTILIZER\\'), set()) - {actor}):\\n                    operations.append([\\'COLLECT_FERTILIZER\\'])\\n                    value += prices[\\'FERTILIZER\\']\\n                if operations:\\n                    distance = abs(pos[0] - xy[0]) + abs(pos[1] - xy[1]) + len(operations) + _R150_HOME_DISTANCE[xy]\\n                    if distance <= horizon - offset:\\n                        bundles.append((xy, operations, value, distance))\\n        bundles.sort(key=lambda b: (-b[2] / b[3], -b[2], b[0]))\\n        variants = [([], carried)] if carried else []\\n        for xy, ops, value, _ in bundles[:6]:\\n            variants.append(([(xy, ops)], carried + value))\\n        for first in bundles[:3]:\\n            for second in bundles[:3]:\\n                if first[0] != second[0]:\\n                    variants.append(([(first[0], first[1]), (second[0], second[1])], carried + first[2] + second[2]))\\n        for stops, value in variants:\\n            route, cursor = ([], pos)\\n            for xy, ops in stops:\\n                route += _walk(cursor, xy) + ops\\n                cursor = xy\\n            route += _return(cursor)\\n            if len(route) > horizon - offset:\\n                continue\\n            route += [[\\'PASS\\']] * (horizon - offset - len(route))\\n            key = (offset, tuple((tuple(c) for c in route)))\\n            if key not in seen:\\n                seen.add(key)\\n                proposals.append((value - obligation, offset, route, len(stops)))\\n    proposals.sort(key=lambda p: (-p[0], p[1], p[2]))\\n    direct = [p for p in proposals if p[3] == 0 and p[0] > 0][:2]\\n    chosen = direct + [p for p in proposals if p not in direct]\\n    return chosen[:max_per_actor]',_PLANNER_NS)\nagent=globals().pop('agent')\n\n\n# EXP283 adaptive arm: clone-gated sale pre-emption with drop-time race escalation.\n# Original mechanism by Ahmed Berat Ozer's project. Live top-band replays (research155) show\n# rivals executing the same public route tape and quoting the same product batches at the\n# same drop turns. While the rival is observed executing our tape, V43's R36 native-tape\n# reservation runs with horizon 8; if the rival is then observed selling a race product at the\n# very turn the same product was dropped into our shed while we did not sell it (public market\n# inventory change beyond town consumption; no own realized sale; own shed stock rose), the rival\n# quotes at the drop and the horizon escalates to 24 for the rest of the game.\n_RACE_PARENT=agent\n_RACE_HORIZON_CLONE=8\n_RACE_HORIZON_ESCALATED=24\n_RACE_HORIZON_MIRROR=24\n_RACE_ITEMS=('CARROT','TOMATO','STRAWBERRY','MELON','EGG','MILK','WOOL')\n_RACE_SHOPS={'BAKERY':('EGG','WHEAT'),'PIZZA_SHOP':('MILK','TOMATO','WHEAT'),'BRUNCH_SPOT':('EGG','WHEAT','STRAWBERRY'),'YARN_STORE':('WOOL',),\n             'ICE_CREAM_SHOP':('STRAWBERRY','MILK','WHEAT'),'PET_CAFE':('CARROT',),'SMOOTHIE_SHOP':('STRAWBERRY','MILK'),'FARMERS_MARKET':('WHEAT','CARROT','TOMATO','STRAWBERRY')}\n_RACE_STATE={}\n_RACE_REPORT=dict(race_clone_turn",
    b's=0,race_horizon_turns=0,race_lost_races=0,race_escalations=0,race_errors=0)\n_RACE_ORIG_RESERVE=_r36_reserve\n\ndef _race_positions_equal(farms,player):\n    own,rival=farms[player],farms[1-player]\n    return len(own[\'hands\'])>0 and own[\'hands\']==rival[\'hands\'] and own[\'farmer\']==rival[\'farmer\']\n\ndef _race_clone(observation,state):\n    farms=observation[\'farms\'];player=int(observation[\'player\'])\n    if len(farms[player][\'hands\'])>0:\n        state[\'hist\'].append(_race_positions_equal(farms,player))\n        if len(state[\'hist\'])>6:state[\'hist\'].pop(0)\n    return len(state[\'hist\'])>=4 and sum(state[\'hist\'])>=4 and _r37_similarity(observation)>=.95\n\ndef _race_town(step,shops):\n    out={}\n    if step%4==0:\n        for shop in shops:\n            items=_RACE_SHOPS.get(shop,())\n            for item in items:out[item]=out.get(item,0)+(2 if len(items)==1 else 1)\n    if step%24==0:\n        for item in _RACE_ITEMS:out[item]=out.get(item,0)+1\n    return out\n\ndef _race_lost(observation,state):\n    """EXP293: True when the rival sold a race product at the previous turn while we held it unsold, the common tape\n    has no sale of it within the lineage\'s own lead/reservation window (5 turns) and sells it within the following\n    24 turns: the rival pre-empts the plan\'s own sale ahead of us."""\n    prev=state.get(\'prev\');prev_action=state.get(\'prev_action\')\n    if prev is None or prev_action is None:return False\n    step=int(observation[\'step\']);player=int(observation[\'player\'])\n    if step!=prev[\'step\']+1 or step%24==0:return False\n    inv=observation[\'market\'][\'inventory\'];pinv=prev[\'inventory\'];prices=prev[\'prices\']\n    town=_race_town(step-1,prev[\'shops\'])\n    sold={}\n    for order in prev_action.get(\'market\',[]):\n        if len(order)>=3 and order[0]==\'SELL\' and order[1] in _RACE_ITEMS:sold[order[1]]=1\n    native=_IMPL.chassis.players[player]\n    for item in _RACE_ITEMS:\n        before=int(prev[\'view\'].shed.get(item,0))\n        if before<=0 or item in sold or prices.get(item,0)<=1:continue\n        rival=int(inv[item])-int(pinv[item])+town.get(item,0)\n        if rival<=0:continue\n        def planned(t):\n            future=_IMPL.chassis.routes[2 if t>=648 else native[\'route\']][t]\n            return any(len(o)>=3 and o[0]==\'SELL\' and o[1]==item for o in future.get(\'market\',[]))\n        # every member of this lineage sells at the scheduled turn, one turn early (sale lead) or up to four turns early\n        # (the base reservation): only a sale further ahead of the plan is a race\n        if any(planned(t) for t in range(step-1,min(719,step+5))):continue\n        if any(planned(t) for t in range(step+5,min(719,step+24))):return True\n    return False\n\ndef _race_snapshot(observation):\n    market=observation[\'market\']\n    return dict(step=int(observation[\'step\']),inventory=dict(market[\'inventory\']),prices=dict(market[\'prices\']),shops=list(observation[\'town\'].get(\'unlocked_shops\',[])),view=FarmView(observation))\n\ndef _r36_reserve(obs,action):\n    player=int(obs[\'player\']);h=_RACE_STATE.get(player,{}).get(\'horizon\',0)\n    if h>_R37_HORIZONS.get(player,2):\n        saved=_R37_HORIZONS.get(player);_R37_HORIZONS[player]=h\n        try:return _RACE_ORIG_RESERVE(obs,action)\n        finally:\n            if saved is None:_R37_HORIZONS.pop(player,None)\n            else:_R37_HORIZONS[player]=saved\n    return _RACE_ORIG_RESERVE(obs,action)\n\ndef agent(observation,configuration=None):\n    state=None\n    try:\n        player=int(observation[\'player\']);step=int(observation[\'step\'])\n        state=_RACE_STATE.get(player)\n        if state is None or step<=state[\'step\']:\n            state=_RACE_STATE[player]={\'step\':-1,\'hist\':[],\'horizon\':0,\'level\':_RACE_HORIZON_CLONE,\'prev\':None,\'prev_action\':None}\n        if step==0:_RACE_REPORT.update(race_clone_turns=0,race_horizon_turns=0,race_lost_races=0,race_escalations=0,race_errors=0)\n        state[\'step\']=step;state[\'horizon\']=0\n        # EXP288 mirror gate: a rival whose cash after the first turn equals ours executed the same first-turn\n        # round trip (a copy of this agent); against a copy the sale race is won only by p',
    b"re-empting the whole day.\n        if step==1:\n            try:\n                farms=observation['farms'];rival=farms[1-player]['money'];own=farms[player]['money']\n                state['level']=_RACE_HORIZON_MIRROR if (abs(float(rival)-float(own))<0.5 and _RACE_HORIZON_MIRROR>state['level']) else state['level']\n                _RACE_REPORT['race_mirror']=int(abs(float(rival)-float(own))<0.5)\n            except Exception:_RACE_REPORT['race_errors']+=1\n        standard=configuration is None or all(configuration.get(k,v)==v for k,v in [('boardSize',10),('turnsPerDay',24),('shedCapacity',100),('maxMarketOrdersPerTurn',10)])\n        if standard and 216<=step<696 and _race_clone(observation,state):\n            _RACE_REPORT['race_clone_turns']+=1\n            if state['level']<_RACE_HORIZON_ESCALATED and _race_lost(observation,state):\n                _RACE_REPORT['race_lost_races']+=1;state['level']=_RACE_HORIZON_ESCALATED;_RACE_REPORT['race_escalations']+=1\n            state['horizon']=state['level'];_RACE_REPORT['race_horizon_turns']+=1\n    except Exception:_RACE_REPORT['race_errors']+=1\n    snapshot=None\n    try:\n        if state is not None and 215<=int(observation['step'])<696:snapshot=_race_snapshot(observation)\n    except Exception:_RACE_REPORT['race_errors']+=1\n    action=_RACE_PARENT(observation,configuration)\n    try:\n        if state is not None:state['prev']=snapshot;state['prev_action']=action if snapshot is not None else None\n    except Exception:_RACE_REPORT['race_errors']+=1\n    _RACE_REPORT.update(getattr(_RACE_PARENT,'telemetry',{}))\n    return action\nagent.telemetry=_RACE_REPORT\nagent=globals().pop('agent')\n\n\n# EXP284 opening arm: step-0 wheat round trip as one buy order followed by one sell order.\n# Original analysis by Ahmed Berat Ozer's project (research155/156 live replays): the parent's split\n# [BUY 5, BUY 10, SELL 60] exposes the second buy to a rival's large round trip in the same turn; two live\n# rivals used [BUY_PRODUCT WHEAT 78, SELL WHEAT 78] and left us one melon seed short. The single large\n# round trip is neutral against the market and symmetric against itself.\n_OPEN_PARENT=agent\n_OPEN_UNITS=70\n_OPEN_FEED_STEP1=5\n_OPEN_STEP0=[['BUY_PRODUCT','WHEAT',7],['SELL','WHEAT',2]]\n_OPEN_ATTACK=30\n_OPEN_ATTACK_MIN_CASH=2860\n_OPEN_REPORT=dict(open_turns=0,open_errors=0)\ndef agent(observation,configuration=None):\n    action=_OPEN_PARENT(observation,configuration)\n    try:\n        step=int(observation['step'])\n        if step==0:_OPEN_REPORT.update(open_turns=0,open_errors=0,open_attack=0)\n        standard=configuration is None or all(configuration.get(k,v)==v for k,v in [('boardSize',10),('turnsPerDay',24),('shedCapacity',100),('maxMarketOrdersPerTurn',10),('startingMoney',3000)])\n        if standard and step==0 and action.get('market')==[['BUY_PRODUCT','WHEAT',5],['BUY_PRODUCT','WHEAT',10],['SELL','WHEAT',60]]:\n            action=dict(action,market=[list(o) for o in _OPEN_STEP0]);_OPEN_REPORT['open_turns']+=1\n        elif standard and step==1:\n            # EXP293: the five feed units were bought at step 0 (index 0, cheapest quotes); the tape's failing SELL 13 and\n            # its step-1 BUY 5 are removed so the shed keeps exactly the tape's five units.\n            market=[list(o) for o in action.get('market',[])]\n            if len(market)>=2 and market[0]==['SELL','WHEAT',13] and market[1]==['BUY_PRODUCT','WHEAT',5]:\n                market=market[2:];_OPEN_REPORT['open_turns']+=1\n                # EXP293 step-1 attack: every tape of this lineage buys its five feed units at index 1 of the first market turn;\n                # a product purchase at index 0 executes before it and lifts its quotes below the tape's day-0 cash slack.\n                # The units are sold back next turn, when no tape trades; the resale meets the lifted quotes, so the trip pays for itself.\n                if _OPEN_ATTACK and float(observation['farms'][int(observation['player'])]['money'])>=_OPEN_ATTACK_MIN_CASH:\n                    market=[['BUY_PRODUCT','WHEAT',_OPEN_ATTACK]]+market;_OPEN_REPORT['open_attack']=1\n                a",
    b'ction=dict(action,market=market)\n        elif standard and step==2 and _OPEN_REPORT.get(\'open_attack\'):\n            action=dict(action,market=[[\'SELL\',\'WHEAT\',_OPEN_ATTACK]]+[list(o) for o in action.get(\'market\',[])][:9]);_OPEN_REPORT[\'open_turns\']+=1\n    except Exception:_OPEN_REPORT[\'open_errors\']+=1\n    _OPEN_REPORT.update(getattr(_OPEN_PARENT,\'telemetry\',{}))\n    return action\nagent.telemetry=_OPEN_REPORT\nagent=globals().pop(\'agent\')\n\n\n# EXP293 sale advance. Mechanism after sdy623 / jaxa623, "Beyond 48-0" (public Kaggle notebook, Apache-2.0): when the\n# native tape sells a pure cash product within the next 3 turns and the units already sit in the shed, sell them now,\n# ahead of a rival executing the same tape.  Own implementation over this project\'s chassis (tape lookup, projected shed,\n# route 2 after step 648); never on the dawn turn (the warehouse-closing layer inspects the shed there); the first-listed\n# sale of the next turn is left alone when it funds the sale-credit feed purchase; quantities are caps, so the tape\'s own\n# later SELL simply sells whatever was deposited since.\n_ADV_PARENT=agent\n_ADV_LOOK=3\n_ADV_FROM=144\n_ADV_TO=718\n_ADV_PROTECT=True\n_ADV_FRONT=True\n_ADV_BOOK=False\n_ADV_SUBTRACT_DEBTS=False\n_ADV_ITEMS=(\'STRAWBERRY\',\'WOOL\',\'EGG\',\'MILK\',\'MELON\',\'CARROT\',\'TOMATO\')\n_ADV_REPORT=dict(adv_turns=0,adv_units=0,adv_errors=0)\ndef _adv_future(player,t):\n    native=_IMPL.chassis.players[player]\n    return _IMPL.chassis.routes[2 if t>=648 else native[\'route\']][t].get(\'market\',[]) or []\ndef _adv_apply(obs,action):\n    step=int(obs[\'step\']);player=int(obs[\'player\'])\n    if step%24==23 or not _ADV_FROM<=step<_ADV_TO:return action\n    native=_IMPL.chassis.players[player];debts=native[\'sell_state\'].setdefault(\'r36_debts\',{})\n    plan=[];first=None\n    for off in range(1,_ADV_LOOK+1):\n        t=step+off\n        if t>718:break\n        for o in _adv_future(player,t):\n            if not o or len(o)<3:continue\n            if first is None:first=o\n            if o[0]==\'SELL\' and o[1] in _ADV_ITEMS:\n                try:q=max(0,int(o[2]))\n                except Exception:q=0\n                if _ADV_SUBTRACT_DEBTS:q-=debts.get(t,{}).get(o[1],0)   # already reserved by the sale-reservation layer\n                if q>0:plan.append((t,o[1],q))\n    protected=first[1] if _ADV_PROTECT and first is not None and first[0]==\'SELL\' else None\n    plan=[(t,item,q) for t,item,q in plan if item!=protected]\n    if not plan:return action\n    market=[list(o) for o in (action.get(\'market\') or [])]\n    if any(len(o)>1 and o[0]==\'BUY_PRODUCT\' for o in market):return action\n    stock=projected_shed(action,FarmView(obs))\n    selling={}\n    for o in market:\n        if len(o)>=3 and o[0]==\'SELL\':\n            try:selling[o[1]]=selling.get(o[1],0)+max(0,int(o[2]))\n            except Exception:return action\n    commands=[action.get(\'farmer\') or [\'PASS\'],*(action.get(\'hands\') or [])]\n    picked={c[1] for c in commands if len(c)>1 and c[0]==\'PICKUP\'}\n    prices=obs[\'market\'][\'prices\'];added=0;extra=[];booked=[]\n    for item in sorted({it for _,it,_ in plan},key=lambda it:-int(prices.get(it,0))):\n        if item in picked or int(prices.get(item,0))<2:continue\n        avail=int(stock.get(item,0))-selling.get(item,0)\n        if avail<1:continue\n        hit=next((o for o in market if len(o)>=3 and o[0]==\'SELL\' and o[1]==item),None)\n        if hit is None and len(market)+len(extra)>=10:continue\n        n=0\n        for t,it,q in plan:\n            if it!=item or avail<=0:continue\n            take=min(q,avail);booked.append((t,item,take));n+=take;avail-=take\n        if n<1:continue\n        if hit is not None:hit[2]=int(hit[2])+n\n        else:extra.append([\'SELL\',item,n])\n        added+=n\n    if not added:return action\n    for t,item,take in (booked if _ADV_BOOK else []):\n        d=debts.setdefault(t,{});d[item]=d.get(item,0)+take\n    _ADV_REPORT[\'adv_turns\']+=1;_ADV_REPORT[\'adv_units\']+=added\n    return dict(action,market=extra+market)\ndef _adv_frontload(obs,action):\n    """Final market-list order: sales first, then product purchases (with the sales of an item t',
    b'he same list also buys, in\n    their original order), then everything else in its original order.  Sales earlier only add cash and shed room before\n    purchases; a product purchase ahead of fixed-price orders meets the rival\'s same-item purchase at the same index or\n    earlier."""\n    market=[list(o) for o in (action.get(\'market\') or []) if o]\n    if len(market)<2 or int(obs[\'step\'])<_ADV_FROM:return action\n    buys={o[1] for o in market if len(o)>1 and o[0]==\'BUY_PRODUCT\'}\n    front=[o for o in market if len(o)>=3 and o[0]==\'SELL\' and o[1] not in buys]\n    mid=[o for o in market if len(o)>=3 and o[0]==\'BUY_PRODUCT\' or (len(o)>=3 and o[0]==\'SELL\' and o[1] in buys)]\n    rest=[o for o in market if o not in front and o not in mid]\n    new=front+mid+rest\n    if new==market:return action\n    _ADV_REPORT[\'front_turns\']=_ADV_REPORT.get(\'front_turns\',0)+1\n    return dict(action,market=new)\ndef agent(observation,configuration=None):\n    action=_ADV_PARENT(observation,configuration)\n    try:\n        if int(observation[\'step\'])==0:_ADV_REPORT.update(adv_turns=0,adv_units=0,adv_errors=0)\n        standard=configuration is None or all(configuration.get(k,v)==v for k,v in [(\'boardSize\',10),(\'turnsPerDay\',24),(\'shedCapacity\',100),(\'maxMarketOrdersPerTurn\',10)])\n        if standard:action=_adv_apply(observation,action)\n        if standard and _ADV_FRONT:action=_adv_frontload(observation,action)\n        # the race layer\'s lost-race detector must judge the final market list (advanced sales included)\n        st=_RACE_STATE.get(int(observation[\'player\']))\n        if st is not None and st.get(\'prev_action\') is not None and st.get(\'step\')==int(observation[\'step\']):st[\'prev_action\']=action\n    except Exception:_ADV_REPORT[\'adv_errors\']+=1\n    _ADV_REPORT.update(getattr(_ADV_PARENT,\'telemetry\',{}))\n    return action\nagent.telemetry=_ADV_REPORT\nagent=globals().pop(\'agent\')\n',
))
assert hashlib.sha256(SOURCE_BYTES).hexdigest() == EXPECTED_MAIN_SHA256, 'Embedded V46 source is incomplete or modified; import the fixed notebook again.'
MAIN = WORKDIR / 'main.py'
MAIN.write_bytes(SOURCE_BYTES)
assert MAIN.read_bytes() == SOURCE_BYTES, 'main.py was not written correctly.'
print('Wrote verified V46 main.py:', len(SOURCE_BYTES), 'bytes', flush=True)


## Verify source and entry point


In [ ]:
import ast, hashlib, json

MAIN = WORKDIR / 'main.py'
EXPECTED_MAIN_SHA256 = '735c370383b70d3bf3aac792f2c147e0afc99166fc9f253ede10e8a030acedb6'
source_bytes = MAIN.read_bytes()
actual_main_sha256 = hashlib.sha256(source_bytes).hexdigest()
assert actual_main_sha256 == EXPECTED_MAIN_SHA256, (
    'main.py differs from tested V46. Rerun cell 2, then this cell. '
    f'Expected: {EXPECTED_MAIN_SHA256}; found: {actual_main_sha256}'
)
compile(source_bytes, 'main.py', 'exec')

imports, trees = set(), [ast.parse(source_bytes)]
while trees:
    for node in ast.walk(trees.pop()):
        if isinstance(node, ast.Import):
            imports.update(a.name.split('.')[0] for a in node.names)
        elif isinstance(node, ast.ImportFrom):
            assert node.level == 0 and node.module
            imports.add(node.module.split('.')[0])
        elif isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == 'exec':
            assert isinstance(node.args[0], ast.Constant) and isinstance(node.args[0].value, str)
            trees.append(ast.parse(node.args[0].value))
assert imports <= sys.stdlib_module_names, 'A non-standard-library import was found.'
namespace = {}
exec(compile(source_bytes, '<v46>', 'exec'), namespace)
entry = [value for value in namespace.values() if callable(value)][-1]
assert entry is namespace['agent'], 'The final callable is not the tested agent.'
assert isinstance(entry({}), dict), 'Safe fallback check failed.'
print('PASS: source hash, Python syntax, embedded imports and entry point.', flush=True)


## Build the submission archive


In [ ]:
import gzip, io, tarfile
from IPython.display import FileLink, display

ARCHIVE = WORKDIR / 'submission_competitive_v46.tar.gz'
EXPECTED_ARCHIVE_SHA256 = '7cfb4ab837e7d809a89ba4876a8508533e115c51556e82c97888069138659a6b'
with ARCHIVE.open('wb') as file:
    with gzip.GzipFile(filename='', mode='wb', fileobj=file, mtime=0) as compressed:
        with tarfile.open(fileobj=compressed, mode='w', format=tarfile.GNU_FORMAT) as tar:
            info = tarfile.TarInfo('main.py')
            info.size, info.mode, info.mtime = len(source_bytes), 0o644, 0
            tar.addfile(info, io.BytesIO(source_bytes))
with tarfile.open(ARCHIVE) as tar:
    assert tar.getnames() == ['main.py']
    assert tar.extractfile('main.py').read() == source_bytes
assert hashlib.sha256(ARCHIVE.read_bytes()).hexdigest() == EXPECTED_ARCHIVE_SHA256
build_manifest = {'version': 'v46', 'classification': 'evaluated_upgrade', 'protocol': 'EXP293 first-turn microstructure, two-turn sale advance, sales-first market order, lost-race detector and imminence-ordered reservation on the frozen EXP288 policy', 'independent_reacting_confirmation_passed': True, 'complete_physical_audit_passed': True, 'isolated_runtime_passed': False, 'isolated_runtime_max_ms': 85.08, 'local_50ms_rule': 'waived by the user for the step-712 closure planner (WAIVER293.json); engine actTimeout 1 s', 'linux_replay_passed': True, 'live_recorded_rival_replay_games': None, 'main_sha256': '735c370383b70d3bf3aac792f2c147e0afc99166fc9f253ede10e8a030acedb6', 'archive_sha256': '7cfb4ab837e7d809a89ba4876a8508533e115c51556e82c97888069138659a6b', 'source_integrity': 'PASS', 'archive_integrity': 'PASS', 'optional_game_checks': 'NOT RUN', 'live_rating_guarantee': None}
(WORKDIR / 'v46_manifest.json').write_text(json.dumps(build_manifest, indent=2))
print('PACKAGE READY:', ARCHIVE.name, '|', ARCHIVE.stat().st_size, 'bytes', flush=True)
print('SHA-256:', EXPECTED_ARCHIVE_SHA256, flush=True)
display(FileLink(ARCHIVE.name))


## Optional: run complete games


In [ ]:
if not RUN_GAME_CHECKS:
    print('Optional game checks skipped. Your verified submission package is already ready.', flush=True)
    print('To play four smoke games, set RUN_GAME_CHECKS = True in cell 1 and run all cells again.', flush=True)
else:
    import contextlib, importlib.metadata
    try:
        engine_version = importlib.metadata.version('kaggle-environments')
    except importlib.metadata.PackageNotFoundError:
        engine_version = None
    if engine_version != '1.32.7':
        print('Optional games require kaggle-environments 1.32.7; installed:', engine_version, flush=True)
        print('No installation was started. The submission archive above remains valid.', flush=True)
    else:
        print('Loading official engine 1.32.7...', flush=True)
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            from kaggle_environments import make
            from kaggle_environments.agent import get_last_callable
        official_entry = get_last_callable(source_bytes.decode('utf-8'), path=str(MAIN))
        assert official_entry is official_entry.__globals__['agent']
        games = []
        for label, opponent in [('self-play', str(MAIN)), ('starter', 'starter')]:
            for seed in (7, 1234):
                print(f'Game {len(games)+1}/4: {label}, seed={seed}', flush=True)
                started = time.perf_counter()
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    env = make('kaggriculture', configuration={'episodeSteps':720, 'seed':seed})
                    final = env.run([str(MAIN), opponent])[-1]
                statuses = [row['status'] for row in final]
                rewards = [row['reward'] for row in final]
                assert statuses == ['DONE', 'DONE'], (label, seed, statuses)
                if label == 'starter':
                    assert rewards[0] > rewards[1], (label, seed, rewards)
                games.append({'opponent':label, 'seed':seed, 'rewards':rewards, 'status':statuses})
                print('DONE:', rewards, '|', round(time.perf_counter()-started, 2), 'seconds', flush=True)
        (WORKDIR / 'v46_smoke_results.json').write_text(json.dumps(games, indent=2))
        build_manifest['optional_game_checks'] = 'PASS'
        build_manifest['games'] = games
        (WORKDIR / 'v46_manifest.json').write_text(json.dumps(build_manifest, indent=2))
        print('All four full-game checks passed. The archive has not changed.', flush=True)



## Evaluation

| Cohort | W/L/T | Strict win rate | Mean cash margin |
|---|---:|---:|---:|
| Development V46 (8 fresh worlds, 19 rivals, both seats) | 269/35/0 | 88.49% | +3,156.47 |
| Development predecessor (EXP288 build) | 217/73/14 | 71.38% | +3,090.83 |
| Independent V46 (32 untouched worlds) | 1080/135/1 | 88.82% | +2,682.61 |
| Independent predecessor | 835/323/58 | 68.67% | +2,365.16 |
| Independent V46, clone family (13 rivals) | 707/124/1 | 84.98% | +869.29 |
| Independent predecessor, clone family | 474/300/58 | 56.97% | +404.00 |
| Independent V46, other rivals (6) | 373/11/0 | 97.14% | +6,611.46 |
| Independent predecessor, other rivals | 361/23/0 | 94.01% | +6,614.34 |

Independent draw (1,216 games per policy): paired match points +17.80 percentage points over the predecessor (world-level bootstrap 95% interval [+10.44, +24.38]); clone family +24.58, other rivals +3.12. Worlds in this cohort are not shop-matched between the two policies (the town draw depends on the game state), so paired cash differences are not reported here; the shop-matched development screens are in the research notes.

| Rival (64 games each) | Predecessor W/L/T | V46 W/L/T | Points |
|---|---:|---:|---:|
| exact_v43 | 62/2/0 | 59/5/0 | -4.7 |
| fastclone24 | 60/4/0 | 53/11/0 | -10.9 |
| guru_dynamic | 60/4/0 | 60/4/0 | +0.0 |
| lynn_v5 | 60/4/0 | 61/3/0 | +1.6 |
| mirror_counter | 60/4/0 | 63/1/0 | +4.7 |
| mirror_v45 | 48/16/0 | 58/6/0 | +15.6 |
| mirror_v46 | 3/3/58 | 58/6/0 | +40.6 |
| pub_aurax_v6 | 48/16/0 | 58/6/0 | +15.6 |
| pub_beyond48 | 0/64/0 | 52/12/0 | +81.2 |
| pub_pipe7_open5 | 3/61/0 | 53/11/0 | +78.1 |
| pub_pipe8 | 3/61/0 | 51/13/0 | +75.0 |
| pub_tetsu_mirror | 2/62/0 | 49/15/0 | +73.4 |
| race8d_v44 | 60/4/0 | 53/10/1 | -10.2 |
| seven_rescue | 60/4/0 | 63/1/0 | +4.7 |
| thomas955 | 63/1/0 | 63/1/0 | +0.0 |
| v43_open78 | 63/1/0 | 59/5/0 | -6.2 |
| v45_open13 | 61/3/0 | 53/11/0 | -12.5 |
| v45_open5 | 61/3/0 | 51/13/0 | -15.6 |
| v49_bandit | 58/6/0 | 63/1/0 | +7.8 |

Mechanism and limits. The turn-0 and turn-1 effects are exact consequences of the per-unit lockstep market and the plan's fixed day-0 cash; they were verified with the official engine on two- and three-turn scripted games against every rival opening observed in 154 live games and in the new public notebooks. The sale-timing changes decide same-turn races against agents executing the same tape; against rivals with a different plan they change little. The predecessor's 70-unit round trip punished the original V43 opening by a melon seed; V46 gives that up for robustness, so against plain V43-family rivals it wins slightly less often in the shop-matched development screens (44-47 of 48 instead of 48).

All 1216 independent candidate/control pairs passed the official cash and product ledger audit; feed shortfalls, pickup shortfalls and animal escapes were 0/0/0 for V46 versus 0/0/0 for the predecessor.

Runtime: eight fresh full games with the candidate in a separate normal-GC process, Windows maximum 85.08 ms per call (step 712, the inherited V43 terminal closure planner with 64 simulations; the live predecessor measures 71.8 ms in the same game; the engine's actTimeout is 1 s; the project's local 50 ms rule was waived for this step); the same 5,752 actions replayed exactly on native Linux, maximum 15.28 ms. Shared-worker timings are not acceptance measurements; all timings are machine-dependent.

Local win rates against a frozen panel are not a ladder rating and not a guarantee; the ladder population changes daily.


# Attribution and license

This agent retains Apache-2.0 notices in its source. Modified in EXP-167 on
September 10, 2026 by Ahmed Berat Özer's Kaggriculture project.

- [Dmitrii Gluzdov — Two Coins, One Sheep](https://www.kaggle.com/code/dmitriigluzdov/kaggriculture-two-coins-one-sheep-lb-2650): bounded two-turn stock reservations, protection of scheduled pickups, partial future-order deductions and placement after worker repairs. Adapted to our per-player chassis state and preserved terminal planner. Earlier seven-turn physical closure work is also retained through v31.
- [prvsiyan — The Moon Counts Melons](https://www.kaggle.com/code/prvsiyan/kaggriculture-frontier-the-moon-counts-melons): the later cattle substitution controller. Earlier fertilizer/feed and tomato production work remains credited in v31's source.
- [yhay81 — Shop Router 0909](https://www.kaggle.com/code/yhay81/shop-router-0909): thirteen public action schedules and ordered shop-pair routing; earlier ShopForge/Fieldbook lineage.
- [aurax7 — Reactive Router](https://www.kaggle.com/code/aurax7/kaggriculture-reactive-router): sale timing and shed projection lineage.
- thomastschinkel: replay-routing research and earlier foundation of this project; tetsutani: market, room and repair mechanisms credited in the retained source.
- [destbreso — X-ray Your Agent](https://www.kaggle.com/code/destbreso/x-ray-your-agent): diagnostic methodology. Its notebook is not bundled as agent runtime.

Other audited Codes appear in SOURCE_AUDIT.md and the evaluation panel; their
presence there does not imply their code was incorporated. Credits do not
imply author endorsement or a verified private-leader implementation.

## EXP-168 changes (v34)

- [lucifer19 — Harvest Nocturne](https://www.kaggle.com/code/lucifer19/harvest-nocturne-the-market-has-a-rhythm): occupied-tile similarity and exact price-loss ordering ideas/code, adapted and independently tested. Per-seat memory/reset replaces its single shared controller state.
- [flexonafft — Most Powerfull Route](https://www.kaggle.com/code/flexonafft/kaggriculture-most-powerfull-route): requested source snapshot; its full executable bundle is byte-identical to Two Coins above. No new route or original capability is attributed to a renamed copy.
- leoprovorov: public-layout comparison lineage credited by Nocturne.
- [Kaggle official implementation](https://github.com/Kaggle/kaggle-environments/tree/master/kaggle_environments/envs/kaggriculture): Apache-2.0 market-price functions from version1.32.7; verified against local engine.

All earlier in-source notices and the full Apache-2.0 license remain in main.py.
This list implies no author endorsement. Exact incorporated switches appear
in agent/manifest.json; unselected experiments are not claimed as improvements.

## EXP-175–178 changes (v35, September 11, 2026)

- [yhay81 — Shop Router 0911 Simple](https://www.kaggle.com/code/yhay81/shop-router-0911-simple): revised opening market-sequence idea. Our selected opening uses `BUY_PRODUCT WHEAT 13`, `BUY_PRODUCT WHEAT 30`, `SELL WHEAT 30`; it retains the earlier thirteen routes rather than adopting the donor's complete fourteen-route revision.
- [leoprovorov — Two Coins at High Noon: Small Improvement](https://www.kaggle.com/code/leoprovorov/two-coins-at-high-noon-small-improvement): Mirror Counter's public cash-response mechanism. Adapted to the existing occupied-tile similarity guard and per-player state, requiring positive matching cash changes after a probe. Our fourth-turn reservation is an independently tested modification, not a claim about the donor's reported results.
- [prvsiyan — The Soil Remembers Rain](https://www.kaggle.com/code/prvsiyan/kaggriculture-frontier-the-soil-remembers-rain): the already reviewed V234 six-sheep southeast expansion, including financing, confirmed hiring, feed, care and credited production. Adapted to our chassis state, combined telemetry and terminal-planner abstention. This capability is newly integrated here; the September 11 exported donor code itself is unchanged from the earlier reviewed version.
- [lucifer19 — Harvest Nocturne V2](https://www.kaggle.com/code/lucifer19/harvest-nocturne-v2-a-lighter-start): audited startup and runtime packaging reference. Our chassis already shared read-only route data, so no new gameplay gain is attributed to its deep-copy removal.
- [Nagata V6.2](https://www.kaggle.com/code/nagatakengo/kaggriculture): added as a reacting evaluation opponent. Its policy is not bundled in this submission.
- [destbreso — X-Ray Your Agent](https://www.kaggle.com/code/destbreso/x-ray-your-agent) and [Georgy Mamarin — What 2600+ Farms Do Differently](https://www.kaggle.com/code/georgymamarin/kaggriculture-what-2600-farms-do-differently): whole-cohort and economic diagnostic references; no runtime code copied from these notebooks.

Integration, public-response safeguards, experiment design, official-engine accounting checks and release packaging: Ahmed Berat Özer's Kaggriculture project. The complete agent retains its Apache-2.0 license and upstream notices. Exact source hashes and the distinction between incorporated code, evaluated opponents and diagnostics are recorded in `research44/SOURCE_AUDIT.md` and `research47/REPORT.md` in the project workspace.

## EXP179–180 changes (V36, September 11, 2026)

- [Tetsutani — Market Smart Farming](https://www.kaggle.com/code/tetsutani/market-smart-farming-kaggriculture): its updated four-turn configuration motivated this isolated extension of our existing Two Coins stock-reservation layer. V36 applies four turns within V35's physical stock, debt, purchase and terminal guards; farm routes and investment controllers are unchanged.
- [Rayk Kretzschmar — Rank Your Agent](https://www.kaggle.com/code/raykkretzschmar/kaggriculture-rank-your-agent) and [Kunal Desale — Kaggriculture2026V1](https://www.kaggle.com/code/kunaldesale2408/kaggriculture-2026-v1): newly evaluated exported opponents. Their separate sale-order scoring ideas were screened and rejected as standalone modifications; those runtime policies and their route libraries are not incorporated into V36.
- The September11 afternoon Flexon export hashes identically to V35; this differs from the older EXP168 snapshot described above. It supplies no new V36 capability.

All earlier Apache-2.0 notices and the complete license remain in main.py. This 130-byte source extension, experiment design, accounting checks and packaging are by Ahmed Berat Özer's Kaggriculture project. Full provenance: research48/SOURCE_AUDIT.md and research49/REPORT.md. Credits imply no author endorsement.

## EXP182–190 changes (V37, September 12, 2026)

V37 retains V36's route library and all earlier source notices. The finite native-crop fertilizer planner, committed-parent priority guard, projected warehouse guard and exact-spawn cooperative tomato labor scheduler were developed and physically audited in this project. Earlier unsuccessful livestock and rival-flow experiments are not included. The latest public Code review is documented in research56/SOURCE_AUDIT.md; those updated analytical/data notebooks supplied no new executable policy to this release.

All earlier Apache-2.0 notices, including thomastschinkel, yhay81 and destbreso, remain intact in main.py along with the full license. Evaluation, guards and packaging are by Ahmed Berat Özer's Kaggriculture project. Credits imply no author endorsement. See research59/REPORT.md for the mixed evidence and release-candidate status.


## EXP193–217 changes (V38, September 12, 2026)

- [Steven Lee Hans — Lord Momo Returns](https://www.kaggle.com/code/stevenleehans/kaggriculture-rank-580-lord-momo-returns): conceptual reference for comparing feed cost with production value and selling surplus fertilizer. Our implementation independently adds care-value accounting, the following-day feeding schedule check, actual carried-food checks, and a reserve for all remaining native and committed crop inputs. No guarantee about future feeding or monotone fertilizer prices is inherited from the donor narrative. Audited exported source SHA-256: `b5c2e156689b41cea5f2e1a0a1cae2e18fd70423931bbac0885bba8b2142c497`.
- The exact-spawn finite-input tour planner, committed-parent fertilizer queue guard, conservative tomato fertilization margin test, next-day service simulation, whole-animal survival audit and integration are by Ahmed Berat Özer's Kaggriculture project. The source preserves the existing route and chassis lineage; tested mechanisms are identified by the frozen manifest.
- [Pilkwang — Structured Economic Policy](https://www.kaggle.com/code/pilkwang/kaggriculture-structured-economic-policy), the Momo source and four other distinct exports were examined in the September 12 public refresh. Six new distinct policies entered the reacting panel; their inclusion as opponents does not mean their runtime was incorporated. Exact V37 duplicates were deduplicated. Complete review: `research84/SOURCE_AUDIT.md` in the project workspace.

All earlier Apache-2.0 notices, including thomastschinkel, yhay81 and destbreso,
remain in `main.py` with the full license. Credits imply no author endorsement.
The final source, complete raw outcomes and frozen-source confirmation are
documented in `research86/REPORT.md` and `research86/results/release_evidence.json`.


## V39 consolidation — September 13, 2026

Production-calendar feeding, bounded physical wheat replenishment, native-pickup
coverage, stock reservation before saturated market turns and current-liability
funding were developed in Ahmed Berat Özer's Kaggriculture project. V39 selects
the unchanged combined EXP231 source after the requested candidate consolidation.
All earlier credits and Apache-2.0 notices remain in main.py, including
thomastschinkel, yhay81 and destbreso. Credits do not imply endorsement.

The latest public inventory audit covered 485 references. Zhihuan Xue's
Kaggriculture Timed Six Cow was reviewed and retained once as an evaluation
opponent. None of its code was copied into this agent. Results and provenance
are recorded in research100, research101 and research102.


## V41 development attribution

The V39 base and all embedded upstream notices are retained. The low-volume
opening was adapted from Rayk Kretzschmar's public [Rank Your Agent notebook](https://www.kaggle.com/code/raykkretzschmar/kaggriculture-rank-your-agent).
Funded atomic planting, first-slot grain funding and final-hour watering
contracts were implemented by Ahmed Berat Ozer. Newly reviewed MetaCounter R1
and Adaptive Land Allocator were evaluated as opponents; their agent source
was not incorporated into this release.


EXP259 owned-input preloading, adjacent pickup/feed ordering, conservative first-sale funding and observation confirmations were implemented by Ahmed Berat Özer. The planned-crop-retirement classifier belongs to validation tooling and is not agent behavior. Master Engine V3 was audited as a V40 runtime copy; its source was not incorporated. All Apache-2.0 license text and upstream notices remain in main.py.


## Production-library lineage retained in V42

## EXP239–241 development — September 13, 2026

- [Yusuke Hayashi (yhay81) — Shop Router 0913](https://www.kaggle.com/code/yhay81/shop-router-0913), version 1 (script version 349474696): the new fixed midgame action plans and ordered shop-pair map. Apache 2.0 was verified in the notebook page's visible License section. Source SHA-256: `77cf67723a753be957cf70fdb8abbce7ef5c020299e8acd16ed8d2dc8b929dfa`.
- Ahmed Berat Özer's project: compatibility repair for late native hiring and already committed crop crews; the fixed choice between the existing V39 policy and the repaired new plans; whole-team cross-fitted selection, integration and official-engine audits.

The common opening, terminal controller and preceding production/market controls retain their existing attribution. The selected source preserves the complete Apache-2.0 text and upstream notices, including thomastschinkel, yhay81 and destbreso. No donor endorsement or live-rating guarantee is implied. This file records research lineage; release qualification is determined separately by the frozen protocol and results.


## EXP242 compatibility correction

Ahmed Berat Özer's project corrected the request window of already committed sheep crews when the selected native plan hires workers at hours 3–6. Initial investment, production routes and the fixed observed shop selector remain unchanged. The correction was motivated by a complete daily animal census, not a loss-only sample. All prior Apache-2.0 notices and source credits are retained.


## EXP260 post-V41 development — September 14, 2026

V42 preserves the V41 opening and service repairs while testing the complete
observed-shop production allocator described above. Integration, the declared
three-arm experiment, official physical audits and packaging are by Ahmed Berat
Özer's project. All existing Apache-2.0 license text and notices are retained.

[Aurax Shop Router Reactive V5](https://www.kaggle.com/code/aurax7/kaggriculture-shop-router-reactive-v5)
motivated the separately tested earlier reservation boundary. Its rescue
wrapper is not incorporated. The results section identifies whether that
boundary was selected along with the production allocator.
[Nathan's No Cow Left Behind](https://www.kaggle.com/code/nathanjacob/no-cow-left-behind-v40-autopsy-fix)
was a diagnostic and evaluation reference; no rescue or rematching code from it
is incorporated. EcoBot V7 and Adaptive Land Allocator R10 are additional
evaluation opponents, not bundled runtime dependencies. Credits imply no
author endorsement.


## EXP261 runtime measurement

The policy source is unchanged. Ahmed Berat Özer’s project added candidate-process isolation to the validation harness, preserving the original shared-process failures. This harness is not bundled as agent runtime.


## V43 — EXP277–279

Exact dawn warehouse replacement contracts and the behavior-preserving copying specialization and exact-distance calculation were developed by Ahmed Berat Özer’s project. No newly reviewed public notebook controller is added in this release. All inherited Apache-2.0 notices, the full license, and credits including thomastschinkel, yhay81 and destbreso remain in main.py. Credits imply no endorsement. The seed repair experiment was inactive in development and is disabled in the selected policy.


## V44 — EXP282–283

The rival-tape observation (public hand and farmer positions, crop/animal layout), the clone-gated extension of the existing native-tape sale reservation and the drop-turn race escalation were developed by Ahmed Berat Özer’s project from its own live replay analysis. No newly reviewed public notebook controller is added in this release. All inherited Apache-2.0 notices, the full license, and credits including thomastschinkel, yhay81, destbreso, aurax7, tetsutani, prvsiyan, Dmitrii Gluzdov, lucifer19, leoprovorov and Rayk Kretzschmar remain in main.py. Credits imply no endorsement.


## V45 — EXP286

The single large first-turn round trip and the exact lockstep analysis behind it were developed by Ahmed Berat Özer’s project from its own live replay analysis (two live rivals used the same trick against us). No newly reviewed public notebook controller is added in this release. All inherited Apache-2.0 notices, the full license, and credits remain in main.py. Credits imply no endorsement.


## V46 — EXP288

The mirror gate (first-turn cash equality as a copy signature) was developed by Ahmed Berat Özer’s project from its own live replay analysis. No newly reviewed public notebook controller is added in this release. All inherited Apache-2.0 notices, the full license, and credits remain in main.py. Credits imply no endorsement.


## V46 — EXP293

The two-turn sale advance and the sales-first market order follow the mechanisms described in the public Kaggle notebook "Beyond 48-0 · 128/128 Worlds with 95% CIs" by sdy623 (jaxa623), Apache-2.0 (which itself embeds this project's V43); the code here is an original implementation over this project's chassis. The finding that the five feed units are cheapest at index 0 of turn 0 follows the public notebook "Kaggriculture: Pipe-8 Clean Opening" by Nathan Jacob (Apache-2.0). The exact lockstep simulations, the cash-slack analysis, the `[BUY 7, SELL 2]` opening, the turn-1 lift with next-turn resale and the lost-race detector are Ahmed Berat Özer's project's own work. All inherited Apache-2.0 notices, the full license, and credits remain in main.py. Credits imply no endorsement.
